In [1]:
# %% [markdown]
# CAMELOT-IDS v2 — PC-optimized full notebook/script (resume-safe)
# Target machine:
# - 64 GB RAM
# - RTX 3060 12 GB
# - Dual Xeon E5-2683 v4
#
# Added without changing model functionality:
# - preprocessing cache
# - epoch checkpoints + resume
# - cached test/cal logits
# - cached calibration/RAPS/stage outputs
# - cached stream inputs/results
#
# Notes:
# - Resumes from the last completed epoch, not mid-batch
# - Raw model/training/calibration/drift/export logic is unchanged

# %% [markdown]
# Cell 0 — Optional installs
# !pip -q install -U pyarrow river onnx onnxruntime joblib tqdm xgboost catboost

# %%
import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

import re
import gc
import json
import time
import math
import random
import warnings
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Optional, Dict, Any, List, Tuple

import numpy as np
import pandas as pd
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, StratifiedGroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_recall_fscore_support,
    confusion_matrix,
    log_loss,
    balanced_accuracy_score,
)

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import joblib

warnings.filterwarnings("ignore")


# %%
def set_seed(seed: int = 42, deterministic: bool = False):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = bool(deterministic)
        torch.backends.cudnn.benchmark = not bool(deterministic)


DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE:", DEVICE)


@dataclass
class ExpConfig:
    # ---------------------
    # Paths
    # ---------------------
    DATA_ROOT: str = r"C:\Users\HaseebWajid\Desktop\Ayesha Code\CICIoT2023"
    OUT_DIR: str = "./results_camelot_ids_v2_pc"

    # ---------------------
    # Runtime / reproducibility
    # ---------------------
    SEED: int = 42
    DETERMINISTIC: bool = False          # True for paper reruns, False for faster runs
    USE_TORCH_COMPILE: bool = False      # safe fallback if compile fails
    TF32_MODE: bool = True               # good for Ampere (RTX 3060)

    # ---------------------
    # Resume / checkpointing
    # ---------------------
    RESUME: bool = True
    CACHE_PREPROCESSED: bool = True
    CACHE_LOGITS: bool = True
    CACHE_STREAM_INPUTS: bool = True
    SAVE_LAST_EVERY_EPOCH: bool = True

    # ---------------------
    # Data loading controls
    # ---------------------
    MAX_FILES: Optional[int] = 100       # start here on 64 GB RAM; use None after a successful run
    SAMPLE_FRAC: Optional[float] = None  # e.g. 0.05 for quick debug
    READ_ENGINE: str = "pyarrow"        # pyarrow | c | python | auto
    LOW_MEMORY_READ: bool = False

    LABEL_COL_CANDIDATES: Tuple[str, ...] = ("label", "Label", "Attack", "attack", "Class", "class", "category", "Category")
    TIME_COL_CANDIDATES:  Tuple[str, ...] = ("ts", "TS", "timestamp", "Timestamp", "time", "Time", "Datetime", "datetime")
    GROUP_COL_CANDIDATES: Tuple[str, ...] = ("Device", "device", "DeviceName", "device_name", "MAC", "mac")

    # Splits
    TRAIN_PCT: float = 0.80
    VAL_PCT: float   = 0.10
    CAL_PCT: float   = 0.05
    TEST_PCT: float  = 0.05
    SPLIT_POLICY: str = "file_holdout"  # auto | time | group | file_holdout | stratified
    N_GROUP_SPLITS: int = 20

    # Leakage-safe dropping
    DROP_FEATURES_NORM: Tuple[str, ...] = (
        "flowid", "timestamp", "ts", "srcip", "dstip", "sourceip", "destinationip",
        "srcport", "dstport", "device", "devicename", "mac",
        "label", "attack", "class", "category",
    )

    CLIP_INF: float = 1e9
    POST_SCALE_CLIP: float = 20.0

    # ---------------------
    # Training
    # ---------------------
    BATCH_SIZE: int = 1024
    EPOCHS: int = 70
    WARMUP_EPOCHS: float = 2.0
    LR: float = 1e-3
    WEIGHT_DECAY: float = 8e-3
    GRAD_CLIP: float = 1.0
    USE_AMP: bool = True
    NUM_WORKERS: int = 0
    PIN_MEMORY: bool = True
    PREFETCH_FACTOR: int = 2

    # ---------------------
    # Model
    # ---------------------
    D_MODEL: int = 256
    NHEAD: int = 8
    LOCAL_LAYERS: int = 2
    GLOBAL_LAYERS: int = 6
    FF_DIM: int = 1024
    DROPOUT: float = 0.10
    ATTN_DROPOUT: float = 0.05

    # Hierarchical multi-task
    USE_MULTITASK: bool = True
    W_COARSE: float = 0.30
    W_BIN: float = 0.20
    W_CONS_COARSE: float = 0.10
    W_CONS_BIN: float = 0.05

    # ---------------------
    # Imbalance: LDAM-DRW
    # ---------------------
    FINE_LOSS_MODE: str = "ldam_drw"    # ldam_drw | logit_adj | balanced_softmax
    DRW_EPOCHS: int = 3
    CB_BETA: float = 0.9999
    LDAM_MAX_M: float = 0.5
    LDAM_S: float = 30.0
    LOGIT_ADJ_TAU: float = 0.5

    # ---------------------
    # Calibration + Conformal
    # ---------------------
    TEMPERATURE_SCALE: bool = True
    N_BINS_ECE: int = 15
    ALPHAS: Tuple[float, ...] = (0.01, 0.05, 0.10)

    RAPS_KREG: int = 3
    RAPS_LAMBDA: float = 0.01
    RAPS_MONDRIAN: str = "pred_coarse"  # global | pred_coarse

    # ---------------------
    # Drift + Adaptation
    # ---------------------
    DO_STREAM_EVAL: bool = True
    STREAM_BATCH: int = 50_000           # logical stream chunk on CPU
    STREAM_MICRO_BATCH: int = 16_384     # actual GPU chunk to avoid OOM

    DRIFT_DETECTOR: str = "ADWIN+MART"  # ADWIN | KSWIN | MART | ADWIN+MART | KSWIN+MART
    DRIFT_DELTA: float = 0.002
    DRIFT_SIGNAL: str = "entropy"       # entropy | 1-maxprob | set_size
    MART_EPS: float = 0.5
    MART_THRESHOLD: float = 25.0

    ADAPT_METHOD: str = "TENT+HEAD"     # NONE | TENT | TENT+HEAD
    ADAPT_STEPS: int = 5
    ADAPT_LR: float = 1e-4
    ADAPT_BUFFER_BATCHES: int = 3

    ADAPT_ALPHA: float = 0.10
    HEAD_TUNE_LR: float = 5e-5
    HEAD_TUNE_STEPS: int = 10
    MIN_PSEUDO: int = 512

    # ---------------------
    # Baselines (optional, slow)
    # ---------------------
    RUN_BASELINES: bool = False
    BASELINE_MAX_TRAIN: int = 800_000
    BASELINE_MAX_TEST: int = 300_000

    # ---------------------
    # Export
    # ---------------------
    EXPORT_ONNX: bool = True
    EXPORT_TORCHSCRIPT: bool = True
    EXPORT_INT8_DYNAMIC: bool = True


cfg = ExpConfig()
Path(cfg.OUT_DIR).mkdir(parents=True, exist_ok=True)

RUN_DIR = Path(cfg.OUT_DIR)
CACHE_DIR = RUN_DIR / "cache"
ARRAY_CACHE_DIR = CACHE_DIR / "arrays"
LOGITS_CACHE_DIR = CACHE_DIR / "logits"
STREAM_CACHE_DIR = CACHE_DIR / "stream"
CKPT_DIR = RUN_DIR / "checkpoints"
STAGE_DIR = RUN_DIR / "stages"
export_dir = RUN_DIR / "export"

for d in [CACHE_DIR, ARRAY_CACHE_DIR, LOGITS_CACHE_DIR, STREAM_CACHE_DIR, CKPT_DIR, STAGE_DIR, export_dir]:
    d.mkdir(parents=True, exist_ok=True)

set_seed(cfg.SEED, deterministic=cfg.DETERMINISTIC)

if DEVICE == "cuda" and cfg.TF32_MODE:
    try:
        torch.backends.cuda.matmul.fp32_precision = "tf32"
        torch.backends.cudnn.fp32_precision = "tf32"
    except Exception:
        torch.backends.cuda.matmul.allow_tf32 = True
    torch.set_float32_matmul_precision("high")

with open(RUN_DIR / "config.json", "w", encoding="utf-8") as f:
    json.dump(asdict(cfg), f, indent=2)

print("OUT_DIR:", cfg.OUT_DIR)


# %%
def save_json_atomic(path: Path, obj: Dict[str, Any]):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = Path(str(path) + ".tmp")
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2)
    os.replace(tmp, path)


def load_json(path: Path, default=None):
    path = Path(path)
    if not path.exists():
        return {} if default is None else default
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def atomic_torch_save(obj: Any, path: Path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = Path(str(path) + ".tmp")
    torch.save(obj, tmp)
    os.replace(tmp, path)


def atomic_save_npy(path: Path, arr: np.ndarray):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = Path(str(path) + ".tmp")
    with open(tmp, "wb") as f:
        np.save(f, arr, allow_pickle=False)
    os.replace(tmp, path)


def save_array_dict(folder: Path, arrays: Dict[str, np.ndarray]):
    folder = Path(folder)
    folder.mkdir(parents=True, exist_ok=True)
    for k, v in arrays.items():
        atomic_save_npy(folder / f"{k}.npy", v)
    save_json_atomic(folder / "_ok.json", {"keys": list(arrays.keys())})


def load_array_dict(folder: Path, keys: Optional[List[str]] = None) -> Dict[str, np.ndarray]:
    folder = Path(folder)
    meta = load_json(folder / "_ok.json", {})
    if keys is None:
        keys = meta.get("keys", [])
    return {k: np.load(folder / f"{k}.npy") for k in keys}


def folder_ready(folder: Path, keys: List[str]) -> bool:
    folder = Path(folder)
    return (folder / "_ok.json").exists() and all((folder / f"{k}.npy").exists() for k in keys)


def get_rng_state() -> Dict[str, Any]:
    state = {
        "python": random.getstate(),
        "numpy": np.random.get_state(),
        "torch": torch.get_rng_state(),
    }
    if torch.cuda.is_available():
        state["torch_cuda"] = torch.cuda.get_rng_state_all()
    return state


def set_rng_state(state: Optional[Dict[str, Any]]):
    if not state:
        return

    if "python" in state:
        random.setstate(state["python"])

    if "numpy" in state:
        np.random.set_state(state["numpy"])

    if "torch" in state:
        cpu_state = state["torch"]

        if isinstance(cpu_state, np.ndarray):
            cpu_state = torch.from_numpy(cpu_state)
        elif not isinstance(cpu_state, torch.Tensor):
            cpu_state = torch.tensor(cpu_state)

        cpu_state = cpu_state.detach().to(device="cpu", dtype=torch.uint8)
        torch.set_rng_state(cpu_state)

    if torch.cuda.is_available() and "torch_cuda" in state:
        cuda_states = []
        for s in state["torch_cuda"]:
            if isinstance(s, np.ndarray):
                s = torch.from_numpy(s)
            elif not isinstance(s, torch.Tensor):
                s = torch.tensor(s)

            s = s.detach().to(device="cpu", dtype=torch.uint8)
            cuda_states.append(s)

        torch.cuda.set_rng_state_all(cuda_states)
        
def stable_json_dumps(obj: Any) -> str:
    return json.dumps(obj, sort_keys=True, default=str)


def same_signature(a: Optional[Dict[str, Any]], b: Optional[Dict[str, Any]]) -> bool:
    return stable_json_dumps(a or {}) == stable_json_dumps(b or {})


def get_training_signature() -> Dict[str, Any]:
    sig = {}
    done_path = STAGE_DIR / "training_done.json"
    best_path = CKPT_DIR / "best_model.pt"
    if done_path.exists():
        done = load_json(done_path, {})
        sig["best_epoch"] = done.get("best_epoch")
        sig["best_score"] = done.get("best_score")
    if best_path.exists():
        sig["best_ckpt_mtime_ns"] = best_path.stat().st_mtime_ns
        sig["best_ckpt_size"] = best_path.stat().st_size
    return sig


ARRAY_KEYS = [
    "X_train", "y_train_f", "y_train_c", "y_train_b",
    "X_val", "y_val_f", "y_val_c", "y_val_b",
    "X_cal", "y_cal_f", "y_cal_c", "y_cal_b",
    "X_test", "y_test_f", "y_test_c", "y_test_b",
]
STREAM_KEYS = ["X_stream_test", "y_stream_f"]
LOGIT_KEYS = ["fine", "coarse", "bin", "yf", "yc", "yb"]


# %%
def norm_colname(c: str) -> str:
    return re.sub(r"[^a-z0-9]+", "", str(c).strip().lower())


def detect_col(columns: List[str], candidates: Tuple[str, ...]) -> Optional[str]:
    colset = set(columns)
    for c in candidates:
        if c in colset:
            return c
    norm_map = {norm_colname(c): c for c in columns}
    for c in candidates:
        nc = norm_colname(c)
        if nc in norm_map:
            return norm_map[nc]
    return None


def sanitize_frame(df: pd.DataFrame, clip_inf: float = 1e9) -> pd.DataFrame:
    df = df.replace({"Infinity": np.inf, "inf": np.inf, "+inf": np.inf, "-inf": -np.inf,
                     "NaN": np.nan, "nan": np.nan, "": np.nan})
    df = df.replace([np.inf, -np.inf], np.nan)
    num_cols = df.select_dtypes(include=[np.number]).columns
    if len(num_cols) > 0:
        df[num_cols] = df[num_cols].clip(-clip_inf, clip_inf)
    return df


def downcast_numeric(df: pd.DataFrame) -> pd.DataFrame:
    for c in df.select_dtypes(include=["float64"]).columns:
        df[c] = pd.to_numeric(df[c], downcast="float", errors="ignore")
    for c in df.select_dtypes(include=["int64"]).columns:
        df[c] = pd.to_numeric(df[c], downcast="integer", errors="ignore")
    return df


def memory_mb(df: pd.DataFrame) -> float:
    try:
        return float(df.memory_usage(deep=True).sum()) / (1024 ** 2)
    except Exception:
        return float("nan")


def discover_csv_files(root: Path, max_files: Optional[int] = None) -> List[Path]:
    files = sorted([p for p in root.rglob("*.csv") if p.is_file()])
    if not files:
        raise FileNotFoundError(f"No .csv files found under: {root}")
    if max_files is not None:
        files = files[:max_files]
    return files


def running_on_kaggle():
    return Path("/kaggle/input").exists()


def find_dataset_root(hint: Optional[str] = None) -> Path:
    base = Path("/kaggle/input")
    if hint is not None and (base / hint).exists():
        return base / hint
    best, best_n = None, -1
    for d in base.iterdir():
        if not d.is_dir():
            continue
        n = len(list(d.rglob("*.csv")))
        if n > best_n:
            best_n = n
            best = d
    if best is None:
        raise FileNotFoundError("No dataset found in /kaggle/input")
    return best


def read_one_csv(path: Path, cfg: ExpConfig) -> pd.DataFrame:
    engines = []
    if cfg.READ_ENGINE == "auto":
        engines = ["pyarrow", "c", "python"]
    else:
        engines = [cfg.READ_ENGINE, "c", "python"]

    last_err = None
    for eng in engines:
        try:
            kwargs = dict(low_memory=cfg.LOW_MEMORY_READ)
            if eng == "pyarrow":
                kwargs.pop("low_memory", None)
                return pd.read_csv(path, engine="pyarrow")
            if eng == "c":
                return pd.read_csv(path, engine="c", **kwargs)
            if eng == "python":
                kwargs.pop("low_memory", None)
                return pd.read_csv(path, engine="python")
        except Exception as e:
            last_err = e
            continue
    raise RuntimeError(f"Failed to read {path}: {last_err}")


def maybe_compress_categoricals(df: pd.DataFrame, max_unique_ratio: float = 0.50) -> pd.DataFrame:
    for c in df.select_dtypes(include=["object"]).columns:
        nunique = df[c].nunique(dropna=False)
        ratio = nunique / max(len(df), 1)
        if ratio <= max_unique_ratio:
            try:
                df[c] = df[c].astype("category")
            except Exception:
                pass
    return df


def sort_stream_df_simple(d: pd.DataFrame, time_col: Optional[str]) -> pd.DataFrame:
    d = d.copy()
    if time_col is not None and time_col in d.columns:
        d[time_col] = pd.to_datetime(d[time_col], errors="coerce")
        d = d.sort_values(time_col, kind="mergesort")
    else:
        d = d.sort_values(["__src_file"], kind="mergesort")
    return d


# %%
DICT_34_CLASSES: Dict[str, int] = {
    "BenignTraffic": 0,
    "DDoS-RSTFINFlood": 1, "DDoS-PSHACK_Flood": 2, "DDoS-SYN_Flood": 3, "DDoS-UDP_Flood": 4,
    "DDoS-TCP_Flood": 5, "DDoS-ICMP_Flood": 6, "DDoS-SynonymousIP_Flood": 7,
    "DDoS-ACK_Fragmentation": 8, "DDoS-UDP_Fragmentation": 9, "DDoS-ICMP_Fragmentation": 10,
    "DDoS-SlowLoris": 11, "DDoS-HTTP_Flood": 12,
    "DoS-UDP_Flood": 13, "DoS-SYN_Flood": 14, "DoS-TCP_Flood": 15, "DoS-HTTP_Flood": 16,
    "Mirai-greeth_flood": 17, "Mirai-greip_flood": 18, "Mirai-udpplain": 19,
    "Recon-PingSweep": 20, "Recon-OSScan": 21, "Recon-PortScan": 22, "VulnerabilityScan": 23, "Recon-HostDiscovery": 24,
    "DNS_Spoofing": 25, "MITM-ArpSpoofing": 26,
    "BrowserHijacking": 27, "Backdoor_Malware": 28, "XSS": 29, "Uploading_Attack": 30, "SqlInjection": 31, "CommandInjection": 32,
    "DictionaryBruteForce": 33
}

DICT_8_FROM_FINEID: Dict[int, int] = {
    0: 0,
    1: 1, 2: 1, 3: 1, 4: 1, 5: 1, 6: 1, 7: 1, 8: 1, 9: 1, 10: 1, 11: 1, 12: 1,
    17: 2, 18: 2, 19: 2,
    20: 3, 21: 3, 22: 3, 23: 3, 24: 3,
    25: 4, 26: 4,
    27: 5, 28: 5, 29: 5, 30: 5, 31: 5, 32: 5,
    33: 6,
    13: 7, 14: 7, 15: 7, 16: 7
}

COARSE_NAMES = ["Benign", "DDoS", "Mirai", "Recon", "Spoofing", "Web-based", "Brute Force", "DoS"]
FINE_CLASS_NAMES = [k for k, v in sorted(DICT_34_CLASSES.items(), key=lambda kv: kv[1])]
NUM_FINE = len(FINE_CLASS_NAMES)
NUM_COARSE = len(COARSE_NAMES)

fine_to_coarse = np.zeros(NUM_FINE, dtype=np.int64)
for fine_name, fine_id in DICT_34_CLASSES.items():
    fine_to_coarse[fine_id] = DICT_8_FROM_FINEID[fine_id]

benign_fine_id = DICT_34_CLASSES["BenignTraffic"]
assert benign_fine_id == 0


# %%
def build_feature_columns(df: pd.DataFrame, cfg: ExpConfig, label_col: str, time_col: Optional[str], group_col: Optional[str]) -> List[str]:
    drop_norm = set(cfg.DROP_FEATURES_NORM)
    cols = []
    for c in df.columns:
        if c in [label_col, "y_fine", "y_coarse", "y_bin", "__src_file"]:
            continue
        if group_col is not None and c == group_col:
            continue
        if time_col is not None and c == time_col:
            continue
        if norm_colname(c) in drop_norm:
            continue
        cols.append(c)
    return cols


def stratified_4way_split(df: pd.DataFrame, y_col: str, cfg: ExpConfig) -> Dict[str, pd.DataFrame]:
    train_df, rem_df = train_test_split(
        df, test_size=(1 - cfg.TRAIN_PCT), stratify=df[y_col], random_state=cfg.SEED
    )
    rem_frac = 1 - cfg.TRAIN_PCT
    val_frac = cfg.VAL_PCT / rem_frac
    val_df, temp_df = train_test_split(
        rem_df, test_size=(1 - val_frac), stratify=rem_df[y_col], random_state=cfg.SEED
    )
    cal_ratio = cfg.CAL_PCT / (cfg.CAL_PCT + cfg.TEST_PCT)
    cal_df, test_df = train_test_split(
        temp_df, test_size=(1 - cal_ratio), stratify=temp_df[y_col], random_state=cfg.SEED
    )
    return {"train": train_df, "val": val_df, "cal": cal_df, "test": test_df}


def time_split(df: pd.DataFrame, time_col: str, cfg: ExpConfig) -> Dict[str, pd.DataFrame]:
    d = df.copy()
    d[time_col] = pd.to_datetime(d[time_col], errors="coerce")
    d = d.sort_values(time_col, kind="mergesort")
    n = len(d)
    n_train = int(cfg.TRAIN_PCT * n)
    n_val = int(cfg.VAL_PCT * n)
    n_cal = int(cfg.CAL_PCT * n)
    return {
        "train": d.iloc[:n_train].copy(),
        "val": d.iloc[n_train:n_train + n_val].copy(),
        "cal": d.iloc[n_train + n_val:n_train + n_val + n_cal].copy(),
        "test": d.iloc[n_train + n_val + n_cal:].copy(),
    }


def stratified_group_4way(df: pd.DataFrame, y_col: str, group_col: str, cfg: ExpConfig) -> Optional[Dict[str, pd.DataFrame]]:
    y = df[y_col].values
    groups = df[group_col].astype(str).fillna("NA").values
    n_groups = pd.Series(groups).nunique()
    n_splits = min(cfg.N_GROUP_SPLITS, n_groups)
    if n_splits < 4:
        return None

    sgkf = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=cfg.SEED)
    fold = np.empty(len(df), dtype=np.int16)
    Xdummy = np.zeros((len(df), 1), dtype=np.int8)

    for i, (_, te) in enumerate(sgkf.split(Xdummy, y, groups)):
        fold[te] = i

    rng = np.random.default_rng(cfg.SEED)
    perm = rng.permutation(n_splits)

    n_test = max(1, int(round(cfg.TEST_PCT * n_splits)))
    n_cal = max(1, int(round(cfg.CAL_PCT * n_splits)))
    n_val = max(1, int(round(cfg.VAL_PCT * n_splits)))
    total = n_test + n_cal + n_val
    if total >= n_splits:
        return None

    test_f = set(perm[:n_test])
    cal_f = set(perm[n_test:n_test + n_cal])
    val_f = set(perm[n_test + n_cal:n_test + n_cal + n_val])
    train_f = set(perm[n_test + n_cal + n_val:])

    splits = {
        "train": df.loc[np.isin(fold, list(train_f))].copy(),
        "val": df.loc[np.isin(fold, list(val_f))].copy(),
        "cal": df.loc[np.isin(fold, list(cal_f))].copy(),
        "test": df.loc[np.isin(fold, list(test_f))].copy(),
    }
    return splits


def infer_numeric_object_cols(df: pd.DataFrame, threshold: float = 0.95) -> List[str]:
    obj_cols = df.select_dtypes(include=["object", "category"]).columns.tolist()
    numeric_like = []
    for c in obj_cols:
        s = pd.to_numeric(df[c], errors="coerce")
        ratio = float(s.notna().mean()) if len(s) else 0.0
        if ratio >= threshold:
            numeric_like.append(c)
    return numeric_like


def fit_preprocessors(train_df: pd.DataFrame, feature_cols: List[str], cfg: ExpConfig):
    X = sanitize_frame(train_df[feature_cols].copy(), cfg.CLIP_INF)
    numeric_like_obj = infer_numeric_object_cols(X, threshold=0.95)
    for c in numeric_like_obj:
        X[c] = pd.to_numeric(X[c], errors="coerce")

    num_cols = X.select_dtypes(include=[np.number]).columns.tolist()
    non_num = [c for c in X.columns if c not in num_cols]
    if non_num:
        print("Dropping non-numeric columns:", non_num[:10], "...")
        X = X[num_cols]

    num_imputer = SimpleImputer(strategy="median")
    scaler = StandardScaler()

    num_imputer.fit(X[num_cols])
    Ximp = num_imputer.transform(X[num_cols]).astype(np.float32)
    scaler.fit(Ximp)

    meta = {"num_cols": num_cols, "numeric_like_obj": numeric_like_obj}
    return num_imputer, scaler, meta


def transform_df(
    df_in: pd.DataFrame,
    feature_cols: List[str],
    num_imputer,
    scaler,
    meta,
    cfg: ExpConfig
) -> np.ndarray:
    X = sanitize_frame(df_in[feature_cols].copy(), cfg.CLIP_INF)
    for c in meta["numeric_like_obj"]:
        if c in X.columns:
            X[c] = pd.to_numeric(X[c], errors="coerce")

    num_cols = meta["num_cols"]
    X = X[num_cols]
    Xn = num_imputer.transform(X).astype(np.float32)
    Xn = scaler.transform(Xn).astype(np.float32)
    Xn = np.clip(Xn, -cfg.POST_SCALE_CLIP, cfg.POST_SCALE_CLIP).astype(np.float32)
    Xn = np.nan_to_num(
        Xn,
        nan=0.0,
        posinf=cfg.POST_SCALE_CLIP,
        neginf=-cfg.POST_SCALE_CLIP
    ).astype(np.float32)
    return Xn


# %%
preprocess_cache_ready = (
    cfg.RESUME
    and cfg.CACHE_PREPROCESSED
    and folder_ready(ARRAY_CACHE_DIR, ARRAY_KEYS)
    and (export_dir / "preprocess_and_meta.joblib").exists()
    and (RUN_DIR / "split_mode.json").exists()
    and ((not cfg.DO_STREAM_EVAL) or folder_ready(STREAM_CACHE_DIR, STREAM_KEYS))
)

if preprocess_cache_ready:
    print("Loading cached preprocessed arrays...")

    arrs = load_array_dict(ARRAY_CACHE_DIR, ARRAY_KEYS)
    X_train = arrs["X_train"]
    y_train_f = arrs["y_train_f"]
    y_train_c = arrs["y_train_c"]
    y_train_b = arrs["y_train_b"]

    X_val = arrs["X_val"]
    y_val_f = arrs["y_val_f"]
    y_val_c = arrs["y_val_c"]
    y_val_b = arrs["y_val_b"]

    X_cal = arrs["X_cal"]
    y_cal_f = arrs["y_cal_f"]
    y_cal_c = arrs["y_cal_c"]
    y_cal_b = arrs["y_cal_b"]

    X_test = arrs["X_test"]
    y_test_f = arrs["y_test_f"]
    y_test_c = arrs["y_test_c"]
    y_test_b = arrs["y_test_b"]

    if cfg.DO_STREAM_EVAL:
        sarrs = load_array_dict(STREAM_CACHE_DIR, STREAM_KEYS)
        X_stream_test = sarrs["X_stream_test"]
        y_stream_f = sarrs["y_stream_f"]
    else:
        X_stream_test, y_stream_f = None, None

    meta_obj = joblib.load(export_dir / "preprocess_and_meta.joblib")
    FEATURE_COLS = meta_obj["FEATURE_COLS"]
    meta = meta_obj["meta"]
    num_imp = meta_obj["num_imputer"]
    scaler = meta_obj["scaler"]

    split_meta = load_json(RUN_DIR / "split_mode.json", {})
    split_mode = split_meta.get("split_mode", "unknown")
    LABEL_COL = split_meta.get("LABEL_COL", None)
    TIME_COL = split_meta.get("TIME_COL", None)
    GROUP_COL = split_meta.get("GROUP_COL", None)

    num_features = X_train.shape[1]
    n_total_samples = int(len(X_train) + len(X_val) + len(X_cal) + len(X_test))

    print("Loaded cached arrays.")
    print("Shapes:", X_train.shape, X_val.shape, X_cal.shape, X_test.shape)
    print("num_features:", num_features, "| num_fine:", NUM_FINE, "| num_coarse:", NUM_COARSE)
else:
    if running_on_kaggle():
        DATA_ROOT = find_dataset_root(getattr(cfg, "DATASET_HINT", None))
    else:
        DATA_ROOT = Path(cfg.DATA_ROOT).expanduser().resolve()

    print("DATA_ROOT:", DATA_ROOT)

    files = discover_csv_files(DATA_ROOT, cfg.MAX_FILES)
    print("CSV files found:", len(files), "| example:", files[0].name)

    dfs = []
    for p in tqdm(files, desc="Reading CSVs"):
        d = read_one_csv(p, cfg)
        d["__src_file"] = p.name
        dfs.append(d)

    df = pd.concat(dfs, ignore_index=True)
    del dfs
    gc.collect()

    df = sanitize_frame(df, cfg.CLIP_INF)
    df = downcast_numeric(df)
    df = maybe_compress_categoricals(df)

    if cfg.SAMPLE_FRAC is not None:
        df = df.sample(frac=cfg.SAMPLE_FRAC, random_state=cfg.SEED).reset_index(drop=True)

    print("df shape:", df.shape, "| memory(MB):", round(memory_mb(df), 2))

    LABEL_COL = detect_col(df.columns.tolist(), cfg.LABEL_COL_CANDIDATES)
    TIME_COL = detect_col(df.columns.tolist(), cfg.TIME_COL_CANDIDATES)
    GROUP_COL = detect_col(df.columns.tolist(), cfg.GROUP_COL_CANDIDATES)

    print("Detected LABEL_COL:", LABEL_COL)
    print("Detected TIME_COL :", TIME_COL)
    print("Detected GROUP_COL:", GROUP_COL)

    if LABEL_COL is None:
        raise ValueError("Could not detect label column.")

    df[LABEL_COL] = df[LABEL_COL].astype(str).str.strip()
    unknown = set(df[LABEL_COL].unique()) - set(DICT_34_CLASSES.keys())
    if unknown:
        print("WARNING: unknown labels found (dropping):", list(sorted(unknown))[:20])
        df = df.loc[~df[LABEL_COL].isin(unknown)].copy()

    df["y_fine"] = df[LABEL_COL].map(DICT_34_CLASSES).astype(np.int64)
    df["y_coarse"] = df["y_fine"].map(lambda i: int(fine_to_coarse[i])).astype(np.int64)
    df["y_bin"] = (df["y_fine"] != benign_fine_id).astype(np.int64)

    df["__src_file"] = df["__src_file"].astype("category")
    try:
        df[LABEL_COL] = df[LABEL_COL].astype("category")
    except Exception:
        pass

    print("After label filter:", df.shape, "| fine classes:", df["y_fine"].nunique(), "| coarse:", df["y_coarse"].nunique())

    n_total_samples = int(len(df))

    FEATURE_COLS = build_feature_columns(df, cfg, LABEL_COL, TIME_COL, GROUP_COL)
    print("Num features:", len(FEATURE_COLS))
    print("First 25 features:", FEATURE_COLS[:25])

    split_mode, splits = None, None

    if cfg.SPLIT_POLICY in ("time", "auto") and TIME_COL is not None:
        splits = time_split(df, TIME_COL, cfg)
        split_mode = f"time_order({TIME_COL})"

    if splits is None and cfg.SPLIT_POLICY in ("group", "auto") and GROUP_COL is not None:
        splits = stratified_group_4way(df, "y_fine", GROUP_COL, cfg)
        if splits is not None:
            split_mode = f"stratified_group({GROUP_COL})"

    if splits is None and cfg.SPLIT_POLICY in ("file_holdout", "auto"):
        splits = stratified_group_4way(df, "y_fine", "__src_file", cfg)
        if splits is not None:
            split_mode = "stratified_group(__src_file)"

    if splits is None:
        splits = stratified_4way_split(df, "y_fine", cfg)
        split_mode = "stratified_random"

    print("Split mode:", split_mode)
    for k, v in splits.items():
        print(k, v.shape, "| fine classes:", v["y_fine"].nunique(), "| coarse:", v["y_coarse"].nunique())

    with open(RUN_DIR / "split_mode.json", "w", encoding="utf-8") as f:
        json.dump({
            "split_mode": split_mode,
            "LABEL_COL": LABEL_COL,
            "TIME_COL": TIME_COL,
            "GROUP_COL": GROUP_COL
        }, f, indent=2)

    train_df, val_df, cal_df, test_df = splits["train"], splits["val"], splits["cal"], splits["test"]

    num_imp, scaler, meta = fit_preprocessors(train_df, FEATURE_COLS, cfg)

    X_train = transform_df(train_df, FEATURE_COLS, num_imp, scaler, meta, cfg)
    y_train_f = train_df["y_fine"].values.astype(np.int64)
    y_train_c = train_df["y_coarse"].values.astype(np.int64)
    y_train_b = train_df["y_bin"].values.astype(np.int64)
    del train_df
    gc.collect()

    X_val = transform_df(val_df, FEATURE_COLS, num_imp, scaler, meta, cfg)
    y_val_f = val_df["y_fine"].values.astype(np.int64)
    y_val_c = val_df["y_coarse"].values.astype(np.int64)
    y_val_b = val_df["y_bin"].values.astype(np.int64)
    del val_df
    gc.collect()

    X_cal = transform_df(cal_df, FEATURE_COLS, num_imp, scaler, meta, cfg)
    y_cal_f = cal_df["y_fine"].values.astype(np.int64)
    y_cal_c = cal_df["y_coarse"].values.astype(np.int64)
    y_cal_b = cal_df["y_bin"].values.astype(np.int64)
    gc.collect()

    X_test = transform_df(test_df, FEATURE_COLS, num_imp, scaler, meta, cfg)
    y_test_f = test_df["y_fine"].values.astype(np.int64)
    y_test_c = test_df["y_coarse"].values.astype(np.int64)
    y_test_b = test_df["y_bin"].values.astype(np.int64)

    if cfg.DO_STREAM_EVAL:
        s_test = sort_stream_df_simple(test_df, TIME_COL)
        X_stream_test = transform_df(s_test, FEATURE_COLS, num_imp, scaler, meta, cfg)
        y_stream_f = s_test["y_fine"].values.astype(np.int64)
    else:
        X_stream_test, y_stream_f = None, None

    num_features = X_train.shape[1]
    print("Shapes:", X_train.shape, X_val.shape, X_cal.shape, X_test.shape)
    print("num_features:", num_features, "| num_fine:", NUM_FINE, "| num_coarse:", NUM_COARSE)

    joblib.dump({
        "FEATURE_COLS": FEATURE_COLS,
        "meta": meta,
        "num_imputer": num_imp,
        "scaler": scaler,
        "cfg": asdict(cfg),
        "fine_class_names": FINE_CLASS_NAMES,
        "coarse_names": COARSE_NAMES,
        "fine_to_coarse": fine_to_coarse.tolist(),
        "benign_fine_id": int(benign_fine_id),
    }, export_dir / "preprocess_and_meta.joblib")

    if cfg.CACHE_PREPROCESSED:
        print("Saving preprocessed cache...")
        save_array_dict(ARRAY_CACHE_DIR, {
            "X_train": X_train, "y_train_f": y_train_f, "y_train_c": y_train_c, "y_train_b": y_train_b,
            "X_val": X_val, "y_val_f": y_val_f, "y_val_c": y_val_c, "y_val_b": y_val_b,
            "X_cal": X_cal, "y_cal_f": y_cal_f, "y_cal_c": y_cal_c, "y_cal_b": y_cal_b,
            "X_test": X_test, "y_test_f": y_test_f, "y_test_c": y_test_c, "y_test_b": y_test_b,
        })
        if cfg.DO_STREAM_EVAL and cfg.CACHE_STREAM_INPUTS:
            save_array_dict(STREAM_CACHE_DIR, {
                "X_stream_test": X_stream_test,
                "y_stream_f": y_stream_f,
            })
        print("Preprocessed cache saved.")

    del cal_df, test_df
    del df
    gc.collect()


# %%
PROTOCOL_HINTS = set([norm_colname(x) for x in [
    "HTTP", "HTTPS", "DNS", "Telnet", "SMTP", "SSH", "IRC", "TCP", "UDP", "DHCP", "ARP", "ICMP", "IPv", "LLC"
]])
STAT_KEYS = ["tot", "sum", "min", "max", "avg", "std", "size", "iat", "number", "magnitude", "magnitue", "radius", "covariance", "variance", "weight"]
RATE_KEYS = ["rate", "srate", "drate"]


def build_feature_groups(feature_cols: List[str]) -> Tuple[List[str], List[List[int]]]:
    groups: Dict[str, List[int]] = {"rates": [], "flags": [], "protocols": [], "stats": [], "other": []}
    for idx, name in enumerate(feature_cols):
        n = norm_colname(name)
        if any(k in n for k in RATE_KEYS):
            groups["rates"].append(idx)
        elif "flag" in n or n.endswith("count") or "count" in n:
            groups["flags"].append(idx)
        elif n in PROTOCOL_HINTS:
            groups["protocols"].append(idx)
        elif any(k in n for k in STAT_KEYS):
            groups["stats"].append(idx)
        else:
            groups["other"].append(idx)

    group_names = [k for k, v in groups.items() if len(v) > 0]
    group_idxs = [groups[k] for k in group_names]
    return group_names, group_idxs


GROUP_NAMES, GROUP_IDXS = build_feature_groups(meta["num_cols"])
print("Groups:", GROUP_NAMES)
for gn, gi in zip(GROUP_NAMES, GROUP_IDXS):
    print(f" - {gn}: {len(gi)} features")

NUM_GROUPS = len(GROUP_NAMES)


# %%
class FlowDataset(Dataset):
    def __init__(self, X: np.ndarray, y_f: np.ndarray, y_c: np.ndarray, y_b: np.ndarray):
        self.X = torch.from_numpy(X.astype(np.float32, copy=False))
        self.yf = torch.from_numpy(y_f.astype(np.int64, copy=False))
        self.yc = torch.from_numpy(y_c.astype(np.int64, copy=False))
        self.yb = torch.from_numpy(y_b.astype(np.int64, copy=False))

    def __len__(self):
        return int(self.X.shape[0])

    def __getitem__(self, idx):
        return self.X[idx], self.yf[idx], self.yc[idx], self.yb[idx]


pin = (cfg.PIN_MEMORY and DEVICE == "cuda")
persistent = (cfg.NUM_WORKERS > 0)

loader_kwargs = dict(
    batch_size=cfg.BATCH_SIZE,
    num_workers=cfg.NUM_WORKERS,
    pin_memory=pin,
    persistent_workers=persistent,
)

if persistent:
    loader_kwargs["prefetch_factor"] = cfg.PREFETCH_FACTOR

ds_train = FlowDataset(X_train, y_train_f, y_train_c, y_train_b)
ds_val = FlowDataset(X_val, y_val_f, y_val_c, y_val_b)
ds_cal = FlowDataset(X_cal, y_cal_f, y_cal_c, y_cal_b)
ds_test = FlowDataset(X_test, y_test_f, y_test_c, y_test_b)

train_loader = DataLoader(ds_train, shuffle=True, **loader_kwargs)
val_loader = DataLoader(ds_val, shuffle=False, **loader_kwargs)
cal_loader = DataLoader(ds_cal, shuffle=False, **loader_kwargs)
test_loader = DataLoader(ds_test, shuffle=False, **loader_kwargs)

print("Train batches:", len(train_loader), "| Val batches:", len(val_loader))


# %%
def class_balanced_weights(y: np.ndarray, num_classes: int, beta: float = 0.9999) -> torch.Tensor:
    counts = np.bincount(y, minlength=num_classes).astype(np.float64)
    effective = 1.0 - np.power(beta, counts)
    w = (1.0 - beta) / np.clip(effective, 1e-12, None)
    w = w / (w.mean() + 1e-12)
    return torch.tensor(w, dtype=torch.float32), counts


class BalancedSoftmaxLoss(nn.Module):
    def __init__(self, class_counts: np.ndarray):
        super().__init__()
        cc = torch.tensor(class_counts.astype(np.float32), dtype=torch.float32)
        self.register_buffer("log_counts", torch.log(cc + 1e-12))

    def forward(self, logits: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
        adj = logits + self.log_counts.unsqueeze(0)
        return F.cross_entropy(adj, target)


class LogitAdjustedCELoss(nn.Module):
    def __init__(self, class_counts: np.ndarray, tau: float = 0.5):
        super().__init__()
        pi = class_counts / (class_counts.sum() + 1e-12)
        adj = tau * np.log(pi + 1e-12)
        self.register_buffer("adj", torch.tensor(adj, dtype=torch.float32))

    def forward(self, logits: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
        return F.cross_entropy(logits + self.adj.unsqueeze(0), target)


class LDAMLoss(nn.Module):
    def __init__(self, class_counts: np.ndarray, max_m: float = 0.5, s: float = 30.0, weight: Optional[torch.Tensor] = None):
        super().__init__()
        counts = torch.tensor(class_counts, dtype=torch.float32)
        m = 1.0 / torch.sqrt(torch.sqrt(counts + 1e-12))
        m = m * (max_m / m.max())
        self.register_buffer("m_list", m)
        self.s = float(s)
        self.weight = weight

    def forward(self, logits: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
        idx = torch.arange(logits.size(0), device=logits.device)
        margins = self.m_list[target]
        logits_m = logits.clone()
        logits_m[idx, target] -= margins
        return F.cross_entropy(self.s * logits_m, target, weight=self.weight)


w_fine_cb, counts_fine = class_balanced_weights(y_train_f, NUM_FINE, cfg.CB_BETA)
w_coarse_cb, counts_coarse = class_balanced_weights(y_train_c, NUM_COARSE, cfg.CB_BETA)
w_bin_cb, counts_bin = class_balanced_weights(y_train_b, 2, cfg.CB_BETA)

print("Fine class count stats:", int(counts_fine.min()), int(counts_fine.max()))


def make_fine_loss(epoch: int) -> nn.Module:
    warm = (epoch <= cfg.DRW_EPOCHS)
    if cfg.FINE_LOSS_MODE == "balanced_softmax":
        return BalancedSoftmaxLoss(counts_fine).to(DEVICE)

    if cfg.FINE_LOSS_MODE == "logit_adj":
        return LogitAdjustedCELoss(counts_fine, tau=cfg.LOGIT_ADJ_TAU).to(DEVICE)

    ww = None if warm else w_fine_cb.to(DEVICE)
    return LDAMLoss(counts_fine, max_m=cfg.LDAM_MAX_M, s=cfg.LDAM_S, weight=ww).to(DEVICE)


def make_aux_losses(epoch: int):
    warm = (epoch <= cfg.DRW_EPOCHS)
    wc = None if warm else w_coarse_cb.to(DEVICE)
    wb = None if warm else w_bin_cb.to(DEVICE)
    loss_coarse = nn.CrossEntropyLoss(weight=wc).to(DEVICE)
    loss_bin = nn.CrossEntropyLoss(weight=wb).to(DEVICE)
    return loss_coarse, loss_bin


fine_to_coarse_t = torch.tensor(fine_to_coarse, device=DEVICE, dtype=torch.long)


# %%
class FeatureTokenizer(nn.Module):
    def __init__(self, n_features: int, d_model: int):
        super().__init__()
        self.W = nn.Parameter(torch.empty(n_features, d_model))
        self.b = nn.Parameter(torch.empty(n_features, d_model))
        nn.init.trunc_normal_(self.W, std=0.02)
        nn.init.trunc_normal_(self.b, std=0.02)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x.unsqueeze(-1) * self.W.unsqueeze(0) + self.b.unsqueeze(0)


def make_encoder_layer(d_model, nhead, ff_dim, dropout, attn_dropout):
    return nn.TransformerEncoderLayer(
        d_model=d_model,
        nhead=nhead,
        dim_feedforward=ff_dim,
        dropout=dropout,
        activation="gelu",
        batch_first=True,
        norm_first=True
    )


class CamelotIDSv2(nn.Module):
    def __init__(self, n_features: int, n_fine: int, n_coarse: int, group_idxs: List[List[int]], cfg: ExpConfig):
        super().__init__()
        self.cfg = cfg
        self.n_features = n_features
        self.n_fine = n_fine
        self.n_coarse = n_coarse
        self.group_idxs = group_idxs
        self.num_groups = len(group_idxs)

        self.tokenizer = FeatureTokenizer(n_features, cfg.D_MODEL)

        self.group_token = nn.Parameter(torch.zeros(self.num_groups, cfg.D_MODEL))
        nn.init.trunc_normal_(self.group_token, std=0.02)
        self.group_type = nn.Embedding(self.num_groups, cfg.D_MODEL)

        for gi, idxs in enumerate(group_idxs):
            self.register_buffer(f"group_idx_{gi}", torch.tensor(idxs, dtype=torch.long))

        local_layer = make_encoder_layer(cfg.D_MODEL, cfg.NHEAD, cfg.FF_DIM, cfg.DROPOUT, cfg.ATTN_DROPOUT)
        self.local_encoder = nn.TransformerEncoder(local_layer, num_layers=cfg.LOCAL_LAYERS)

        self.cls = nn.Parameter(torch.zeros(1, 1, cfg.D_MODEL))
        nn.init.trunc_normal_(self.cls, std=0.02)

        global_layer = make_encoder_layer(cfg.D_MODEL, cfg.NHEAD, cfg.FF_DIM, cfg.DROPOUT, cfg.ATTN_DROPOUT)
        self.global_encoder = nn.TransformerEncoder(global_layer, num_layers=cfg.GLOBAL_LAYERS)

        self.pos_global = nn.Parameter(torch.zeros(1, 1 + self.num_groups, cfg.D_MODEL))
        nn.init.trunc_normal_(self.pos_global, std=0.02)

        self.head_fine = nn.Sequential(nn.LayerNorm(cfg.D_MODEL), nn.Dropout(cfg.DROPOUT), nn.Linear(cfg.D_MODEL, n_fine))
        self.head_coarse = nn.Sequential(nn.LayerNorm(cfg.D_MODEL), nn.Dropout(cfg.DROPOUT), nn.Linear(cfg.D_MODEL, n_coarse))
        self.head_bin = nn.Sequential(nn.LayerNorm(cfg.D_MODEL), nn.Dropout(cfg.DROPOUT), nn.Linear(cfg.D_MODEL, 2))

    def forward(self, x: torch.Tensor) -> Dict[str, torch.Tensor]:
        B = x.size(0)
        tok = self.tokenizer(x)

        group_reps = []
        for g in range(self.num_groups):
            idxs_t = getattr(self, f"group_idx_{g}")
            t = tok.index_select(dim=1, index=idxs_t)
            t = t + self.group_type.weight[g].view(1, 1, -1)

            gt = self.group_token[g].view(1, 1, -1).expand(B, 1, -1)
            z = torch.cat([gt, t], dim=1)
            z = self.local_encoder(z)
            group_reps.append(z[:, 0])

        G = torch.stack(group_reps, dim=1)
        cls = self.cls.expand(B, -1, -1)
        Z = torch.cat([cls, G], dim=1)
        Z = Z + self.pos_global
        Z = self.global_encoder(Z)
        h = Z[:, 0]

        return {
            "logits_fine": self.head_fine(h),
            "logits_coarse": self.head_coarse(h),
            "logits_bin": self.head_bin(h),
        }


model = CamelotIDSv2(
    n_features=num_features,
    n_fine=NUM_FINE,
    n_coarse=NUM_COARSE,
    group_idxs=GROUP_IDXS,
    cfg=cfg
).to(DEVICE)

if cfg.USE_TORCH_COMPILE and hasattr(torch, "compile"):
    try:
        model = torch.compile(model, mode="reduce-overhead")
        print("torch.compile enabled")
    except Exception as e:
        print("torch.compile skipped:", e)

n_params = sum(p.numel() for p in model.parameters()) / 1e6
print("Model params (M):", round(n_params, 3))


# %%
def softmax_np(x: np.ndarray) -> np.ndarray:
    x = x - x.max(axis=1, keepdims=True)
    ex = np.exp(x)
    return ex / np.clip(ex.sum(axis=1, keepdims=True), 1e-12, None)


def metrics_mc(y_true: np.ndarray, probs: np.ndarray) -> Dict[str, float]:
    y_pred = probs.argmax(axis=1)
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
        "macro_f1": float(f1_score(y_true, y_pred, average="macro")),
        "weighted_f1": float(f1_score(y_true, y_pred, average="weighted")),
    }


class EMA:
    def __init__(self, model: nn.Module, decay: float = 0.999):
        self.decay = float(decay)
        self.shadow = {}
        self.backup = {}
        for name, p in model.named_parameters():
            if p.requires_grad:
                self.shadow[name] = p.detach().clone()

    @torch.no_grad()
    def update(self, model: nn.Module):
        for name, p in model.named_parameters():
            if name in self.shadow:
                self.shadow[name].mul_(self.decay).add_(p.detach(), alpha=(1.0 - self.decay))

    def apply(self, model: nn.Module):
        self.backup = {}
        sd = model.state_dict()
        for name in self.shadow:
            self.backup[name] = sd[name].detach().clone()
            sd[name].copy_(self.shadow[name])

    def restore(self, model: nn.Module):
        sd = model.state_dict()
        for name, p in self.backup.items():
            sd[name].copy_(p)
        self.backup = {}

    def state_dict(self):
        return {
            "decay": self.decay,
            "shadow": self.shadow,
        }

    def load_state_dict(self, state):
        self.decay = float(state["decay"])
        self.shadow = {k: v.clone() for k, v in state["shadow"].items()}
        self.backup = {}


@torch.inference_mode()
def predict_logits(model: nn.Module, loader: DataLoader) -> Dict[str, np.ndarray]:
    model.eval()
    out = {"fine": [], "coarse": [], "bin": [], "yf": [], "yc": [], "yb": []}
    use_amp = (cfg.USE_AMP and DEVICE == "cuda")

    for xb, yf, yc, yb in loader:
        xb = xb.to(DEVICE, non_blocking=True)
        with torch.amp.autocast(device_type="cuda" if DEVICE == "cuda" else "cpu", enabled=use_amp):
            r = model(xb)
        out["fine"].append(r["logits_fine"].detach().cpu().numpy())
        out["coarse"].append(r["logits_coarse"].detach().cpu().numpy())
        out["bin"].append(r["logits_bin"].detach().cpu().numpy())
        out["yf"].append(yf.numpy())
        out["yc"].append(yc.numpy())
        out["yb"].append(yb.numpy())

    for k in ["fine", "coarse", "bin", "yf", "yc", "yb"]:
        out[k] = np.concatenate(out[k], axis=0)
    return out


def predict_logits_cached(name: str, model: nn.Module, loader: DataLoader, signature: Optional[Dict[str, Any]] = None) -> Dict[str, np.ndarray]:
    folder = LOGITS_CACHE_DIR / name
    meta_path = folder / "_meta.json"

    if cfg.RESUME and cfg.CACHE_LOGITS and folder_ready(folder, LOGIT_KEYS):
        cache_meta = load_json(meta_path, {})
        if same_signature(cache_meta.get("signature", {}), signature or {}):
            print(f"Loading cached logits: {name}")
            return load_array_dict(folder, LOGIT_KEYS)

    out = predict_logits(model, loader)

    if cfg.CACHE_LOGITS:
        save_array_dict(folder, out)
        save_json_atomic(meta_path, {
            "name": name,
            "signature": signature or {}
        })
    return out


def hierarchical_consistency_loss(logits_fine, logits_coarse, logits_bin):
    p_f = torch.softmax(logits_fine, dim=1)
    p_c = torch.softmax(logits_coarse, dim=1)
    p_b = torch.softmax(logits_bin, dim=1)

    B = p_f.size(0)
    idx = fine_to_coarse_t.unsqueeze(0).expand(B, -1)
    p_from_f = torch.zeros(B, NUM_COARSE, device=p_f.device)
    p_from_f.scatter_add_(1, idx, p_f)
    cons_c = F.kl_div(torch.log(p_c + 1e-8), p_from_f, reduction="batchmean")

    p_from_f_bin = torch.stack([p_f[:, benign_fine_id], 1.0 - p_f[:, benign_fine_id]], dim=1)
    cons_b = F.kl_div(torch.log(p_b + 1e-8), p_from_f_bin, reduction="batchmean")
    return cons_c, cons_b


def train_model(model: nn.Module, train_loader: DataLoader, val_loader: DataLoader, cfg: ExpConfig):
    last_ckpt_path = CKPT_DIR / "last_checkpoint.pt"
    best_ckpt_path = CKPT_DIR / "best_model.pt"
    done_path = STAGE_DIR / "training_done.json"
    history_csv_path = RUN_DIR / "epoch_history.csv"

    def make_empty_history():
        return {
            "epoch": [],
            "train_loss": [],

            "val_fine_accuracy": [],
            "val_fine_balanced_accuracy": [],
            "val_fine_macro_f1": [],
            "val_fine_weighted_f1": [],

            "val_coarse_accuracy": [],
            "val_coarse_balanced_accuracy": [],
            "val_coarse_macro_f1": [],
            "val_coarse_weighted_f1": [],

            "val_bin_f1": [],

            "best_score_so_far": [],
            "best_epoch_so_far": [],
        }

    def upgrade_history(history_obj):
        base = make_empty_history()

        if not isinstance(history_obj, dict):
            return base

        # copy existing keys
        for k in base.keys():
            if k in history_obj and isinstance(history_obj[k], list):
                base[k] = history_obj[k]

        # backward compatibility with your older history format
        if len(base["epoch"]) == 0:
            n_old = 0
            for k in history_obj.keys():
                if isinstance(history_obj[k], list):
                    n_old = max(n_old, len(history_obj[k]))
            if n_old > 0:
                base["epoch"] = list(range(1, n_old + 1))

        if len(base["val_fine_macro_f1"]) == 0 and "val_macro_f1_fine" in history_obj:
            base["val_fine_macro_f1"] = history_obj["val_macro_f1_fine"]

        if len(base["val_coarse_macro_f1"]) == 0 and "val_macro_f1_coarse" in history_obj:
            base["val_coarse_macro_f1"] = history_obj["val_macro_f1_coarse"]

        if len(base["val_bin_f1"]) == 0 and "val_f1_bin" in history_obj:
            base["val_bin_f1"] = history_obj["val_f1_bin"]

        if len(base["train_loss"]) == 0 and "train_loss" in history_obj:
            base["train_loss"] = history_obj["train_loss"]

        # make all lists same length
        max_len = max(len(v) for v in base.values()) if len(base) > 0 else 0
        for k in base.keys():
            if len(base[k]) < max_len:
                if k == "epoch" and len(base[k]) == 0:
                    base[k] = list(range(1, max_len + 1))
                else:
                    base[k] = base[k] + [float("nan")] * (max_len - len(base[k]))

        return base

    def save_history_csv(history_obj, csv_path):
        df_hist = pd.DataFrame(history_obj)
        df_hist.to_csv(csv_path, index=False)

    def print_history(history_obj):
        if len(history_obj["epoch"]) == 0:
            print("No completed epochs found in saved history.")
            return

        print("\nSaved completed epochs:")
        for i in range(len(history_obj["epoch"])):
            ep = history_obj["epoch"][i]
            tr_loss = history_obj["train_loss"][i]
            ff1 = history_obj["val_fine_macro_f1"][i]
            cf1 = history_obj["val_coarse_macro_f1"][i]
            bf1 = history_obj["val_bin_f1"][i]

            tr_loss_s = f"{tr_loss:.4f}" if pd.notna(tr_loss) else "nan"
            ff1_s = f"{ff1:.4f}" if pd.notna(ff1) else "nan"
            cf1_s = f"{cf1:.4f}" if pd.notna(cf1) else "nan"
            bf1_s = f"{bf1:.4f}" if pd.notna(bf1) else "nan"

            print(
                f"[Completed epoch {ep}] "
                f"loss={tr_loss_s} | "
                f"val_fine_macroF1={ff1_s} | "
                f"val_coarse_macroF1={cf1_s} | "
                f"val_binF1={bf1_s}"
            )

    empty_history = make_empty_history()

    if cfg.RESUME and done_path.exists() and best_ckpt_path.exists():
        done_meta = load_json(done_path, {})
        best_blob = torch.load(best_ckpt_path, map_location=DEVICE, weights_only=False)
        model.load_state_dict(best_blob["model_state"])
        history = upgrade_history(done_meta.get("history", empty_history))
        save_history_csv(history, history_csv_path)
        print(f"Training already complete. Loaded best model from epoch {done_meta.get('best_epoch', '?')}.")
        print_history(history)
        return history

    opt = torch.optim.AdamW(model.parameters(), lr=cfg.LR, weight_decay=cfg.WEIGHT_DECAY)

    total_steps = max(1, cfg.EPOCHS * len(train_loader))
    warmup_steps = max(1, int(cfg.WARMUP_EPOCHS * len(train_loader)))

    def lr_lambda(step):
        if step < warmup_steps:
            return max(1e-6, step / max(1, warmup_steps))
        t = (step - warmup_steps) / max(1, (total_steps - warmup_steps))
        return 0.5 * (1.0 + math.cos(math.pi * t))

    sched = torch.optim.lr_scheduler.LambdaLR(opt, lr_lambda)

    use_amp = (cfg.USE_AMP and DEVICE == "cuda")
    scaler_obj = torch.amp.GradScaler("cuda", enabled=use_amp) if DEVICE == "cuda" else None
    ema = EMA(model, decay=0.999)

    best_score = -1.0
    best_epoch = 0
    patience, bad = 7, 0
    start_epoch = 1
    history = make_empty_history()

    if cfg.RESUME and last_ckpt_path.exists():
        print("Resuming from last checkpoint...")
        ckpt = torch.load(last_ckpt_path, map_location=DEVICE, weights_only=False)

        model.load_state_dict(ckpt["model_state"])
        opt.load_state_dict(ckpt["optimizer_state"])
        sched.load_state_dict(ckpt["scheduler_state"])

        if scaler_obj is not None and ckpt.get("scaler_state") is not None:
            scaler_obj.load_state_dict(ckpt["scaler_state"])

        if ckpt.get("ema_state") is not None:
            ema.load_state_dict(ckpt["ema_state"])

        history = upgrade_history(ckpt.get("history", history))
        best_score = float(ckpt.get("best_score", -1.0))
        best_epoch = int(ckpt.get("best_epoch", 0))
        bad = int(ckpt.get("bad", 0))
        start_epoch = int(ckpt["epoch"]) + 1

        set_rng_state(ckpt.get("rng_state", None))
        print(f"Resumed at epoch {start_epoch}/{cfg.EPOCHS}")

        print_history(history)
        save_history_csv(history, history_csv_path)

    for epoch in range(start_epoch, cfg.EPOCHS + 1):
        model.train()
        loss_fine = make_fine_loss(epoch)
        loss_coarse, loss_bin = make_aux_losses(epoch)

        total_loss, n = 0.0, 0

        for xb, yf, yc, yb in tqdm(train_loader, desc=f"Epoch {epoch}/{cfg.EPOCHS}", leave=False):
            xb = xb.to(DEVICE, non_blocking=True)
            yf = yf.to(DEVICE, non_blocking=True)
            yc = yc.to(DEVICE, non_blocking=True)
            yb = yb.to(DEVICE, non_blocking=True)

            opt.zero_grad(set_to_none=True)

            with torch.amp.autocast(device_type="cuda" if DEVICE == "cuda" else "cpu", enabled=use_amp):
                out = model(xb)
                lf = loss_fine(out["logits_fine"], yf)
                lc = loss_coarse(out["logits_coarse"], yc)
                lb = loss_bin(out["logits_bin"], yb)
                cons_c, cons_b = hierarchical_consistency_loss(
                    out["logits_fine"], out["logits_coarse"], out["logits_bin"]
                )
                loss = (
                    lf
                    + cfg.W_COARSE * lc
                    + cfg.W_BIN * lb
                    + cfg.W_CONS_COARSE * cons_c
                    + cfg.W_CONS_BIN * cons_b
                )

            if use_amp:
                scaler_obj.scale(loss).backward()
                scaler_obj.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.GRAD_CLIP)
                scaler_obj.step(opt)
                scaler_obj.update()
            else:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.GRAD_CLIP)
                opt.step()

            sched.step()
            ema.update(model)

            total_loss += float(loss.item()) * int(xb.size(0))
            n += int(xb.size(0))

        train_loss = total_loss / max(n, 1)

        ema.apply(model)
        val_out = predict_logits(model, val_loader)
        ema.restore(model)

        val_probs_f = softmax_np(val_out["fine"])
        val_probs_c = softmax_np(val_out["coarse"])
        val_probs_b = softmax_np(val_out["bin"])

        m_f = metrics_mc(val_out["yf"], val_probs_f)
        m_c = metrics_mc(val_out["yc"], val_probs_c)
        yb_pred = val_probs_b.argmax(axis=1)
        f1_bin = float(f1_score(val_out["yb"], yb_pred, average="binary", pos_label=1))

        score = m_f["macro_f1"]
        if score > best_score:
            best_score = score
            best_epoch = epoch
            bad = 0

            atomic_torch_save({
                "epoch": epoch,
                "best_epoch": best_epoch,
                "best_score": float(best_score),
                "model_state": {k: v.detach().cpu().clone() for k, v in model.state_dict().items()},
            }, best_ckpt_path)
        else:
            bad += 1

        history["epoch"].append(epoch)
        history["train_loss"].append(train_loss)

        history["val_fine_accuracy"].append(m_f["accuracy"])
        history["val_fine_balanced_accuracy"].append(m_f["balanced_accuracy"])
        history["val_fine_macro_f1"].append(m_f["macro_f1"])
        history["val_fine_weighted_f1"].append(m_f["weighted_f1"])

        history["val_coarse_accuracy"].append(m_c["accuracy"])
        history["val_coarse_balanced_accuracy"].append(m_c["balanced_accuracy"])
        history["val_coarse_macro_f1"].append(m_c["macro_f1"])
        history["val_coarse_weighted_f1"].append(m_c["weighted_f1"])

        history["val_bin_f1"].append(f1_bin)

        history["best_score_so_far"].append(float(best_score))
        history["best_epoch_so_far"].append(int(best_epoch))

        print(
            f"[Epoch {epoch}] loss={train_loss:.4f} | "
            f"val_fine_acc={m_f['accuracy']:.4f} | "
            f"val_fine_macroF1={m_f['macro_f1']:.4f} | "
            f"val_coarse_acc={m_c['accuracy']:.4f} | "
            f"val_coarse_macroF1={m_c['macro_f1']:.4f} | "
            f"val_binF1={f1_bin:.4f}"
        )

        save_history_csv(history, history_csv_path)

        if cfg.SAVE_LAST_EVERY_EPOCH:
            atomic_torch_save({
                "epoch": epoch,
                "best_epoch": best_epoch,
                "best_score": float(best_score),
                "bad": int(bad),
                "model_state": model.state_dict(),
                "optimizer_state": opt.state_dict(),
                "scheduler_state": sched.state_dict(),
                "scaler_state": scaler_obj.state_dict() if scaler_obj is not None else None,
                "ema_state": ema.state_dict(),
                "history": history,
                "rng_state": get_rng_state(),
            }, last_ckpt_path)

        if bad >= patience:
            print("Early stopping.")
            break

    if best_ckpt_path.exists():
        best_blob = torch.load(best_ckpt_path, map_location=DEVICE, weights_only=False)
        model.load_state_dict(best_blob["model_state"])

    save_json_atomic(done_path, {
        "best_epoch": int(best_epoch),
        "best_score": float(best_score),
        "history": history,
    })

    save_history_csv(history, history_csv_path)
    return history
    
history = train_model(model, train_loader, val_loader, cfg)
training_signature = get_training_signature()


# %%
test_out = predict_logits_cached("test", model, test_loader, signature=training_signature)

test_probs_f = softmax_np(test_out["fine"])
test_probs_c = softmax_np(test_out["coarse"])
test_probs_b = softmax_np(test_out["bin"])

test_metrics_f = metrics_mc(test_out["yf"], test_probs_f)
test_metrics_c = metrics_mc(test_out["yc"], test_probs_c)

yb_pred = test_probs_b.argmax(axis=1)
test_f1_bin = float(f1_score(test_out["yb"], yb_pred, average="binary", pos_label=1))

print("TEST fine:", test_metrics_f)
print("TEST coarse:", test_metrics_c)
print("TEST binF1(malicious):", test_f1_bin)

with open(RUN_DIR / "test_metrics.json", "w", encoding="utf-8") as f:
    json.dump({"fine": test_metrics_f, "coarse": test_metrics_c, "binF1_mal": test_f1_bin}, f, indent=2)

y_pred_f = test_probs_f.argmax(axis=1)
prec, rec, f1c, sup = precision_recall_fscore_support(
    test_out["yf"], y_pred_f, labels=np.arange(NUM_FINE), zero_division=0
)
per_class = [
    {
        "class": FINE_CLASS_NAMES[i],
        "support": int(sup[i]),
        "precision": float(prec[i]),
        "recall": float(rec[i]),
        "f1": float(f1c[i]),
    }
    for i in range(NUM_FINE)
]
per_class_sorted = sorted(per_class, key=lambda d: d["f1"])

with open(RUN_DIR / "per_class_metrics_sorted_by_f1.json", "w", encoding="utf-8") as f:
    json.dump(per_class_sorted, f, indent=2)

print("Worst-10 fine classes by F1:")
for r in per_class_sorted[:10]:
    print(r)

cmc = confusion_matrix(test_out["yc"], test_probs_c.argmax(axis=1), labels=np.arange(NUM_COARSE))
plt.figure(figsize=(8, 6))
plt.imshow(cmc, aspect="auto")
plt.title("Confusion Matrix (Coarse 8-class)")
plt.xlabel("Pred")
plt.ylabel("True")
plt.colorbar()
plt.tight_layout()
plt.savefig(str(RUN_DIR / "confusion_matrix_coarse.png"), dpi=200)
plt.close()


# %%
def expected_calibration_error(y_true: np.ndarray, probs: np.ndarray, n_bins: int = 15) -> float:
    y_pred = probs.argmax(axis=1)
    conf = probs[np.arange(len(probs)), y_pred]
    acc = (y_pred == y_true).astype(np.float32)
    bins = np.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0
    for i in range(n_bins):
        lo, hi = bins[i], bins[i + 1]
        m = (conf >= lo) & (conf < hi) if i < n_bins - 1 else (conf >= lo) & (conf <= hi)
        if m.any():
            ece += (m.mean()) * abs(acc[m].mean() - conf[m].mean())
    return float(ece)


def brier_multi(y_true: np.ndarray, probs: np.ndarray, n_classes: int) -> float:
    y_onehot = np.eye(n_classes)[y_true]
    return float(np.mean(np.sum((probs - y_onehot) ** 2, axis=1)))


class TemperatureScaler(nn.Module):
    def __init__(self):
        super().__init__()
        self.log_t = nn.Parameter(torch.zeros(()))

    def forward(self, logits: torch.Tensor) -> torch.Tensor:
        t = torch.exp(self.log_t).clamp(1e-3, 100.0)
        return logits / t


def fit_temperature_np(logits_np: np.ndarray, y_np: np.ndarray) -> float:
    logits = torch.from_numpy(logits_np).to(DEVICE)
    y = torch.from_numpy(y_np).to(DEVICE)
    ts = TemperatureScaler().to(DEVICE)
    opt = torch.optim.LBFGS(ts.parameters(), lr=0.5, max_iter=50)
    nll = nn.CrossEntropyLoss()

    def closure():
        opt.zero_grad(set_to_none=True)
        loss = nll(ts(logits), y)
        loss.backward()
        return loss

    opt.step(closure)
    return float(torch.exp(ts.log_t).detach().cpu().item())


calibration_done_path = STAGE_DIR / "calibration_done.json"
cal_out = predict_logits_cached("cal", model, cal_loader, signature=training_signature)

if cfg.RESUME and calibration_done_path.exists() and (RUN_DIR / "calibration_summary.json").exists():
    calibration_done = load_json(calibration_done_path, {})
    if same_signature(calibration_done.get("signature", {}), training_signature):
        calibration_summary = load_json(RUN_DIR / "calibration_summary.json", {})
        temp_f = float(calibration_summary["temperature"]["fine"])
        temp_c = float(calibration_summary["temperature"]["coarse"])
        temp_b = float(calibration_summary["temperature"]["bin"])
        print("Loaded cached calibration summary.")
    else:
        calibration_summary = None
        temp_f, temp_c, temp_b = 1.0, 1.0, 1.0
else:
    calibration_summary = None
    temp_f, temp_c, temp_b = 1.0, 1.0, 1.0

if calibration_summary is None:
    if cfg.TEMPERATURE_SCALE:
        temp_f = fit_temperature_np(cal_out["fine"], cal_out["yf"])
        temp_c = fit_temperature_np(cal_out["coarse"], cal_out["yc"])
        temp_b = fit_temperature_np(cal_out["bin"], cal_out["yb"])

    print("Temperatures:", {"fine": temp_f, "coarse": temp_c, "bin": temp_b})

    test_probs_f_cal = softmax_np(test_out["fine"] / max(temp_f, 1e-6))
    test_probs_c_cal = softmax_np(test_out["coarse"] / max(temp_c, 1e-6))
    test_probs_b_cal = softmax_np(test_out["bin"] / max(temp_b, 1e-6))

    nll_before = float(log_loss(test_out["yf"], test_probs_f, labels=list(range(NUM_FINE))))
    nll_after = float(log_loss(test_out["yf"], test_probs_f_cal, labels=list(range(NUM_FINE))))
    ece_before = expected_calibration_error(test_out["yf"], test_probs_f, cfg.N_BINS_ECE)
    ece_after = expected_calibration_error(test_out["yf"], test_probs_f_cal, cfg.N_BINS_ECE)
    brier_before = brier_multi(test_out["yf"], test_probs_f, NUM_FINE)
    brier_after = brier_multi(test_out["yf"], test_probs_f_cal, NUM_FINE)

    calibration_summary = {
        "temperature": {"fine": temp_f, "coarse": temp_c, "bin": temp_b},
        "fine_before": {"nll": nll_before, "ece": ece_before, "brier": brier_before},
        "fine_after": {"nll": nll_after, "ece": ece_after, "brier": brier_after},
    }
    with open(RUN_DIR / "calibration_summary.json", "w", encoding="utf-8") as f:
        json.dump(calibration_summary, f, indent=2)
    save_json_atomic(calibration_done_path, {"signature": training_signature})
else:
    test_probs_f_cal = softmax_np(test_out["fine"] / max(temp_f, 1e-6))
    test_probs_c_cal = softmax_np(test_out["coarse"] / max(temp_c, 1e-6))
    test_probs_b_cal = softmax_np(test_out["bin"] / max(temp_b, 1e-6))

print("Calibration summary:", calibration_summary)

meta_obj = joblib.load(export_dir / "preprocess_and_meta.joblib")
meta_obj["temperature"] = {"fine": float(temp_f), "coarse": float(temp_c), "bin": float(temp_b)}
joblib.dump(meta_obj, export_dir / "preprocess_and_meta.joblib")


# %%
def conformal_quantile(scores: np.ndarray, alpha: float) -> float:
    n = len(scores)
    if n == 0:
        return 1.0
    q_level = math.ceil((n + 1) * (1 - alpha)) / n
    return float(np.quantile(scores, min(q_level, 1.0), method="higher"))


def raps_scores_true(probs: np.ndarray, y_true: np.ndarray, k_reg: int, lam: float) -> np.ndarray:
    n, C = probs.shape
    idx_sorted = np.argsort(-probs, axis=1)
    probs_sorted = np.take_along_axis(probs, idx_sorted, axis=1)
    cumsum = np.cumsum(probs_sorted, axis=1)

    inv = np.empty_like(idx_sorted)
    rows = np.arange(n)[:, None]
    inv[rows, idx_sorted] = np.arange(C)[None, :]
    rank0 = inv[rows[:, 0], y_true]
    rank1 = rank0 + 1

    score = cumsum[rows[:, 0], rank0] + lam * np.maximum(rank1 - k_reg, 0)
    return score.astype(np.float32)


def raps_predict_k(probs: np.ndarray, q: np.ndarray, k_reg: int, lam: float) -> np.ndarray:
    n, C = probs.shape
    idx_sorted = np.argsort(-probs, axis=1)
    probs_sorted = np.take_along_axis(probs, idx_sorted, axis=1)
    cumsum = np.cumsum(probs_sorted, axis=1)
    ranks = np.arange(1, C + 1, dtype=np.float32)[None, :]
    reg = lam * np.maximum(ranks - float(k_reg), 0.0)
    score_k = cumsum + reg
    ok = score_k <= q[:, None]
    k_star = ok.sum(axis=1).astype(np.int64)
    return np.maximum(k_star, 1)


def raps_eval(probs: np.ndarray, y_true: np.ndarray, q_row: np.ndarray, k_reg: int, lam: float) -> Dict[str, float]:
    n, C = probs.shape
    idx_sorted = np.argsort(-probs, axis=1)

    inv = np.empty_like(idx_sorted)
    rows = np.arange(n)[:, None]
    inv[rows, idx_sorted] = np.arange(C)[None, :]
    rank0 = inv[rows[:, 0], y_true]

    k_star = raps_predict_k(probs, q_row, k_reg, lam)
    covered = (rank0 < k_star).astype(np.float32)
    return {"coverage": float(covered.mean()), "avg_set_size": float(k_star.mean())}


raps_done_path = STAGE_DIR / "raps_done.json"
if cfg.RESUME and raps_done_path.exists() and (RUN_DIR / "raps_calibration.json").exists() and (RUN_DIR / "raps_results.json").exists():
    raps_done = load_json(raps_done_path, {})
    if same_signature(raps_done.get("signature", {}), training_signature):
        raps_calib = load_json(RUN_DIR / "raps_calibration.json", {})
        raps_results = load_json(RUN_DIR / "raps_results.json", {})
        print("Loaded cached RAPS outputs.")
    else:
        raps_calib = None
        raps_results = None
else:
    raps_calib = None
    raps_results = None

if raps_calib is None or raps_results is None:
    cal_probs_f = softmax_np(cal_out["fine"] / max(temp_f, 1e-6))
    cal_probs_c = softmax_np(cal_out["coarse"] / max(temp_c, 1e-6))

    scores_global = raps_scores_true(cal_probs_f, cal_out["yf"], cfg.RAPS_KREG, cfg.RAPS_LAMBDA)
    g_hat = cal_probs_c.argmax(axis=1)
    scores_by_g = {g: scores_global[g_hat == g] for g in range(NUM_COARSE)}

    raps_calib = {
        "mode": cfg.RAPS_MONDRIAN,
        "k_reg": int(cfg.RAPS_KREG),
        "lambda": float(cfg.RAPS_LAMBDA),
        "alphas": [float(a) for a in cfg.ALPHAS],
        "q_global": {},
        "q_by_pred_coarse": {},
        "min_group_n": 200
    }

    for a in cfg.ALPHAS:
        qg = conformal_quantile(scores_global, a)
        raps_calib["q_global"][str(a)] = float(qg)

        q_by = []
        for g in range(NUM_COARSE):
            sg = scores_by_g.get(g, np.array([], dtype=np.float32))
            if len(sg) >= raps_calib["min_group_n"]:
                q_by.append(conformal_quantile(sg, a))
            else:
                q_by.append(qg)
        raps_calib["q_by_pred_coarse"][str(a)] = [float(x) for x in q_by]

    with open(RUN_DIR / "raps_calibration.json", "w", encoding="utf-8") as f:
        json.dump(raps_calib, f, indent=2)

    g_hat_test = test_probs_c_cal.argmax(axis=1)
    raps_results = {}
    for a in cfg.ALPHAS:
        q_by = np.array(raps_calib["q_by_pred_coarse"][str(a)], dtype=np.float32)
        q_row = q_by[g_hat_test]
        r = raps_eval(test_probs_f_cal, test_out["yf"], q_row, cfg.RAPS_KREG, cfg.RAPS_LAMBDA)
        raps_results[str(a)] = r

    with open(RUN_DIR / "raps_results.json", "w", encoding="utf-8") as f:
        json.dump(raps_results, f, indent=2)

    save_json_atomic(raps_done_path, {"signature": training_signature})

print("RAPS results:", raps_results)


def singleton_risk_coverage(y_true, probs_f, probs_c, raps_calib, alpha, k_reg, lam, benign_id=0):
    q_by = np.array(raps_calib["q_by_pred_coarse"][str(alpha)], dtype=np.float32)
    g_hat = probs_c.argmax(axis=1)
    q_row = q_by[g_hat]
    k_star = raps_predict_k(probs_f, q_row, k_reg, lam)

    single = (k_star == 1)
    y_pred = probs_f.argmax(axis=1)

    cov = float(single.mean())
    if single.any():
        acc = float((y_pred[single] == y_true[single]).mean())
        mf1 = float(f1_score(y_true[single], y_pred[single], average="macro"))
    else:
        acc, mf1 = float("nan"), float("nan")

    benign_mask = (y_true == benign_id) & single
    benign_fpr = float((y_pred[benign_mask] != benign_id).mean()) if benign_mask.any() else float("nan")

    return {
        "alpha": float(alpha),
        "coverage_singleton": cov,
        "risk_1_minus_acc": float(1.0 - acc),
        "macro_f1_on_decisions": mf1,
        "benign_FPR_on_decisions": benign_fpr,
        "avg_set_size": float(k_star.mean()),
    }


sel_rows = [
    singleton_risk_coverage(
        test_out["yf"], test_probs_f_cal, test_probs_c_cal,
        raps_calib, a, cfg.RAPS_KREG, cfg.RAPS_LAMBDA, benign_fine_id
    )
    for a in cfg.ALPHAS
]
df_sel = pd.DataFrame(sel_rows)
df_sel.to_csv(RUN_DIR / "risk_coverage_table_raps.csv", index=False)
print(df_sel)

meta_obj = joblib.load(export_dir / "preprocess_and_meta.joblib")
meta_obj["raps_calib"] = raps_calib
joblib.dump(meta_obj, export_dir / "preprocess_and_meta.joblib")


# %%
def make_drift_detector(cfg: ExpConfig):
    try:
        from river import drift
        if "KSWIN" in cfg.DRIFT_DETECTOR.upper():
            return drift.KSWIN(alpha=cfg.DRIFT_DELTA, window_size=100, stat_size=30)
        if "ADWIN" in cfg.DRIFT_DETECTOR.upper():
            return drift.ADWIN(delta=cfg.DRIFT_DELTA)
    except Exception:
        pass
    return None


class ConformalMartingale:
    def __init__(self, ref_values: np.ndarray, eps: float = 0.5, threshold: float = 25.0):
        self.ref = np.asarray(ref_values, dtype=np.float32)
        self.eps = float(eps)
        self.threshold = float(threshold)
        self.M = 1.0

    def p_value(self, x: float) -> float:
        ref = self.ref
        return float((1.0 + np.sum(ref >= x)) / (len(ref) + 1.0))

    def update(self, x: float) -> Dict[str, float]:
        p = self.p_value(x)
        self.M *= self.eps * (p ** (self.eps - 1.0))
        drift = float(self.M > self.threshold)
        if drift:
            self.M = 1.0
        return {"p": float(p), "M": float(self.M), "drift": float(drift)}


class TentAdapter:
    def __init__(self, model: nn.Module, lr: float = 1e-4, steps: int = 5):
        self.model = model
        self.steps = int(steps)

        for p in self.model.parameters():
            p.requires_grad = False

        self.params = []
        for m in self.model.modules():
            if isinstance(m, (nn.LayerNorm, nn.BatchNorm1d, nn.GroupNorm)):
                for p in m.parameters():
                    p.requires_grad = True
                    self.params.append(p)

        self.opt = torch.optim.Adam(self.params, lr=lr)

        for m in self.model.modules():
            if isinstance(m, nn.Dropout):
                m.eval()

    def adapt(self, xb_cpu: torch.Tensor, micro_bs: int):
        if len(self.params) == 0:
            return
        self.model.train()
        for _ in range(self.steps):
            for j in range(0, xb_cpu.size(0), micro_bs):
                xb = xb_cpu[j:j + micro_bs].to(DEVICE, non_blocking=True)
                out = self.model(xb)
                p = torch.softmax(out["logits_fine"], dim=1)
                ent = -(p * torch.log(p + 1e-8)).sum(dim=1).mean()
                self.opt.zero_grad(set_to_none=True)
                ent.backward()
                self.opt.step()
                del xb, out, p, ent


class HeadTuner:
    def __init__(self, model: nn.Module, lr: float = 5e-5, steps: int = 10):
        self.model = model
        self.steps = int(steps)

        for p in self.model.parameters():
            p.requires_grad = False

        params = []
        for head in [self.model.head_fine, self.model.head_coarse, self.model.head_bin]:
            for p in head.parameters():
                p.requires_grad = True
                params.append(p)

        self.opt = torch.optim.Adam(params, lr=lr)
        self.model.eval()

    def tune(self, xb_cpu: torch.Tensor, y_f: torch.Tensor, y_c: torch.Tensor, y_b: torch.Tensor, micro_bs: int):
        self.model.train()
        y_f = y_f.cpu()
        y_c = y_c.cpu()
        y_b = y_b.cpu()

        for _ in range(self.steps):
            for j in range(0, xb_cpu.size(0), micro_bs):
                xb = xb_cpu[j:j + micro_bs].to(DEVICE, non_blocking=True)
                yf = y_f[j:j + micro_bs].to(DEVICE, non_blocking=True)
                yc = y_c[j:j + micro_bs].to(DEVICE, non_blocking=True)
                yb = y_b[j:j + micro_bs].to(DEVICE, non_blocking=True)

                out = self.model(xb)
                lf = F.cross_entropy(out["logits_fine"], yf)
                lc = F.cross_entropy(out["logits_coarse"], yc)
                lb = F.cross_entropy(out["logits_bin"], yb)
                loss = lf + 0.3 * lc + 0.2 * lb
                self.opt.zero_grad(set_to_none=True)
                loss.backward()
                self.opt.step()

                del xb, yf, yc, yb, out, lf, lc, lb, loss


@torch.inference_mode()
def batch_predict_probs(
    model: nn.Module,
    xb_np: np.ndarray,
    temp_f: float,
    temp_c: float,
    micro_bs: int = 16_384
):
    model.eval()
    pf_parts, pc_parts = [], []
    use_amp = (cfg.USE_AMP and DEVICE == "cuda")

    for j in range(0, len(xb_np), micro_bs):
        xb = torch.from_numpy(xb_np[j:j + micro_bs].astype(np.float32, copy=False)).to(DEVICE, non_blocking=True)
        with torch.amp.autocast(device_type="cuda" if DEVICE == "cuda" else "cpu", enabled=use_amp):
            out = model(xb)
            pf = torch.softmax(out["logits_fine"] / max(temp_f, 1e-6), dim=1).cpu()
            pc = torch.softmax(out["logits_coarse"] / max(temp_c, 1e-6), dim=1).cpu()

        pf_parts.append(pf)
        pc_parts.append(pc)
        del xb, out, pf, pc

    if DEVICE == "cuda":
        torch.cuda.empty_cache()

    return torch.cat(pf_parts, dim=0).numpy(), torch.cat(pc_parts, dim=0).numpy()


def entropy_from_probs(p: np.ndarray) -> float:
    return float(np.mean(-np.sum(p * np.log(p + 1e-12), axis=1)))


def signal_from_probs(pf: np.ndarray, pc: np.ndarray, cfg: ExpConfig, raps_calib: Dict[str, Any]) -> float:
    if cfg.DRIFT_SIGNAL == "1-maxprob":
        return float(np.mean(1.0 - pf.max(axis=1)))
    if cfg.DRIFT_SIGNAL == "set_size":
        alpha = float(cfg.ADAPT_ALPHA)
        q_by = np.array(raps_calib["q_by_pred_coarse"][str(alpha)], dtype=np.float32)
        g_hat = pc.argmax(axis=1)
        q_row = q_by[g_hat]
        k_star = raps_predict_k(pf, q_row, cfg.RAPS_KREG, cfg.RAPS_LAMBDA)
        return float(k_star.mean())
    return entropy_from_probs(pf)


def stream_eval(cfg: ExpConfig, Xs: np.ndarray, ys_f: np.ndarray):
    ref_probs_f = softmax_np(cal_out["fine"] / max(temp_f, 1e-6))
    ref_ent = -np.sum(ref_probs_f * np.log(ref_probs_f + 1e-12), axis=1)
    ref_ent = ref_ent[:min(len(ref_ent), 200_000)]

    det = make_drift_detector(cfg)
    mart = ConformalMartingale(ref_ent, eps=cfg.MART_EPS, threshold=cfg.MART_THRESHOLD) if "MART" in cfg.DRIFT_DETECTOR.upper() else None

    tent = TentAdapter(model, lr=cfg.ADAPT_LR, steps=cfg.ADAPT_STEPS) if "TENT" in cfg.ADAPT_METHOD.upper() else None
    head_tuner = HeadTuner(model, lr=cfg.HEAD_TUNE_LR, steps=cfg.HEAD_TUNE_STEPS) if "HEAD" in cfg.ADAPT_METHOD.upper() else None

    B = cfg.STREAM_BATCH
    MB = cfg.STREAM_MICRO_BATCH
    acc_hist, sig_hist, drift_points = [], [], []
    p_hist, m_hist = [], []
    buffer = []

    for bi, i in enumerate(tqdm(range(0, len(Xs), B), desc="Stream eval")):
        xb_np = Xs[i:i + B]
        yb = ys_f[i:i + B]
        if len(xb_np) == 0:
            break

        pf, pc = batch_predict_probs(model, xb_np, temp_f, temp_c, micro_bs=MB)
        pred = pf.argmax(axis=1)
        acc = float(np.mean(pred == yb))
        acc_hist.append(acc)

        sgn = signal_from_probs(pf, pc, cfg, raps_calib)
        sig_hist.append(sgn)

        drifted = False

        if det is not None and ("ADWIN" in cfg.DRIFT_DETECTOR.upper() or "KSWIN" in cfg.DRIFT_DETECTOR.upper()):
            det.update(sgn)
            drifted = drifted or bool(getattr(det, "drift_detected", False))

        if mart is not None:
            upd = mart.update(sgn)
            p_hist.append(upd["p"])
            m_hist.append(upd["M"])
            drifted = drifted or bool(upd["drift"] > 0)

        xb_cpu = torch.from_numpy(xb_np.astype(np.float32, copy=False))
        buffer.append((xb_cpu, pf, pc))
        if len(buffer) > cfg.ADAPT_BUFFER_BATCHES:
            buffer.pop(0)

        if drifted:
            drift_points.append(bi)

            if tent is not None:
                for xb_buf, _, _ in buffer:
                    tent.adapt(xb_buf, micro_bs=MB)

            if head_tuner is not None:
                alpha = float(cfg.ADAPT_ALPHA)
                q_by = np.array(raps_calib["q_by_pred_coarse"][str(alpha)], dtype=np.float32)

                xb_buf, pf_buf, pc_buf = buffer[-1]
                g_hat = pc_buf.argmax(axis=1)
                q_row = q_by[g_hat]
                k_star = raps_predict_k(pf_buf, q_row, cfg.RAPS_KREG, cfg.RAPS_LAMBDA)
                single = (k_star == 1)

                if int(single.sum()) >= cfg.MIN_PSEUDO:
                    y_pseudo_f = pf_buf.argmax(axis=1)[single]
                    y_pseudo_c = fine_to_coarse[y_pseudo_f]
                    y_pseudo_b = (y_pseudo_f != benign_fine_id).astype(np.int64)

                    xb_sel = xb_buf[single]
                    yf_t = torch.from_numpy(y_pseudo_f.astype(np.int64))
                    yc_t = torch.from_numpy(y_pseudo_c.astype(np.int64))
                    yb_t = torch.from_numpy(y_pseudo_b.astype(np.int64))
                    head_tuner.tune(xb_sel, yf_t, yc_t, yb_t, micro_bs=MB)

    fig, ax = plt.subplots(figsize=(12, 4))
    ax.plot(acc_hist, label="fine accuracy (labels only for analysis)")
    for p in drift_points:
        ax.axvline(p, linestyle=":", alpha=0.7)
    ax.set_title("Streaming accuracy with drift points")
    ax.set_xlabel("batch idx")
    ax.set_ylabel("accuracy")
    ax.grid(True, alpha=0.3)
    ax.legend()
    fig.savefig(str(RUN_DIR / "stream_accuracy.png"), dpi=200)
    plt.close(fig)

    fig, ax = plt.subplots(figsize=(12, 4))
    ax.plot(sig_hist, label=f"signal={cfg.DRIFT_SIGNAL}")
    for p in drift_points:
        ax.axvline(p, linestyle=":", alpha=0.7)
    ax.set_title("Streaming drift signal")
    ax.set_xlabel("batch idx")
    ax.set_ylabel(cfg.DRIFT_SIGNAL)
    ax.grid(True, alpha=0.3)
    ax.legend()
    fig.savefig(str(RUN_DIR / "stream_signal.png"), dpi=200)
    plt.close(fig)

    if mart is not None and len(p_hist) == len(sig_hist):
        fig, ax = plt.subplots(figsize=(12, 4))
        ax.plot(m_hist, label="martingale M")
        ax.axhline(cfg.MART_THRESHOLD, linestyle="--", alpha=0.7, label="threshold")
        ax.set_title("Conformal martingale")
        ax.set_xlabel("batch idx")
        ax.set_ylabel("M")
        ax.grid(True, alpha=0.3)
        ax.legend()
        fig.savefig(str(RUN_DIR / "stream_martingale.png"), dpi=200)
        plt.close(fig)

    out = {"signal": cfg.DRIFT_SIGNAL, "acc_hist": acc_hist, "sig_hist": sig_hist, "drift_points": drift_points}
    with open(RUN_DIR / "stream_results.json", "w", encoding="utf-8") as f:
        json.dump(out, f, indent=2)
    return out


stream_results = None
stream_done_path = STAGE_DIR / "stream_done.json"
stream_json = RUN_DIR / "stream_results.json"

if cfg.DO_STREAM_EVAL:
    cached_ok = False
    if cfg.RESUME and stream_done_path.exists() and stream_json.exists():
        stream_done = load_json(stream_done_path, {})
        cached_ok = same_signature(stream_done.get("signature", {}), training_signature)

    if cached_ok:
        stream_results = load_json(stream_json, None)
        print("Loaded cached stream results.")
    else:
        stream_results = stream_eval(cfg, X_stream_test, y_stream_f)
        save_json_atomic(stream_done_path, {"signature": training_signature})

    print("Drift points:", stream_results["drift_points"], "total:", len(stream_results["drift_points"]))


# %%
def sample_subset(X: np.ndarray, y: np.ndarray, max_n: int, seed: int = 42):
    if len(X) <= max_n:
        return X, y
    rng = np.random.default_rng(seed)
    idx = rng.choice(len(X), size=max_n, replace=False)
    return X[idx], y[idx]


baseline_results = {}

if cfg.RUN_BASELINES:
    Xtr_b, ytr_b = sample_subset(X_train, y_train_f, cfg.BASELINE_MAX_TRAIN, cfg.SEED)
    Xte_b, yte_b = sample_subset(X_test, y_test_f, cfg.BASELINE_MAX_TEST, cfg.SEED)

    try:
        from catboost import CatBoostClassifier
        cb = CatBoostClassifier(
            loss_function="MultiClass",
            iterations=2500,
            depth=10,
            learning_rate=0.08,
            auto_class_weights="Balanced",
            verbose=250,
            task_type="GPU" if DEVICE == "cuda" else "CPU"
        )
        cb.fit(Xtr_b, ytr_b, eval_set=(Xte_b[:min(200000, len(Xte_b))], yte_b[:min(200000, len(yte_b))]))
        p = cb.predict_proba(Xte_b)
        baseline_results["CatBoost_fine34"] = metrics_mc(yte_b, p)
    except Exception as e:
        baseline_results["CatBoost_fine34"] = {"error": str(e)}

    try:
        import xgboost as xgb
        xgbm = xgb.XGBClassifier(
            n_estimators=1600,
            max_depth=10,
            learning_rate=0.06,
            subsample=0.85,
            colsample_bytree=0.85,
            tree_method="hist",
            n_jobs=-1
        )
        xgbm.fit(Xtr_b, ytr_b)
        p = xgbm.predict_proba(Xte_b)
        baseline_results["XGBoost_fine34"] = metrics_mc(yte_b, p)
    except Exception as e:
        baseline_results["XGBoost_fine34"] = {"error": str(e)}

with open(RUN_DIR / "baseline_results.json", "w", encoding="utf-8") as f:
    json.dump(baseline_results, f, indent=2)

print("Baselines:", baseline_results)


# %%
@torch.inference_mode()
def measure_throughput(model: nn.Module, X_np: np.ndarray, batch_size: int = 4096, device: str = DEVICE):
    model.eval()
    n = len(X_np)
    if n == 0:
        return {"samples_per_sec": float("nan"), "ms_per_sample": float("nan")}

    warm = min(batch_size, n)
    xb = torch.from_numpy(X_np[:warm].astype(np.float32, copy=False)).to(device)
    for _ in range(5):
        _ = model(xb)

    if device == "cuda":
        torch.cuda.synchronize()

    t0 = time.time()
    seen = 0
    for i in range(0, n, batch_size):
        xb = torch.from_numpy(X_np[i:i + batch_size].astype(np.float32, copy=False)).to(device)
        _ = model(xb)
        seen += xb.size(0)

    if device == "cuda":
        torch.cuda.synchronize()

    dt = time.time() - t0
    sps = seen / max(dt, 1e-9)
    return {"samples_per_sec": float(sps), "ms_per_sample": float(1000.0 / sps)}


throughput_gpu = measure_throughput(model, X_test[:min(len(X_test), 200_000)], batch_size=4096, device=DEVICE)

model_cpu_for_bench = model.to("cpu")
throughput_cpu = measure_throughput(model_cpu_for_bench, X_test[:min(len(X_test), 200_000)], batch_size=4096, device="cpu")
model = model_cpu_for_bench.to(DEVICE)

print("Throughput GPU:", throughput_gpu)
print("Throughput CPU:", throughput_cpu)

torch.save({"state_dict": model.state_dict()}, export_dir / "camelot_ids_v2.pt")


class LogitsWrapper(nn.Module):
    def __init__(self, base: nn.Module):
        super().__init__()
        self.base = base

    def forward(self, x: torch.Tensor):
        o = self.base(x)
        return o["logits_fine"], o["logits_coarse"], o["logits_bin"]


wrapper = LogitsWrapper(model).to(DEVICE).eval()

if cfg.EXPORT_TORCHSCRIPT:
    try:
        example = torch.randn(1, num_features, device=DEVICE)
        ts = torch.jit.trace(wrapper, example)
        ts.save(str(export_dir / "camelot_ids_v2_logits_torchscript.pt"))
        print("TorchScript saved.")
    except Exception as e:
        print("TorchScript failed:", e)

if cfg.EXPORT_ONNX:
    try:
        dummy = torch.randn(1, num_features, device=DEVICE)
        torch.onnx.export(
            wrapper,
            dummy,
            str(export_dir / "camelot_ids_v2_logits.onnx"),
            input_names=["input"],
            output_names=["logits_fine", "logits_coarse", "logits_bin"],
            dynamic_axes={
                "input": {0: "batch"},
                "logits_fine": {0: "batch"},
                "logits_coarse": {0: "batch"},
                "logits_bin": {0: "batch"},
            },
            opset_version=13
        )
        print("ONNX saved.")
    except Exception as e:
        print("ONNX failed:", e)

if cfg.EXPORT_INT8_DYNAMIC:
    try:
        wrapper_cpu = LogitsWrapper(model.to("cpu")).eval()
        qmodel = torch.quantization.quantize_dynamic(wrapper_cpu, {nn.Linear}, dtype=torch.qint8)
        example = torch.randn(1, num_features)
        qts = torch.jit.trace(qmodel, example)
        qts.save(str(export_dir / "camelot_ids_v2_logits_torchscript_int8.pt"))
        print("Dynamic int8 TorchScript saved.")
        model = model.to(DEVICE)
    except Exception as e:
        print("Dynamic quantization failed:", e)
        model = model.to(DEVICE)

cli_code = r'''
import argparse
from pathlib import Path
import numpy as np
import pandas as pd
import joblib
import torch


def sanitize_frame(df: pd.DataFrame, clip_inf: float = 1e9) -> pd.DataFrame:
    df = df.replace({"Infinity": np.inf, "inf": np.inf, "+inf": np.inf, "-inf": -np.inf,
                     "NaN": np.nan, "nan": np.nan, "": np.nan})
    df = df.replace([np.inf, -np.inf], np.nan)
    num_cols = df.select_dtypes(include=[np.number]).columns
    if len(num_cols) > 0:
        df[num_cols] = df[num_cols].clip(-clip_inf, clip_inf)
    return df


def transform_df(df_in: pd.DataFrame, feature_cols, meta_obj):
    cfg = meta_obj["cfg"]
    X = sanitize_frame(df_in[feature_cols].copy(), cfg["CLIP_INF"])
    for c in meta_obj["meta"]["numeric_like_obj"]:
        if c in X.columns:
            X[c] = pd.to_numeric(X[c], errors="coerce")
    num_cols = meta_obj["meta"]["num_cols"]
    X = X[num_cols]
    Xn = meta_obj["num_imputer"].transform(X).astype(np.float32)
    Xn = meta_obj["scaler"].transform(Xn).astype(np.float32)
    Xn = np.clip(Xn, -cfg["POST_SCALE_CLIP"], cfg["POST_SCALE_CLIP"]).astype(np.float32)
    Xn = np.nan_to_num(Xn, nan=0.0, posinf=cfg["POST_SCALE_CLIP"], neginf=-cfg["POST_SCALE_CLIP"]).astype(np.float32)
    return Xn


def softmax_np(x):
    x = x - x.max(axis=1, keepdims=True)
    ex = np.exp(x)
    return ex / np.clip(ex.sum(axis=1, keepdims=True), 1e-12, None)


def raps_predict_set(probs_f, probs_c, raps_calib, alpha):
    k_reg = int(raps_calib["k_reg"])
    lam = float(raps_calib["lambda"])
    q_by = np.array(raps_calib["q_by_pred_coarse"][str(alpha)], dtype=np.float32)
    g_hat = probs_c.argmax(axis=1)
    q_row = q_by[g_hat]

    idx_sorted = np.argsort(-probs_f, axis=1)
    probs_sorted = np.take_along_axis(probs_f, idx_sorted, axis=1)
    cumsum = np.cumsum(probs_sorted, axis=1)
    C = probs_f.shape[1]
    ranks = np.arange(1, C + 1, dtype=np.float32)[None, :]
    reg = lam * np.maximum(ranks - float(k_reg), 0.0)
    score_k = cumsum + reg
    ok = score_k <= q_row[:, None]
    k_star = np.maximum(ok.sum(axis=1).astype(np.int64), 1)

    sets = []
    for i in range(len(probs_f)):
        sets.append(idx_sorted[i, :k_star[i]])
    return sets, k_star


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--export_dir", type=str, required=True)
    ap.add_argument("--input_csv", type=str, required=True)
    ap.add_argument("--out_csv", type=str, default="preds.csv")
    ap.add_argument("--device", type=str, default="cpu")
    ap.add_argument("--int8", action="store_true")
    ap.add_argument("--alpha", type=float, default=0.10)
    args = ap.parse_args()

    exp = Path(args.export_dir)
    meta = joblib.load(exp / "preprocess_and_meta.joblib")
    feature_cols = meta["FEATURE_COLS"]
    fine_names = meta["fine_class_names"]
    coarse_names = meta["coarse_names"]
    temp = meta.get("temperature", {"fine": 1.0, "coarse": 1.0, "bin": 1.0})
    raps_calib = meta.get("raps_calib", None)
    if raps_calib is None:
        raise RuntimeError("No raps_calib found in preprocess_and_meta.joblib")

    model_path = exp / ("camelot_ids_v2_logits_torchscript_int8.pt" if args.int8 else "camelot_ids_v2_logits_torchscript.pt")
    model = torch.jit.load(str(model_path), map_location=args.device)
    model.eval()

    df = pd.read_csv(args.input_csv, low_memory=False)
    X = transform_df(df, feature_cols, meta)
    xb = torch.from_numpy(X).to(args.device)

    with torch.inference_mode():
        logits_f, logits_c, logits_b = model(xb)
        logits_f = logits_f.cpu().numpy()
        logits_c = logits_c.cpu().numpy()
        logits_b = logits_b.cpu().numpy()

    pf = softmax_np(logits_f / max(float(temp["fine"]), 1e-6))
    pc = softmax_np(logits_c / max(float(temp["coarse"]), 1e-6))
    pb = softmax_np(logits_b / max(float(temp["bin"]), 1e-6))

    top_f = pf.argmax(axis=1)
    top_conf = pf[np.arange(len(pf)), top_f]
    top_c = pc.argmax(axis=1)
    top_b = pb.argmax(axis=1)

    sets, k_star = raps_predict_set(pf, pc, raps_calib, float(args.alpha))
    set_str = [",".join([fine_names[j] for j in s]) for s in sets]

    out = pd.DataFrame({
        "top1_fine": [fine_names[i] for i in top_f],
        "top1_conf": top_conf,
        "pred_coarse": [coarse_names[i] for i in top_c],
        "pred_bin": ["Malicious" if i == 1 else "Benign" for i in top_b],
        "raps_set_size": k_star,
        "raps_set": set_str,
    })
    out.to_csv(args.out_csv, index=False)
    print("Saved:", args.out_csv)


if __name__ == "__main__":
    main()
'''
(export_dir / "camelot_infer_v2.py").write_text(cli_code, encoding="utf-8")
print("CLI written:", export_dir / "camelot_infer_v2.py")


# %%
final_summary = {
    "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
    "split_mode": split_mode,
    "num_samples": n_total_samples,
    "num_features": int(num_features),
    "num_fine": int(NUM_FINE),
    "num_coarse": int(NUM_COARSE),
    "test_fine": test_metrics_f,
    "test_coarse": test_metrics_c,
    "test_binF1_mal": test_f1_bin,
    "calibration": calibration_summary,
    "raps": raps_results,
    "baselines": baseline_results,
    "throughput_gpu": throughput_gpu,
    "throughput_cpu": throughput_cpu,
    "stream_results": stream_results,
}
with open(RUN_DIR / "final_summary.json", "w", encoding="utf-8") as f:
    json.dump(final_summary, f, indent=2)

print("Saved:", RUN_DIR / "final_summary.json")
print("All outputs in:", cfg.OUT_DIR)


DEVICE: cuda
OUT_DIR: ./results_camelot_ids_v2_pc
Loading cached preprocessed arrays...
Loaded cached arrays.
Shapes: (21705973, 46) (2631074, 46) (1389497, 46) (1372896, 46)
num_features: 46 | num_fine: 34 | num_coarse: 8
Groups: ['rates', 'flags', 'protocols', 'stats', 'other']
 - rates: 3 features
 - flags: 12 features
 - protocols: 14 features
 - stats: 13 features
 - other: 4 features
Train batches: 21198 | Val batches: 2570
Fine class count stats: 604 3347293
Model params (M): 6.359
Training already complete. Loaded best model from epoch 61.

Saved completed epochs:
[Completed epoch 1] loss=1.7803 | val_fine_macroF1=0.6532 | val_coarse_macroF1=0.6886 | val_binF1=0.9968
[Completed epoch 2] loss=0.3281 | val_fine_macroF1=0.6563 | val_coarse_macroF1=0.6771 | val_binF1=0.9965
[Completed epoch 3] loss=0.2321 | val_fine_macroF1=0.6635 | val_coarse_macroF1=0.6880 | val_binF1=0.9966
[Completed epoch 4] loss=0.2460 | val_fine_macroF1=0.6827 | val_coarse_macroF1=0.6935 | val_binF1=0.9968
[

W0828 09:52:52.086000 19380 site-packages\torch\onnx\_internal\exporter\_compat.py:114] Setting ONNX exporter to use operator set version 18 because the requested opset_version 13 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features


[torch.onnx] Obtain model graph for `LogitsWrapper([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `LogitsWrapper([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decomposition...
[torch.onnx] Run decomposition... ✅
[torch.onnx] Translate the graph into ONNX...


The model version conversion is not supported by the onnxscript version converter and fallback is enabled. The model will be converted using the onnx C API (target version: 13).


[torch.onnx] Translate the graph into ONNX... ✅


Failed to convert the model to the target version 13 using the ONNX C API. The model was not modified
Traceback (most recent call last):
  File "C:\Users\HaseebWajid\AppData\Local\anaconda3\envs\torch_env\Lib\site-packages\onnxscript\version_converter\__init__.py", line 120, in call
    converted_proto = _c_api_utils.call_onnx_api(
                      ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\HaseebWajid\AppData\Local\anaconda3\envs\torch_env\Lib\site-packages\onnxscript\version_converter\_c_api_utils.py", line 65, in call_onnx_api
    result = func(proto)
             ^^^^^^^^^^^
  File "C:\Users\HaseebWajid\AppData\Local\anaconda3\envs\torch_env\Lib\site-packages\onnxscript\version_converter\__init__.py", line 115, in _partial_convert_version
    return onnx.version_converter.convert_version(
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\HaseebWajid\AppData\Local\anaconda3\envs\torch_env\Lib\site-packages\onnx\version_converter.py", line 39, in convert_vers

Applied 26 of general pattern rewrite rules.
ONNX saved.
Dynamic quantization failed: 'function' object has no attribute 'device'
CLI written: results_camelot_ids_v2_pc\export\camelot_infer_v2.py
Saved: results_camelot_ids_v2_pc\final_summary.json
All outputs in: ./results_camelot_ids_v2_pc


In [5]:
# ============================================================
# BOOTSTRAP CELL FOR PUBLICATION EXTENSION
# Run this BEFORE the publication extension cell.
# It reconstructs the saved run context from disk after a restart.
# ============================================================

from pathlib import Path
from types import SimpleNamespace
import json
import math
import gc
import warnings

import numpy as np
import pandas as pd
import joblib

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
)

warnings.filterwarnings("ignore")

# ------------------------------------------------------------
# 0) Basic runtime
# ------------------------------------------------------------
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE:", DEVICE)

# CHANGE THIS ONLY IF YOUR saved run folder is somewhere else
DEFAULT_OUT_DIR = Path("./results_camelot_ids_v2_pc")

# ------------------------------------------------------------
# 1) Locate saved run folder + config
# ------------------------------------------------------------
if "cfg" not in globals():
    config_path = DEFAULT_OUT_DIR / "config.json"
    if not config_path.exists():
        raise FileNotFoundError(
            f"Could not find config.json at: {config_path}\n"
            f"Edit DEFAULT_OUT_DIR in this bootstrap cell to your saved run folder."
        )

    with open(config_path, "r", encoding="utf-8") as f:
        cfg_dict = json.load(f)
    cfg = SimpleNamespace(**cfg_dict)
    print("Loaded cfg from:", config_path)

if "RUN_DIR" not in globals():
    RUN_DIR = Path(cfg.OUT_DIR)
RUN_DIR.mkdir(parents=True, exist_ok=True)

CACHE_DIR = RUN_DIR / "cache"
ARRAY_CACHE_DIR = CACHE_DIR / "arrays"
LOGITS_CACHE_DIR = CACHE_DIR / "logits"
STREAM_CACHE_DIR = CACHE_DIR / "stream"
CKPT_DIR = RUN_DIR / "checkpoints"
STAGE_DIR = RUN_DIR / "stages"
export_dir = RUN_DIR / "export"

for d in [CACHE_DIR, ARRAY_CACHE_DIR, LOGITS_CACHE_DIR, STREAM_CACHE_DIR, CKPT_DIR, STAGE_DIR, export_dir]:
    d.mkdir(parents=True, exist_ok=True)

print("RUN_DIR:", RUN_DIR)

# ------------------------------------------------------------
# 2) Utility helpers
# ------------------------------------------------------------
def load_json(path: Path, default=None):
    path = Path(path)
    if not path.exists():
        return {} if default is None else default
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def save_json_atomic(path: Path, obj):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = Path(str(path) + ".tmp")
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2)
    tmp.replace(path)

def atomic_save_npy(path: Path, arr: np.ndarray):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = Path(str(path) + ".tmp")
    with open(tmp, "wb") as f:
        np.save(f, arr, allow_pickle=False)
    tmp.replace(path)

def folder_ready(folder: Path, keys):
    folder = Path(folder)
    return (folder / "_ok.json").exists() and all((folder / f"{k}.npy").exists() for k in keys)

def load_array_dict(folder: Path, keys=None):
    folder = Path(folder)
    meta = load_json(folder / "_ok.json", {})
    if keys is None:
        keys = meta.get("keys", [])
    return {k: np.load(folder / f"{k}.npy") for k in keys}

def save_array_dict(folder: Path, arrays):
    folder = Path(folder)
    folder.mkdir(parents=True, exist_ok=True)
    for k, v in arrays.items():
        atomic_save_npy(folder / f"{k}.npy", v)
    save_json_atomic(folder / "_ok.json", {"keys": list(arrays.keys())})

def stable_json_dumps(obj):
    return json.dumps(obj, sort_keys=True, default=str)

def same_signature(a, b):
    return stable_json_dumps(a or {}) == stable_json_dumps(b or {})

def get_training_signature():
    sig = {}
    done_path = STAGE_DIR / "training_done.json"
    best_path = CKPT_DIR / "best_model.pt"
    if done_path.exists():
        done = load_json(done_path, {})
        sig["best_epoch"] = done.get("best_epoch")
        sig["best_score"] = done.get("best_score")
    if best_path.exists():
        sig["best_ckpt_mtime_ns"] = best_path.stat().st_mtime_ns
        sig["best_ckpt_size"] = best_path.stat().st_size
    return sig

def softmax_np(x: np.ndarray) -> np.ndarray:
    x = x - x.max(axis=1, keepdims=True)
    ex = np.exp(x)
    return ex / np.clip(ex.sum(axis=1, keepdims=True), 1e-12, None)

def metrics_mc(y_true: np.ndarray, probs: np.ndarray):
    y_pred = probs.argmax(axis=1)
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
        "macro_f1": float(f1_score(y_true, y_pred, average="macro")),
        "weighted_f1": float(f1_score(y_true, y_pred, average="weighted")),
    }

# ------------------------------------------------------------
# 3) Load metadata saved by main run
# ------------------------------------------------------------
meta_path = export_dir / "preprocess_and_meta.joblib"
if not meta_path.exists():
    raise FileNotFoundError(
        f"Missing saved metadata: {meta_path}\n"
        f"Your main notebook must finish preprocessing/model setup at least once."
    )

meta_obj = joblib.load(meta_path)

FEATURE_COLS = meta_obj["FEATURE_COLS"]
meta = meta_obj["meta"]
num_imp = meta_obj["num_imputer"]
scaler = meta_obj["scaler"]

FINE_CLASS_NAMES = meta_obj["fine_class_names"]
COARSE_NAMES = meta_obj["coarse_names"]
fine_to_coarse = np.array(meta_obj["fine_to_coarse"], dtype=np.int64)
benign_fine_id = int(meta_obj["benign_fine_id"])

NUM_FINE = len(FINE_CLASS_NAMES)
NUM_COARSE = len(COARSE_NAMES)

# ------------------------------------------------------------
# 4) Rebuild feature groups
# ------------------------------------------------------------
import re

def norm_colname(c: str) -> str:
    return re.sub(r"[^a-z0-9]+", "", str(c).strip().lower())

PROTOCOL_HINTS = set([norm_colname(x) for x in [
    "HTTP", "HTTPS", "DNS", "Telnet", "SMTP", "SSH", "IRC", "TCP", "UDP", "DHCP", "ARP", "ICMP", "IPv", "LLC"
]])
STAT_KEYS = ["tot", "sum", "min", "max", "avg", "std", "size", "iat", "number", "magnitude", "magnitue", "radius", "covariance", "variance", "weight"]
RATE_KEYS = ["rate", "srate", "drate"]

def build_feature_groups(feature_cols):
    groups = {"rates": [], "flags": [], "protocols": [], "stats": [], "other": []}
    for idx, name in enumerate(feature_cols):
        n = norm_colname(name)
        if any(k in n for k in RATE_KEYS):
            groups["rates"].append(idx)
        elif "flag" in n or n.endswith("count") or "count" in n:
            groups["flags"].append(idx)
        elif n in PROTOCOL_HINTS:
            groups["protocols"].append(idx)
        elif any(k in n for k in STAT_KEYS):
            groups["stats"].append(idx)
        else:
            groups["other"].append(idx)

    group_names = [k for k, v in groups.items() if len(v) > 0]
    group_idxs = [groups[k] for k in group_names]
    return group_names, group_idxs

GROUP_NAMES, GROUP_IDXS = build_feature_groups(meta["num_cols"])
num_features = len(meta["num_cols"])

print("num_features:", num_features)
print("GROUP_NAMES:", GROUP_NAMES)

# ------------------------------------------------------------
# 5) Load cached arrays + rebuild loaders
# ------------------------------------------------------------
ARRAY_KEYS = [
    "X_train", "y_train_f", "y_train_c", "y_train_b",
    "X_val", "y_val_f", "y_val_c", "y_val_b",
    "X_cal", "y_cal_f", "y_cal_c", "y_cal_b",
    "X_test", "y_test_f", "y_test_c", "y_test_b",
]

if not folder_ready(ARRAY_CACHE_DIR, ARRAY_KEYS):
    raise FileNotFoundError(
        f"Cached preprocessed arrays not found in: {ARRAY_CACHE_DIR}\n"
        f"Your main notebook must have saved preprocessing cache."
    )

arrs = load_array_dict(ARRAY_CACHE_DIR, ARRAY_KEYS)

X_train = arrs["X_train"]
y_train_f = arrs["y_train_f"]
y_train_c = arrs["y_train_c"]
y_train_b = arrs["y_train_b"]

X_val = arrs["X_val"]
y_val_f = arrs["y_val_f"]
y_val_c = arrs["y_val_c"]
y_val_b = arrs["y_val_b"]

X_cal = arrs["X_cal"]
y_cal_f = arrs["y_cal_f"]
y_cal_c = arrs["y_cal_c"]
y_cal_b = arrs["y_cal_b"]

X_test = arrs["X_test"]
y_test_f = arrs["y_test_f"]
y_test_c = arrs["y_test_c"]
y_test_b = arrs["y_test_b"]

class FlowDataset(Dataset):
    def __init__(self, X, y_f, y_c, y_b):
        self.X = torch.from_numpy(X.astype(np.float32, copy=False))
        self.yf = torch.from_numpy(y_f.astype(np.int64, copy=False))
        self.yc = torch.from_numpy(y_c.astype(np.int64, copy=False))
        self.yb = torch.from_numpy(y_b.astype(np.int64, copy=False))

    def __len__(self):
        return int(self.X.shape[0])

    def __getitem__(self, idx):
        return self.X[idx], self.yf[idx], self.yc[idx], self.yb[idx]

pin = (DEVICE == "cuda")
loader_kwargs = dict(
    batch_size=getattr(cfg, "BATCH_SIZE", 1024),
    num_workers=0,
    pin_memory=pin,
    persistent_workers=False,
)

ds_val = FlowDataset(X_val, y_val_f, y_val_c, y_val_b)
ds_cal = FlowDataset(X_cal, y_cal_f, y_cal_c, y_cal_b)
ds_test = FlowDataset(X_test, y_test_f, y_test_c, y_test_b)

val_loader = DataLoader(ds_val, shuffle=False, **loader_kwargs)
cal_loader = DataLoader(ds_cal, shuffle=False, **loader_kwargs)
test_loader = DataLoader(ds_test, shuffle=False, **loader_kwargs)

print("Cached arrays loaded:")
print("  X_val :", X_val.shape)
print("  X_cal :", X_cal.shape)
print("  X_test:", X_test.shape)

# ------------------------------------------------------------
# 6) Recreate model definition
# ------------------------------------------------------------
class FeatureTokenizer(nn.Module):
    def __init__(self, n_features: int, d_model: int):
        super().__init__()
        self.W = nn.Parameter(torch.empty(n_features, d_model))
        self.b = nn.Parameter(torch.empty(n_features, d_model))
        nn.init.trunc_normal_(self.W, std=0.02)
        nn.init.trunc_normal_(self.b, std=0.02)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x.unsqueeze(-1) * self.W.unsqueeze(0) + self.b.unsqueeze(0)

def make_encoder_layer(d_model, nhead, ff_dim, dropout, attn_dropout):
    return nn.TransformerEncoderLayer(
        d_model=d_model,
        nhead=nhead,
        dim_feedforward=ff_dim,
        dropout=dropout,
        activation="gelu",
        batch_first=True,
        norm_first=True
    )

class CamelotIDSv2(nn.Module):
    def __init__(self, n_features: int, n_fine: int, n_coarse: int, group_idxs, cfg):
        super().__init__()
        self.cfg = cfg
        self.n_features = n_features
        self.n_fine = n_fine
        self.n_coarse = n_coarse
        self.group_idxs = group_idxs
        self.num_groups = len(group_idxs)

        self.tokenizer = FeatureTokenizer(n_features, cfg.D_MODEL)

        self.group_token = nn.Parameter(torch.zeros(self.num_groups, cfg.D_MODEL))
        nn.init.trunc_normal_(self.group_token, std=0.02)
        self.group_type = nn.Embedding(self.num_groups, cfg.D_MODEL)

        for gi, idxs in enumerate(group_idxs):
            self.register_buffer(f"group_idx_{gi}", torch.tensor(idxs, dtype=torch.long))

        local_layer = make_encoder_layer(cfg.D_MODEL, cfg.NHEAD, cfg.FF_DIM, cfg.DROPOUT, cfg.ATTN_DROPOUT)
        self.local_encoder = nn.TransformerEncoder(local_layer, num_layers=cfg.LOCAL_LAYERS)

        self.cls = nn.Parameter(torch.zeros(1, 1, cfg.D_MODEL))
        nn.init.trunc_normal_(self.cls, std=0.02)

        global_layer = make_encoder_layer(cfg.D_MODEL, cfg.NHEAD, cfg.FF_DIM, cfg.DROPOUT, cfg.ATTN_DROPOUT)
        self.global_encoder = nn.TransformerEncoder(global_layer, num_layers=cfg.GLOBAL_LAYERS)

        self.pos_global = nn.Parameter(torch.zeros(1, 1 + self.num_groups, cfg.D_MODEL))
        nn.init.trunc_normal_(self.pos_global, std=0.02)

        self.head_fine = nn.Sequential(nn.LayerNorm(cfg.D_MODEL), nn.Dropout(cfg.DROPOUT), nn.Linear(cfg.D_MODEL, n_fine))
        self.head_coarse = nn.Sequential(nn.LayerNorm(cfg.D_MODEL), nn.Dropout(cfg.DROPOUT), nn.Linear(cfg.D_MODEL, n_coarse))
        self.head_bin = nn.Sequential(nn.LayerNorm(cfg.D_MODEL), nn.Dropout(cfg.DROPOUT), nn.Linear(cfg.D_MODEL, 2))

    def forward(self, x: torch.Tensor):
        B = x.size(0)
        tok = self.tokenizer(x)

        group_reps = []
        for g in range(self.num_groups):
            idxs_t = getattr(self, f"group_idx_{g}")
            t = tok.index_select(dim=1, index=idxs_t)
            t = t + self.group_type.weight[g].view(1, 1, -1)

            gt = self.group_token[g].view(1, 1, -1).expand(B, 1, -1)
            z = torch.cat([gt, t], dim=1)
            z = self.local_encoder(z)
            group_reps.append(z[:, 0])

        G = torch.stack(group_reps, dim=1)
        cls = self.cls.expand(B, -1, -1)
        Z = torch.cat([cls, G], dim=1)
        Z = Z + self.pos_global
        Z = self.global_encoder(Z)
        h = Z[:, 0]

        return {
            "logits_fine": self.head_fine(h),
            "logits_coarse": self.head_coarse(h),
            "logits_bin": self.head_bin(h),
        }

# ------------------------------------------------------------
# 7) Prediction + calibration helpers
# ------------------------------------------------------------
@torch.inference_mode()
def predict_logits(model: nn.Module, loader: DataLoader):
    model.eval()
    out = {"fine": [], "coarse": [], "bin": [], "yf": [], "yc": [], "yb": []}
    use_amp = (getattr(cfg, "USE_AMP", True) and DEVICE == "cuda")

    for xb, yf, yc, yb in loader:
        xb = xb.to(DEVICE, non_blocking=True)
        with torch.amp.autocast(device_type="cuda" if DEVICE == "cuda" else "cpu", enabled=use_amp):
            r = model(xb)
        out["fine"].append(r["logits_fine"].detach().cpu().numpy())
        out["coarse"].append(r["logits_coarse"].detach().cpu().numpy())
        out["bin"].append(r["logits_bin"].detach().cpu().numpy())
        out["yf"].append(yf.numpy())
        out["yc"].append(yc.numpy())
        out["yb"].append(yb.numpy())

    for k in ["fine", "coarse", "bin", "yf", "yc", "yb"]:
        out[k] = np.concatenate(out[k], axis=0)
    return out

LOGIT_KEYS = ["fine", "coarse", "bin", "yf", "yc", "yb"]

def predict_logits_cached(name: str, model: nn.Module, loader: DataLoader, signature=None):
    folder = LOGITS_CACHE_DIR / name
    meta_path = folder / "_meta.json"

    if folder_ready(folder, LOGIT_KEYS):
        cache_meta = load_json(meta_path, {})
        if same_signature(cache_meta.get("signature", {}), signature or {}):
            print(f"Loading cached logits: {name}")
            return load_array_dict(folder, LOGIT_KEYS)

    out = predict_logits(model, loader)
    save_array_dict(folder, out)
    save_json_atomic(meta_path, {"name": name, "signature": signature or {}})
    return out

class TemperatureScaler(nn.Module):
    def __init__(self):
        super().__init__()
        self.log_t = nn.Parameter(torch.zeros(()))

    def forward(self, logits: torch.Tensor) -> torch.Tensor:
        t = torch.exp(self.log_t).clamp(1e-3, 100.0)
        return logits / t

def fit_temperature_np(logits_np: np.ndarray, y_np: np.ndarray) -> float:
    logits = torch.from_numpy(logits_np).to(DEVICE)
    y = torch.from_numpy(y_np).to(DEVICE)
    ts = TemperatureScaler().to(DEVICE)
    opt = torch.optim.LBFGS(ts.parameters(), lr=0.5, max_iter=50)
    nll = nn.CrossEntropyLoss()

    def closure():
        opt.zero_grad(set_to_none=True)
        loss = nll(ts(logits), y)
        loss.backward()
        return loss

    opt.step(closure)
    return float(torch.exp(ts.log_t).detach().cpu().item())

# ------------------------------------------------------------
# 8) Optional throughput fallback
# ------------------------------------------------------------
if "throughput_gpu" not in globals():
    throughput_gpu = None
if "throughput_cpu" not in globals():
    throughput_cpu = None

print("\nBootstrap complete.")
print("You can now run the publication extension cell.")

DEVICE: cuda
Loaded cfg from: results_camelot_ids_v2_pc\config.json
RUN_DIR: results_camelot_ids_v2_pc
num_features: 46
GROUP_NAMES: ['rates', 'flags', 'protocols', 'stats', 'other']
Cached arrays loaded:
  X_val : (2631074, 46)
  X_cal : (1389497, 46)
  X_test: (1372896, 46)

Bootstrap complete.
You can now run the publication extension cell.


In [3]:
# ============================================================
# PUBLICATION EXTENSION CELL (NO RETRAINING, CACHE-FIRST)
# ============================================================
# What this cell does:
# 1) Picks the best already-available model candidate without retraining:
#       - best_model.pt (raw best checkpoint)
#       - last_checkpoint.pt (raw final checkpoint)
#       - last_checkpoint.pt EMA shadow (usually strongest)
# 2) Recomputes clean validation/test/calibration outputs for the chosen candidate
# 3) Refits temperature scaling (fast, no retraining)
# 4) Regenerates publication-ready figures/tables/latex/csv/xlsx/json/md
# 5) Exports stable TorchScript/ONNX + validates ONNX if possible
#
# NOTE:
# - This does NOT run for days.
# - It does NOT retrain.
# - It fixes the broken export stage and avoids the old failing int8 path.
# - It gives you a publication package from the current run.
# ============================================================

from pathlib import Path
import json
import math
import gc
import warnings
from copy import deepcopy
from sklearn.metrics import precision_recall_fscore_support
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    confusion_matrix,
    log_loss,
    matthews_corrcoef,
    cohen_kappa_score,
    roc_auc_score,
    average_precision_score,
    roc_curve,
    precision_recall_curve,
)

warnings.filterwarnings("ignore")

# ---------------------------
# Directories
# ---------------------------
PUB_DIR = RUN_DIR / "publication_package"
PUB_FIG_DIR = PUB_DIR / "figures"
PUB_TAB_DIR = PUB_DIR / "tables"
PUB_JSON_DIR = PUB_DIR / "json"
PUB_EXPORT_DIR = PUB_DIR / "export"

for d in [PUB_DIR, PUB_FIG_DIR, PUB_TAB_DIR, PUB_JSON_DIR, PUB_EXPORT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("Publication package dir:", PUB_DIR)

# ---------------------------
# Safety / config
# ---------------------------
PUB_BOOT_N = 120          # fast but useful CIs
PUB_BOOT_MAX_POOL = 150_000
PUB_RISK_POINTS = 31
PUB_RAPS_ALPHA_DEFAULT = 0.10
PUB_ONNX_OPSET = 18
PUB_VALIDATE_ONNX = True

# ---------------------------
# Helpers
# ---------------------------
def topk_acc_from_probs(y_true: np.ndarray, probs: np.ndarray, k: int) -> float:
    k = int(min(k, probs.shape[1]))
    topk = np.argpartition(probs, -k, axis=1)[:, -k:]
    return float((topk == y_true[:, None]).any(axis=1).mean())

def multiclass_extended_metrics(y_true: np.ndarray, probs: np.ndarray) -> dict:
    y_pred = probs.argmax(axis=1)
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
        "macro_f1": float(f1_score(y_true, y_pred, average="macro")),
        "weighted_f1": float(f1_score(y_true, y_pred, average="weighted")),
        "macro_precision": float(precision_score(y_true, y_pred, average="macro", zero_division=0)),
        "macro_recall": float(recall_score(y_true, y_pred, average="macro", zero_division=0)),
        "mcc": float(matthews_corrcoef(y_true, y_pred)),
        "kappa": float(cohen_kappa_score(y_true, y_pred)),
        "top3_accuracy": float(topk_acc_from_probs(y_true, probs, 3)),
        "top5_accuracy": float(topk_acc_from_probs(y_true, probs, 5)),
    }

def binary_extended_metrics(y_true: np.ndarray, probs_2: np.ndarray) -> dict:
    y_pred = probs_2.argmax(axis=1)
    p1 = probs_2[:, 1]
    out = {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
        "f1_malicious": float(f1_score(y_true, y_pred, average="binary", pos_label=1)),
        "precision_malicious": float(precision_score(y_true, y_pred, average="binary", pos_label=1, zero_division=0)),
        "recall_malicious": float(recall_score(y_true, y_pred, average="binary", pos_label=1, zero_division=0)),
        "mcc": float(matthews_corrcoef(y_true, y_pred)),
        "kappa": float(cohen_kappa_score(y_true, y_pred)),
    }
    try:
        out["auroc"] = float(roc_auc_score(y_true, p1))
    except Exception:
        out["auroc"] = float("nan")
    try:
        out["auprc"] = float(average_precision_score(y_true, p1))
    except Exception:
        out["auprc"] = float("nan")
    return out

def calibration_stats(y_true: np.ndarray, probs: np.ndarray, n_bins: int = 15) -> dict:
    y_pred = probs.argmax(axis=1)
    conf = probs[np.arange(len(probs)), y_pred]
    acc = (y_pred == y_true).astype(np.float32)
    bins = np.linspace(0.0, 1.0, n_bins + 1)

    ece = 0.0
    mce = 0.0
    rows = []
    for i in range(n_bins):
        lo, hi = bins[i], bins[i + 1]
        if i < n_bins - 1:
            m = (conf >= lo) & (conf < hi)
        else:
            m = (conf >= lo) & (conf <= hi)

        if m.any():
            bin_acc = float(acc[m].mean())
            bin_conf = float(conf[m].mean())
            gap = abs(bin_acc - bin_conf)
            frac = float(m.mean())
            ece += frac * gap
            mce = max(mce, gap)
            rows.append({
                "bin_lo": float(lo),
                "bin_hi": float(hi),
                "count": int(m.sum()),
                "frac": frac,
                "acc": bin_acc,
                "conf": bin_conf,
                "gap": float(gap),
            })
        else:
            rows.append({
                "bin_lo": float(lo),
                "bin_hi": float(hi),
                "count": 0,
                "frac": 0.0,
                "acc": float("nan"),
                "conf": float("nan"),
                "gap": float("nan"),
            })

    return {
        "nll": float(log_loss(y_true, probs, labels=list(range(probs.shape[1])))),
        "ece": float(ece),
        "mce": float(mce),
        "brier": float(np.mean(np.sum((probs - np.eye(probs.shape[1])[y_true]) ** 2, axis=1))),
        "bins": rows,
    }

def plot_reliability(ax, y_true: np.ndarray, probs: np.ndarray, title: str, n_bins: int = 15):
    stats = calibration_stats(y_true, probs, n_bins=n_bins)
    bins = pd.DataFrame(stats["bins"])
    mids = (bins["bin_lo"].values + bins["bin_hi"].values) / 2.0
    ax.plot([0, 1], [0, 1], linestyle="--", linewidth=1)
    ax.plot(mids, bins["acc"].values, marker="o", label="accuracy")
    ax.plot(mids, bins["conf"].values, marker="s", label="confidence")
    ax.set_title(title)
    ax.set_xlabel("confidence bin")
    ax.set_ylabel("value")
    ax.grid(True, alpha=0.3)
    ax.legend()
    return stats

def stratified_pool_indices(y: np.ndarray, max_n: int, seed: int = 42) -> np.ndarray:
    if len(y) <= max_n:
        return np.arange(len(y))
    rng = np.random.default_rng(seed)
    idxs = []
    classes, counts = np.unique(y, return_counts=True)
    frac = max_n / len(y)
    for c, cnt in zip(classes, counts):
        class_idx = np.where(y == c)[0]
        take = max(1, int(round(cnt * frac)))
        take = min(take, len(class_idx))
        pick = rng.choice(class_idx, size=take, replace=False)
        idxs.append(pick)
    idxs = np.concatenate(idxs)
    if len(idxs) > max_n:
        idxs = rng.choice(idxs, size=max_n, replace=False)
    return np.sort(idxs)

def bootstrap_ci_classification(yf_true, yf_pred, yc_true, yc_pred, yb_true, yb_pred,
                                n_boot=120, max_pool=150_000, seed=42):
    pool = stratified_pool_indices(yf_true, max_pool, seed)
    yf_true_p = yf_true[pool]
    yf_pred_p = yf_pred[pool]
    yc_true_p = yc_true[pool]
    yc_pred_p = yc_pred[pool]
    yb_true_p = yb_true[pool]
    yb_pred_p = yb_pred[pool]

    n = len(pool)
    rng = np.random.default_rng(seed + 123)

    rows = []
    for _ in range(n_boot):
        b = rng.integers(0, n, size=n)
        rows.append({
            "fine_accuracy": float(accuracy_score(yf_true_p[b], yf_pred_p[b])),
            "fine_macro_f1": float(f1_score(yf_true_p[b], yf_pred_p[b], average="macro")),
            "coarse_accuracy": float(accuracy_score(yc_true_p[b], yc_pred_p[b])),
            "coarse_macro_f1": float(f1_score(yc_true_p[b], yc_pred_p[b], average="macro")),
            "bin_f1_malicious": float(f1_score(yb_true_p[b], yb_pred_p[b], average="binary", pos_label=1)),
        })

    dfb = pd.DataFrame(rows)

    out = {}
    for col in dfb.columns:
        vals = dfb[col].values
        out[col] = {
            "mean": float(np.mean(vals)),
            "ci95_lo": float(np.quantile(vals, 0.025)),
            "ci95_hi": float(np.quantile(vals, 0.975)),
        }
    return out, dfb

def save_df_many_formats(df: pd.DataFrame, stem: Path, index: bool = False):
    stem.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(str(stem.with_suffix(".csv")), index=index)
    try:
        df.to_latex(str(stem.with_suffix(".tex")), index=index, float_format="%.4f")
    except Exception:
        pass

def plot_cm(cm: np.ndarray, labels: list, title: str, out_path: Path, normalize: bool = False, figsize=(10, 8)):
    arr = cm.astype(np.float64)
    if normalize:
        denom = arr.sum(axis=1, keepdims=True)
        arr = np.divide(arr, np.clip(denom, 1e-12, None))
    plt.figure(figsize=figsize)
    plt.imshow(arr, aspect="auto")
    plt.title(title)
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.colorbar()
    plt.xticks(np.arange(len(labels)), labels, rotation=90, fontsize=7)
    plt.yticks(np.arange(len(labels)), labels, fontsize=7)
    plt.tight_layout()
    plt.savefig(str(out_path), dpi=220, bbox_inches="tight")
    plt.close()

def build_fresh_model():
    m = CamelotIDSv2(
        n_features=num_features,
        n_fine=NUM_FINE,
        n_coarse=NUM_COARSE,
        group_idxs=GROUP_IDXS,
        cfg=cfg
    ).to(DEVICE)
    m.eval()
    return m

def load_state_into_fresh(state_dict: dict):
    m = build_fresh_model()
    m.load_state_dict(state_dict, strict=True)
    m.eval()
    return m

def apply_ema_shadow_(m: nn.Module, ema_state: dict):
    if ema_state is None:
        return m
    shadow = ema_state.get("shadow", {})
    sd = m.state_dict()
    for k, v in shadow.items():
        if k in sd:
            sd[k].copy_(v.detach().to(sd[k].device, dtype=sd[k].dtype))
    m.load_state_dict(sd, strict=False)
    m.eval()
    return m

@torch.inference_mode()
def eval_candidate_on_val(m: nn.Module, name: str):
    out = predict_logits(m, val_loader)
    probs_f = softmax_np(out["fine"])
    probs_c = softmax_np(out["coarse"])
    probs_b = softmax_np(out["bin"])
    fine_m = metrics_mc(out["yf"], probs_f)
    coarse_m = metrics_mc(out["yc"], probs_c)
    bin_f1 = float(f1_score(out["yb"], probs_b.argmax(axis=1), average="binary", pos_label=1))
    score = float(fine_m["macro_f1"])
    return {
        "name": name,
        "score_val_fine_macro_f1": score,
        "val_fine": fine_m,
        "val_coarse": coarse_m,
        "val_bin_f1_malicious": bin_f1,
    }

def write_markdown(path: Path, text: str):
    path.write_text(text, encoding="utf-8")

# ---------------------------
# 1) Candidate selection from existing checkpoints
# ---------------------------
candidate_rows = []
candidate_models = {}

best_ckpt_path = CKPT_DIR / "best_model.pt"
last_ckpt_path = CKPT_DIR / "last_checkpoint.pt"

if not best_ckpt_path.exists():
    raise FileNotFoundError(f"Missing: {best_ckpt_path}")
if not last_ckpt_path.exists():
    raise FileNotFoundError(f"Missing: {last_ckpt_path}")

best_blob = torch.load(best_ckpt_path, map_location=DEVICE, weights_only=False)
last_blob = torch.load(last_ckpt_path, map_location=DEVICE, weights_only=False)

# Candidate A: raw best checkpoint
cand_best_raw = load_state_into_fresh(best_blob["model_state"])
res_best_raw = eval_candidate_on_val(cand_best_raw, "best_raw_epoch_checkpoint")
candidate_rows.append(res_best_raw)
candidate_models["best_raw_epoch_checkpoint"] = cand_best_raw
print("Candidate:", res_best_raw["name"], "| val fine macro-F1 =", round(res_best_raw["score_val_fine_macro_f1"], 6))

# Candidate B: raw last checkpoint
cand_last_raw = load_state_into_fresh(last_blob["model_state"])
res_last_raw = eval_candidate_on_val(cand_last_raw, "last_raw_checkpoint")
candidate_rows.append(res_last_raw)
candidate_models["last_raw_checkpoint"] = cand_last_raw
print("Candidate:", res_last_raw["name"], "| val fine macro-F1 =", round(res_last_raw["score_val_fine_macro_f1"], 6))

# Candidate C: EMA shadow from last checkpoint applied to the last model
cand_last_ema = load_state_into_fresh(last_blob["model_state"])
cand_last_ema = apply_ema_shadow_(cand_last_ema, last_blob.get("ema_state", None))
res_last_ema = eval_candidate_on_val(cand_last_ema, "last_ema_shadow_checkpoint")
candidate_rows.append(res_last_ema)
candidate_models["last_ema_shadow_checkpoint"] = cand_last_ema
print("Candidate:", res_last_ema["name"], "| val fine macro-F1 =", round(res_last_ema["score_val_fine_macro_f1"], 6))

candidate_df = pd.DataFrame([
    {
        "candidate": r["name"],
        "val_fine_macro_f1": r["score_val_fine_macro_f1"],
        "val_fine_accuracy": r["val_fine"]["accuracy"],
        "val_coarse_macro_f1": r["val_coarse"]["macro_f1"],
        "val_bin_f1_malicious": r["val_bin_f1_malicious"],
    }
    for r in candidate_rows
]).sort_values("val_fine_macro_f1", ascending=False).reset_index(drop=True)

save_df_many_formats(candidate_df, PUB_TAB_DIR / "candidate_model_selection", index=False)
best_name = candidate_df.iloc[0]["candidate"]
pub_model = candidate_models[best_name]
pub_model.eval()

print("\nChosen publication model:", best_name)

# Save chosen model state for publication package
torch.save(
    {
        "candidate_name": best_name,
        "state_dict": {k: v.detach().cpu().clone() for k, v in pub_model.state_dict().items()},
        "num_features": int(num_features),
        "num_fine": int(NUM_FINE),
        "num_coarse": int(NUM_COARSE),
    },
    PUB_EXPORT_DIR / "publication_best_model.pt"
)

# ---------------------------
# 2) Fresh logits for chosen candidate
# ---------------------------
pub_signature = {
    "publication_candidate": best_name,
    "source_training_signature": get_training_signature(),
}

pub_test_out = predict_logits_cached("publication_test_" + best_name, pub_model, test_loader, signature=pub_signature)
pub_cal_out  = predict_logits_cached("publication_cal_" + best_name,  pub_model, cal_loader,  signature=pub_signature)

# ---------------------------
# 3) Fresh calibration (fast, no retraining)
# ---------------------------
pub_temp_f = fit_temperature_np(pub_cal_out["fine"], pub_cal_out["yf"]) if cfg.TEMPERATURE_SCALE else 1.0
pub_temp_c = fit_temperature_np(pub_cal_out["coarse"], pub_cal_out["yc"]) if cfg.TEMPERATURE_SCALE else 1.0
pub_temp_b = fit_temperature_np(pub_cal_out["bin"], pub_cal_out["yb"]) if cfg.TEMPERATURE_SCALE else 1.0

print("Refit temperatures:", {"fine": pub_temp_f, "coarse": pub_temp_c, "bin": pub_temp_b})

# Raw probs
pub_test_probs_f_raw = softmax_np(pub_test_out["fine"])
pub_test_probs_c_raw = softmax_np(pub_test_out["coarse"])
pub_test_probs_b_raw = softmax_np(pub_test_out["bin"])

# Calibrated probs
pub_test_probs_f = softmax_np(pub_test_out["fine"] / max(pub_temp_f, 1e-6))
pub_test_probs_c = softmax_np(pub_test_out["coarse"] / max(pub_temp_c, 1e-6))
pub_test_probs_b = softmax_np(pub_test_out["bin"] / max(pub_temp_b, 1e-6))

pub_yf = pub_test_out["yf"]
pub_yc = pub_test_out["yc"]
pub_yb = pub_test_out["yb"]

pub_pred_f = pub_test_probs_f.argmax(axis=1)
pub_pred_c = pub_test_probs_c.argmax(axis=1)
pub_pred_b = pub_test_probs_b.argmax(axis=1)

# ---------------------------
# 4) Core publication metrics
# ---------------------------
metrics_fine = multiclass_extended_metrics(pub_yf, pub_test_probs_f)
metrics_coarse = multiclass_extended_metrics(pub_yc, pub_test_probs_c)
metrics_bin = binary_extended_metrics(pub_yb, pub_test_probs_b)

cal_f_raw = calibration_stats(pub_yf, pub_test_probs_f_raw, n_bins=cfg.N_BINS_ECE)
cal_f_cal = calibration_stats(pub_yf, pub_test_probs_f, n_bins=cfg.N_BINS_ECE)

cal_c_raw = calibration_stats(pub_yc, pub_test_probs_c_raw, n_bins=cfg.N_BINS_ECE)
cal_c_cal = calibration_stats(pub_yc, pub_test_probs_c, n_bins=cfg.N_BINS_ECE)

cal_b_raw = calibration_stats(pub_yb, pub_test_probs_b_raw, n_bins=cfg.N_BINS_ECE)
cal_b_cal = calibration_stats(pub_yb, pub_test_probs_b, n_bins=cfg.N_BINS_ECE)

ci_summary, ci_boot_df = bootstrap_ci_classification(
    pub_yf, pub_pred_f,
    pub_yc, pub_pred_c,
    pub_yb, pub_pred_b,
    n_boot=PUB_BOOT_N,
    max_pool=PUB_BOOT_MAX_POOL,
    seed=cfg.SEED
)

with pd.ExcelWriter(PUB_TAB_DIR / "publication_tables.xlsx") as writer:
    candidate_df.to_excel(writer, sheet_name="candidate_selection", index=False)
    pd.DataFrame([metrics_fine]).to_excel(writer, sheet_name="fine_metrics", index=False)
    pd.DataFrame([metrics_coarse]).to_excel(writer, sheet_name="coarse_metrics", index=False)
    pd.DataFrame([metrics_bin]).to_excel(writer, sheet_name="binary_metrics", index=False)
    pd.DataFrame(ci_summary).T.reset_index().rename(columns={"index": "metric"}).to_excel(writer, sheet_name="bootstrap_ci", index=False)

# ---------------------------
# 5) Per-class and per-coarse tables
# ---------------------------
fine_prec, fine_rec, fine_f1, fine_sup = precision_recall_fscore_support(
    pub_yf, pub_pred_f, labels=np.arange(NUM_FINE), zero_division=0
)
fine_table = pd.DataFrame({
    "fine_class": FINE_CLASS_NAMES,
    "support": fine_sup.astype(int),
    "precision": fine_prec,
    "recall": fine_rec,
    "f1": fine_f1,
    "coarse_class": [COARSE_NAMES[fine_to_coarse[i]] for i in range(NUM_FINE)],
}).sort_values(["f1", "support"], ascending=[True, False]).reset_index(drop=True)

coarse_prec, coarse_rec, coarse_f1, coarse_sup = precision_recall_fscore_support(
    pub_yc, pub_pred_c, labels=np.arange(NUM_COARSE), zero_division=0
)
coarse_table = pd.DataFrame({
    "coarse_class": COARSE_NAMES,
    "support": coarse_sup.astype(int),
    "precision": coarse_prec,
    "recall": coarse_rec,
    "f1": coarse_f1,
}).sort_values("f1", ascending=True).reset_index(drop=True)

save_df_many_formats(fine_table, PUB_TAB_DIR / "fine_per_class_metrics", index=False)
save_df_many_formats(coarse_table, PUB_TAB_DIR / "coarse_per_class_metrics", index=False)
save_df_many_formats(ci_boot_df, PUB_TAB_DIR / "bootstrap_draws_key_metrics", index=False)

# ---------------------------
# 6) Confusion matrices
# ---------------------------
cm_fine = confusion_matrix(pub_yf, pub_pred_f, labels=np.arange(NUM_FINE))
cm_coarse = confusion_matrix(pub_yc, pub_pred_c, labels=np.arange(NUM_COARSE))

plot_cm(cm_fine, FINE_CLASS_NAMES, "Fine 34-class Confusion Matrix (raw counts)", PUB_FIG_DIR / "cm_fine_counts.png", normalize=False, figsize=(14, 12))
plot_cm(cm_fine, FINE_CLASS_NAMES, "Fine 34-class Confusion Matrix (row-normalized)", PUB_FIG_DIR / "cm_fine_normalized.png", normalize=True, figsize=(14, 12))
plot_cm(cm_coarse, COARSE_NAMES, "Coarse 8-class Confusion Matrix (raw counts)", PUB_FIG_DIR / "cm_coarse_counts.png", normalize=False, figsize=(8, 6))
plot_cm(cm_coarse, COARSE_NAMES, "Coarse 8-class Confusion Matrix (row-normalized)", PUB_FIG_DIR / "cm_coarse_normalized.png", normalize=True, figsize=(8, 6))

# ---------------------------
# 7) Reliability diagrams
# ---------------------------
fig, axs = plt.subplots(2, 3, figsize=(16, 9))

plot_reliability(axs[0, 0], pub_yf, pub_test_probs_f_raw, "Fine reliability (before)")
plot_reliability(axs[1, 0], pub_yf, pub_test_probs_f,     "Fine reliability (after)")

plot_reliability(axs[0, 1], pub_yc, pub_test_probs_c_raw, "Coarse reliability (before)")
plot_reliability(axs[1, 1], pub_yc, pub_test_probs_c,     "Coarse reliability (after)")

plot_reliability(axs[0, 2], pub_yb, pub_test_probs_b_raw, "Binary reliability (before)")
plot_reliability(axs[1, 2], pub_yb, pub_test_probs_b,     "Binary reliability (after)")

plt.tight_layout()
plt.savefig(str(PUB_FIG_DIR / "reliability_diagrams_all.png"), dpi=220, bbox_inches="tight")
plt.close()

# ---------------------------
# 8) ROC / PR for binary malicious-vs-benign
# ---------------------------
try:
    fpr, tpr, _ = roc_curve(pub_yb, pub_test_probs_b[:, 1])
    prec_curve, rec_curve, _ = precision_recall_curve(pub_yb, pub_test_probs_b[:, 1])

    plt.figure(figsize=(6, 5))
    plt.plot(fpr, tpr, label=f"AUROC = {metrics_bin['auroc']:.4f}")
    plt.plot([0, 1], [0, 1], linestyle="--", linewidth=1)
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title("Binary ROC (Malicious vs Benign)")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.savefig(str(PUB_FIG_DIR / "binary_roc.png"), dpi=220, bbox_inches="tight")
    plt.close()

    plt.figure(figsize=(6, 5))
    plt.plot(rec_curve, prec_curve, label=f"AUPRC = {metrics_bin['auprc']:.4f}")
    plt.xlabel("Recall")
    plt.ylabel("Precision")
    plt.title("Binary Precision-Recall (Malicious vs Benign)")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.savefig(str(PUB_FIG_DIR / "binary_pr.png"), dpi=220, bbox_inches="tight")
    plt.close()
except Exception as e:
    print("ROC/PR plot skipped:", e)

# ---------------------------
# 9) Per-class figures
# ---------------------------
plt.figure(figsize=(14, 6))
tmp = fine_table.sort_values("f1", ascending=True)
plt.bar(np.arange(len(tmp)), tmp["f1"].values)
plt.xticks(np.arange(len(tmp)), tmp["fine_class"].values, rotation=90, fontsize=7)
plt.ylabel("F1")
plt.title("Per-class F1 (fine classes, sorted)")
plt.grid(True, axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(str(PUB_FIG_DIR / "per_class_f1_sorted.png"), dpi=220, bbox_inches="tight")
plt.close()

plt.figure(figsize=(8, 6))
plt.scatter(np.log10(np.maximum(fine_table["support"].values, 1)), fine_table["f1"].values)
for _, r in fine_table.nsmallest(10, "f1").iterrows():
    plt.annotate(r["fine_class"], (np.log10(max(r["support"], 1)), r["f1"]), fontsize=8)
plt.xlabel("log10(support)")
plt.ylabel("F1")
plt.title("Class support vs F1")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(str(PUB_FIG_DIR / "support_vs_f1.png"), dpi=220, bbox_inches="tight")
plt.close()

# ---------------------------
# 10) Training curves (from saved epoch history)
# ---------------------------
hist_path = RUN_DIR / "epoch_history.csv"
if hist_path.exists():
    hist_df = pd.read_csv(hist_path)

    plt.figure(figsize=(10, 5))
    plt.plot(hist_df["epoch"], hist_df["train_loss"], label="train_loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title("Training loss")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.savefig(str(PUB_FIG_DIR / "training_loss.png"), dpi=220, bbox_inches="tight")
    plt.close()

    plt.figure(figsize=(10, 5))
    plt.plot(hist_df["epoch"], hist_df["val_fine_macro_f1"], label="val fine macro-F1")
    plt.plot(hist_df["epoch"], hist_df["val_coarse_macro_f1"], label="val coarse macro-F1")
    plt.plot(hist_df["epoch"], hist_df["val_bin_f1"], label="val binary F1")
    plt.xlabel("Epoch")
    plt.ylabel("Score")
    plt.title("Validation curves")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.savefig(str(PUB_FIG_DIR / "validation_curves.png"), dpi=220, bbox_inches="tight")
    plt.close()

# ---------------------------
# 11) Selective prediction / risk-coverage
# ---------------------------
conf_f = pub_test_probs_f.max(axis=1)
thresholds = np.unique(np.quantile(conf_f, np.linspace(0.0, 0.995, PUB_RISK_POINTS)))

sel_rows = []
for th in thresholds:
    m = conf_f >= th
    if m.sum() == 0:
        continue
    acc = float((pub_pred_f[m] == pub_yf[m]).mean())
    mf1 = float(f1_score(pub_yf[m], pub_pred_f[m], average="macro"))
    sel_rows.append({
        "threshold": float(th),
        "coverage": float(m.mean()),
        "accuracy": float(acc),
        "risk_1_minus_accuracy": float(1.0 - acc),
        "macro_f1": float(mf1),
    })

sel_df = pd.DataFrame(sel_rows).sort_values("coverage")
save_df_many_formats(sel_df, PUB_TAB_DIR / "confidence_selective_prediction_curve", index=False)

plt.figure(figsize=(7, 5))
plt.plot(sel_df["coverage"], sel_df["risk_1_minus_accuracy"], marker="o")
plt.xlabel("Coverage")
plt.ylabel("Risk = 1 - accuracy")
plt.title("Selective prediction risk-coverage (fine top-1 confidence)")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(str(PUB_FIG_DIR / "risk_coverage_confidence.png"), dpi=220, bbox_inches="tight")
plt.close()

# ---------------------------
# 12) Fresh RAPS using chosen model + calibrated probs
# ---------------------------
def conformal_quantile(scores: np.ndarray, alpha: float) -> float:
    n = len(scores)
    if n == 0:
        return 1.0
    q_level = math.ceil((n + 1) * (1 - alpha)) / n
    return float(np.quantile(scores, min(q_level, 1.0), method="higher"))

def raps_scores_true(probs: np.ndarray, y_true: np.ndarray, k_reg: int, lam: float) -> np.ndarray:
    n, C = probs.shape
    idx_sorted = np.argsort(-probs, axis=1)
    probs_sorted = np.take_along_axis(probs, idx_sorted, axis=1)
    cumsum = np.cumsum(probs_sorted, axis=1)

    inv = np.empty_like(idx_sorted)
    rows = np.arange(n)[:, None]
    inv[rows, idx_sorted] = np.arange(C)[None, :]
    rank0 = inv[rows[:, 0], y_true]
    rank1 = rank0 + 1

    score = cumsum[rows[:, 0], rank0] + lam * np.maximum(rank1 - k_reg, 0)
    return score.astype(np.float32)

def raps_predict_k(probs: np.ndarray, q: np.ndarray, k_reg: int, lam: float) -> np.ndarray:
    n, C = probs.shape
    idx_sorted = np.argsort(-probs, axis=1)
    probs_sorted = np.take_along_axis(probs, idx_sorted, axis=1)
    cumsum = np.cumsum(probs_sorted, axis=1)
    ranks = np.arange(1, C + 1, dtype=np.float32)[None, :]
    reg = lam * np.maximum(ranks - float(k_reg), 0.0)
    score_k = cumsum + reg
    ok = score_k <= q[:, None]
    k_star = ok.sum(axis=1).astype(np.int64)
    return np.maximum(k_star, 1)

def raps_eval(probs: np.ndarray, y_true: np.ndarray, q_row: np.ndarray, k_reg: int, lam: float) -> dict:
    n, C = probs.shape
    idx_sorted = np.argsort(-probs, axis=1)

    inv = np.empty_like(idx_sorted)
    rows = np.arange(n)[:, None]
    inv[rows, idx_sorted] = np.arange(C)[None, :]
    rank0 = inv[rows[:, 0], y_true]

    k_star = raps_predict_k(probs, q_row, k_reg, lam)
    covered = (rank0 < k_star).astype(np.float32)
    return {"coverage": float(covered.mean()), "avg_set_size": float(k_star.mean())}

pub_cal_probs_f = softmax_np(pub_cal_out["fine"] / max(pub_temp_f, 1e-6))
pub_cal_probs_c = softmax_np(pub_cal_out["coarse"] / max(pub_temp_c, 1e-6))

raps_calib_pub = {
    "mode": cfg.RAPS_MONDRIAN,
    "k_reg": int(cfg.RAPS_KREG),
    "lambda": float(cfg.RAPS_LAMBDA),
    "alphas": [float(a) for a in cfg.ALPHAS],
    "q_global": {},
    "q_by_pred_coarse": {},
    "min_group_n": 200
}

scores_global = raps_scores_true(pub_cal_probs_f, pub_cal_out["yf"], cfg.RAPS_KREG, cfg.RAPS_LAMBDA)
g_hat_cal = pub_cal_probs_c.argmax(axis=1)
scores_by_g = {g: scores_global[g_hat_cal == g] for g in range(NUM_COARSE)}

for a in cfg.ALPHAS:
    qg = conformal_quantile(scores_global, a)
    raps_calib_pub["q_global"][str(a)] = float(qg)
    q_by = []
    for g in range(NUM_COARSE):
        sg = scores_by_g.get(g, np.array([], dtype=np.float32))
        if len(sg) >= raps_calib_pub["min_group_n"]:
            q_by.append(conformal_quantile(sg, a))
        else:
            q_by.append(qg)
    raps_calib_pub["q_by_pred_coarse"][str(a)] = [float(x) for x in q_by]

raps_results_pub = {}
for a in cfg.ALPHAS:
    q_by = np.array(raps_calib_pub["q_by_pred_coarse"][str(a)], dtype=np.float32)
    q_row = q_by[pub_test_probs_c.argmax(axis=1)]
    raps_results_pub[str(a)] = raps_eval(pub_test_probs_f, pub_yf, q_row, cfg.RAPS_KREG, cfg.RAPS_LAMBDA)

# singleton risk / coverage table
def singleton_risk_coverage(y_true, probs_f, probs_c, raps_calib, alpha, k_reg, lam, benign_id=0):
    q_by = np.array(raps_calib["q_by_pred_coarse"][str(alpha)], dtype=np.float32)
    g_hat = probs_c.argmax(axis=1)
    q_row = q_by[g_hat]
    k_star = raps_predict_k(probs_f, q_row, k_reg, lam)

    single = (k_star == 1)
    y_pred = probs_f.argmax(axis=1)

    cov = float(single.mean())
    if single.any():
        acc = float((y_pred[single] == y_true[single]).mean())
        mf1 = float(f1_score(y_true[single], y_pred[single], average="macro"))
    else:
        acc, mf1 = float("nan"), float("nan")

    benign_mask = (y_true == benign_id) & single
    benign_fpr = float((y_pred[benign_mask] != benign_id).mean()) if benign_mask.any() else float("nan")

    return {
        "alpha": float(alpha),
        "coverage_singleton": cov,
        "risk_1_minus_acc": float(1.0 - acc),
        "macro_f1_on_decisions": mf1,
        "benign_FPR_on_decisions": benign_fpr,
        "avg_set_size": float(k_star.mean()),
    }

raps_sel_rows = [
    singleton_risk_coverage(
        pub_yf, pub_test_probs_f, pub_test_probs_c,
        raps_calib_pub, a, cfg.RAPS_KREG, cfg.RAPS_LAMBDA, benign_fine_id
    )
    for a in cfg.ALPHAS
]
raps_sel_df = pd.DataFrame(raps_sel_rows)

save_df_many_formats(pd.DataFrame(raps_results_pub).T.reset_index().rename(columns={"index": "alpha"}), PUB_TAB_DIR / "raps_results", index=False)
save_df_many_formats(raps_sel_df, PUB_TAB_DIR / "raps_singleton_risk_coverage", index=False)

# RAPS set size histogram for default alpha
alpha0 = float(PUB_RAPS_ALPHA_DEFAULT)
if str(alpha0) in raps_calib_pub["q_by_pred_coarse"]:
    q_by = np.array(raps_calib_pub["q_by_pred_coarse"][str(alpha0)], dtype=np.float32)
    q_row = q_by[pub_test_probs_c.argmax(axis=1)]
    k_star = raps_predict_k(pub_test_probs_f, q_row, cfg.RAPS_KREG, cfg.RAPS_LAMBDA)

    plt.figure(figsize=(7, 5))
    vals, counts = np.unique(k_star, return_counts=True)
    plt.bar(vals, counts / counts.sum())
    plt.xlabel("RAPS set size")
    plt.ylabel("Fraction")
    plt.title(f"RAPS set-size distribution (alpha={alpha0})")
    plt.grid(True, axis="y", alpha=0.3)
    plt.tight_layout()
    plt.savefig(str(PUB_FIG_DIR / f"raps_set_size_alpha_{alpha0:.2f}.png"), dpi=220, bbox_inches="tight")
    plt.close()

# ---------------------------
# 13) Throughput summary reuse
# ---------------------------
pub_throughput = {
    "gpu": throughput_gpu if "throughput_gpu" in globals() else None,
    "cpu": throughput_cpu if "throughput_cpu" in globals() else None,
}

# ---------------------------
# 14) Safe export (fixes old export problems)
# ---------------------------
class PubLogitsWrapper(nn.Module):
    def __init__(self, base: nn.Module):
        super().__init__()
        self.base = base

    def forward(self, x: torch.Tensor):
        o = self.base(x)
        return o["logits_fine"], o["logits_coarse"], o["logits_bin"]

export_status = {
    "torchscript": {"ok": False, "path": None, "error": None},
    "onnx": {"ok": False, "path": None, "error": None, "validated": False},
}

pub_model_cpu = build_fresh_model().to("cpu").eval()
pub_model_cpu.load_state_dict({k: v.detach().cpu() for k, v in pub_model.state_dict().items()}, strict=True)
wrapper_cpu = PubLogitsWrapper(pub_model_cpu).eval()

# TorchScript via script (safer than trace here)
try:
    scripted = torch.jit.script(wrapper_cpu)
    ts_path = PUB_EXPORT_DIR / "camelot_ids_v2_publication_torchscript.pt"
    scripted.save(str(ts_path))
    export_status["torchscript"]["ok"] = True
    export_status["torchscript"]["path"] = str(ts_path)
    print("TorchScript export OK:", ts_path)
except Exception as e:
    export_status["torchscript"]["error"] = str(e)
    print("TorchScript export skipped:", e)

# ONNX opset 18
try:
    onnx_path = PUB_EXPORT_DIR / "camelot_ids_v2_publication.onnx"
    dummy = torch.randn(1, num_features, dtype=torch.float32)
    torch.onnx.export(
        wrapper_cpu,
        dummy,
        str(onnx_path),
        input_names=["input"],
        output_names=["logits_fine", "logits_coarse", "logits_bin"],
        dynamic_axes={
            "input": {0: "batch"},
            "logits_fine": {0: "batch"},
            "logits_coarse": {0: "batch"},
            "logits_bin": {0: "batch"},
        },
        opset_version=PUB_ONNX_OPSET,
    )
    export_status["onnx"]["ok"] = True
    export_status["onnx"]["path"] = str(onnx_path)
    print("ONNX export OK:", onnx_path)

    if PUB_VALIDATE_ONNX:
        try:
            import onnxruntime as ort
            sess = ort.InferenceSession(str(onnx_path), providers=["CPUExecutionProvider"])
            x_small = X_test[:8].astype(np.float32, copy=False)

            with torch.inference_mode():
                pt_out = wrapper_cpu(torch.from_numpy(x_small))
                pt_f = pt_out[0].numpy()
                pt_c = pt_out[1].numpy()
                pt_b = pt_out[2].numpy()

            onnx_out = sess.run(None, {"input": x_small})
            ok_f = np.allclose(pt_f, onnx_out[0], atol=1e-4, rtol=1e-4)
            ok_c = np.allclose(pt_c, onnx_out[1], atol=1e-4, rtol=1e-4)
            ok_b = np.allclose(pt_b, onnx_out[2], atol=1e-4, rtol=1e-4)

            export_status["onnx"]["validated"] = bool(ok_f and ok_c and ok_b)
            export_status["onnx"]["validation_allclose"] = {
                "fine": bool(ok_f),
                "coarse": bool(ok_c),
                "bin": bool(ok_b),
            }
            print("ONNX validation:", export_status["onnx"]["validation_allclose"])
        except Exception as e:
            export_status["onnx"]["validated"] = False
            export_status["onnx"]["validation_error"] = str(e)
            print("ONNX validation skipped:", e)

except Exception as e:
    export_status["onnx"]["error"] = str(e)
    print("ONNX export failed:", e)

# Robust inference CLI (TorchScript first, ONNX fallback)
pub_cli = r'''
import argparse
from pathlib import Path
import numpy as np
import pandas as pd
import joblib

def sanitize_frame(df: pd.DataFrame, clip_inf: float = 1e9) -> pd.DataFrame:
    df = df.replace({"Infinity": np.inf, "inf": np.inf, "+inf": np.inf, "-inf": -np.inf,
                     "NaN": np.nan, "nan": np.nan, "": np.nan})
    df = df.replace([np.inf, -np.inf], np.nan)
    num_cols = df.select_dtypes(include=[np.number]).columns
    if len(num_cols) > 0:
        df[num_cols] = df[num_cols].clip(-clip_inf, clip_inf)
    return df

def transform_df(df_in: pd.DataFrame, feature_cols, meta_obj):
    cfg = meta_obj["cfg"]
    X = sanitize_frame(df_in[feature_cols].copy(), cfg["CLIP_INF"])
    for c in meta_obj["meta"]["numeric_like_obj"]:
        if c in X.columns:
            X[c] = pd.to_numeric(X[c], errors="coerce")
    num_cols = meta_obj["meta"]["num_cols"]
    X = X[num_cols]
    Xn = meta_obj["num_imputer"].transform(X).astype(np.float32)
    Xn = meta_obj["scaler"].transform(Xn).astype(np.float32)
    Xn = np.clip(Xn, -cfg["POST_SCALE_CLIP"], cfg["POST_SCALE_CLIP"]).astype(np.float32)
    Xn = np.nan_to_num(Xn, nan=0.0, posinf=cfg["POST_SCALE_CLIP"], neginf=-cfg["POST_SCALE_CLIP"]).astype(np.float32)
    return Xn

def softmax_np(x):
    x = x - x.max(axis=1, keepdims=True)
    ex = np.exp(x)
    return ex / np.clip(ex.sum(axis=1, keepdims=True), 1e-12, None)

def raps_predict_set(probs_f, probs_c, raps_calib, alpha):
    k_reg = int(raps_calib["k_reg"])
    lam = float(raps_calib["lambda"])
    q_by = np.array(raps_calib["q_by_pred_coarse"][str(alpha)], dtype=np.float32)
    g_hat = probs_c.argmax(axis=1)
    q_row = q_by[g_hat]

    idx_sorted = np.argsort(-probs_f, axis=1)
    probs_sorted = np.take_along_axis(probs_f, idx_sorted, axis=1)
    cumsum = np.cumsum(probs_sorted, axis=1)
    C = probs_f.shape[1]
    ranks = np.arange(1, C + 1, dtype=np.float32)[None, :]
    reg = lam * np.maximum(ranks - float(k_reg), 0.0)
    score_k = cumsum + reg
    ok = score_k <= q_row[:, None]
    k_star = np.maximum(ok.sum(axis=1).astype(np.int64), 1)

    sets = []
    for i in range(len(probs_f)):
        sets.append(idx_sorted[i, :k_star[i]])
    return sets, k_star

def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--meta_joblib", type=str, required=True)
    ap.add_argument("--input_csv", type=str, required=True)
    ap.add_argument("--out_csv", type=str, default="preds_publication.csv")
    ap.add_argument("--torchscript", type=str, default="")
    ap.add_argument("--onnx", type=str, default="")
    ap.add_argument("--device", type=str, default="cpu")
    ap.add_argument("--alpha", type=float, default=0.10)
    args = ap.parse_args()

    meta = joblib.load(args.meta_joblib)
    feature_cols = meta["FEATURE_COLS"]
    fine_names = meta["fine_class_names"]
    coarse_names = meta["coarse_names"]
    temp = meta.get("temperature", {"fine": 1.0, "coarse": 1.0, "bin": 1.0})
    raps_calib = meta.get("raps_calib_publication", meta.get("raps_calib", None))
    if raps_calib is None:
        raise RuntimeError("No RAPS calibration found in meta joblib.")

    df = pd.read_csv(args.input_csv, low_memory=False)
    X = transform_df(df, feature_cols, meta)

    logits_f = logits_c = logits_b = None

    if args.torchscript:
        import torch
        model = torch.jit.load(args.torchscript, map_location=args.device)
        model.eval()
        xb = torch.from_numpy(X).to(args.device)
        with torch.inference_mode():
            logits_f, logits_c, logits_b = model(xb)
            logits_f = logits_f.cpu().numpy()
            logits_c = logits_c.cpu().numpy()
            logits_b = logits_b.cpu().numpy()
    elif args.onnx:
        import onnxruntime as ort
        sess = ort.InferenceSession(args.onnx, providers=["CPUExecutionProvider"])
        logits_f, logits_c, logits_b = sess.run(None, {"input": X.astype(np.float32)})
    else:
        raise RuntimeError("Provide either --torchscript or --onnx")

    pf = softmax_np(logits_f / max(float(temp["fine"]), 1e-6))
    pc = softmax_np(logits_c / max(float(temp["coarse"]), 1e-6))
    pb = softmax_np(logits_b / max(float(temp["bin"]), 1e-6))

    top_f = pf.argmax(axis=1)
    top_conf = pf[np.arange(len(pf)), top_f]
    top_c = pc.argmax(axis=1)
    top_b = pb.argmax(axis=1)

    sets, k_star = raps_predict_set(pf, pc, raps_calib, float(args.alpha))
    set_str = [",".join([fine_names[j] for j in s]) for s in sets]

    out = pd.DataFrame({
        "top1_fine": [fine_names[i] for i in top_f],
        "top1_conf": top_conf,
        "pred_coarse": [coarse_names[i] for i in top_c],
        "pred_bin": ["Malicious" if i == 1 else "Benign" for i in top_b],
        "raps_set_size": k_star,
        "raps_set": set_str,
    })
    out.to_csv(args.out_csv, index=False)
    print("Saved:", args.out_csv)

if __name__ == "__main__":
    main()
'''
(PUB_EXPORT_DIR / "camelot_infer_publication.py").write_text(pub_cli, encoding="utf-8")

# ---------------------------
# 15) Save publication meta joblib
# ---------------------------
pub_meta = joblib.load(export_dir / "preprocess_and_meta.joblib")
pub_meta["temperature"] = {"fine": float(pub_temp_f), "coarse": float(pub_temp_c), "bin": float(pub_temp_b)}
pub_meta["raps_calib_publication"] = raps_calib_pub
joblib.dump(pub_meta, PUB_EXPORT_DIR / "preprocess_and_meta_publication.joblib")

# ---------------------------
# 16) Stream results reuse (fast)
# ---------------------------
stream_summary = None
stream_json = RUN_DIR / "stream_results.json"
if stream_json.exists():
    stream_summary = load_json(stream_json, None)

# ---------------------------
# 17) Publication summary files
# ---------------------------
publication_summary = {
    "chosen_publication_model": best_name,
    "candidate_selection": candidate_df.to_dict(orient="records"),
    "fine_metrics": metrics_fine,
    "coarse_metrics": metrics_coarse,
    "binary_metrics": metrics_bin,
    "fine_calibration_before": {k: v for k, v in cal_f_raw.items() if k != "bins"},
    "fine_calibration_after":  {k: v for k, v in cal_f_cal.items() if k != "bins"},
    "coarse_calibration_before": {k: v for k, v in cal_c_raw.items() if k != "bins"},
    "coarse_calibration_after":  {k: v for k, v in cal_c_cal.items() if k != "bins"},
    "binary_calibration_before": {k: v for k, v in cal_b_raw.items() if k != "bins"},
    "binary_calibration_after":  {k: v for k, v in cal_b_cal.items() if k != "bins"},
    "bootstrap_ci_key_metrics": ci_summary,
    "raps_results": raps_results_pub,
    "raps_singleton_risk_coverage": raps_sel_df.to_dict(orient="records"),
    "throughput": pub_throughput,
    "stream_results_reused": stream_summary,
    "export_status": export_status,
    "num_test_samples": int(len(pub_yf)),
    "num_features": int(num_features),
    "num_fine": int(NUM_FINE),
    "num_coarse": int(NUM_COARSE),
}

with open(PUB_JSON_DIR / "publication_summary.json", "w", encoding="utf-8") as f:
    json.dump(publication_summary, f, indent=2)

summary_md = f"""
# CAMELOT-IDS v2 — Publication Summary

## Chosen publication model
- Candidate: **{best_name}**

## Fine (34-class)
- Accuracy: **{metrics_fine['accuracy']:.6f}**
- Balanced accuracy: **{metrics_fine['balanced_accuracy']:.6f}**
- Macro-F1: **{metrics_fine['macro_f1']:.6f}**
- Weighted-F1: **{metrics_fine['weighted_f1']:.6f}**
- Macro-Precision: **{metrics_fine['macro_precision']:.6f}**
- Macro-Recall: **{metrics_fine['macro_recall']:.6f}**
- MCC: **{metrics_fine['mcc']:.6f}**
- Cohen's kappa: **{metrics_fine['kappa']:.6f}**
- Top-3 accuracy: **{metrics_fine['top3_accuracy']:.6f}**
- Top-5 accuracy: **{metrics_fine['top5_accuracy']:.6f}**

## Coarse (8-class)
- Accuracy: **{metrics_coarse['accuracy']:.6f}**
- Balanced accuracy: **{metrics_coarse['balanced_accuracy']:.6f}**
- Macro-F1: **{metrics_coarse['macro_f1']:.6f}**
- Weighted-F1: **{metrics_coarse['weighted_f1']:.6f}**
- MCC: **{metrics_coarse['mcc']:.6f}**
- Cohen's kappa: **{metrics_coarse['kappa']:.6f}**

## Binary (malicious vs benign)
- Accuracy: **{metrics_bin['accuracy']:.6f}**
- Balanced accuracy: **{metrics_bin['balanced_accuracy']:.6f}**
- F1 (malicious): **{metrics_bin['f1_malicious']:.6f}**
- Precision (malicious): **{metrics_bin['precision_malicious']:.6f}**
- Recall (malicious): **{metrics_bin['recall_malicious']:.6f}**
- AUROC: **{metrics_bin['auroc']:.6f}**
- AUPRC: **{metrics_bin['auprc']:.6f}**

## Calibration (fine)
- Before: NLL={cal_f_raw['nll']:.6f}, ECE={cal_f_raw['ece']:.6f}, Brier={cal_f_raw['brier']:.6f}
- After:  NLL={cal_f_cal['nll']:.6f}, ECE={cal_f_cal['ece']:.6f}, Brier={cal_f_cal['brier']:.6f}

## RAPS
{json.dumps(raps_results_pub, indent=2)}

## Export status
{json.dumps(export_status, indent=2)}

## Files generated
- Figures: `{PUB_FIG_DIR}`
- Tables: `{PUB_TAB_DIR}`
- JSON: `{PUB_JSON_DIR}`
- Export: `{PUB_EXPORT_DIR}`
"""
write_markdown(PUB_DIR / "publication_summary.md", summary_md)

print("\nSaved publication summary:", PUB_JSON_DIR / "publication_summary.json")
print("Saved markdown summary:", PUB_DIR / "publication_summary.md")
print("Publication package complete at:", PUB_DIR)

Publication package dir: results_camelot_ids_v2_pc\publication_package
Candidate: best_raw_epoch_checkpoint | val fine macro-F1 = 0.770156
Candidate: last_raw_checkpoint | val fine macro-F1 = 0.769814
Candidate: last_ema_shadow_checkpoint | val fine macro-F1 = 0.769492

Chosen publication model: best_raw_epoch_checkpoint
Loading cached logits: publication_test_best_raw_epoch_checkpoint
Loading cached logits: publication_cal_best_raw_epoch_checkpoint
Refit temperatures: {'fine': 1.0, 'coarse': 1.0, 'bin': 1.0}
TorchScript export skipped: 
getattr's second argument must be a string literal:
  File "C:\Users\HaseebWajid\AppData\Local\Temp\ipykernel_17984\859139402.py", line 352
        group_reps = []
        for g in range(self.num_groups):
            idxs_t = getattr(self, f"group_idx_{g}")
                     ~~~~~~~~~~~~~~~~~~~~~~~~~~~~ <--- HERE
            t = tok.index_select(dim=1, index=idxs_t)
            t = t + self.group_type.weight[g].view(1, 1, -1)

[torch.onnx] Obtain mo

In [6]:
# ============================================================
# EXPORT PATCH FOR PUBLICATION PACKAGE
# Run AFTER the publication extension cell.
# This fixes the remaining export issues without retraining.
# ============================================================

from pathlib import Path
import json
import numpy as np
import torch
import torch.nn as nn

PATCH_EXPORT_DIR = PUB_EXPORT_DIR
PATCH_EXPORT_DIR.mkdir(parents=True, exist_ok=True)

class PubLogitsWrapper(nn.Module):
    def __init__(self, base: nn.Module):
        super().__init__()
        self.base = base

    def forward(self, x: torch.Tensor):
        o = self.base(x)
        return o["logits_fine"], o["logits_coarse"], o["logits_bin"]

# build CPU export wrapper from chosen publication model
pub_model_cpu = build_fresh_model().to("cpu").eval()
pub_model_cpu.load_state_dict({k: v.detach().cpu() for k, v in pub_model.state_dict().items()}, strict=True)
wrapper_cpu = PubLogitsWrapper(pub_model_cpu).eval()

export_patch_status = {
    "torchscript_trace": {"ok": False, "path": None, "error": None},
    "onnx_static_batch1": {"ok": False, "path": None, "validated": False, "error": None},
}

# ------------------------------------------------------------
# 1) TorchScript via TRACE, not SCRIPT
#    This avoids the getattr string-literal scripting error.
# ------------------------------------------------------------
try:
    example = torch.randn(1, num_features, dtype=torch.float32)
    traced = torch.jit.trace(wrapper_cpu, example, check_trace=False)
    ts_trace_path = PATCH_EXPORT_DIR / "camelot_ids_v2_publication_torchscript_trace.pt"
    traced.save(str(ts_trace_path))
    export_patch_status["torchscript_trace"]["ok"] = True
    export_patch_status["torchscript_trace"]["path"] = str(ts_trace_path)
    print("TorchScript TRACE export OK:", ts_trace_path)
except Exception as e:
    export_patch_status["torchscript_trace"]["error"] = str(e)
    print("TorchScript TRACE export failed:", e)

# ------------------------------------------------------------
# 2) ONNX STATIC BATCH=1
#    This avoids the bad dynamic reshape issue from the prior export.
# ------------------------------------------------------------
try:
    onnx_static_path = PATCH_EXPORT_DIR / "camelot_ids_v2_publication_static_b1.onnx"
    dummy = torch.randn(1, num_features, dtype=torch.float32)

    torch.onnx.export(
        wrapper_cpu,
        dummy,
        str(onnx_static_path),
        input_names=["input"],
        output_names=["logits_fine", "logits_coarse", "logits_bin"],
        opset_version=18,
        dynamic_axes=None,   # <-- important: static batch export
        do_constant_folding=True,
    )

    export_patch_status["onnx_static_batch1"]["ok"] = True
    export_patch_status["onnx_static_batch1"]["path"] = str(onnx_static_path)
    print("Static ONNX export OK:", onnx_static_path)

    # validate ONNX on batch size 1
    try:
        import onnxruntime as ort

        sess = ort.InferenceSession(str(onnx_static_path), providers=["CPUExecutionProvider"])

        x_small = X_test[:1].astype(np.float32, copy=False)
        with torch.inference_mode():
            pt_out = wrapper_cpu(torch.from_numpy(x_small))
            pt_f = pt_out[0].numpy()
            pt_c = pt_out[1].numpy()
            pt_b = pt_out[2].numpy()

        ort_out = sess.run(None, {"input": x_small})

        ok_f = np.allclose(pt_f, ort_out[0], atol=1e-4, rtol=1e-4)
        ok_c = np.allclose(pt_c, ort_out[1], atol=1e-4, rtol=1e-4)
        ok_b = np.allclose(pt_b, ort_out[2], atol=1e-4, rtol=1e-4)

        export_patch_status["onnx_static_batch1"]["validated"] = bool(ok_f and ok_c and ok_b)
        export_patch_status["onnx_static_batch1"]["validation_allclose"] = {
            "fine": bool(ok_f),
            "coarse": bool(ok_c),
            "bin": bool(ok_b),
        }
        print("Static ONNX validation:", export_patch_status["onnx_static_batch1"]["validation_allclose"])

    except Exception as e:
        export_patch_status["onnx_static_batch1"]["validated"] = False
        export_patch_status["onnx_static_batch1"]["validation_error"] = str(e)
        print("Static ONNX validation failed:", e)

except Exception as e:
    export_patch_status["onnx_static_batch1"]["error"] = str(e)
    print("Static ONNX export failed:", e)

# ------------------------------------------------------------
# 3) Write a static-batch ONNX inference CLI
#    It runs one row at a time, so it is slower, but robust.
# ------------------------------------------------------------
static_cli = r'''
import argparse
from pathlib import Path
import numpy as np
import pandas as pd
import joblib
import onnxruntime as ort

def sanitize_frame(df: pd.DataFrame, clip_inf: float = 1e9) -> pd.DataFrame:
    df = df.replace({"Infinity": np.inf, "inf": np.inf, "+inf": np.inf, "-inf": -np.inf,
                     "NaN": np.nan, "nan": np.nan, "": np.nan})
    df = df.replace([np.inf, -np.inf], np.nan)
    num_cols = df.select_dtypes(include=[np.number]).columns
    if len(num_cols) > 0:
        df[num_cols] = df[num_cols].clip(-clip_inf, clip_inf)
    return df

def transform_df(df_in: pd.DataFrame, feature_cols, meta_obj):
    cfg = meta_obj["cfg"]
    X = sanitize_frame(df_in[feature_cols].copy(), cfg["CLIP_INF"])
    for c in meta_obj["meta"]["numeric_like_obj"]:
        if c in X.columns:
            X[c] = pd.to_numeric(X[c], errors="coerce")
    num_cols = meta_obj["meta"]["num_cols"]
    X = X[num_cols]
    Xn = meta_obj["num_imputer"].transform(X).astype(np.float32)
    Xn = meta_obj["scaler"].transform(Xn).astype(np.float32)
    Xn = np.clip(Xn, -cfg["POST_SCALE_CLIP"], cfg["POST_SCALE_CLIP"]).astype(np.float32)
    Xn = np.nan_to_num(Xn, nan=0.0, posinf=cfg["POST_SCALE_CLIP"], neginf=-cfg["POST_SCALE_CLIP"]).astype(np.float32)
    return Xn

def softmax_np(x):
    x = x - x.max(axis=1, keepdims=True)
    ex = np.exp(x)
    return ex / np.clip(ex.sum(axis=1, keepdims=True), 1e-12, None)

def raps_predict_set(probs_f, probs_c, raps_calib, alpha):
    k_reg = int(raps_calib["k_reg"])
    lam = float(raps_calib["lambda"])
    q_by = np.array(raps_calib["q_by_pred_coarse"][str(alpha)], dtype=np.float32)
    g_hat = probs_c.argmax(axis=1)
    q_row = q_by[g_hat]

    idx_sorted = np.argsort(-probs_f, axis=1)
    probs_sorted = np.take_along_axis(probs_f, idx_sorted, axis=1)
    cumsum = np.cumsum(probs_sorted, axis=1)
    C = probs_f.shape[1]
    ranks = np.arange(1, C + 1, dtype=np.float32)[None, :]
    reg = lam * np.maximum(ranks - float(k_reg), 0.0)
    score_k = cumsum + reg
    ok = score_k <= q_row[:, None]
    k_star = np.maximum(ok.sum(axis=1).astype(np.int64), 1)

    sets = []
    for i in range(len(probs_f)):
        sets.append(idx_sorted[i, :k_star[i]])
    return sets, k_star

def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--meta_joblib", type=str, required=True)
    ap.add_argument("--onnx", type=str, required=True)
    ap.add_argument("--input_csv", type=str, required=True)
    ap.add_argument("--out_csv", type=str, default="preds_publication_static_onnx.csv")
    ap.add_argument("--alpha", type=float, default=0.10)
    args = ap.parse_args()

    meta = joblib.load(args.meta_joblib)
    feature_cols = meta["FEATURE_COLS"]
    fine_names = meta["fine_class_names"]
    coarse_names = meta["coarse_names"]
    temp = meta.get("temperature", {"fine": 1.0, "coarse": 1.0, "bin": 1.0})
    raps_calib = meta.get("raps_calib_publication", meta.get("raps_calib", None))
    if raps_calib is None:
        raise RuntimeError("No RAPS calibration found in meta joblib.")

    df = pd.read_csv(args.input_csv, low_memory=False)
    X = transform_df(df, feature_cols, meta)

    sess = ort.InferenceSession(args.onnx, providers=["CPUExecutionProvider"])

    logits_f_list, logits_c_list, logits_b_list = [], [], []
    for i in range(len(X)):
        xi = X[i:i+1].astype(np.float32, copy=False)
        lf, lc, lb = sess.run(None, {"input": xi})
        logits_f_list.append(lf)
        logits_c_list.append(lc)
        logits_b_list.append(lb)

    logits_f = np.concatenate(logits_f_list, axis=0)
    logits_c = np.concatenate(logits_c_list, axis=0)
    logits_b = np.concatenate(logits_b_list, axis=0)

    pf = softmax_np(logits_f / max(float(temp["fine"]), 1e-6))
    pc = softmax_np(logits_c / max(float(temp["coarse"]), 1e-6))
    pb = softmax_np(logits_b / max(float(temp["bin"]), 1e-6))

    top_f = pf.argmax(axis=1)
    top_conf = pf[np.arange(len(pf)), top_f]
    top_c = pc.argmax(axis=1)
    top_b = pb.argmax(axis=1)

    sets, k_star = raps_predict_set(pf, pc, raps_calib, float(args.alpha))
    set_str = [",".join([fine_names[j] for j in s]) for s in sets]

    out = pd.DataFrame({
        "top1_fine": [fine_names[i] for i in top_f],
        "top1_conf": top_conf,
        "pred_coarse": [coarse_names[i] for i in top_c],
        "pred_bin": ["Malicious" if i == 1 else "Benign" for i in top_b],
        "raps_set_size": k_star,
        "raps_set": set_str,
    })
    out.to_csv(args.out_csv, index=False)
    print("Saved:", args.out_csv)

if __name__ == "__main__":
    main()
'''
(PATCH_EXPORT_DIR / "camelot_infer_publication_static_onnx.py").write_text(static_cli, encoding="utf-8")

# ------------------------------------------------------------
# 4) Save patch status
# ------------------------------------------------------------
with open(PATCH_EXPORT_DIR / "export_patch_status.json", "w", encoding="utf-8") as f:
    json.dump(export_patch_status, f, indent=2)

print("\nSaved:", PATCH_EXPORT_DIR / "export_patch_status.json")
print("Export patch complete.")

NameError: name 'PUB_EXPORT_DIR' is not defined

In [7]:
# ============================================================
# COMPATIBILITY PATCH FOR FAST EXTRA PUBLICATION BLOCK
# Run this BEFORE the fast extra publication block.
# ============================================================

import os
import random
import numpy as np
import torch
import torch.nn.functional as F

# ---------------------------
# Seed helper
# ---------------------------
def set_seed(seed: int = 42, deterministic: bool = False):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = bool(deterministic)
        torch.backends.cudnn.benchmark = not bool(deterministic)

# ---------------------------
# Hierarchical consistency loss
# ---------------------------
if "fine_to_coarse" not in globals():
    raise NameError("fine_to_coarse is not defined. Run the bootstrap/main context first.")

if "NUM_COARSE" not in globals():
    raise NameError("NUM_COARSE is not defined. Run the bootstrap/main context first.")

if "benign_fine_id" not in globals():
    raise NameError("benign_fine_id is not defined. Run the bootstrap/main context first.")

fine_to_coarse_t = torch.tensor(fine_to_coarse, device=DEVICE, dtype=torch.long)

def hierarchical_consistency_loss(logits_fine, logits_coarse, logits_bin):
    p_f = torch.softmax(logits_fine, dim=1)
    p_c = torch.softmax(logits_coarse, dim=1)
    p_b = torch.softmax(logits_bin, dim=1)

    B = p_f.size(0)
    idx = fine_to_coarse_t.unsqueeze(0).expand(B, -1)
    p_from_f = torch.zeros(B, NUM_COARSE, device=p_f.device)
    p_from_f.scatter_add_(1, idx, p_f)
    cons_c = F.kl_div(torch.log(p_c + 1e-8), p_from_f, reduction="batchmean")

    p_from_f_bin = torch.stack([p_f[:, benign_fine_id], 1.0 - p_f[:, benign_fine_id]], dim=1)
    cons_b = F.kl_div(torch.log(p_b + 1e-8), p_from_f_bin, reduction="batchmean")
    return cons_c, cons_b

print("Compatibility patch loaded: set_seed + hierarchical_consistency_loss")

Compatibility patch loaded: set_seed + hierarchical_consistency_loss


In [8]:
# ============================================================
# FAST EXTRA PUBLICATION RESULTS (SHORT-RUN VERSION)
# Run after the main notebook.
# This is designed to avoid multi-day runs.
# ============================================================

import os
import gc
import json
import math
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    matthews_corrcoef,
    cohen_kappa_score,
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier

warnings.filterwarnings("ignore")

# ---------------------------
# Fast controls
# ---------------------------
FAST_PUB_DIR = RUN_DIR / "fast_extra_publication"
FAST_PUB_DIR.mkdir(parents=True, exist_ok=True)

FAST_CFG = {
    # bounded subset sizes for fast publication extras
    "TRAIN_CAP": 250_000,
    "VAL_CAP": 60_000,
    "TEST_CAP": 100_000,

    # repeatability
    "REPEAT_SEEDS": [11, 42, 123],
    "QUICK_EPOCHS": 8,
    "QUICK_PATIENCE": 3,
    "QUICK_BATCH_SIZE": 2048,
    "QUICK_LR": 1e-3,
    "QUICK_WD": 5e-3,
    "QUICK_GRAD_CLIP": 1.0,

    # baseline caps
    "BASELINE_TRAIN_CAP": 200_000,
    "BASELINE_TEST_CAP": 100_000,

    # toggles
    "RUN_REPEATABILITY": True,
    "RUN_BASELINES": True,
    "RUN_MULTITASK_ABLATION": True,
    "ABLATION_SEEDS": [42],      # keep to 1 seed for fast run
    "RUN_EXTERNAL": False,       # set True only if you have a second dataset
    "EXT_DATA_ROOT": None,       # set folder path if available
    "EXT_MAX_FILES": 25,

    # conformal default
    "RAPS_ALPHA": 0.10,
}

print("FAST publication dir:", FAST_PUB_DIR)
print(json.dumps(FAST_CFG, indent=2))

# ---------------------------
# Helpers
# ---------------------------
def save_json(path, obj):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2)

def save_df(df: pd.DataFrame, stem: Path, index=False):
    stem.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(str(stem.with_suffix(".csv")), index=index)
    try:
        df.to_latex(str(stem.with_suffix(".tex")), index=index, float_format="%.4f")
    except Exception:
        pass

def stratified_cap_indices(y: np.ndarray, cap: int, seed: int = 42) -> np.ndarray:
    if len(y) <= cap:
        return np.arange(len(y))
    rng = np.random.default_rng(seed)
    idxs = []
    classes, counts = np.unique(y, return_counts=True)
    frac = cap / len(y)
    for c, cnt in zip(classes, counts):
        c_idx = np.where(y == c)[0]
        take = max(1, int(round(cnt * frac)))
        take = min(take, len(c_idx))
        pick = rng.choice(c_idx, size=take, replace=False)
        idxs.append(pick)
    idxs = np.concatenate(idxs)
    if len(idxs) > cap:
        idxs = rng.choice(idxs, size=cap, replace=False)
    return np.sort(idxs)

def subset_arrays(X, yf, yc, yb, cap, seed):
    idx = stratified_cap_indices(yf, cap, seed)
    return X[idx], yf[idx], yc[idx], yb[idx], idx

def quick_metrics_mc(y_true: np.ndarray, probs: np.ndarray):
    y_pred = probs.argmax(axis=1)
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
        "macro_f1": float(f1_score(y_true, y_pred, average="macro")),
        "weighted_f1": float(f1_score(y_true, y_pred, average="weighted")),
        "macro_precision": float(precision_score(y_true, y_pred, average="macro", zero_division=0)),
        "macro_recall": float(recall_score(y_true, y_pred, average="macro", zero_division=0)),
        "mcc": float(matthews_corrcoef(y_true, y_pred)),
        "kappa": float(cohen_kappa_score(y_true, y_pred)),
    }

def quick_metrics_bin(y_true: np.ndarray, probs_2: np.ndarray):
    y_pred = probs_2.argmax(axis=1)
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
        "f1_malicious": float(f1_score(y_true, y_pred, average="binary", pos_label=1)),
        "precision_malicious": float(precision_score(y_true, y_pred, average="binary", pos_label=1, zero_division=0)),
        "recall_malicious": float(recall_score(y_true, y_pred, average="binary", pos_label=1, zero_division=0)),
        "mcc": float(matthews_corrcoef(y_true, y_pred)),
        "kappa": float(cohen_kappa_score(y_true, y_pred)),
    }

def cb_weights_from_counts(y: np.ndarray, n_classes: int):
    counts = np.bincount(y, minlength=n_classes).astype(np.float64)
    w = counts.sum() / np.clip(counts, 1.0, None)
    w = w / np.mean(w)
    return torch.tensor(w, dtype=torch.float32)

def build_quick_loaders(seed=42):
    Xtr, ytr_f, ytr_c, ytr_b, _ = subset_arrays(X_train, y_train_f, y_train_c, y_train_b, FAST_CFG["TRAIN_CAP"], seed)
    Xva, yva_f, yva_c, yva_b, _ = subset_arrays(X_val,   y_val_f,   y_val_c,   y_val_b,   FAST_CFG["VAL_CAP"], seed)
    Xte, yte_f, yte_c, yte_b, _ = subset_arrays(X_test,  y_test_f,  y_test_c,  y_test_b,  FAST_CFG["TEST_CAP"], seed)

    ds_tr = FlowDataset(Xtr, ytr_f, ytr_c, ytr_b)
    ds_va = FlowDataset(Xva, yva_f, yva_c, yva_b)
    ds_te = FlowDataset(Xte, yte_f, yte_c, yte_b)

    loader_kwargs_local = dict(
        batch_size=FAST_CFG["QUICK_BATCH_SIZE"],
        num_workers=0,
        pin_memory=(DEVICE == "cuda"),
        persistent_workers=False,
    )

    tr_loader = DataLoader(ds_tr, shuffle=True, **loader_kwargs_local)
    va_loader = DataLoader(ds_va, shuffle=False, **loader_kwargs_local)
    te_loader = DataLoader(ds_te, shuffle=False, **loader_kwargs_local)

    return {
        "train": (Xtr, ytr_f, ytr_c, ytr_b, tr_loader),
        "val":   (Xva, yva_f, yva_c, yva_b, va_loader),
        "test":  (Xte, yte_f, yte_c, yte_b, te_loader),
    }

def build_fresh_model_local():
    m = CamelotIDSv2(
        n_features=num_features,
        n_fine=NUM_FINE,
        n_coarse=NUM_COARSE,
        group_idxs=GROUP_IDXS,
        cfg=cfg
    ).to(DEVICE)
    return m

@torch.inference_mode()
def eval_model_on_loader(model_local, loader):
    model_local.eval()
    out = predict_logits(model_local, loader)
    pf = softmax_np(out["fine"])
    pc = softmax_np(out["coarse"])
    pb = softmax_np(out["bin"])
    return {
        "fine": quick_metrics_mc(out["yf"], pf),
        "coarse": quick_metrics_mc(out["yc"], pc),
        "binary": quick_metrics_bin(out["yb"], pb),
        "raw": out,
        "pf": pf,
        "pc": pc,
        "pb": pb,
    }

def train_quick_model(seed=42, use_multitask=True):
    set_seed(seed, deterministic=False)
    bundles = build_quick_loaders(seed)

    Xtr, ytr_f, ytr_c, ytr_b, tr_loader = bundles["train"]
    Xva, yva_f, yva_c, yva_b, va_loader = bundles["val"]
    Xte, yte_f, yte_c, yte_b, te_loader = bundles["test"]

    model_local = build_fresh_model_local()
    opt = torch.optim.AdamW(model_local.parameters(), lr=FAST_CFG["QUICK_LR"], weight_decay=FAST_CFG["QUICK_WD"])
    use_amp_local = (cfg.USE_AMP and DEVICE == "cuda")
    scaler_local = torch.amp.GradScaler("cuda", enabled=use_amp_local) if DEVICE == "cuda" else None

    w_f = cb_weights_from_counts(ytr_f, NUM_FINE).to(DEVICE)
    w_c = cb_weights_from_counts(ytr_c, NUM_COARSE).to(DEVICE)
    w_b = cb_weights_from_counts(ytr_b, 2).to(DEVICE)

    fine_loss = nn.CrossEntropyLoss(weight=w_f).to(DEVICE)
    coarse_loss = nn.CrossEntropyLoss(weight=w_c).to(DEVICE)
    bin_loss = nn.CrossEntropyLoss(weight=w_b).to(DEVICE)

    best_state = None
    best_val = -1.0
    bad = 0
    hist_rows = []

    for epoch in range(1, FAST_CFG["QUICK_EPOCHS"] + 1):
        model_local.train()
        total_loss = 0.0
        n_seen = 0

        for xb, yf, yc, yb in tr_loader:
            xb = xb.to(DEVICE, non_blocking=True)
            yf = yf.to(DEVICE, non_blocking=True)
            yc = yc.to(DEVICE, non_blocking=True)
            yb = yb.to(DEVICE, non_blocking=True)

            opt.zero_grad(set_to_none=True)

            with torch.amp.autocast(device_type="cuda" if DEVICE == "cuda" else "cpu", enabled=use_amp_local):
                out = model_local(xb)
                lf = fine_loss(out["logits_fine"], yf)

                if use_multitask:
                    lc = coarse_loss(out["logits_coarse"], yc)
                    lb = bin_loss(out["logits_bin"], yb)
                    cons_c, cons_b = hierarchical_consistency_loss(
                        out["logits_fine"], out["logits_coarse"], out["logits_bin"]
                    )
                    loss = (
                        lf
                        + cfg.W_COARSE * lc
                        + cfg.W_BIN * lb
                        + cfg.W_CONS_COARSE * cons_c
                        + cfg.W_CONS_BIN * cons_b
                    )
                else:
                    loss = lf

            if use_amp_local:
                scaler_local.scale(loss).backward()
                scaler_local.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(model_local.parameters(), FAST_CFG["QUICK_GRAD_CLIP"])
                scaler_local.step(opt)
                scaler_local.update()
            else:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model_local.parameters(), FAST_CFG["QUICK_GRAD_CLIP"])
                opt.step()

            total_loss += float(loss.item()) * int(xb.size(0))
            n_seen += int(xb.size(0))

        train_loss = total_loss / max(n_seen, 1)

        val_eval = eval_model_on_loader(model_local, va_loader)
        val_macro_f1 = val_eval["fine"]["macro_f1"]

        hist_rows.append({
            "epoch": epoch,
            "train_loss": train_loss,
            "val_fine_macro_f1": val_macro_f1,
            "val_fine_accuracy": val_eval["fine"]["accuracy"],
            "val_coarse_macro_f1": val_eval["coarse"]["macro_f1"],
            "val_bin_f1": val_eval["binary"]["f1_malicious"],
        })

        if val_macro_f1 > best_val:
            best_val = val_macro_f1
            best_state = {k: v.detach().cpu().clone() for k, v in model_local.state_dict().items()}
            bad = 0
        else:
            bad += 1

        print(f"[seed={seed} | multitask={use_multitask}] epoch={epoch} train_loss={train_loss:.4f} val_macroF1={val_macro_f1:.4f}")

        if bad >= FAST_CFG["QUICK_PATIENCE"]:
            break

    if best_state is not None:
        model_local.load_state_dict(best_state, strict=True)

    test_eval = eval_model_on_loader(model_local, te_loader)

    history_df = pd.DataFrame(hist_rows)
    return {
        "seed": seed,
        "use_multitask": bool(use_multitask),
        "history": history_df,
        "test_fine": test_eval["fine"],
        "test_coarse": test_eval["coarse"],
        "test_binary": test_eval["binary"],
    }

# ============================================================
# A) 3-seed repeatability (fast subset version)
# ============================================================
repeat_rows = []
if FAST_CFG["RUN_REPEATABILITY"]:
    print("\n=== FAST 3-SEED REPEATABILITY (subset) ===")
    repeat_dir = FAST_PUB_DIR / "repeatability"
    repeat_dir.mkdir(parents=True, exist_ok=True)

    for seed in FAST_CFG["REPEAT_SEEDS"]:
        res = train_quick_model(seed=seed, use_multitask=True)

        res["history"].to_csv(repeat_dir / f"history_seed_{seed}.csv", index=False)

        repeat_rows.append({
            "seed": seed,
            "fine_accuracy": res["test_fine"]["accuracy"],
            "fine_macro_f1": res["test_fine"]["macro_f1"],
            "coarse_accuracy": res["test_coarse"]["accuracy"],
            "coarse_macro_f1": res["test_coarse"]["macro_f1"],
            "bin_f1_malicious": res["test_binary"]["f1_malicious"],
        })

        gc.collect()
        if DEVICE == "cuda":
            torch.cuda.empty_cache()

    repeat_df = pd.DataFrame(repeat_rows)
    repeat_summary = {
        "fine_accuracy_mean": float(repeat_df["fine_accuracy"].mean()),
        "fine_accuracy_std": float(repeat_df["fine_accuracy"].std(ddof=1)) if len(repeat_df) > 1 else 0.0,
        "fine_macro_f1_mean": float(repeat_df["fine_macro_f1"].mean()),
        "fine_macro_f1_std": float(repeat_df["fine_macro_f1"].std(ddof=1)) if len(repeat_df) > 1 else 0.0,
        "coarse_accuracy_mean": float(repeat_df["coarse_accuracy"].mean()),
        "coarse_accuracy_std": float(repeat_df["coarse_accuracy"].std(ddof=1)) if len(repeat_df) > 1 else 0.0,
        "coarse_macro_f1_mean": float(repeat_df["coarse_macro_f1"].mean()),
        "coarse_macro_f1_std": float(repeat_df["coarse_macro_f1"].std(ddof=1)) if len(repeat_df) > 1 else 0.0,
        "bin_f1_mean": float(repeat_df["bin_f1_malicious"].mean()),
        "bin_f1_std": float(repeat_df["bin_f1_malicious"].std(ddof=1)) if len(repeat_df) > 1 else 0.0,
        "note": "Fast repeatability on capped stratified subset, not full-data reruns.",
    }

    save_df(repeat_df, repeat_dir / "repeatability_results", index=False)
    save_json(repeat_dir / "repeatability_summary.json", repeat_summary)
    print("Repeatability summary:", repeat_summary)
else:
    repeat_df = None
    repeat_summary = None

# ============================================================
# B) Fast baseline comparisons
# ============================================================
baseline_rows = []
if FAST_CFG["RUN_BASELINES"]:
    print("\n=== FAST BASELINE COMPARISONS ===")
    base_dir = FAST_PUB_DIR / "baselines"
    base_dir.mkdir(parents=True, exist_ok=True)

    Xtr_b, ytr_f_b, ytr_c_b, ytr_b_b, _ = subset_arrays(
        X_train, y_train_f, y_train_c, y_train_b,
        FAST_CFG["BASELINE_TRAIN_CAP"], cfg.SEED
    )
    Xte_b, yte_f_b, yte_c_b, yte_b_b, _ = subset_arrays(
        X_test, y_test_f, y_test_c, y_test_b,
        FAST_CFG["BASELINE_TEST_CAP"], cfg.SEED
    )

    def eval_baseline_preds(model_name, y_pred_f):
        y_pred_f = np.asarray(y_pred_f).astype(np.int64)
        y_pred_c = fine_to_coarse[y_pred_f]
        y_pred_b = (y_pred_f != benign_fine_id).astype(np.int64)

        return {
            "model": model_name,
            "fine_accuracy": float(accuracy_score(yte_f_b, y_pred_f)),
            "fine_macro_f1": float(f1_score(yte_f_b, y_pred_f, average="macro")),
            "fine_weighted_f1": float(f1_score(yte_f_b, y_pred_f, average="weighted")),
            "coarse_accuracy": float(accuracy_score(yte_c_b, y_pred_c)),
            "coarse_macro_f1": float(f1_score(yte_c_b, y_pred_c, average="macro")),
            "bin_f1_malicious": float(f1_score(yte_b_b, y_pred_b, average="binary", pos_label=1)),
        }

    # 1) Logistic Regression
    try:
        lr = LogisticRegression(
            max_iter=120,
            solver="saga",
            multi_class="multinomial",
            n_jobs=-1,
            class_weight="balanced",
            verbose=0,
        )
        lr.fit(Xtr_b, ytr_f_b)
        pred = lr.predict(Xte_b)
        baseline_rows.append(eval_baseline_preds("LogisticRegression", pred))
        print("LogisticRegression done")
    except Exception as e:
        baseline_rows.append({"model": "LogisticRegression", "error": str(e)})
        print("LogisticRegression failed:", e)

    # 2) HistGradientBoosting
    try:
        hgb = HistGradientBoostingClassifier(
            max_iter=220,
            learning_rate=0.08,
            max_depth=10,
            random_state=cfg.SEED,
        )
        hgb.fit(Xtr_b, ytr_f_b)
        pred = hgb.predict(Xte_b)
        baseline_rows.append(eval_baseline_preds("HistGradientBoosting", pred))
        print("HistGradientBoosting done")
    except Exception as e:
        baseline_rows.append({"model": "HistGradientBoosting", "error": str(e)})
        print("HistGradientBoosting failed:", e)

    # 3) XGBoost
    try:
        import xgboost as xgb
        xgbm = xgb.XGBClassifier(
            n_estimators=500,
            max_depth=8,
            learning_rate=0.08,
            subsample=0.85,
            colsample_bytree=0.85,
            tree_method="hist",
            n_jobs=-1,
            objective="multi:softmax",
            num_class=NUM_FINE,
            random_state=cfg.SEED,
        )
        xgbm.fit(Xtr_b, ytr_f_b)
        pred = xgbm.predict(Xte_b)
        baseline_rows.append(eval_baseline_preds("XGBoost", pred))
        print("XGBoost done")
    except Exception as e:
        baseline_rows.append({"model": "XGBoost", "error": str(e)})
        print("XGBoost failed:", e)

    # 4) CatBoost
    try:
        from catboost import CatBoostClassifier
        cb = CatBoostClassifier(
            loss_function="MultiClass",
            iterations=500,
            depth=8,
            learning_rate=0.08,
            auto_class_weights="Balanced",
            verbose=False,
            task_type="GPU" if DEVICE == "cuda" else "CPU",
        )
        cb.fit(Xtr_b, ytr_f_b)
        pred = cb.predict(Xte_b).reshape(-1).astype(int)
        baseline_rows.append(eval_baseline_preds("CatBoost", pred))
        print("CatBoost done")
    except Exception as e:
        baseline_rows.append({"model": "CatBoost", "error": str(e)})
        print("CatBoost failed:", e)

    baseline_df = pd.DataFrame(baseline_rows)
    save_df(baseline_df, base_dir / "baseline_comparison_fast", index=False)
else:
    baseline_df = None

# ============================================================
# C) Fast ablation study
#    1. zero-cost inference ablations (already-trained model)
#    2. quick multitask on/off subset retraining
# ============================================================
print("\n=== FAST ABLATION STUDY ===")
ablation_dir = FAST_PUB_DIR / "ablations"
ablation_dir.mkdir(parents=True, exist_ok=True)

ablation_rows = []

# zero-cost calibration ablation from existing logits
if "test_out" in globals():
    probs_f_raw = softmax_np(test_out["fine"])
    probs_c_raw = softmax_np(test_out["coarse"])
    probs_b_raw = softmax_np(test_out["bin"])

    probs_f_temp = softmax_np(test_out["fine"] / max(float(temp_f), 1e-6))
    probs_c_temp = softmax_np(test_out["coarse"] / max(float(temp_c), 1e-6))
    probs_b_temp = softmax_np(test_out["bin"] / max(float(temp_b), 1e-6))

    ablation_rows.append({
        "component": "calibration_off_top1",
        "fine_accuracy": accuracy_score(test_out["yf"], probs_f_raw.argmax(axis=1)),
        "fine_macro_f1": f1_score(test_out["yf"], probs_f_raw.argmax(axis=1), average="macro"),
        "coarse_accuracy": accuracy_score(test_out["yc"], probs_c_raw.argmax(axis=1)),
        "coarse_macro_f1": f1_score(test_out["yc"], probs_c_raw.argmax(axis=1), average="macro"),
        "bin_f1_malicious": f1_score(test_out["yb"], probs_b_raw.argmax(axis=1), average="binary", pos_label=1),
    })

    ablation_rows.append({
        "component": "calibration_on_top1",
        "fine_accuracy": accuracy_score(test_out["yf"], probs_f_temp.argmax(axis=1)),
        "fine_macro_f1": f1_score(test_out["yf"], probs_f_temp.argmax(axis=1), average="macro"),
        "coarse_accuracy": accuracy_score(test_out["yc"], probs_c_temp.argmax(axis=1)),
        "coarse_macro_f1": f1_score(test_out["yc"], probs_c_temp.argmax(axis=1), average="macro"),
        "bin_f1_malicious": f1_score(test_out["yb"], probs_b_temp.argmax(axis=1), average="binary", pos_label=1),
    })

# zero-cost conformal ablation
if "raps_calib" in globals() and "test_probs_f_cal" in globals() and "test_probs_c_cal" in globals():
    alpha0 = FAST_CFG["RAPS_ALPHA"]
    if str(alpha0) in raps_calib["q_by_pred_coarse"]:
        q_by = np.array(raps_calib["q_by_pred_coarse"][str(alpha0)], dtype=np.float32)
        g_hat = test_probs_c_cal.argmax(axis=1)
        q_row = q_by[g_hat]
        k_star = raps_predict_k(test_probs_f_cal, q_row, cfg.RAPS_KREG, cfg.RAPS_LAMBDA)
        single = (k_star == 1)
        y_pred = test_probs_f_cal.argmax(axis=1)

        if single.any():
            sing_acc = float((y_pred[single] == test_out["yf"][single]).mean())
            sing_mf1 = float(f1_score(test_out["yf"][single], y_pred[single], average="macro"))
        else:
            sing_acc = float("nan")
            sing_mf1 = float("nan")

        ablation_rows.append({
            "component": f"conformal_RAPS_alpha_{alpha0}",
            "singleton_coverage": float(single.mean()),
            "singleton_risk_1_minus_acc": float(1.0 - sing_acc),
            "singleton_macro_f1": sing_mf1,
            "avg_set_size": float(k_star.mean()),
        })

# quick multitask ablation
mt_rows = []
if FAST_CFG["RUN_MULTITASK_ABLATION"]:
    print("Running quick multitask ablation on capped subset...")
    for seed in FAST_CFG["ABLATION_SEEDS"]:
        res_mt_on = train_quick_model(seed=seed, use_multitask=True)
        res_mt_off = train_quick_model(seed=seed, use_multitask=False)

        mt_rows.append({
            "seed": seed,
            "variant": "multitask_on",
            "fine_accuracy": res_mt_on["test_fine"]["accuracy"],
            "fine_macro_f1": res_mt_on["test_fine"]["macro_f1"],
            "coarse_macro_f1": res_mt_on["test_coarse"]["macro_f1"],
            "bin_f1_malicious": res_mt_on["test_binary"]["f1_malicious"],
        })
        mt_rows.append({
            "seed": seed,
            "variant": "multitask_off",
            "fine_accuracy": res_mt_off["test_fine"]["accuracy"],
            "fine_macro_f1": res_mt_off["test_fine"]["macro_f1"],
            "coarse_macro_f1": res_mt_off["test_coarse"]["macro_f1"],
            "bin_f1_malicious": res_mt_off["test_binary"]["f1_malicious"],
        })

        gc.collect()
        if DEVICE == "cuda":
            torch.cuda.empty_cache()

mt_df = pd.DataFrame(mt_rows) if len(mt_rows) > 0 else pd.DataFrame()
ablation_df = pd.DataFrame(ablation_rows) if len(ablation_rows) > 0 else pd.DataFrame()

if len(ablation_df) > 0:
    save_df(ablation_df, ablation_dir / "inference_ablation_fast", index=False)

if len(mt_df) > 0:
    save_df(mt_df, ablation_dir / "multitask_ablation_fast", index=False)

# ============================================================
# D) External / cross-dataset validation
# ============================================================
external_summary = {"ran": False, "reason": "not_requested"}

def build_eval_model_for_external():
    # use current in-memory model from main run
    m = build_fresh_model_local()
    m.load_state_dict(model.state_dict(), strict=True)
    m.eval()
    return m

if FAST_CFG["RUN_EXTERNAL"] and FAST_CFG["EXT_DATA_ROOT"] is not None:
    print("\n=== EXTERNAL VALIDATION ===")
    ext_dir = FAST_PUB_DIR / "external_validation"
    ext_dir.mkdir(parents=True, exist_ok=True)

    try:
        ext_root = Path(FAST_CFG["EXT_DATA_ROOT"]).expanduser().resolve()
        ext_files = discover_csv_files(ext_root, FAST_CFG["EXT_MAX_FILES"])
        ext_dfs = []

        for p in ext_files:
            d = read_one_csv(p, cfg)
            d["__src_file"] = p.name
            ext_dfs.append(d)

        ext_df = pd.concat(ext_dfs, ignore_index=True)
        del ext_dfs
        gc.collect()

        ext_df = sanitize_frame(ext_df, cfg.CLIP_INF)
        ext_df = downcast_numeric(ext_df)

        ext_label_col = detect_col(ext_df.columns.tolist(), cfg.LABEL_COL_CANDIDATES)
        ext_time_col = detect_col(ext_df.columns.tolist(), cfg.TIME_COL_CANDIDATES)
        ext_group_col = detect_col(ext_df.columns.tolist(), cfg.GROUP_COL_CANDIDATES)

        if ext_label_col is None:
            raise ValueError("Could not detect label column in external dataset.")

        ext_df[ext_label_col] = ext_df[ext_label_col].astype(str).str.strip()
        known_mask = ext_df[ext_label_col].isin(DICT_34_CLASSES.keys())
        ext_df = ext_df.loc[known_mask].copy()

        ext_df["y_fine"] = ext_df[ext_label_col].map(DICT_34_CLASSES).astype(np.int64)
        ext_df["y_coarse"] = ext_df["y_fine"].map(lambda i: int(fine_to_coarse[i])).astype(np.int64)
        ext_df["y_bin"] = (ext_df["y_fine"] != benign_fine_id).astype(np.int64)

        # keep only training-time feature columns that exist
        ext_feature_cols = [c for c in FEATURE_COLS if c in ext_df.columns]
        if len(ext_feature_cols) != len(FEATURE_COLS):
            missing = sorted(set(FEATURE_COLS) - set(ext_feature_cols))
            print("External data missing feature columns:", missing[:20], "...")

        X_ext = transform_df(ext_df, ext_feature_cols, num_imp, scaler, meta, cfg)
        y_ext_f = ext_df["y_fine"].values.astype(np.int64)
        y_ext_c = ext_df["y_coarse"].values.astype(np.int64)
        y_ext_b = ext_df["y_bin"].values.astype(np.int64)

        ds_ext = FlowDataset(X_ext, y_ext_f, y_ext_c, y_ext_b)
        ext_loader = DataLoader(
            ds_ext,
            batch_size=FAST_CFG["QUICK_BATCH_SIZE"],
            shuffle=False,
            num_workers=0,
            pin_memory=(DEVICE == "cuda"),
        )

        ext_model = build_eval_model_for_external()
        ext_out = predict_logits(ext_model, ext_loader)

        ext_pf = softmax_np(ext_out["fine"] / max(float(temp_f), 1e-6))
        ext_pc = softmax_np(ext_out["coarse"] / max(float(temp_c), 1e-6))
        ext_pb = softmax_np(ext_out["bin"] / max(float(temp_b), 1e-6))

        external_summary = {
            "ran": True,
            "data_root": str(ext_root),
            "num_samples": int(len(y_ext_f)),
            "fine": quick_metrics_mc(ext_out["yf"], ext_pf),
            "coarse": quick_metrics_mc(ext_out["yc"], ext_pc),
            "binary": quick_metrics_bin(ext_out["yb"], ext_pb),
            "note": "External validation used training-time preprocessor without refit.",
        }

        save_json(ext_dir / "external_validation_summary.json", external_summary)
        print("External validation summary:", external_summary)

    except Exception as e:
        external_summary = {"ran": False, "reason": str(e)}
        print("External validation skipped/failed:", e)

# ============================================================
# E) Final combined summary
# ============================================================
final_fast_summary = {
    "config": FAST_CFG,
    "repeatability_summary": repeat_summary,
    "baseline_results_available": None if baseline_df is None else baseline_df.to_dict(orient="records"),
    "inference_ablation_available": None if len(ablation_df) == 0 else ablation_df.to_dict(orient="records"),
    "multitask_ablation_available": None if len(mt_df) == 0 else mt_df.to_dict(orient="records"),
    "external_validation": external_summary,
    "notes": [
        "This package is the fast publication version.",
        "3-seed repeatability and multitask ablation are done on capped stratified subsets, not full-data reruns.",
        "Calibration and conformal ablations reuse cached logits and are effectively zero-cost.",
        "External validation only runs if a second dataset folder is provided.",
    ],
}

save_json(FAST_PUB_DIR / "fast_extra_publication_summary.json", final_fast_summary)

md_text = f"""
# Fast Extra Publication Results

## What this adds
- 3-seed repeatability (fast subset)
- 2–4 strong tabular baselines
- calibration/conformal ablations
- quick multitask ablation
- optional external validation

## Important note
These are **fast publication extras** designed to avoid multi-day runs.
The repeatability and multitask ablation results are on capped stratified subsets.

## Files
- Summary JSON: `{FAST_PUB_DIR / "fast_extra_publication_summary.json"}`
- Repeatability dir: `{FAST_PUB_DIR / "repeatability"}`
- Baselines dir: `{FAST_PUB_DIR / "baselines"}`
- Ablations dir: `{FAST_PUB_DIR / "ablations"}`
- External validation dir: `{FAST_PUB_DIR / "external_validation"}`
"""
(FAST_PUB_DIR / "README_fast_extra_publication.md").write_text(md_text, encoding="utf-8")

print("\nSaved:", FAST_PUB_DIR / "fast_extra_publication_summary.json")
print("All fast extra publication outputs are in:", FAST_PUB_DIR)

FAST publication dir: results_camelot_ids_v2_pc\fast_extra_publication
{
  "TRAIN_CAP": 250000,
  "VAL_CAP": 60000,
  "TEST_CAP": 100000,
  "REPEAT_SEEDS": [
    11,
    42,
    123
  ],
  "QUICK_EPOCHS": 8,
  "QUICK_PATIENCE": 3,
  "QUICK_BATCH_SIZE": 2048,
  "QUICK_LR": 0.001,
  "QUICK_WD": 0.005,
  "QUICK_GRAD_CLIP": 1.0,
  "BASELINE_TRAIN_CAP": 200000,
  "BASELINE_TEST_CAP": 100000,
  "RUN_REPEATABILITY": true,
  "RUN_BASELINES": true,
  "RUN_MULTITASK_ABLATION": true,
  "ABLATION_SEEDS": [
    42
  ],
  "RUN_EXTERNAL": false,
  "EXT_DATA_ROOT": null,
  "EXT_MAX_FILES": 25,
  "RAPS_ALPHA": 0.1
}

=== FAST 3-SEED REPEATABILITY (subset) ===
[seed=11 | multitask=True] epoch=1 train_loss=4.4975 val_macroF1=0.0014
[seed=11 | multitask=True] epoch=2 train_loss=4.4333 val_macroF1=0.0000
[seed=11 | multitask=True] epoch=3 train_loss=4.4218 val_macroF1=0.0000
[seed=11 | multitask=True] epoch=4 train_loss=4.3809 val_macroF1=0.0001
[seed=42 | multitask=True] epoch=1 train_loss=4.5045 val_macr

In [9]:
print("RUN_DIR exists:", "RUN_DIR" in globals())
print("X_train exists:", "X_train" in globals())
print("FlowDataset exists:", "FlowDataset" in globals())
print("CamelotIDSv2 exists:", "CamelotIDSv2" in globals())
print("predict_logits exists:", "predict_logits" in globals())
print("softmax_np exists:", "softmax_np" in globals())
print("set_seed exists:", "set_seed" in globals())
print("hierarchical_consistency_loss exists:", "hierarchical_consistency_loss" in globals())

RUN_DIR exists: True
X_train exists: True
FlowDataset exists: True
CamelotIDSv2 exists: True
predict_logits exists: True
softmax_np exists: True
set_seed exists: True
hierarchical_consistency_loss exists: True


In [2]:
print('[P00] START: paper-completion runtime, logging, and atomic resume layer', flush=True)

import os
import sys
import gc
import re
import json
import math
import time
import copy
import random
import shutil
import hashlib
import inspect
import platform
import traceback
import subprocess
import importlib.util
from pathlib import Path
from dataclasses import dataclass, asdict, field
from contextlib import contextmanager, nullcontext
from typing import Any, Dict, Iterable, Iterator, List, Mapping, MutableMapping, Optional, Sequence, Tuple, Callable

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

try:
    from IPython.display import display, Markdown
except Exception:
    display = print
    Markdown = str


def _env_flag(name: str, default: bool = False) -> bool:
    raw = os.environ.get(name)
    if raw is None:
        return bool(default)
    return str(raw).strip().lower() in {'1', 'true', 'yes', 'y', 'on'}


def _safe_int(value: Any, default: int) -> int:
    try:
        return int(value)
    except Exception:
        return int(default)


def _safe_float(value: Any, default: float) -> float:
    try:
        return float(value)
    except Exception:
        return float(default)


@dataclass
class PaperRunConfig:
    # Main profile: strict | accelerated | smoke
    PROFILE: str = field(default_factory=lambda: os.environ.get('CAMELOT_PROFILE', 'accelerated').strip().lower())
    RUN_ALL: bool = field(default_factory=lambda: _env_flag('CAMELOT_RUN_ALL', True))
    FORCE_RERUN: bool = field(default_factory=lambda: _env_flag('CAMELOT_FORCE_RERUN', False))

    # One invocation is deliberately time-bounded. It can be restarted indefinitely.
    TIME_BUDGET_HOURS: float = field(default_factory=lambda: _safe_float(os.environ.get('CAMELOT_TIME_BUDGET_HOURS', 96), 96.0))
    CHECK_DEADLINE_EVERY_STEPS: int = 250

    # Paper-aligned training settings.
    EPOCHS: int = 70
    BATCH_SIZE: int = 1024
    WARMUP_EPOCHS: float = 2.0
    EARLY_STOPPING_PATIENCE: int = 7
    SEEDS: Tuple[int, ...] = (42, 123, 456, 789, 1011)
    CONFIRMATORY_SEEDS: Tuple[int, ...] = (42, 123, 456)
    VALIDATE_EVERY_EPOCHS: int = 1
    VALIDATION_BATCH_SIZE: int = 16384
    INFERENCE_BATCH_SIZE: int = 16384
    LOG_EVERY_STEPS: int = 500
    # Checkpoint inside long epochs so an unexpected shutdown loses at most a
    # bounded number of optimizer updates. The checkpoint is atomic and stores
    # model/optimizer/scheduler/scaler/EMA/RNG state.
    INTRA_EPOCH_CHECKPOINT_STEPS: int = 500
    EMA_DECAY: float = 0.999

    # Execution controls. Strict mode uses the full paper protocol. Accelerated
    # mode keeps the architecture, losses, label hierarchy, and batch size but
    # uses a deterministic class-stratified training/validation subset and a
    # shorter epoch budget so the full experiment graph can finish on one RTX
    # 3060 in days rather than weeks. Accelerated outputs are prominently tagged
    # as non-equivalent to strict manuscript results.
    USE_AMP: bool = True
    USE_TF32: bool = True
    USE_FUSED_ADAMW: bool = True
    USE_TORCH_COMPILE: bool = field(default_factory=lambda: _env_flag('CAMELOT_TORCH_COMPILE', False))
    TORCH_COMPILE_MODE: str = 'reduce-overhead'
    GPU_RESIDENT_TRAIN: str = 'auto'  # auto | yes | no
    GPU_RESIDENT_FRACTION: float = 0.58
    CPU_THREADS: int = field(default_factory=lambda: max(1, min(32, (os.cpu_count() or 4) // 2)))
    TREE_THREADS: int = field(default_factory=lambda: max(1, min(32, (os.cpu_count() or 4) - 2)))

    # Stage policy.
    AUTO_RUN_LIGHT: bool = True
    RUN_GROUPS: Tuple[str, ...] = ('light', 'medium', 'long')
    RUN_TREE_HYPERPARAMETER_SEARCH: bool = False
    RUN_OPTIONAL_CALIBRATORS: bool = False
    RUN_EXPLORATORY_SYNTHETIC_SHIFT_TESTS: bool = False
    RUN_EXTERNAL_DATASETS: bool = field(default_factory=lambda: _env_flag('CAMELOT_RUN_EXTERNAL', False))

    # Statistical and uncertainty settings.
    ALPHAS: Tuple[float, ...] = (0.01, 0.05, 0.10)
    RAPS_KREG: int = 3
    RAPS_LAMBDA: float = 0.01
    RAPS_MIN_GROUP: int = 200
    ECE_BINS: int = 15
    CLUSTER_BOOTSTRAP_REPS: int = 5000
    PAIRED_SEED_CORRELATION: float = 0.6

    # Strict protocol artifacts. Missing artifacts produce BLOCKED/PENDING output,
    # never invented data.
    STRICT_REQUIRE_PROTOCOL_ARTIFACTS: bool = True
    PROVENANCE_MANIFEST: str = field(default_factory=lambda: os.environ.get('CAMELOT_PROVENANCE_MANIFEST', ''))
    TRAIN_CLUSTER_IDS: str = field(default_factory=lambda: os.environ.get('CAMELOT_TRAIN_CLUSTER_IDS', ''))
    TEST_CLUSTER_IDS: str = field(default_factory=lambda: os.environ.get('CAMELOT_TEST_CLUSTER_IDS', ''))
    FEATURE_MAP_DIR: str = field(default_factory=lambda: os.environ.get('CAMELOT_FEATURE_MAP_DIR', ''))
    ROBUSTNESS_ARTIFACT_DIR: str = field(default_factory=lambda: os.environ.get('CAMELOT_ROBUSTNESS_ARTIFACT_DIR', ''))

    # External datasets. No automatic download is performed.
    CIC_IDS2017_ROOT: str = field(default_factory=lambda: os.environ.get('CIC_IDS2017_ROOT', ''))
    CSE_CIC_IDS2018_ROOT: str = field(default_factory=lambda: os.environ.get('CSE_CIC_IDS2018_ROOT', ''))
    TON_IOT_ROOT: str = field(default_factory=lambda: os.environ.get('TON_IOT_ROOT', ''))
    UNSW_NB15_ROOT: str = field(default_factory=lambda: os.environ.get('UNSW_NB15_ROOT', ''))

    # Accelerated mode is the practical default for the stated workstation.
    # Strict mode ignores these caps and uses all rows for 70 epochs.
    ACCEL_TRAIN_CAP: int = 1_000_000
    ACCEL_VAL_CAP: int = 200_000
    ACCEL_CAL_CAP: int = 500_000
    ACCEL_TEST_CAP: int = 1_000_000
    ACCEL_EPOCHS: int = 12

    # Smoke mode only; these are never used in strict/accelerated manuscript tables.
    SMOKE_TRAIN_CAP: int = 250_000
    SMOKE_VAL_CAP: int = 60_000
    SMOKE_CAL_CAP: int = 60_000
    SMOKE_TEST_CAP: int = 100_000
    SMOKE_EPOCHS: int = 3

    # Table tolerance is for audit display only, never for forcing a pass.
    AUDIT_ABS_TOL_PERCENTAGE_POINTS: float = 0.50

    def normalize(self) -> 'PaperRunConfig':
        self.PROFILE = str(self.PROFILE).strip().lower()
        if self.PROFILE not in {'strict', 'accelerated', 'smoke'}:
            raise ValueError(f'Unknown PROFILE={self.PROFILE!r}; use strict, accelerated, or smoke.')
        if self.PROFILE == 'strict':
            self.USE_TORCH_COMPILE = False
            self.GPU_RESIDENT_TRAIN = 'no'  # closest to the archived loader path
            self.VALIDATE_EVERY_EPOCHS = 1
        elif self.PROFILE == 'accelerated':
            self.EPOCHS = int(self.ACCEL_EPOCHS)
            self.EARLY_STOPPING_PATIENCE = min(int(self.EARLY_STOPPING_PATIENCE), 4)
            self.VALIDATE_EVERY_EPOCHS = 1
        elif self.PROFILE == 'smoke':
            self.EPOCHS = int(self.SMOKE_EPOCHS)
            self.BATCH_SIZE = min(int(self.BATCH_SIZE), 1024)
            self.VALIDATE_EVERY_EPOCHS = 1
        return self


PAPER = PaperRunConfig().normalize()

_base_out = Path(getattr(globals().get('cfg', None), 'OUT_DIR', './results_camelot_ids_v2_pc'))
PAPER_DIR = _base_out / 'paper_complete_v4_merged'
PAPER_STAGE_DIR = PAPER_DIR / 'stages'
PAPER_LOG_DIR = PAPER_DIR / 'logs'
PAPER_TABLE_DIR = PAPER_DIR / 'tables'
PAPER_FIG_DIR = PAPER_DIR / 'figures'
PAPER_CACHE_DIR = PAPER_DIR / 'cache'
PAPER_MODEL_DIR = PAPER_DIR / 'models'
PAPER_REPORT_DIR = PAPER_DIR / 'reports'
for _d in [PAPER_DIR, PAPER_STAGE_DIR, PAPER_LOG_DIR, PAPER_TABLE_DIR, PAPER_FIG_DIR,
           PAPER_CACHE_DIR, PAPER_MODEL_DIR, PAPER_REPORT_DIR]:
    _d.mkdir(parents=True, exist_ok=True)

PAPER_EVENT_LOG = PAPER_LOG_DIR / 'events.jsonl'
PAPER_CELL_LOG = PAPER_LOG_DIR / 'cells.jsonl'
_PAPER_DEADLINE: Optional[float] = None
_PAPER_PIPELINE_STARTED_AT: Optional[float] = None


def _jsonable(value: Any) -> Any:
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating,)):
        return float(value)
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, torch.Tensor):
        return value.detach().cpu().tolist()
    if isinstance(value, set):
        return sorted(value)
    if hasattr(value, '__dict__') and not isinstance(value, type):
        try:
            return {k: _jsonable(v) for k, v in vars(value).items()}
        except Exception:
            return str(value)
    return value


def stable_json(obj: Any) -> str:
    return json.dumps(obj, sort_keys=True, separators=(',', ':'), default=_jsonable)


def signature_hash(payload: Any) -> str:
    return hashlib.sha256(stable_json(payload).encode('utf-8')).hexdigest()


def atomic_write_text(path: Path, text: str, encoding: str = 'utf-8') -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + '.tmp')
    with open(tmp, 'w', encoding=encoding, newline='') as handle:
        handle.write(text)
        handle.flush()
        try:
            os.fsync(handle.fileno())
        except OSError:
            pass
    os.replace(tmp, path)


def atomic_json(path: Path, payload: Any) -> None:
    atomic_write_text(Path(path), json.dumps(payload, indent=2, default=_jsonable, sort_keys=True))


def atomic_npy(path: Path, array: np.ndarray) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + '.tmp')
    with open(tmp, 'wb') as handle:
        np.save(handle, np.asarray(array), allow_pickle=False)
        handle.flush()
        try:
            os.fsync(handle.fileno())
        except OSError:
            pass
    os.replace(tmp, path)


def atomic_torch(path: Path, payload: Any) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + '.tmp')
    torch.save(payload, tmp)
    os.replace(tmp, path)


def atomic_dataframe(df: pd.DataFrame, stem: Path, index: bool = False) -> Dict[str, str]:
    stem = Path(stem)
    stem.parent.mkdir(parents=True, exist_ok=True)
    outputs: Dict[str, str] = {}
    csv_path = stem.with_suffix('.csv')
    tmp_csv = csv_path.with_name(csv_path.name + '.tmp')
    df.to_csv(tmp_csv, index=index)
    os.replace(tmp_csv, csv_path)
    outputs['csv'] = str(csv_path)
    try:
        xlsx_path = stem.with_suffix('.xlsx')
        tmp_xlsx = xlsx_path.with_name(xlsx_path.stem + '.tmp.xlsx')
        df.to_excel(tmp_xlsx, index=index)
        os.replace(tmp_xlsx, xlsx_path)
        outputs['xlsx'] = str(xlsx_path)
    except Exception as exc:
        outputs['xlsx_error'] = str(exc)
    return outputs


def read_json(path: Path, default: Any = None) -> Any:
    path = Path(path)
    if not path.exists():
        return {} if default is None else default
    try:
        return json.loads(path.read_text(encoding='utf-8'))
    except Exception:
        return {} if default is None else default


def append_jsonl(path: Path, record: Mapping[str, Any]) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    line = json.dumps(dict(record), default=_jsonable, sort_keys=True)
    with open(path, 'a', encoding='utf-8') as handle:
        handle.write(line + '\n')
        handle.flush()
        try:
            os.fsync(handle.fileno())
        except OSError:
            pass


def paper_log(message: str, *, level: str = 'INFO', stage: Optional[str] = None, **fields: Any) -> None:
    now = time.strftime('%Y-%m-%d %H:%M:%S')
    record = {'time': now, 'level': level, 'stage': stage, 'message': message, **fields}
    append_jsonl(PAPER_EVENT_LOG, record)
    prefix = f'[{now}] [{level}]'
    if stage:
        prefix += f' [{stage}]'
    extras = ' '.join(f'{key}={value}' for key, value in fields.items())
    print(f'{prefix} {message}' + (f' | {extras}' if extras else ''), flush=True)


def paper_cell_start(cell_id: str, title: str) -> float:
    started = time.time()
    rec = {'time': time.strftime('%Y-%m-%d %H:%M:%S'), 'cell': cell_id, 'event': 'START', 'title': title}
    append_jsonl(PAPER_CELL_LOG, rec)
    print(f'[{cell_id}] START: {title}', flush=True)
    return started


def paper_cell_done(cell_id: str, title: str, started: float) -> None:
    elapsed = time.time() - started
    rec = {'time': time.strftime('%Y-%m-%d %H:%M:%S'), 'cell': cell_id, 'event': 'DONE',
           'title': title, 'elapsed_seconds': elapsed}
    append_jsonl(PAPER_CELL_LOG, rec)
    print(f'[{cell_id}] DONE: {title} | elapsed={elapsed:.2f}s', flush=True)


class TimeBudgetReached(RuntimeError):
    pass


def set_pipeline_deadline(hours: Optional[float] = None) -> Optional[float]:
    global _PAPER_DEADLINE, _PAPER_PIPELINE_STARTED_AT
    _PAPER_PIPELINE_STARTED_AT = time.time()
    use_hours = PAPER.TIME_BUDGET_HOURS if hours is None else float(hours)
    _PAPER_DEADLINE = None if use_hours <= 0 else (_PAPER_PIPELINE_STARTED_AT + 3600.0 * use_hours)
    atomic_json(PAPER_DIR / 'active_deadline.json', {
        'started_at': time.strftime('%Y-%m-%d %H:%M:%S'),
        'budget_hours': use_hours,
        'deadline_epoch': _PAPER_DEADLINE,
    })
    return _PAPER_DEADLINE


def remaining_budget_seconds() -> float:
    if _PAPER_DEADLINE is None:
        return float('inf')
    return max(0.0, _PAPER_DEADLINE - time.time())


def check_deadline(where: str = '') -> None:
    if _PAPER_DEADLINE is not None and time.time() >= _PAPER_DEADLINE:
        raise TimeBudgetReached(f'Time budget reached at {where or "stage boundary"}. Checkpoints are current.')


def stage_path(name: str) -> Path:
    safe = re.sub(r'[^A-Za-z0-9_.-]+', '_', str(name)).strip('_')
    return PAPER_STAGE_DIR / safe


def stage_status(name: str) -> Dict[str, Any]:
    return read_json(stage_path(name) / 'status.json', {})


def mark_stage(name: str, status: str, **payload: Any) -> None:
    root = stage_path(name)
    root.mkdir(parents=True, exist_ok=True)
    previous = read_json(root / 'status.json', {})
    attempts = int(previous.get('attempts', 0))
    if status == 'RUNNING':
        attempts += 1
    data = {
        **previous,
        **payload,
        'stage': name,
        'status': status,
        'attempts': attempts,
        'updated_at': time.strftime('%Y-%m-%d %H:%M:%S'),
    }
    atomic_json(root / 'status.json', data)


class PaperStage:
    def __init__(self, name: str, signature: Mapping[str, Any], force: Optional[bool] = None):
        self.name = name
        self.signature_payload = dict(signature)
        self.signature = signature_hash(self.signature_payload)
        self.force = PAPER.FORCE_RERUN if force is None else bool(force)
        self.skip = False
        self.started = 0.0
        self.root = stage_path(name)

    def __enter__(self) -> 'PaperStage':
        check_deadline(f'before {self.name}')
        self.root.mkdir(parents=True, exist_ok=True)
        prior = stage_status(self.name)
        if (not self.force and prior.get('status') == 'DONE' and prior.get('signature') == self.signature):
            self.skip = True
            paper_log('completed stage reused', stage=self.name, signature=self.signature[:12])
            return self
        self.started = time.time()
        mark_stage(self.name, 'RUNNING', signature=self.signature,
                   signature_payload=self.signature_payload,
                   started_at=time.strftime('%Y-%m-%d %H:%M:%S'))
        paper_log('stage started', stage=self.name, signature=self.signature[:12])
        return self

    def __exit__(self, exc_type, exc, tb) -> bool:
        if self.skip:
            return False
        elapsed = time.time() - self.started
        if exc is None:
            mark_stage(self.name, 'DONE', signature=self.signature,
                       elapsed_seconds=elapsed,
                       completed_at=time.strftime('%Y-%m-%d %H:%M:%S'))
            paper_log('stage completed', stage=self.name, elapsed_seconds=round(elapsed, 3))
            return False
        if isinstance(exc, TimeBudgetReached):
            mark_stage(self.name, 'PAUSED', signature=self.signature,
                       elapsed_seconds=elapsed, reason=str(exc))
            paper_log('stage paused at a safe checkpoint', level='WARNING', stage=self.name,
                      elapsed_seconds=round(elapsed, 3), reason=str(exc))
            return False
        mark_stage(self.name, 'FAILED', signature=self.signature,
                   elapsed_seconds=elapsed, error=repr(exc), traceback=''.join(traceback.format_exception(exc_type, exc, tb)))
        paper_log('stage failed', level='ERROR', stage=self.name, error=repr(exc))
        return False


def paper_stage(name: str, signature: Mapping[str, Any], force: Optional[bool] = None) -> PaperStage:
    return PaperStage(name=name, signature=signature, force=force)


def seed_everything(seed: int, deterministic: Optional[bool] = None) -> None:
    det = (PAPER.PROFILE == 'strict') if deterministic is None else bool(deterministic)
    os.environ['PYTHONHASHSEED'] = str(int(seed))
    random.seed(int(seed))
    np.random.seed(int(seed))
    torch.manual_seed(int(seed))
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(int(seed))
        torch.backends.cudnn.deterministic = det
        torch.backends.cudnn.benchmark = not det
    try:
        torch.use_deterministic_algorithms(det, warn_only=True)
    except Exception:
        pass


def get_rng_bundle(extra_generator: Optional[torch.Generator] = None) -> Dict[str, Any]:
    state: Dict[str, Any] = {
        'python': random.getstate(),
        'numpy': np.random.get_state(),
        'torch_cpu': torch.get_rng_state(),
    }
    if torch.cuda.is_available():
        state['torch_cuda'] = torch.cuda.get_rng_state_all()
    if extra_generator is not None:
        state['shuffle_generator'] = extra_generator.get_state()
    return state


def set_rng_bundle(state: Optional[Mapping[str, Any]], extra_generator: Optional[torch.Generator] = None) -> None:
    if not state:
        return
    if 'python' in state:
        random.setstate(state['python'])
    if 'numpy' in state:
        np.random.set_state(state['numpy'])
    if 'torch_cpu' in state:
        torch.set_rng_state(state['torch_cpu'].detach().cpu().to(torch.uint8))
    if torch.cuda.is_available() and 'torch_cuda' in state:
        torch.cuda.set_rng_state_all([s.detach().cpu().to(torch.uint8) for s in state['torch_cuda']])
    if extra_generator is not None and 'shuffle_generator' in state:
        extra_generator.set_state(state['shuffle_generator'].detach().cpu().to(torch.uint8))


def model_state_cpu(model: nn.Module) -> Dict[str, torch.Tensor]:
    return {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}


def optimizer_to_device(optimizer: torch.optim.Optimizer, device: torch.device) -> None:
    for state in optimizer.state.values():
        for key, value in list(state.items()):
            if torch.is_tensor(value):
                state[key] = value.to(device)


def make_grad_scaler(enabled: bool):
    if not enabled:
        return None
    try:
        return torch.amp.GradScaler('cuda', enabled=True)
    except Exception:
        return torch.cuda.amp.GradScaler(enabled=True)


@contextmanager
def amp_autocast(enabled: bool):
    if not enabled:
        yield
        return
    try:
        with torch.amp.autocast(device_type='cuda', dtype=torch.float16, enabled=True):
            yield
    except Exception:
        with torch.cuda.amp.autocast(dtype=torch.float16, enabled=True):
            yield


def configure_runtime() -> Dict[str, Any]:
    os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
    os.environ.setdefault('OMP_NUM_THREADS', str(PAPER.CPU_THREADS))
    os.environ.setdefault('MKL_NUM_THREADS', str(PAPER.CPU_THREADS))
    try:
        torch.set_num_threads(int(PAPER.CPU_THREADS))
        torch.set_num_interop_threads(max(1, min(4, int(PAPER.CPU_THREADS) // 4)))
    except RuntimeError:
        pass
    if torch.cuda.is_available():
        torch.backends.cudnn.benchmark = (PAPER.PROFILE != 'strict')
        if PAPER.USE_TF32:
            try:
                torch.backends.cuda.matmul.fp32_precision = 'tf32'
                torch.backends.cudnn.fp32_precision = 'tf32'
            except Exception:
                try:
                    torch.backends.cuda.matmul.allow_tf32 = True
                    torch.backends.cudnn.allow_tf32 = True
                except Exception:
                    pass
            try:
                torch.set_float32_matmul_precision('high')
            except Exception:
                pass
        for fn_name in ('enable_flash_sdp', 'enable_mem_efficient_sdp'):
            fn = getattr(getattr(torch.backends, 'cuda', None), fn_name, None)
            if callable(fn):
                try:
                    fn(True)
                except Exception:
                    pass
    info: Dict[str, Any] = {
        'profile': PAPER.PROFILE,
        'python': sys.version,
        'platform': platform.platform(),
        'processor': platform.processor(),
        'cpu_logical_count': os.cpu_count(),
        'torch': torch.__version__,
        'cuda_available': torch.cuda.is_available(),
        'cuda_runtime': torch.version.cuda,
        'cudnn': torch.backends.cudnn.version() if torch.backends.cudnn.is_available() else None,
        'paper_config': asdict(PAPER),
    }
    if torch.cuda.is_available():
        props = torch.cuda.get_device_properties(0)
        info.update({
            'gpu_name': props.name,
            'gpu_total_gb': props.total_memory / (1024 ** 3),
            'gpu_capability': f'{props.major}.{props.minor}',
        })
    try:
        smi = subprocess.run(['nvidia-smi', '--query-gpu=name,driver_version,memory.total', '--format=csv,noheader'],
                             check=False, capture_output=True, text=True, timeout=15)
        if smi.stdout.strip():
            info['nvidia_smi'] = smi.stdout.strip()
    except Exception:
        pass
    atomic_json(PAPER_DIR / 'runtime_environment.json', info)
    return info


def optional_dependency_report() -> pd.DataFrame:
    modules = {
        'river': 'ADWIN stream monitor',
        'xgboost': 'XGBoost baseline',
        'lightgbm': 'LightGBM baseline',
        'imblearn': 'reference SMOTE utilities',
        'pytorch_tabnet': 'TabNet baseline',
        'pyarrow': 'Parquet/external-data acceleration',
        'openpyxl': 'XLSX table export',
        'fvcore': 'best-effort FLOP profiling',
        'psutil': 'enhanced memory reporting',
    }
    rows = []
    for module, purpose in modules.items():
        rows.append({'module': module, 'available': importlib.util.find_spec(module) is not None, 'purpose': purpose})
    report = pd.DataFrame(rows)
    atomic_dataframe(report, PAPER_REPORT_DIR / 'optional_dependencies')
    missing = report.loc[~report['available'], 'module'].tolist()
    if missing:
        paper_log('optional packages missing; stages that need them will be BLOCKED, not auto-installed',
                  level='WARNING', missing=missing)
    return report


runtime_info = configure_runtime()
seed_everything(42, deterministic=(PAPER.PROFILE == 'strict'))
atomic_json(PAPER_DIR / 'paper_run_config.json', asdict(PAPER))
dependency_report = optional_dependency_report()

paper_log('paper-completion runtime ready', output_dir=str(PAPER_DIR), profile=PAPER.PROFILE,
          run_all=PAPER.RUN_ALL, time_budget_hours=PAPER.TIME_BUDGET_HOURS)
display(dependency_report)
print('Output root:', PAPER_DIR)
print('[P00] DONE: paper-completion runtime, logging, and atomic resume layer', flush=True)


[P00] START: paper-completion runtime, logging, and atomic resume layer
[2026-08-28 09:53:30] [INFO] paper-completion runtime ready | output_dir=results_camelot_ids_v2_pc\paper_complete_v4_merged profile=accelerated run_all=True time_budget_hours=96.0


,module,available,purpose
0,river,True,ADWIN stream monitor
1,xgboost,True,XGBoost baseline
2,lightgbm,True,LightGBM baseline
3,imblearn,True,reference SMOTE utilities
4,pytorch_tabnet,True,TabNet baseline
5,pyarrow,True,Parquet/external-data acceleration
6,openpyxl,True,XLSX table export
7,fvcore,True,best-effort FLOP profiling
8,psutil,True,enhanced memory reporting


Output root: results_camelot_ids_v2_pc\paper_complete_v4_merged
[P00] DONE: paper-completion runtime, logging, and atomic resume layer


In [3]:
_started = paper_cell_start('P01', 'validate and restore the archived v4 execution context')

# The append-only section can restore arrays and the selected model from the v4
# cache after a kernel restart, provided the original v4 definition cell has
# been executed in the restarted kernel. It never rewrites an original v4 cell.

_REQUIRED_DEFINITIONS = [
    'cfg', 'DEVICE', 'CamelotIDSv2', 'FeatureTokenizer', 'make_encoder_layer',
    'FlowDataset', 'LDAMLoss', 'softmax_np', 'fit_temperature_np',
    'conformal_quantile', 'raps_scores_true', 'raps_predict_k',
    'FINE_CLASS_NAMES', 'COARSE_NAMES', 'fine_to_coarse', 'benign_fine_id',
    'FEATURE_COLS', 'GROUP_IDXS', 'GROUP_NAMES',
]
_missing_definitions = [name for name in _REQUIRED_DEFINITIONS if name not in globals()]
if _missing_definitions:
    raise RuntimeError(
        'The original v4 definition/execution cell must run once in this kernel before the appended section. '
        'Missing definitions: ' + ', '.join(_missing_definitions)
    )

_ARRAY_NAMES = [
    'X_train', 'y_train_f', 'y_train_c', 'y_train_b',
    'X_val', 'y_val_f', 'y_val_c', 'y_val_b',
    'X_cal', 'y_cal_f', 'y_cal_c', 'y_cal_b',
    'X_test', 'y_test_f', 'y_test_c', 'y_test_b',
]


def _restore_v4_arrays_if_needed() -> None:
    missing = [name for name in _ARRAY_NAMES if name not in globals()]
    if not missing:
        return
    cache_dir = Path(globals().get('ARRAY_CACHE_DIR', Path(cfg.OUT_DIR) / 'cache' / 'arrays'))
    unavailable = [name for name in missing if not (cache_dir / f'{name}.npy').exists()]
    if unavailable:
        raise RuntimeError(
            'v4 arrays are not in memory and their cache files are missing: ' + ', '.join(unavailable) +
            '. Re-run the original v4 cell; completed preprocessing/training stages will be reused.'
        )
    for name in missing:
        globals()[name] = np.load(cache_dir / f'{name}.npy', mmap_mode=None)
        paper_log('restored v4 array from cache', stage='context_restore', name=name,
                  shape=list(globals()[name].shape), dtype=str(globals()[name].dtype))


_restore_v4_arrays_if_needed()

NUM_FINE_PAPER = int(len(FINE_CLASS_NAMES))
NUM_COARSE_PAPER = int(len(COARSE_NAMES))
DEVICE_T = torch.device(DEVICE)


def build_fresh_camelot_model(group_idxs: Optional[Sequence[Sequence[int]]] = None,
                              n_features: Optional[int] = None,
                              n_fine: Optional[int] = None,
                              n_coarse: Optional[int] = None,
                              cfg_local: Optional[Any] = None) -> nn.Module:
    return CamelotIDSv2(
        n_features=int(X_train.shape[1] if n_features is None else n_features),
        n_fine=int(NUM_FINE_PAPER if n_fine is None else n_fine),
        n_coarse=int(NUM_COARSE_PAPER if n_coarse is None else n_coarse),
        group_idxs=[list(map(int, g)) for g in (GROUP_IDXS if group_idxs is None else group_idxs)],
        cfg=copy.deepcopy(cfg if cfg_local is None else cfg_local),
    )


def _restore_selected_model() -> nn.Module:
    if 'pub_model' in globals() and isinstance(globals()['pub_model'], nn.Module):
        return globals()['pub_model'].to(DEVICE_T).eval()
    if 'model' in globals() and isinstance(globals()['model'], nn.Module):
        return globals()['model'].to(DEVICE_T).eval()
    ckpt_candidates = [
        Path(cfg.OUT_DIR) / 'publication_package' / 'checkpoints' / 'publication_selected_model.pt',
        Path(cfg.OUT_DIR) / 'checkpoints' / 'best_model.pt',
    ]
    for path in ckpt_candidates:
        if not path.exists():
            continue
        blob = torch.load(path, map_location='cpu', weights_only=False)
        state = blob.get('model_state') or blob.get('state_dict')
        if state is None:
            continue
        restored = build_fresh_camelot_model()
        restored.load_state_dict(state, strict=True)
        paper_log('restored selected v4 model', stage='context_restore', checkpoint=str(path))
        return restored.to(DEVICE_T).eval()
    raise RuntimeError('Could not restore the selected v4 model. Re-run original v4 cell 1 or the publication bootstrap cell.')


PAPER_MODEL = _restore_selected_model()


def _load_logit_cache(cache_name: str) -> Optional[Dict[str, np.ndarray]]:
    folder = Path(cfg.OUT_DIR) / 'cache' / 'logits' / cache_name
    keys = ('fine', 'coarse', 'bin', 'yf', 'yc', 'yb')
    if not all((folder / f'{key}.npy').exists() for key in keys):
        return None
    return {key: np.load(folder / f'{key}.npy') for key in keys}


def _resolve_outputs(kind: str) -> Dict[str, np.ndarray]:
    pub_name = f'pub_{kind}_out'
    base_name = f'{kind}_out'
    if pub_name in globals() and isinstance(globals()[pub_name], Mapping):
        return {k: np.asarray(v) for k, v in globals()[pub_name].items()}
    if base_name in globals() and isinstance(globals()[base_name], Mapping):
        return {k: np.asarray(v) for k, v in globals()[base_name].items()}
    # Publication cache names include the chosen checkpoint name. Search the
    # metadata folders only when direct globals are unavailable.
    root = Path(cfg.OUT_DIR) / 'cache' / 'logits'
    candidates: List[Path] = []
    if root.exists():
        candidates.extend(sorted(root.glob(f'publication_{kind}_*')))
        candidates.append(root / kind)
    for folder in candidates:
        keys = ('fine', 'coarse', 'bin', 'yf', 'yc', 'yb')
        if folder.exists() and all((folder / f'{key}.npy').exists() for key in keys):
            paper_log('restored cached logits', stage='context_restore', split=kind, folder=str(folder))
            return {key: np.load(folder / f'{key}.npy') for key in keys}
    raise RuntimeError(f'No cached {kind} logits found. Re-run the original v4 main/publication cells.')


PAPER_TEST_OUT = _resolve_outputs('test')
PAPER_CAL_OUT = _resolve_outputs('cal')

_cal_summary = read_json(Path(cfg.OUT_DIR) / 'calibration_summary.json', {})
PAPER_TEMP_FINE = float(globals().get('pub_temp_f', globals().get('temp_f', _cal_summary.get('temperature', {}).get('fine', 1.0))))
PAPER_TEMP_COARSE = float(globals().get('pub_temp_c', globals().get('temp_c', _cal_summary.get('temperature', {}).get('coarse', 1.0))))
PAPER_TEMP_BINARY = float(globals().get('pub_temp_b', globals().get('temp_b', _cal_summary.get('temperature', {}).get('bin', 1.0))))

PAPER_CAL_PROBS_FINE = softmax_np(PAPER_CAL_OUT['fine'] / max(PAPER_TEMP_FINE, 1e-6))
PAPER_CAL_PROBS_COARSE = softmax_np(PAPER_CAL_OUT['coarse'] / max(PAPER_TEMP_COARSE, 1e-6))
PAPER_CAL_PROBS_BINARY = softmax_np(PAPER_CAL_OUT['bin'] / max(PAPER_TEMP_BINARY, 1e-6))
PAPER_TEST_PROBS_FINE = softmax_np(PAPER_TEST_OUT['fine'] / max(PAPER_TEMP_FINE, 1e-6))
PAPER_TEST_PROBS_COARSE = softmax_np(PAPER_TEST_OUT['coarse'] / max(PAPER_TEMP_COARSE, 1e-6))
PAPER_TEST_PROBS_BINARY = softmax_np(PAPER_TEST_OUT['bin'] / max(PAPER_TEMP_BINARY, 1e-6))

# Canonical labels come from the archived cached outputs to prevent accidental
# row-order mismatch with a separately loaded array.
PAPER_Y_CAL_FINE = np.asarray(PAPER_CAL_OUT['yf'], dtype=np.int64)
PAPER_Y_CAL_COARSE = np.asarray(PAPER_CAL_OUT['yc'], dtype=np.int64)
PAPER_Y_CAL_BINARY = np.asarray(PAPER_CAL_OUT['yb'], dtype=np.int64)
PAPER_Y_TEST_FINE = np.asarray(PAPER_TEST_OUT['yf'], dtype=np.int64)
PAPER_Y_TEST_COARSE = np.asarray(PAPER_TEST_OUT['yc'], dtype=np.int64)
PAPER_Y_TEST_BINARY = np.asarray(PAPER_TEST_OUT['yb'], dtype=np.int64)

for label, expected, actual in [
    ('calibration rows', len(X_cal), len(PAPER_Y_CAL_FINE)),
    ('test rows', len(X_test), len(PAPER_Y_TEST_FINE)),
]:
    if int(expected) != int(actual):
        raise RuntimeError(f'{label} mismatch: array={expected}, cached logits={actual}. Refuse to mix row orders.')


def _array_identity(name: str, arr: np.ndarray) -> Dict[str, Any]:
    a = np.asarray(arr)
    sample = a.reshape(-1)
    if sample.size:
        idx = np.linspace(0, sample.size - 1, min(32, sample.size), dtype=np.int64)
        digest = hashlib.sha256(np.ascontiguousarray(sample[idx]).tobytes()).hexdigest()
    else:
        digest = hashlib.sha256(b'').hexdigest()
    return {'name': name, 'shape': list(a.shape), 'dtype': str(a.dtype), 'sample_sha256': digest}


PAPER_BASE_SIGNATURE = {
    'profile': PAPER.PROFILE,
    'v4_out_dir': str(Path(cfg.OUT_DIR)),
    'feature_columns': list(map(str, FEATURE_COLS)),
    'group_names': list(map(str, GROUP_NAMES)),
    'group_indices': [list(map(int, g)) for g in GROUP_IDXS],
    'fine_classes': list(map(str, FINE_CLASS_NAMES)),
    'coarse_classes': list(map(str, COARSE_NAMES)),
    'temperatures': {'fine': PAPER_TEMP_FINE, 'coarse': PAPER_TEMP_COARSE, 'binary': PAPER_TEMP_BINARY},
    'arrays': [
        _array_identity('X_train', X_train),
        _array_identity('y_train_f', y_train_f),
        _array_identity('X_val', X_val),
        _array_identity('X_cal', X_cal),
        _array_identity('X_test', X_test),
    ],
}
PAPER_BASE_SIGNATURE_HASH = signature_hash(PAPER_BASE_SIGNATURE)
atomic_json(PAPER_DIR / 'v4_context_signature.json', {
    'signature': PAPER_BASE_SIGNATURE_HASH,
    'payload': PAPER_BASE_SIGNATURE,
})

PAPER_DATA = {
    'train': {'X': X_train, 'yf': y_train_f, 'yc': y_train_c, 'yb': y_train_b},
    'val': {'X': X_val, 'yf': y_val_f, 'yc': y_val_c, 'yb': y_val_b},
    'cal': {'X': X_cal, 'yf': y_cal_f, 'yc': y_cal_c, 'yb': y_cal_b},
    'test': {'X': X_test, 'yf': y_test_f, 'yc': y_test_c, 'yb': y_test_b},
}

context_summary = pd.DataFrame([
    {'partition': 'Train', 'rows': len(X_train), 'features': X_train.shape[1]},
    {'partition': 'Validation', 'rows': len(X_val), 'features': X_val.shape[1]},
    {'partition': 'Calibration', 'rows': len(X_cal), 'features': X_cal.shape[1]},
    {'partition': 'Test', 'rows': len(X_test), 'features': X_test.shape[1]},
])
atomic_dataframe(context_summary, PAPER_REPORT_DIR / 'context_summary')
display(context_summary)
paper_log('v4 context validated', stage='context_restore',
          signature=PAPER_BASE_SIGNATURE_HASH[:12],
          model_parameters=sum(p.numel() for p in PAPER_MODEL.parameters()),
          temperatures={'fine': PAPER_TEMP_FINE, 'coarse': PAPER_TEMP_COARSE, 'binary': PAPER_TEMP_BINARY})
paper_cell_done('P01', 'validate and restore the archived v4 execution context', _started)


[P01] START: validate and restore the archived v4 execution context


,partition,rows,features
0,Train,21705973,46
1,Validation,2631074,46
2,Calibration,1389497,46
3,Test,1372896,46


[2026-08-28 09:53:53] [INFO] [context_restore] v4 context validated | signature=da5bd32c8faa model_parameters=6358828 temperatures={'fine': 1.0, 'coarse': 1.0, 'binary': 1.0}
[P01] DONE: validate and restore the archived v4 execution context | elapsed=22.76s


## Current-manuscript alignment and evidence boundaries

The final manuscript contains a mixture of (a) exact values embedded in the executed v4 run, (b) confirmatory analyses that can be reproduced from cached v4 logits, and (c) supplementary values described as author-reported pending independent reproduction. The code below preserves those categories.

| Evidence class | Notebook behavior |
|---|---|
| Executed v4 evidence | Regenerated from the preserved v4 arrays, logits, checkpoints, history, and JSON outputs. |
| Confirmatory split conformal | Recomputed from disjoint probability-calibration and threshold-calibration subsets for seeds 42, 123, and 456. |
| Provenance/cluster analyses | Run only when a row-aligned capture manifest or cluster-ID vector is supplied. Missing identifiers produce `BLOCKED`, not flow-level substitutes. |
| Ablations, baselines, multi-seed, robustness, scalability, external datasets | Fully defined as independent resumable stages. Their measured tables are kept separate from manuscript reference targets until the stages actually finish. |
| Under-specified supplementary protocols | Require explicit mapping/index artifacts in strict mode. Accelerated exploratory fallbacks, where available, are labeled `EXPLORATORY` and cannot populate the paper table. |
| Synthetic shift or embedded-device projections | Optional diagnostic harness only; not treated as evidence for the current paper. |

This separation prevents the old v4.1/v4.2 assumptions from silently changing the final paper protocol.


In [4]:
_started = paper_cell_start('P02', 'paper experiment registry and manuscript-reference targets')

# The reference values below are transcribed from the supplied manuscript solely
# for side-by-side auditing. They are never substituted for measured outputs.
# Every generated result table includes a source_kind/status field.

PAPER_REFERENCE_ROWS: List[Dict[str, Any]] = [
    # Main tables.
    {'artifact': 'Table 2', 'metric': 'Train samples', 'value': 21_705_973, 'unit': 'flows'},
    {'artifact': 'Table 2', 'metric': 'Validation samples', 'value': 2_631_074, 'unit': 'flows'},
    {'artifact': 'Table 2', 'metric': 'Calibration samples', 'value': 1_389_497, 'unit': 'flows'},
    {'artifact': 'Table 2', 'metric': 'Test samples', 'value': 1_372_896, 'unit': 'flows'},
    {'artifact': 'Table 5', 'metric': 'Fine accuracy', 'value': 99.20, 'unit': '%'},
    {'artifact': 'Table 5', 'metric': 'Fine balanced accuracy', 'value': 76.56, 'unit': '%'},
    {'artifact': 'Table 5', 'metric': 'Fine macro-F1', 'value': 76.76, 'unit': '%'},
    {'artifact': 'Table 5', 'metric': 'Fine MCC', 'value': 0.9912, 'unit': 'fraction'},
    {'artifact': 'Table 5', 'metric': 'Coarse accuracy', 'value': 99.39, 'unit': '%'},
    {'artifact': 'Table 5', 'metric': 'Coarse balanced accuracy', 'value': 75.72, 'unit': '%'},
    {'artifact': 'Table 5', 'metric': 'Coarse macro-F1', 'value': 78.89, 'unit': '%'},
    {'artifact': 'Table 5', 'metric': 'Coarse MCC', 'value': 0.9861, 'unit': 'fraction'},
    {'artifact': 'Table 5', 'metric': 'Binary accuracy', 'value': 99.59, 'unit': '%'},
    {'artifact': 'Table 5', 'metric': 'Binary balanced accuracy', 'value': 96.16, 'unit': '%'},
    {'artifact': 'Table 5', 'metric': 'Malicious-class F1', 'value': 99.79, 'unit': '%'},
    {'artifact': 'Table 5', 'metric': 'Binary MCC', 'value': 0.9126, 'unit': 'fraction'},
    {'artifact': 'Table 6', 'metric': 'Uploading Attack F1', 'value': 4.96, 'unit': '%'},
    {'artifact': 'Table 6', 'metric': 'XSS F1', 'value': 12.36, 'unit': '%'},
    {'artifact': 'Table 6', 'metric': 'Recon Ping Sweep F1', 'value': 19.91, 'unit': '%'},
    {'artifact': 'Table 6', 'metric': 'Backdoor Malware F1', 'value': 24.37, 'unit': '%'},
    {'artifact': 'Table 6', 'metric': 'SQL Injection F1', 'value': 28.89, 'unit': '%'},
    {'artifact': 'Table 6', 'metric': 'Command Injection F1', 'value': 31.20, 'unit': '%'},
    {'artifact': 'Table 6', 'metric': 'Recon OS Scan F1', 'value': 38.27, 'unit': '%'},
    {'artifact': 'Table 6', 'metric': 'Dictionary Brute Force F1', 'value': 38.97, 'unit': '%'},
    {'artifact': 'Table 7', 'metric': 'alpha=.01 coverage', 'value': 99.99, 'unit': '%'},
    {'artifact': 'Table 7', 'metric': 'alpha=.01 mean set size', 'value': 3.64, 'unit': 'classes'},
    {'artifact': 'Table 7', 'metric': 'alpha=.05 coverage', 'value': 99.96, 'unit': '%'},
    {'artifact': 'Table 7', 'metric': 'alpha=.05 mean set size', 'value': 1.84, 'unit': 'classes'},
    {'artifact': 'Table 7', 'metric': 'alpha=.10 coverage', 'value': 99.95, 'unit': '%'},
    {'artifact': 'Table 7', 'metric': 'alpha=.10 mean set size', 'value': 1.60, 'unit': 'classes'},
    {'artifact': 'Table 7', 'metric': 'alpha=.10 singleton fraction', 'value': 57.31, 'unit': '%'},
    {'artifact': 'Table 7', 'metric': 'alpha=.10 singleton risk', 'value': 0.063, 'unit': '%'},
    {'artifact': 'Table 7', 'metric': 'alpha=.10 benign FPR', 'value': 0.184, 'unit': '%'},
    {'artifact': 'Table 8', 'metric': 'GPU throughput', 'value': 18_141.563, 'unit': 'samples/s'},
    {'artifact': 'Table 8', 'metric': 'CPU throughput', 'value': 1_062.328, 'unit': 'samples/s'},
    {'artifact': 'Table 9', 'metric': 'Test-stream flows', 'value': 1_372_896, 'unit': 'flows'},
    {'artifact': 'Table 9', 'metric': 'Logical batches', 'value': 28, 'unit': 'batches'},
    {'artifact': 'Table 9', 'metric': 'Combined drift points', 'value': 0, 'unit': 'events'},
    {'artifact': 'Table 9', 'metric': 'Adaptation events', 'value': 0, 'unit': 'events'},
    # Main ablation Table 10.
    {'artifact': 'Table 10', 'metric': 'Full fine macro-F1', 'value': 76.76, 'unit': '%'},
    {'artifact': 'Table 10', 'metric': 'Flat-46 fine macro-F1', 'value': 73.60, 'unit': '%'},
    {'artifact': 'Table 10', 'metric': 'Random grouping fine macro-F1', 'value': 74.90, 'unit': '%'},
    {'artifact': 'Table 10', 'metric': 'Fine-head only fine macro-F1', 'value': 75.40, 'unit': '%'},
    {'artifact': 'Table 10', 'metric': 'No-KL fine macro-F1', 'value': 76.00, 'unit': '%'},
    {'artifact': 'Table 10', 'metric': 'Weighted-CE fine macro-F1', 'value': 74.60, 'unit': '%'},
    # Supplementary exact/confirmatory statistics.
    {'artifact': 'Table S16', 'metric': 'Seed 42 alpha=.10 coverage', 'value': 90.12, 'unit': '%'},
    {'artifact': 'Table S16', 'metric': 'Seed 123 alpha=.10 coverage', 'value': 89.95, 'unit': '%'},
    {'artifact': 'Table S16', 'metric': 'Seed 456 alpha=.10 coverage', 'value': 90.08, 'unit': '%'},
    {'artifact': 'Table S16', 'metric': 'Mean alpha=.10 coverage', 'value': 90.05, 'unit': '%'},
    {'artifact': 'Table S16', 'metric': 'Mean alpha=.10 set size', 'value': 1.59, 'unit': 'classes'},
    {'artifact': 'Table S17', 'metric': 'CAMELOT fine macro-F1 CI low', 'value': 76.42, 'unit': '%'},
    {'artifact': 'Table S17', 'metric': 'CAMELOT fine macro-F1 CI high', 'value': 77.10, 'unit': '%'},
]

PAPER_REFERENCE = pd.DataFrame(PAPER_REFERENCE_ROWS)
PAPER_REFERENCE['source_kind'] = 'manuscript_reference_only'
atomic_dataframe(PAPER_REFERENCE, PAPER_REPORT_DIR / 'manuscript_reference_targets')

# COMPLETE coverage map. “static” means a literature/configuration/schematic
# artifact rather than a trainable experiment. “blocked-if-missing” means the
# exact experiment requires an external manifest, mapping, raw data, or package.
PAPER_COVERAGE_ROWS: List[Dict[str, Any]] = [
    {'artifact': 'Table 1', 'stage': 'reference_and_schematics', 'kind': 'static literature comparison', 'policy': 'export manuscript matrix; not an executable benchmark'},
    {'artifact': 'Table 2', 'stage': 'archived_core', 'kind': 'measured from v4 arrays', 'policy': 'automatic'},
    {'artifact': 'Table 3', 'stage': 'provenance_audit', 'kind': 'metadata audit', 'policy': 'blocked-if-missing manifest'},
    {'artifact': 'Figure 1', 'stage': 'archived_core', 'kind': 'dataset/split plot', 'policy': 'automatic'},
    {'artifact': 'Equations 1-11', 'stage': 'reference_and_schematics', 'kind': 'code-formula alignment', 'policy': 'documented in helper implementations'},
    {'artifact': 'Algorithm 1', 'stage': 'reference_and_schematics', 'kind': 'pipeline pseudocode', 'policy': 'exported from executable stage graph'},
    {'artifact': 'Figure 2', 'stage': 'reference_and_schematics', 'kind': 'pipeline schematic', 'policy': 'automatic'},
    {'artifact': 'Table 4', 'stage': 'archived_core', 'kind': 'configuration audit', 'policy': 'automatic'},
    {'artifact': 'Figure 3', 'stage': 'reference_and_schematics', 'kind': 'calibration protocol schematic', 'policy': 'automatic'},
    {'artifact': 'Table 5', 'stage': 'archived_core', 'kind': 'cached v4 evaluation', 'policy': 'automatic'},
    {'artifact': 'Figure 4', 'stage': 'archived_core', 'kind': 'performance plot', 'policy': 'automatic'},
    {'artifact': 'Table 6', 'stage': 'archived_core', 'kind': 'class-level evaluation', 'policy': 'automatic'},
    {'artifact': 'Table 7', 'stage': 'archived_core', 'kind': 'archived empirical RAPS', 'policy': 'automatic'},
    {'artifact': 'Table 8', 'stage': 'throughput_profile', 'kind': 'native PyTorch benchmark', 'policy': 'automatic; cacheable'},
    {'artifact': 'Table 9', 'stage': 'stream_stationarity', 'kind': 'held-out stream replay', 'policy': 'automatic; cacheable'},
    {'artifact': 'Table 10', 'stage': 'ablations', 'kind': 'six model/loss variants', 'policy': 'long; epoch-resumable'},
    {'artifact': 'Table S1 + Figure S1', 'stage': 'archived_core', 'kind': 'checkpoint comparison', 'policy': 'automatic when checkpoints/history exist'},
    {'artifact': 'Table S2', 'stage': 'archived_core', 'kind': 'rare-class confusion excerpt', 'policy': 'automatic'},
    {'artifact': 'Table S3', 'stage': 'archived_core', 'kind': 'Mondrian diagnostics', 'policy': 'automatic'},
    {'artifact': 'Table S4', 'stage': 'archived_core', 'kind': 'decision-interface comparison', 'policy': 'automatic'},
    {'artifact': 'Table S5', 'stage': 'scalability', 'kind': 'feature-count scaling', 'policy': 'blocked-if-missing exact feature mappings; long'},
    {'artifact': 'Table S6', 'stage': 'sensitivity', 'kind': 'hyperparameter/group sensitivity', 'policy': 'partly executable; exact K mappings required for strict mode'},
    {'artifact': 'Table S7', 'stage': 'robustness', 'kind': 'perturbation/holdout/few-shot suite', 'policy': 'clean/noise/FGSM automatic; holdouts require metadata/splits'},
    {'artifact': 'Tables S8-S9', 'stage': 'external_datasets', 'kind': 'four external datasets + mappings', 'policy': 'blocked-until dataset roots supplied; long'},
    {'artifact': 'Tables S10-S11', 'stage': 'baselines', 'kind': 'ten baselines and configurations', 'policy': 'long; optional packages; capture-aware CV requires IDs'},
    {'artifact': 'Tables S12-S12b', 'stage': 'multi_seed', 'kind': 'five matched seeds + paired intervals', 'policy': 'very long; seed/epoch resumable'},
    {'artifact': 'Table S13', 'stage': 'baselines', 'kind': 'seed-42 accuracy/throughput', 'policy': 'automatic after baseline checkpoints'},
    {'artifact': 'Table S14', 'stage': 'archived_core', 'kind': 'class-name glossary', 'policy': 'automatic'},
    {'artifact': 'Table S15', 'stage': 'archived_core', 'kind': 'numerical audit', 'policy': 'automatic'},
    {'artifact': 'Table S16', 'stage': 'confirmatory_split_conformal', 'kind': 'disjoint calibration', 'policy': 'automatic; cacheable'},
    {'artifact': 'Table S17', 'stage': 'cluster_bootstrap', 'kind': 'capture-level bootstrap', 'policy': 'blocked-if-missing row-aligned capture IDs/predictions'},
]
PAPER_COVERAGE = pd.DataFrame(PAPER_COVERAGE_ROWS)
PAPER_COVERAGE['status'] = 'REGISTERED'
atomic_dataframe(PAPER_COVERAGE, PAPER_REPORT_DIR / 'paper_artifact_coverage_registry')

# Source-aligned literature/component matrix from manuscript Table 1. It is
# intentionally tagged as static and is not presented as a rerun result.
TABLE1_REFERENCE = pd.DataFrame([
    ['Semantic feature grouping', 'No', 'No', 'No', 'No', 'No', 'Yes'],
    ['Local-global attention', 'No', 'No (KAN FFN)', 'No', 'No', 'Latent alignment', 'Yes'],
    ['Hierarchical supervision', 'No', 'No', 'No', 'No', 'No', '3 levels + KL'],
    ['Imbalance-aware loss', 'Transfer setup', 'Implicit', 'Targeted', 'Meta-loss', 'N/A', 'LDAM-DRW'],
    ['Conformal prediction', 'No', 'No', 'No', 'No', 'No', 'Deterministic Mondrian RAPS'],
    ['Stream monitoring/adaptation', 'No', 'No', 'No', 'Partial', 'Yes', 'ADWIN + martingale + Tent-LN/head tuning'],
], columns=['component', 'IDS-INT', 'TFKAN', 'Christinal', 'Ghazal', 'Wasswa', 'CAMELOT-IDS'])
TABLE1_REFERENCE.insert(0, 'source_kind', 'manuscript_literature_matrix')
atomic_dataframe(TABLE1_REFERENCE, PAPER_TABLE_DIR / 'table_01_component_comparison_reference')

print(f'Registered {len(PAPER_COVERAGE)} paper artifacts and {len(PAPER_REFERENCE)} audit targets.')
display(PAPER_COVERAGE)
paper_cell_done('P02', 'paper experiment registry and manuscript-reference targets', _started)


[P02] START: paper experiment registry and manuscript-reference targets
Registered 31 paper artifacts and 52 audit targets.


,artifact,stage,kind,policy,status
0,Table 1,reference_and_schematics,static literature comparison,export manuscript matrix; not an executable be...,REGISTERED
1,Table 2,archived_core,measured from v4 arrays,automatic,REGISTERED
2,Table 3,provenance_audit,metadata audit,blocked-if-missing manifest,REGISTERED
3,Figure 1,archived_core,dataset/split plot,automatic,REGISTERED
4,Equations 1-11,reference_and_schematics,code-formula alignment,documented in helper implementations,REGISTERED
5,Algorithm 1,reference_and_schematics,pipeline pseudocode,exported from executable stage graph,REGISTERED
6,Figure 2,reference_and_schematics,pipeline schematic,automatic,REGISTERED
7,Table 4,archived_core,configuration audit,automatic,REGISTERED
8,Figure 3,reference_and_schematics,calibration protocol schematic,automatic,REGISTERED
9,Table 5,archived_core,cached v4 evaluation,automatic,REGISTERED


[P02] DONE: paper experiment registry and manuscript-reference targets | elapsed=0.17s


In [5]:
_started = paper_cell_start('P03', 'shared metrics, calibration, RAPS, plotting, and inference helpers')

from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, f1_score, precision_recall_fscore_support,
    confusion_matrix, matthews_corrcoef, classification_report, log_loss,
)
from scipy.stats import spearmanr, pearsonr, t as student_t
import matplotlib.pyplot as plt


def safe_torch_load(path: Path, map_location: Any = 'cpu') -> Any:
    path = Path(path)
    try:
        return torch.load(path, map_location=map_location, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=map_location)


def atomic_npz(path: Path, **arrays: np.ndarray) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + '.tmp')
    with open(tmp, 'wb') as handle:
        np.savez_compressed(handle, **{k: np.asarray(v) for k, v in arrays.items()})
        handle.flush()
        try:
            os.fsync(handle.fileno())
        except OSError:
            pass
    os.replace(tmp, path)


def save_figure(fig: Any, stem: Path, dpi: int = 300) -> Dict[str, str]:
    stem = Path(stem)
    stem.parent.mkdir(parents=True, exist_ok=True)
    paths: Dict[str, str] = {}
    for suffix in ('.png', '.pdf'):
        out = stem.with_suffix(suffix)
        fig.savefig(out, dpi=dpi, bbox_inches='tight')
        paths[suffix.lstrip('.')] = str(out)
    plt.close(fig)
    return paths


def canonical_label_name(name: Any) -> str:
    raw = str(name).strip()
    replacements = {
        'Uploading_Attack': 'Uploading Attack',
        'SqlInjection': 'SQL Injection',
        'Sql_Injection': 'SQL Injection',
        'CommandInjection': 'Command Injection',
        'Command_Injection': 'Command Injection',
        'Recon-PingSweep': 'Recon Ping Sweep',
        'Recon_Ping_Sweep': 'Recon Ping Sweep',
        'Recon-OSScan': 'Recon OS Scan',
        'Recon_OS_Scan': 'Recon OS Scan',
        'Backdoor_Malware': 'Backdoor Malware',
        'DictionaryBruteForce': 'Dictionary Brute Force',
        'Dictionary_Brute_Force': 'Dictionary Brute Force',
        'DDoS-ICMP_Flood': 'DDoS ICMP Flood',
    }
    if raw in replacements:
        return replacements[raw]
    return raw.replace('_', ' ').replace('-', ' ').strip()


CANONICAL_FINE_NAMES = [canonical_label_name(x) for x in FINE_CLASS_NAMES]
CANONICAL_COARSE_NAMES = [canonical_label_name(x) for x in COARSE_NAMES]


def multiclass_metrics(y_true: np.ndarray, probs: np.ndarray) -> Dict[str, float]:
    y = np.asarray(y_true, dtype=np.int64)
    p = np.asarray(probs, dtype=np.float64)
    pred = p.argmax(axis=1)
    return {
        'accuracy': float(accuracy_score(y, pred)),
        'balanced_accuracy': float(balanced_accuracy_score(y, pred)),
        'macro_f1': float(f1_score(y, pred, average='macro', zero_division=0)),
        'weighted_f1': float(f1_score(y, pred, average='weighted', zero_division=0)),
        'mcc': float(matthews_corrcoef(y, pred)),
    }


def binary_metrics(y_true: np.ndarray, probs: np.ndarray) -> Dict[str, float]:
    y = np.asarray(y_true, dtype=np.int64)
    p = np.asarray(probs, dtype=np.float64)
    pred = p.argmax(axis=1)
    precision, recall, f1, support = precision_recall_fscore_support(
        y, pred, labels=[0, 1], zero_division=0
    )
    return {
        'accuracy': float(accuracy_score(y, pred)),
        'balanced_accuracy': float(balanced_accuracy_score(y, pred)),
        'macro_f1': float(f1_score(y, pred, average='macro', zero_division=0)),
        'weighted_f1': float(f1_score(y, pred, average='weighted', zero_division=0)),
        'mcc': float(matthews_corrcoef(y, pred)),
        'benign_precision': float(precision[0]),
        'benign_recall': float(recall[0]),
        'benign_f1': float(f1[0]),
        'malicious_precision': float(precision[1]),
        'malicious_recall': float(recall[1]),
        'malicious_f1': float(f1[1]),
        'benign_support': int(support[0]),
        'malicious_support': int(support[1]),
    }


def classwise_report(y_true: np.ndarray, probs: np.ndarray, names: Sequence[str]) -> pd.DataFrame:
    y = np.asarray(y_true, dtype=np.int64)
    pred = np.asarray(probs).argmax(axis=1)
    labels = np.arange(len(names), dtype=np.int64)
    precision, recall, f1, support = precision_recall_fscore_support(
        y, pred, labels=labels, zero_division=0
    )
    return pd.DataFrame({
        'class_id': labels,
        'class_name_raw': [str(x) for x in names],
        'class_name': [canonical_label_name(x) for x in names],
        'support': support.astype(np.int64),
        'precision': precision.astype(float),
        'recall': recall.astype(float),
        'f1': f1.astype(float),
    })


def expected_calibration_error_local(y_true: np.ndarray, probs: np.ndarray, n_bins: int = 15) -> float:
    y = np.asarray(y_true, dtype=np.int64)
    p = np.asarray(probs, dtype=np.float64)
    confidence = p.max(axis=1)
    correct = (p.argmax(axis=1) == y).astype(np.float64)
    edges = np.linspace(0.0, 1.0, int(n_bins) + 1)
    value = 0.0
    for i, (lo, hi) in enumerate(zip(edges[:-1], edges[1:])):
        mask = (confidence >= lo) & (confidence <= hi) if i == 0 else (confidence > lo) & (confidence <= hi)
        if mask.any():
            value += float(mask.mean()) * abs(float(correct[mask].mean()) - float(confidence[mask].mean()))
    return float(value)


def brier_multiclass(y_true: np.ndarray, probs: np.ndarray) -> float:
    y = np.asarray(y_true, dtype=np.int64)
    p = np.asarray(probs, dtype=np.float64)
    one_hot = np.zeros_like(p)
    one_hot[np.arange(len(y)), y] = 1.0
    return float(np.mean(np.sum((p - one_hot) ** 2, axis=1)))


def calibration_metrics(y_true: np.ndarray, probs: np.ndarray, bins: int = 15) -> Dict[str, float]:
    y = np.asarray(y_true, dtype=np.int64)
    p = np.clip(np.asarray(probs, dtype=np.float64), 1e-12, 1.0)
    return {
        'nll': float(-np.mean(np.log(p[np.arange(len(y)), y]))),
        'ece': expected_calibration_error_local(y, p, n_bins=bins),
        'brier': brier_multiclass(y, p),
    }


def finite_quantile_level(n: int, alpha: float) -> float:
    if int(n) <= 0:
        return 1.0
    return float(min(1.0, math.ceil((int(n) + 1) * (1.0 - float(alpha))) / int(n)))


def higher_quantile(values: np.ndarray, level: float) -> float:
    arr = np.asarray(values, dtype=np.float64)
    if arr.size == 0:
        return 1.0
    try:
        return float(np.quantile(arr, float(level), method='higher'))
    except TypeError:
        return float(np.quantile(arr, float(level), interpolation='higher'))


def raps_true_scores_chunked(
    probs: np.ndarray,
    y_true: np.ndarray,
    k_reg: int,
    lam: float,
    chunk_size: int = 100_000,
) -> np.ndarray:
    p = np.asarray(probs)
    y = np.asarray(y_true, dtype=np.int64)
    out = np.empty(len(y), dtype=np.float32)
    for start in range(0, len(y), int(chunk_size)):
        stop = min(len(y), start + int(chunk_size))
        pc = np.asarray(p[start:stop], dtype=np.float32)
        yc = y[start:stop]
        order = np.argsort(-pc, axis=1)
        sorted_p = np.take_along_axis(pc, order, axis=1)
        cumulative = np.cumsum(sorted_p, axis=1, dtype=np.float32)
        inverse = np.empty_like(order)
        rows = np.arange(stop - start)[:, None]
        inverse[rows, order] = np.arange(pc.shape[1], dtype=order.dtype)[None, :]
        rank0 = inverse[np.arange(stop - start), yc]
        rank1 = rank0 + 1
        out[start:stop] = cumulative[np.arange(stop - start), rank0] + float(lam) * np.maximum(rank1 - int(k_reg), 0)
    return out


def build_mondrian_raps_thresholds(
    probs_fine: np.ndarray,
    probs_coarse: np.ndarray,
    y_fine: np.ndarray,
    alphas: Sequence[float],
    k_reg: int,
    lam: float,
    min_group: int = 200,
    chunk_size: int = 100_000,
) -> Dict[str, Any]:
    scores = raps_true_scores_chunked(probs_fine, y_fine, k_reg, lam, chunk_size=chunk_size)
    predicted_group = np.asarray(probs_coarse).argmax(axis=1).astype(np.int64)
    result: Dict[str, Any] = {
        'mode': 'predicted_coarse',
        'k_reg': int(k_reg),
        'lambda': float(lam),
        'min_group': int(min_group),
        'alphas': [float(a) for a in alphas],
        'global': {},
        'groups': {},
    }
    for alpha in alphas:
        akey = f'{float(alpha):.12g}'
        global_level = finite_quantile_level(len(scores), float(alpha))
        q_global = higher_quantile(scores, global_level)
        groups = []
        for group_id in range(int(probs_coarse.shape[1])):
            mask = predicted_group == group_id
            n_group = int(mask.sum())
            level = finite_quantile_level(n_group, float(alpha))
            if n_group >= int(min_group):
                threshold = higher_quantile(scores[mask], level)
                fallback = False
            else:
                threshold = q_global
                fallback = True
            groups.append({
                'group_id': int(group_id),
                'group_name': CANONICAL_COARSE_NAMES[group_id] if group_id < len(CANONICAL_COARSE_NAMES) else str(group_id),
                'n': n_group,
                'quantile_level': float(level),
                'threshold': float(threshold),
                'used_global_fallback': bool(fallback),
            })
        result['global'][akey] = {
            'n': int(len(scores)),
            'quantile_level': float(global_level),
            'threshold': float(q_global),
        }
        result['groups'][akey] = groups
    return result


def raps_evaluate_detailed_chunked(
    probs_fine: np.ndarray,
    probs_coarse: np.ndarray,
    y_true: np.ndarray,
    thresholds: Mapping[str, Any],
    alpha: float,
    benign_id: int,
    chunk_size: int = 100_000,
) -> Dict[str, Any]:
    p_f = np.asarray(probs_fine)
    p_c = np.asarray(probs_coarse)
    y = np.asarray(y_true, dtype=np.int64)
    akey = f'{float(alpha):.12g}'
    group_thresholds = np.asarray(
        [float(row['threshold']) for row in thresholds['groups'][akey]], dtype=np.float32
    )
    k_reg = int(thresholds['k_reg'])
    lam = float(thresholds['lambda'])
    n_total = len(y)
    covered_n = 0
    set_size_sum = 0
    singleton_n = 0
    singleton_correct = 0
    singleton_benign_n = 0
    singleton_benign_fp = 0
    recovered_top1_errors = 0
    top1_error_n = 0
    per_group = {
        g: {'n': 0, 'covered': 0, 'set_size_sum': 0, 'singleton': 0, 'singleton_correct': 0}
        for g in range(p_c.shape[1])
    }
    for start in range(0, n_total, int(chunk_size)):
        check_deadline(f'RAPS evaluation alpha={alpha} row={start}')
        stop = min(n_total, start + int(chunk_size))
        pf = np.asarray(p_f[start:stop], dtype=np.float32)
        pc = np.asarray(p_c[start:stop], dtype=np.float32)
        yy = y[start:stop]
        group = pc.argmax(axis=1).astype(np.int64)
        q_row = group_thresholds[group]
        order = np.argsort(-pf, axis=1)
        sorted_p = np.take_along_axis(pf, order, axis=1)
        cumulative = np.cumsum(sorted_p, axis=1, dtype=np.float32)
        ranks = np.arange(1, pf.shape[1] + 1, dtype=np.float32)[None, :]
        score = cumulative + float(lam) * np.maximum(ranks - float(k_reg), 0.0)
        k_star = np.maximum((score <= q_row[:, None]).sum(axis=1).astype(np.int64), 1)
        inverse = np.empty_like(order)
        rows = np.arange(stop - start)[:, None]
        inverse[rows, order] = np.arange(pf.shape[1], dtype=order.dtype)[None, :]
        rank0 = inverse[np.arange(stop - start), yy]
        covered = rank0 < k_star
        top1 = order[:, 0]
        top1_wrong = top1 != yy
        singleton = k_star == 1
        covered_n += int(covered.sum())
        set_size_sum += int(k_star.sum())
        singleton_n += int(singleton.sum())
        singleton_correct += int((singleton & (top1 == yy)).sum())
        singleton_benign = singleton & (yy == int(benign_id))
        singleton_benign_n += int(singleton_benign.sum())
        singleton_benign_fp += int((singleton_benign & (top1 != int(benign_id))).sum())
        top1_error_n += int(top1_wrong.sum())
        recovered_top1_errors += int((top1_wrong & covered).sum())
        for g in np.unique(group):
            mask = group == int(g)
            rec = per_group[int(g)]
            rec['n'] += int(mask.sum())
            rec['covered'] += int(covered[mask].sum())
            rec['set_size_sum'] += int(k_star[mask].sum())
            rec['singleton'] += int(singleton[mask].sum())
            rec['singleton_correct'] += int((singleton[mask] & (top1[mask] == yy[mask])).sum())
    singleton_accuracy = singleton_correct / singleton_n if singleton_n else float('nan')
    result: Dict[str, Any] = {
        'alpha': float(alpha),
        'n': int(n_total),
        'coverage': covered_n / n_total if n_total else float('nan'),
        'avg_set_size': set_size_sum / n_total if n_total else float('nan'),
        'singleton_fraction': singleton_n / n_total if n_total else float('nan'),
        'singleton_coverage': singleton_n / n_total if n_total else float('nan'),
        'singleton_accuracy': float(singleton_accuracy),
        'singleton_risk': float(1.0 - singleton_accuracy) if singleton_n else float('nan'),
        'benign_fpr_singletons': singleton_benign_fp / singleton_benign_n if singleton_benign_n else float('nan'),
        'top1_error': top1_error_n / n_total if n_total else float('nan'),
        'top1_errors_recovered_by_set': recovered_top1_errors / top1_error_n if top1_error_n else float('nan'),
        'committed_decisions': int(singleton_n),
        'committed_errors': int(singleton_n - singleton_correct),
    }
    group_rows = []
    for group_id, rec in per_group.items():
        n = int(rec['n'])
        single_n = int(rec['singleton'])
        group_rows.append({
            'group_id': int(group_id),
            'group_name': CANONICAL_COARSE_NAMES[group_id] if group_id < len(CANONICAL_COARSE_NAMES) else str(group_id),
            'n': n,
            'coverage': rec['covered'] / n if n else float('nan'),
            'avg_set_size': rec['set_size_sum'] / n if n else float('nan'),
            'singleton_fraction': single_n / n if n else float('nan'),
            'singleton_risk': 1.0 - rec['singleton_correct'] / single_n if single_n else float('nan'),
        })
    result['group_rows'] = group_rows
    return result


@torch.inference_mode()
def predict_hier_logits_array(
    model: nn.Module,
    X: np.ndarray,
    batch_size: Optional[int] = None,
    device: Optional[torch.device] = None,
    log_every_batches: int = 100,
) -> Dict[str, np.ndarray]:
    bs = int(PAPER.INFERENCE_BATCH_SIZE if batch_size is None else batch_size)
    dev = DEVICE_T if device is None else torch.device(device)
    model = model.to(dev).eval()
    fine_parts: List[np.ndarray] = []
    coarse_parts: List[np.ndarray] = []
    binary_parts: List[np.ndarray] = []
    total_batches = int(math.ceil(len(X) / bs)) if len(X) else 0
    started = time.perf_counter()
    for batch_idx, start in enumerate(range(0, len(X), bs), start=1):
        stop = min(len(X), start + bs)
        xb = torch.from_numpy(np.array(X[start:stop], dtype=np.float32, copy=True, order='C')).to(dev, non_blocking=True)
        with amp_autocast(PAPER.USE_AMP and dev.type == 'cuda'):
            output = model(xb)
        fine_parts.append(output['logits_fine'].float().cpu().numpy())
        coarse_parts.append(output['logits_coarse'].float().cpu().numpy())
        binary_parts.append(output['logits_bin'].float().cpu().numpy())
        if batch_idx == 1 or batch_idx % max(1, int(log_every_batches)) == 0 or batch_idx == total_batches:
            elapsed = time.perf_counter() - started
            rate = stop / max(elapsed, 1e-9)
            paper_log('inference progress', stage='predict', batch=f'{batch_idx}/{total_batches}', rows=stop,
                      rate_samples_s=round(rate, 1))
        check_deadline(f'inference batch {batch_idx}/{total_batches}')
    return {
        'fine': np.concatenate(fine_parts, axis=0) if fine_parts else np.empty((0, NUM_FINE_PAPER), dtype=np.float32),
        'coarse': np.concatenate(coarse_parts, axis=0) if coarse_parts else np.empty((0, NUM_COARSE_PAPER), dtype=np.float32),
        'bin': np.concatenate(binary_parts, axis=0) if binary_parts else np.empty((0, 2), dtype=np.float32),
    }


def profile_cap(partition: str, n: int) -> int:
    if PAPER.PROFILE == 'strict':
        return int(n)
    if PAPER.PROFILE == 'accelerated':
        caps = {
            'train': PAPER.ACCEL_TRAIN_CAP,
            'val': PAPER.ACCEL_VAL_CAP,
            'cal': PAPER.ACCEL_CAL_CAP,
            'test': PAPER.ACCEL_TEST_CAP,
        }
    else:
        caps = {
            'train': PAPER.SMOKE_TRAIN_CAP,
            'val': PAPER.SMOKE_VAL_CAP,
            'cal': PAPER.SMOKE_CAL_CAP,
            'test': PAPER.SMOKE_TEST_CAP,
        }
    return int(min(n, caps.get(partition, n)))


def deterministic_subset_indices(y: np.ndarray, max_n: int, seed: int) -> np.ndarray:
    y = np.asarray(y, dtype=np.int64)
    if int(max_n) >= len(y):
        return np.arange(len(y), dtype=np.int64)
    rng = np.random.default_rng(int(seed))
    selected: List[np.ndarray] = []
    classes, counts = np.unique(y, return_counts=True)
    allocation = np.maximum(1, np.floor(counts / counts.sum() * int(max_n)).astype(int))
    while allocation.sum() > int(max_n):
        candidate = int(np.argmax(allocation))
        if allocation[candidate] > 1:
            allocation[candidate] -= 1
        else:
            break
    while allocation.sum() < int(max_n):
        candidate = int(np.argmax(counts - allocation))
        allocation[candidate] += 1
    for cls, n_take in zip(classes, allocation):
        pool = np.flatnonzero(y == cls)
        selected.append(rng.choice(pool, size=min(int(n_take), len(pool)), replace=False))
    idx = np.concatenate(selected)
    rng.shuffle(idx)
    return idx[:int(max_n)].astype(np.int64)


def benchmark_native_pytorch(
    model: nn.Module,
    X: np.ndarray,
    device: torch.device,
    batch_size: int = 4096,
    max_rows: int = 200_000,
    warmup_passes: int = 5,
) -> Dict[str, float]:
    dev = torch.device(device)
    model = model.to(dev).eval()
    n = int(min(len(X), int(max_rows)))
    if n <= 0:
        return {'n': 0, 'throughput_samples_s': float('nan'), 'ms_per_sample': float('nan')}
    sample = torch.from_numpy(np.array(X[:n], dtype=np.float32, copy=True, order='C'))
    warm = sample[:min(int(batch_size), n)].to(dev)
    with torch.inference_mode():
        for _ in range(int(warmup_passes)):
            with amp_autocast(PAPER.USE_AMP and dev.type == 'cuda'):
                _ = model(warm)
        if dev.type == 'cuda':
            torch.cuda.synchronize(dev)
        started = time.perf_counter()
        seen = 0
        for start in range(0, n, int(batch_size)):
            xb = sample[start:start + int(batch_size)].to(dev, non_blocking=True)
            with amp_autocast(PAPER.USE_AMP and dev.type == 'cuda'):
                _ = model(xb)
            seen += len(xb)
        if dev.type == 'cuda':
            torch.cuda.synchronize(dev)
        elapsed = time.perf_counter() - started
    throughput = seen / max(elapsed, 1e-12)
    return {
        'n': int(seen),
        'batch_size': int(batch_size),
        'warmup_passes': int(warmup_passes),
        'elapsed_seconds': float(elapsed),
        'throughput_samples_s': float(throughput),
        'ms_per_sample': float(1000.0 / throughput),
    }


def write_blocked_stage(stage_name: str, reason: str, **details: Any) -> pd.DataFrame:
    mark_stage(stage_name, 'BLOCKED', reason=str(reason), **details)
    row = {'stage': stage_name, 'status': 'BLOCKED', 'reason': str(reason), **details}
    out = pd.DataFrame([row])
    atomic_dataframe(out, PAPER_REPORT_DIR / f'{stage_name}_blocked')
    paper_log('stage blocked', level='WARNING', stage=stage_name, reason=reason)
    return out


def update_coverage_status(artifact: str, status: str, output: Optional[str] = None, note: str = '') -> None:
    path = PAPER_REPORT_DIR / 'paper_artifact_coverage_registry.csv'
    if path.exists():
        table = pd.read_csv(path)
    else:
        table = PAPER_COVERAGE.copy()
    mask = table['artifact'].astype(str) == str(artifact)
    table.loc[mask, 'status'] = str(status)
    if 'output' not in table.columns:
        table['output'] = ''
    else:
        table['output'] = table['output'].fillna('').astype(str)
    if 'note' not in table.columns:
        table['note'] = ''
    else:
        table['note'] = table['note'].fillna('').astype(str)
    table['status'] = table['status'].fillna('').astype(str)
    if output is not None:
        table.loc[mask, 'output'] = str(output)
    if note:
        table.loc[mask, 'note'] = str(note)
    atomic_dataframe(table, PAPER_REPORT_DIR / 'paper_artifact_coverage_registry')


paper_log('shared analysis helpers ready', stage='helpers', fine_classes=NUM_FINE_PAPER,
          coarse_classes=NUM_COARSE_PAPER, inference_batch=PAPER.INFERENCE_BATCH_SIZE)
paper_cell_done('P03', 'shared metrics, calibration, RAPS, plotting, and inference helpers', _started)


[P03] START: shared metrics, calibration, RAPS, plotting, and inference helpers
[2026-08-28 09:53:54] [INFO] [helpers] shared analysis helpers ready | fine_classes=34 coarse_classes=8 inference_batch=16384
[P03] DONE: shared metrics, calibration, RAPS, plotting, and inference helpers | elapsed=0.33s


In [6]:
_started = paper_cell_start('P04', 'rebuild archived v4 tables, figures, diagnostics, and numerical audit')


def _norm_label_key(text: Any) -> str:
    return re.sub(r'[^a-z0-9]+', '', str(text).lower())


def _select_class_row(report: pd.DataFrame, desired: str) -> Optional[pd.Series]:
    key = _norm_label_key(desired)
    candidates = report.copy()
    candidates['_key'] = candidates['class_name'].map(_norm_label_key)
    exact = candidates[candidates['_key'] == key]
    if len(exact):
        return exact.iloc[0]
    partial = candidates[candidates['_key'].map(lambda x: key in x or x in key)]
    return partial.iloc[0] if len(partial) else None


def run_archived_core_tables() -> Dict[str, Any]:
    signature = {
        'base': PAPER_BASE_SIGNATURE_HASH,
        'stage_version': 4,
        'alphas': list(PAPER.ALPHAS),
        'k_reg': PAPER.RAPS_KREG,
        'lambda': PAPER.RAPS_LAMBDA,
    }
    summary_path = PAPER_REPORT_DIR / 'archived_core_summary.json'
    with paper_stage('archived_core', signature) as stage:
        if stage.skip and summary_path.exists():
            return read_json(summary_path, {})

        outputs: Dict[str, Any] = {}

        # -----------------------
        # Table 2: split structure
        # -----------------------
        partition_rows = [
            ('Train', len(X_train)),
            ('Validation', len(X_val)),
            ('Calibration', len(X_cal)),
            ('Test', len(X_test)),
        ]
        n_total = int(sum(n for _, n in partition_rows))
        table2 = pd.DataFrame([
            {'partition': name, 'samples': int(n), 'share_percent': 100.0 * n / n_total,
             'source_kind': 'measured_from_v4_arrays'}
            for name, n in partition_rows
        ])
        table2.loc[len(table2)] = {
            'partition': 'Fine/coarse/binary labels', 'samples': f'{NUM_FINE_PAPER}/{NUM_COARSE_PAPER}/2',
            'share_percent': np.nan, 'source_kind': 'measured_from_v4_metadata'
        }
        outputs['table_02'] = atomic_dataframe(table2, PAPER_TABLE_DIR / 'table_02_evaluation_split')
        update_coverage_status('Table 2', 'DONE', outputs['table_02']['csv'])

        # ------------------------------------
        # Table 4: code-aligned configuration
        # ------------------------------------
        table4 = pd.DataFrame([
            ['Retained features/groups', f'{len(FEATURE_COLS)} features; group sizes ' + '/'.join(str(len(g)) for g in GROUP_IDXS)],
            ['Embedding/attention', f'd={cfg.D_MODEL}; {cfg.NHEAD} heads; FF dimension {cfg.FF_DIM}'],
            ['Encoder depth', f'{cfg.LOCAL_LAYERS} local layers; {cfg.GLOBAL_LAYERS} global layers'],
            ['Dropout', f'{cfg.DROPOUT:.2f}; attention dropout {cfg.ATTN_DROPOUT:.2f}'],
            ['Batch/epochs', f'{cfg.BATCH_SIZE}/{cfg.EPOCHS}; checkpoint selected by validation fine macro-F1'],
            ['Optimizer', f'AdamW; LR {cfg.LR:g}; weight decay {cfg.WEIGHT_DECAY:g}; cosine decay; {cfg.WARMUP_EPOCHS:g}-epoch warm-up; gradient clip {cfg.GRAD_CLIP:g}'],
            ['Execution', f'AMP={bool(cfg.USE_AMP)}; TF32={bool(cfg.TF32_MODE)}; deterministic={bool(cfg.DETERMINISTIC)}; seed {cfg.SEED}'],
            ['RAPS', f'Temperature scaling; k_reg={PAPER.RAPS_KREG}; lambda={PAPER.RAPS_LAMBDA}; deterministic; predicted-coarse Mondrian groups; top-1 fallback'],
            ['Stream monitor', f'{cfg.STREAM_BATCH:,}-flow logical batches; {cfg.STREAM_MICRO_BATCH:,} micro-batch; ADWIN delta={cfg.DRIFT_DELTA}; martingale epsilon={cfg.MART_EPS}, threshold {cfg.MART_THRESHOLD:g}'],
            ['Tent-LN', f'LayerNorm affine parameters; Adam LR {cfg.ADAPT_LR:g}; {cfg.ADAPT_STEPS} passes; weight decay 0'],
            ['Pseudo-label head tuning', f'Fine/coarse/binary heads; Adam LR {cfg.HEAD_TUNE_LR:g}; {cfg.HEAD_TUNE_STEPS} passes; minimum {cfg.MIN_PSEUDO} singleton pseudo-labels'],
            ['Model size', f'{sum(p.numel() for p in PAPER_MODEL.parameters()) / 1e6:.3f} million trainable parameters'],
        ], columns=['component', 'archived_setting'])
        table4['source_kind'] = 'code_configuration_audit'
        outputs['table_04'] = atomic_dataframe(table4, PAPER_TABLE_DIR / 'table_04_model_calibration_adaptation_configuration')
        update_coverage_status('Table 4', 'DONE', outputs['table_04']['csv'])

        # ----------------------------------
        # Table 5: three decision granularities
        # ----------------------------------
        fine_metrics = multiclass_metrics(PAPER_Y_TEST_FINE, PAPER_TEST_PROBS_FINE)
        coarse_metrics = multiclass_metrics(PAPER_Y_TEST_COARSE, PAPER_TEST_PROBS_COARSE)
        bin_metrics = binary_metrics(PAPER_Y_TEST_BINARY, PAPER_TEST_PROBS_BINARY)
        table5 = pd.DataFrame([
            {'task': 'Fine-34', 'accuracy_percent': 100 * fine_metrics['accuracy'],
             'balanced_accuracy_percent': 100 * fine_metrics['balanced_accuracy'],
             'macro_f1_percent': 100 * fine_metrics['macro_f1'], 'mcc': fine_metrics['mcc'],
             'macro_f1_definition': 'macro over 34 classes'},
            {'task': 'Coarse-8', 'accuracy_percent': 100 * coarse_metrics['accuracy'],
             'balanced_accuracy_percent': 100 * coarse_metrics['balanced_accuracy'],
             'macro_f1_percent': 100 * coarse_metrics['macro_f1'], 'mcc': coarse_metrics['mcc'],
             'macro_f1_definition': 'macro over 8 classes'},
            {'task': 'Binary', 'accuracy_percent': 100 * bin_metrics['accuracy'],
             'balanced_accuracy_percent': 100 * bin_metrics['balanced_accuracy'],
             'macro_f1_percent': 100 * bin_metrics['malicious_f1'], 'mcc': bin_metrics['mcc'],
             'macro_f1_definition': 'malicious-class F1 (paper display)'},
        ])
        table5['source_kind'] = 'recomputed_from_archived_v4_logits'
        outputs['table_05'] = atomic_dataframe(table5, PAPER_TABLE_DIR / 'table_05_primary_test_performance')
        update_coverage_status('Table 5', 'DONE', outputs['table_05']['csv'])

        # ----------------------------------
        # Table 6 and support/F1 association
        # ----------------------------------
        fine_report = classwise_report(PAPER_Y_TEST_FINE, PAPER_TEST_PROBS_FINE, FINE_CLASS_NAMES)
        outputs['fine_class_report'] = atomic_dataframe(fine_report, PAPER_TABLE_DIR / 'fine_34_class_metrics_full')
        selected_names = [
            'Uploading Attack', 'XSS', 'Recon Ping Sweep', 'Backdoor Malware',
            'SQL Injection', 'Command Injection', 'Recon OS Scan', 'Dictionary Brute Force',
        ]
        rare_rows: List[Dict[str, Any]] = []
        for desired in selected_names:
            row = _select_class_row(fine_report, desired)
            if row is None:
                rare_rows.append({'class': desired, 'status': 'NOT_FOUND'})
            else:
                rare_rows.append({
                    'class': desired, 'class_id': int(row['class_id']), 'support': int(row['support']),
                    'precision_percent': 100 * float(row['precision']),
                    'recall_percent': 100 * float(row['recall']),
                    'f1_percent': 100 * float(row['f1']), 'status': 'MEASURED',
                })
        table6 = pd.DataFrame(rare_rows)
        table6['source_kind'] = 'recomputed_from_archived_v4_logits'
        outputs['table_06'] = atomic_dataframe(table6, PAPER_TABLE_DIR / 'table_06_underrepresented_fine_classes')
        update_coverage_status('Table 6', 'DONE', outputs['table_06']['csv'])
        valid_corr = fine_report[(fine_report['support'] > 0) & np.isfinite(fine_report['f1'])]
        support_log = np.log10(valid_corr['support'].astype(float).values + 1.0)
        spearman_value, spearman_p = spearmanr(valid_corr['support'].astype(float).values, valid_corr['f1'].values)
        pearson_value, pearson_p = pearsonr(support_log, valid_corr['f1'].values)
        corr_table = pd.DataFrame([{
            'spearman_support_vs_f1': float(spearman_value), 'spearman_p': float(spearman_p),
            'pearson_log10_support_vs_f1': float(pearson_value), 'pearson_p': float(pearson_p),
            'n_classes_with_support': int(len(valid_corr)), 'source_kind': 'recomputed_from_archived_v4_logits',
        }])
        outputs['support_f1_correlation'] = atomic_dataframe(corr_table, PAPER_TABLE_DIR / 'support_f1_correlation')

        # ----------------------------------
        # Archived empirical RAPS (one pool)
        # ----------------------------------
        thresholds = build_mondrian_raps_thresholds(
            PAPER_CAL_PROBS_FINE, PAPER_CAL_PROBS_COARSE, PAPER_Y_CAL_FINE,
            PAPER.ALPHAS, PAPER.RAPS_KREG, PAPER.RAPS_LAMBDA, PAPER.RAPS_MIN_GROUP,
        )
        globals()['ARCHIVED_RAPS_THRESHOLDS'] = thresholds
        atomic_json(PAPER_CACHE_DIR / 'archived_empirical_raps_thresholds.json', thresholds)
        raps_rows = []
        group_frames = []
        detailed_by_alpha: Dict[str, Any] = {}
        for alpha in PAPER.ALPHAS:
            detail = raps_evaluate_detailed_chunked(
                PAPER_TEST_PROBS_FINE, PAPER_TEST_PROBS_COARSE, PAPER_Y_TEST_FINE,
                thresholds, alpha, int(benign_fine_id),
            )
            detailed_by_alpha[str(alpha)] = {k: v for k, v in detail.items() if k != 'group_rows'}
            raps_rows.append({
                'alpha': alpha,
                'coverage_percent': 100 * detail['coverage'],
                'average_set_size': detail['avg_set_size'],
                'singleton_fraction_percent': 100 * detail['singleton_fraction'],
                'singleton_risk_percent': 100 * detail['singleton_risk'],
                'benign_fpr_singletons_percent': 100 * detail['benign_fpr_singletons'],
                'committed_decisions': detail['committed_decisions'],
                'committed_errors': detail['committed_errors'],
                'source_kind': 'archived_empirical_diagnostic_same_calibration_pool',
            })
            gf = pd.DataFrame(detail['group_rows'])
            gf.insert(0, 'alpha', alpha)
            group_frames.append(gf)
        table7 = pd.DataFrame(raps_rows)
        outputs['table_07'] = atomic_dataframe(table7, PAPER_TABLE_DIR / 'table_07_empirical_mondrian_raps')
        update_coverage_status('Table 7', 'DONE', outputs['table_07']['csv'],
                               'Empirical diagnostic: probability scaling and thresholds reuse one labeled pool.')
        atomic_json(PAPER_REPORT_DIR / 'archived_empirical_raps_detailed.json', detailed_by_alpha)

        # S3: calibration group sizes/levels/thresholds plus test behavior at alpha=.10.
        alpha_s3 = 0.10
        akey = f'{alpha_s3:.12g}'
        threshold_df = pd.DataFrame(thresholds['groups'][akey])
        group_eval_df = pd.DataFrame(detailed_by_alpha[str(alpha_s3)]['group_rows'] if 'group_rows' in detailed_by_alpha[str(alpha_s3)] else [])
        # detailed_by_alpha intentionally omits group_rows; use the already computed frame.
        group_eval_df = group_frames[list(PAPER.ALPHAS).index(alpha_s3)].copy()
        s3 = threshold_df.merge(
            group_eval_df[['group_id', 'coverage', 'avg_set_size', 'singleton_fraction', 'singleton_risk']],
            on='group_id', how='left'
        )
        s3['coverage_percent'] = 100 * s3['coverage']
        s3['source_kind'] = 'archived_empirical_diagnostic_same_calibration_pool'
        outputs['table_s03'] = atomic_dataframe(s3, PAPER_TABLE_DIR / 'table_s03_mondrian_group_diagnostics')
        update_coverage_status('Table S3', 'DONE', outputs['table_s03']['csv'])

        # S4: top-1 versus set-valued interface.
        top1_accuracy = float((PAPER_TEST_PROBS_FINE.argmax(axis=1) == PAPER_Y_TEST_FINE).mean())
        a10 = table7.loc[np.isclose(table7['alpha'], 0.10)].iloc[0]
        table_s4 = pd.DataFrame([
            {'decision_interface': 'Top-1 point prediction', 'empirical_coverage_percent': 100 * top1_accuracy,
             'average_set_size': 1.0, 'committed_decisions_percent': 100.0,
             'committed_risk_percent': 100 * (1.0 - top1_accuracy),
             'source_kind': 'recomputed_from_archived_v4_logits'},
            {'decision_interface': 'Deterministic Mondrian RAPS',
             'empirical_coverage_percent': float(a10['coverage_percent']),
             'average_set_size': float(a10['average_set_size']),
             'committed_decisions_percent': float(a10['singleton_fraction_percent']),
             'committed_risk_percent': float(a10['singleton_risk_percent']),
             'source_kind': 'archived_empirical_diagnostic_same_calibration_pool'},
        ])
        outputs['table_s04'] = atomic_dataframe(table_s4, PAPER_TABLE_DIR / 'table_s04_point_vs_raps')
        update_coverage_status('Table S4', 'DONE', outputs['table_s04']['csv'])

        # S2: selected confusion excerpt.
        pred_fine = PAPER_TEST_PROBS_FINE.argmax(axis=1)
        cm = confusion_matrix(PAPER_Y_TEST_FINE, pred_fine, labels=np.arange(NUM_FINE_PAPER))
        excerpt_names = ['Uploading Attack', 'XSS', 'Recon Ping Sweep', 'Backdoor Malware', 'SQL Injection', 'Command Injection', 'Benign']
        excerpt_ids = []
        for name in excerpt_names:
            row = _select_class_row(fine_report, name)
            if row is not None:
                excerpt_ids.append((name, int(row['class_id'])))
        excerpt = pd.DataFrame(
            cm[np.ix_([i for _, i in excerpt_ids], [i for _, i in excerpt_ids])],
            index=[name for name, _ in excerpt_ids], columns=[name for name, _ in excerpt_ids]
        ).reset_index(names='true_class')
        excerpt['source_kind'] = 'recomputed_from_archived_v4_logits'
        outputs['table_s02'] = atomic_dataframe(excerpt, PAPER_TABLE_DIR / 'table_s02_rare_class_confusion_excerpt')
        update_coverage_status('Table S2', 'DONE', outputs['table_s02']['csv'])

        # S14 canonical class-name glossary.
        glossary = pd.DataFrame({
            'raw_code_label': list(map(str, FINE_CLASS_NAMES)),
            'manuscript_display_name': CANONICAL_FINE_NAMES,
            'source_kind': 'code_to_manuscript_name_normalization',
        })
        outputs['table_s14'] = atomic_dataframe(glossary, PAPER_TABLE_DIR / 'table_s14_class_name_glossary')
        update_coverage_status('Table S14', 'DONE', outputs['table_s14']['csv'])

        # S15 exact numerical audit and corrected analytical footprint.
        local_footprint = int(sum((len(group) + 1) ** 2 for group in GROUP_IDXS))
        global_footprint = int((len(GROUP_IDXS) + 1) ** 2)
        total_footprint = local_footprint + global_footprint
        flat_footprint = int(len(FEATURE_COLS) ** 2)
        reduction = 1.0 - total_footprint / flat_footprint
        table_s15 = pd.DataFrame([
            ['Fine accuracy', fine_metrics['accuracy'], f'{100 * fine_metrics["accuracy"]:.2f}%', 'recomputed archived logits'],
            ['Fine balanced accuracy', fine_metrics['balanced_accuracy'], f'{100 * fine_metrics["balanced_accuracy"]:.2f}%', 'recomputed archived logits'],
            ['Fine macro-F1', fine_metrics['macro_f1'], f'{100 * fine_metrics["macro_f1"]:.2f}%', 'recomputed archived logits'],
            ['Coarse accuracy', coarse_metrics['accuracy'], f'{100 * coarse_metrics["accuracy"]:.2f}%', 'recomputed archived logits'],
            ['Coarse balanced accuracy', coarse_metrics['balanced_accuracy'], f'{100 * coarse_metrics["balanced_accuracy"]:.2f}%', 'recomputed archived logits'],
            ['Coarse macro-F1', coarse_metrics['macro_f1'], f'{100 * coarse_metrics["macro_f1"]:.2f}%', 'recomputed archived logits'],
            ['Malicious-class F1', bin_metrics['malicious_f1'], f'{100 * bin_metrics["malicious_f1"]:.2f}%', 'recomputed archived logits'],
            ['RAPS coverage alpha=.10', float(a10['coverage_percent']) / 100.0, f'{float(a10["coverage_percent"]):.2f}%', 'archived empirical diagnostic'],
            ['RAPS mean set size alpha=.10', float(a10['average_set_size']), f'{float(a10["average_set_size"]):.2f}', 'archived empirical diagnostic'],
            ['RAPS singleton fraction alpha=.10', float(a10['singleton_fraction_percent']) / 100.0, f'{float(a10["singleton_fraction_percent"]):.2f}%', 'archived empirical diagnostic'],
            ['Local attention footprint', local_footprint, str(local_footprint), 'sum (group_size+1)^2'],
            ['Global attention footprint', global_footprint, str(global_footprint), '(number_groups+1)^2'],
            ['Total attention footprint', total_footprint, str(total_footprint), 'local + global'],
            ['Reduction versus 46^2', reduction, f'{100 * reduction:.1f}%', '1-total/flat'],
        ], columns=['quantity', 'exact_value', 'display_value', 'calculation_or_source'])
        table_s15['source_kind'] = 'recomputed_or_algebraically_derived'
        outputs['table_s15'] = atomic_dataframe(table_s15, PAPER_TABLE_DIR / 'table_s15_exact_numerical_audit')
        update_coverage_status('Table S15', 'DONE', outputs['table_s15']['csv'])

        # ----------------------------------
        # S1 checkpoint selection (cache-first)
        # ----------------------------------
        candidate_path = Path(cfg.OUT_DIR) / 'publication_package' / 'tables' / 'candidate_model_selection.csv'
        if candidate_path.exists():
            candidates = pd.read_csv(candidate_path)
            candidates['source_kind'] = 'existing_v4_publication_checkpoint_evaluation'
            outputs['table_s01'] = atomic_dataframe(candidates, PAPER_TABLE_DIR / 'table_s01_checkpoint_selection')
            update_coverage_status('Table S1 + Figure S1', 'DONE', outputs['table_s01']['csv'])
            # Use detected columns rather than assuming one version of the v4 package.
            fine_col = next((c for c in candidates.columns if 'fine' in c.lower() and 'macro' in c.lower()), None)
            coarse_col = next((c for c in candidates.columns if 'coarse' in c.lower() and 'macro' in c.lower()), None)
            label_col = next((c for c in candidates.columns if c.lower() in {'candidate', 'name', 'model'}), candidates.columns[0])
            if fine_col and coarse_col:
                fig, ax = plt.subplots(figsize=(7.2, 5.2))
                ax.scatter(100 * candidates[fine_col].astype(float), 100 * candidates[coarse_col].astype(float), s=90)
                for _, row in candidates.iterrows():
                    ax.annotate(str(row[label_col]), (100 * float(row[fine_col]), 100 * float(row[coarse_col])), xytext=(5, 5), textcoords='offset points', fontsize=8)
                ax.set_xlabel('Validation fine macro-F1 (%)')
                ax.set_ylabel('Validation coarse macro-F1 (%)')
                ax.set_title('Validation checkpoint comparison')
                ax.grid(True, alpha=0.25)
                outputs['figure_s01'] = save_figure(fig, PAPER_FIG_DIR / 'figure_s01_checkpoint_comparison')
        else:
            update_coverage_status('Table S1 + Figure S1', 'PENDING', note='Run original v4 publication package cell to create candidate_model_selection.csv.')

        # -----------------------
        # Figure 1: imbalance/split
        # -----------------------
        all_coarse = np.concatenate([y_train_c, y_val_c, y_cal_c, y_test_c]).astype(np.int64)
        coarse_counts = np.bincount(all_coarse, minlength=NUM_COARSE_PAPER)
        all_fine = np.concatenate([y_train_f, y_val_f, y_cal_f, y_test_f]).astype(np.int64)
        fine_counts = np.bincount(all_fine, minlength=NUM_FINE_PAPER)
        fig, axes = plt.subplots(2, 2, figsize=(13.5, 9.5))
        ax = axes[0, 0]
        order = np.argsort(-coarse_counts)
        ax.barh([CANONICAL_COARSE_NAMES[i] for i in order][::-1], (100 * coarse_counts[order] / coarse_counts.sum())[::-1])
        ax.set_xlabel('Share of reported run (%)')
        ax.set_title('(a) Coarse-family prevalence')
        ax = axes[0, 1]
        ax.barh(table2.iloc[:4]['partition'][::-1], table2.iloc[:4]['share_percent'][::-1])
        ax.set_xlabel('Share of full run (%)')
        ax.set_title('(b) Reported processed-file-disjoint split')
        ax = axes[1, 0]
        ddos_dos_ids = [i for i, name in enumerate(CANONICAL_COARSE_NAMES) if _norm_label_key(name) in {'ddos', 'dos'}]
        dominant = 100 * coarse_counts[ddos_dos_ids].sum() / coarse_counts.sum() if ddos_dos_ids else np.nan
        ax.axis('off')
        ax.text(0.05, 0.70, f'DDoS + DoS\n{dominant:.2f}% of all flows', bbox=dict(boxstyle='round', facecolor='white'), fontsize=12)
        ax.text(0.55, 0.70, f'All other traffic\n{100-dominant:.2f}%', bbox=dict(boxstyle='round', facecolor='white'), fontsize=12)
        ax.text(0.05, 0.25, f'Largest fine class\n{fine_counts.max():,} flows', bbox=dict(boxstyle='round', facecolor='white'), fontsize=11)
        positive = fine_counts[fine_counts > 0]
        ax.text(0.55, 0.25, f'Smallest fine class\n{positive.min():,} flows', bbox=dict(boxstyle='round', facecolor='white'), fontsize=11)
        ax.set_title('(c) Long-tail concentration')
        ax = axes[1, 1]
        ax.bar(['Largest fine class', 'Smallest fine class'], [fine_counts.max(), positive.min()])
        ax.set_yscale('log')
        ax.set_ylabel('Count (log scale)')
        ax.set_title('(d) Largest-to-smallest contrast')
        fig.suptitle('CICIoT2023 class imbalance and reported evaluation split', fontsize=14)
        fig.tight_layout()
        outputs['figure_01'] = save_figure(fig, PAPER_FIG_DIR / 'figure_01_class_imbalance_and_split')
        update_coverage_status('Figure 1', 'DONE', outputs['figure_01']['png'])

        # Figure 4: decision-granularity metrics.
        fig, axes = plt.subplots(2, 2, figsize=(11.5, 8.0))
        tasks = table5['task'].tolist()
        panels = [
            ('Accuracy (%)', table5['accuracy_percent'].values),
            ('Balanced accuracy (%)', table5['balanced_accuracy_percent'].values),
            ('Macro-F1 / malicious F1 (%)', table5['macro_f1_percent'].values),
            ('Matthews correlation coefficient', table5['mcc'].values),
        ]
        for ax, (title, values) in zip(axes.flat, panels):
            ax.plot(values, tasks, marker='o')
            for value, task in zip(values, tasks):
                ax.annotate(f'{value:.3f}' if 'coefficient' in title else f'{value:.2f}', (value, task), xytext=(5, 0), textcoords='offset points', va='center')
            ax.set_title(title)
            ax.grid(True, alpha=0.25)
        fig.suptitle('Performance by decision granularity', fontsize=14)
        fig.tight_layout()
        outputs['figure_04'] = save_figure(fig, PAPER_FIG_DIR / 'figure_04_performance_by_granularity')
        update_coverage_status('Figure 4', 'DONE', outputs['figure_04']['png'])

        summary = {
            'status': 'DONE', 'signature': signature_hash(signature), 'outputs': outputs,
            'fine_metrics': fine_metrics, 'coarse_metrics': coarse_metrics, 'binary_metrics': bin_metrics,
            'support_f1_correlation': corr_table.iloc[0].to_dict(),
            'attention_footprint': {'local': local_footprint, 'global': global_footprint,
                                    'total': total_footprint, 'reduction_fraction': reduction},
        }
        atomic_json(summary_path, summary)
        return summary


ARCHIVED_CORE_SUMMARY = run_archived_core_tables() if PAPER.AUTO_RUN_LIGHT else {'status': 'DEFINED_NOT_RUN'}
print('Archived core status:', ARCHIVED_CORE_SUMMARY.get('status'))
paper_cell_done('P04', 'rebuild archived v4 tables, figures, diagnostics, and numerical audit', _started)


[P04] START: rebuild archived v4 tables, figures, diagnostics, and numerical audit
[2026-08-28 09:53:54] [INFO] [archived_core] stage started | signature=94d322c5346c
[2026-08-28 09:54:26] [INFO] [archived_core] stage completed | elapsed_seconds=31.441
Archived core status: DONE
[P04] DONE: rebuild archived v4 tables, figures, diagnostics, and numerical audit | elapsed=31.48s


In [7]:
_started = paper_cell_start('P05', 'generate code-aligned schematics, algorithm table, and formula map')

from matplotlib.patches import FancyBboxPatch, FancyArrowPatch


def _box(ax, xy, width, height, text, fontsize=9):
    x, y = xy
    patch = FancyBboxPatch((x, y), width, height, boxstyle='round,pad=0.02', linewidth=1.2,
                           edgecolor='black', facecolor='white')
    ax.add_patch(patch)
    ax.text(x + width / 2, y + height / 2, text, ha='center', va='center', fontsize=fontsize, wrap=True)
    return patch


def _arrow(ax, start, end):
    ax.add_patch(FancyArrowPatch(start, end, arrowstyle='-|>', mutation_scale=12, linewidth=1.1))


def run_reference_and_schematics() -> Dict[str, Any]:
    signature = {'stage_version': 3, 'base': PAPER_BASE_SIGNATURE_HASH}
    summary_path = PAPER_REPORT_DIR / 'reference_and_schematics_summary.json'
    with paper_stage('reference_and_schematics', signature) as stage:
        if stage.skip and summary_path.exists():
            return read_json(summary_path, {})
        outputs: Dict[str, Any] = {}

        algorithm_rows = [
            [1, 'Tokenize 46 sanitized numerical features and form five semantic group sequences.'],
            [2, 'Apply local self-attention within each group and global attention across the five summaries.'],
            [3, 'Compute fine, coarse, and binary logits; optimize LDAM-DRW, auxiliary cross-entropy, and hierarchical KL losses.'],
            [4, 'Select the checkpoint using validation fine macro-F1.'],
            [5, 'Fit scalar temperatures and compute deterministic RAPS scores with k_reg=3 and lambda=0.01.'],
            [6, 'Estimate predicted-coarse Mondrian thresholds with the higher empirical quantile; fall back to the global threshold for groups with n<200.'],
            [7, 'Return all ranked labels satisfying the threshold and force a minimum set size of one.'],
            [8, 'For streams, send mean fine-head entropy from each 50,000-flow batch to ADWIN and the conformal martingale.'],
            [9, 'If a detector triggers, apply Tent-LN and, when at least 512 singleton pseudo-labels exist, update all three heads.'],
            [10, 'Treat the archived one-pool RAPS metrics as empirical diagnostics; use the disjoint D_prob/D_conf run for split-conformal validity.'],
        ]
        algorithm = pd.DataFrame(algorithm_rows, columns=['step', 'operation'])
        algorithm['executable_stage_or_function'] = [
            'v4 preprocessing / P01 context', 'CamelotIDSv2.forward', 'train_resumable_hierarchical',
            'validation checkpoint selection', 'fit_temperature_np + build_mondrian_raps_thresholds',
            'build_mondrian_raps_thresholds', 'raps_evaluate_detailed_chunked',
            'run_stream_stationarity', 'TentLayerNormAdapter + pseudo_label_head_tune',
            'P04 archived_core + P06 confirmatory_split_conformal',
        ]
        outputs['algorithm_01'] = atomic_dataframe(algorithm, PAPER_TABLE_DIR / 'algorithm_01_executable_pipeline')
        update_coverage_status('Algorithm 1', 'DONE', outputs['algorithm_01']['csv'])

        formula_map = pd.DataFrame([
            [1, 'Attention footprint', 'sum_k (|G_k|+1)^2 + (K+1)^2', 'P04 attention_footprint'],
            [2, 'Feature-specific tokenization', 'x_i W_i + b_i', 'FeatureTokenizer.forward'],
            [3, 'Local group encoding', 'Transformer(group token + feature tokens)', 'CamelotIDSv2.local_encoder'],
            [4, 'Global fusion', 'Transformer(CLS + group summaries)', 'CamelotIDSv2.global_encoder'],
            [5, 'Hierarchical objective', 'LDAM + 0.30 CE_c + 0.20 CE_b + 0.10 KL_c + 0.05 KL_b', 'hierarchical_objective'],
            [6, 'Temperature scaling', 'softmax(logits / T)', 'fit_temperature_np / softmax_np'],
            [7, 'RAPS true-label score', 'cumulative ranked probability + lambda max(rank-k_reg,0)', 'raps_true_scores_chunked'],
            [8, 'Finite-sample quantile', 'ceil((n+1)(1-alpha))/n with higher order statistic', 'finite_quantile_level / higher_quantile'],
            [9, 'Prediction set', 'ranked labels with score <= group threshold; max set size at least 1', 'raps_evaluate_detailed_chunked'],
            [10, 'Entropy monitor', '-sum_c p_c log p_c', 'stream_batch_entropy'],
            [11, 'Logical-batch statistic', 'mean entropy over 50,000 flows', 'run_stream_stationarity'],
        ], columns=['equation', 'concept', 'formula_summary', 'implementation'])
        formula_map['source_kind'] = 'code_formula_alignment'
        outputs['equation_map'] = atomic_dataframe(formula_map, PAPER_TABLE_DIR / 'equations_01_11_code_alignment')
        update_coverage_status('Equations 1-11', 'DONE', outputs['equation_map']['csv'])

        # Figure 2: executable CAMELOT-IDS pipeline.
        fig, ax = plt.subplots(figsize=(10.5, 12.0))
        ax.set_xlim(0, 10)
        ax.set_ylim(0, 14)
        ax.axis('off')
        _box(ax, (1.5, 12.6), 7.0, 0.7, '46 sanitized numerical flow features')
        _box(ax, (1.0, 11.35), 8.0, 0.8, 'Five semantic groups: rates (3), flags/counts (12), protocols (14), statistics (13), residual (4)')
        _box(ax, (1.5, 10.0), 7.0, 0.8, 'Local self-attention in each group (+ one learnable group token)')
        _box(ax, (1.5, 8.65), 7.0, 0.8, 'Global Transformer over five group summaries (+ CLS token)')
        _box(ax, (3.0, 7.4), 4.0, 0.65, 'Shared CLS representation')
        _box(ax, (0.35, 5.85), 2.75, 0.9, 'Fine head\n34 classes\nLDAM-DRW')
        _box(ax, (3.62, 5.85), 2.75, 0.9, 'Coarse head\n8 families\nweighted CE')
        _box(ax, (6.9, 5.85), 2.75, 0.9, 'Binary head\n2 labels\nweighted CE')
        _box(ax, (1.25, 4.45), 7.5, 0.75, 'Hierarchical consistency: KL(fine-aggregated || coarse) + KL(fine-aggregated || binary)')
        _box(ax, (0.55, 2.75), 4.25, 1.05, 'Uncertainty path\nTemperature scaling; deterministic predicted-coarse Mondrian RAPS; top-1 fallback')
        _box(ax, (5.2, 2.75), 4.25, 1.05, 'Streaming path\nMean fine-head entropy per 50,000-flow batch; ADWIN + martingale; Tent-LN/head tuning only after trigger')
        _box(ax, (1.25, 1.25), 7.5, 0.75, 'Operational output: top-1 for singleton sets; otherwise full candidate set')
        _box(ax, (1.25, 0.15), 7.5, 0.55, 'Validity boundary: archived one-pool analysis is empirical; disjoint D_prob/D_conf is confirmatory split conformal', fontsize=8)
        vertical_centers = [12.6, 11.35, 10.0, 8.65, 7.4]
        for y1, y2 in zip([12.6, 11.35, 10.0, 8.65], [12.15, 10.8, 9.45, 8.05]):
            _arrow(ax, (5.0, y1), (5.0, y2))
        for x in [1.72, 5.0, 8.28]:
            _arrow(ax, (5.0, 7.4), (x, 6.75))
        for x in [1.72, 5.0, 8.28]:
            _arrow(ax, (x, 5.85), (5.0, 5.2))
        _arrow(ax, (5.0, 4.45), (2.7, 3.8))
        _arrow(ax, (5.0, 4.45), (7.3, 3.8))
        _arrow(ax, (2.7, 2.75), (5.0, 2.0))
        _arrow(ax, (7.3, 2.75), (5.0, 2.0))
        _arrow(ax, (5.0, 1.25), (5.0, 0.7))
        ax.set_title('CAMELOT-IDS archived pipeline and validity boundary', fontsize=15, pad=12)
        outputs['figure_02'] = save_figure(fig, PAPER_FIG_DIR / 'figure_02_code_aligned_pipeline')
        update_coverage_status('Figure 2', 'DONE', outputs['figure_02']['png'])

        # Figure 3: archived and confirmatory calibration protocols.
        fig, axes = plt.subplots(1, 2, figsize=(13.5, 6.7))
        for ax in axes:
            ax.set_xlim(0, 10)
            ax.set_ylim(0, 10)
            ax.axis('off')
        axes[0].set_title('(a) Archived reported analysis')
        _box(axes[0], (1.0, 7.8), 3.5, 0.9, f'Full run\n{sum(len(PAPER_DATA[k]["X"]) for k in PAPER_DATA):,} flows')
        y_positions = [('Train', len(X_train), 7.8), ('Validation', len(X_val), 6.2), ('Calibration', len(X_cal), 4.6), ('Test', len(X_test), 3.0)]
        for label, n, y in y_positions:
            _box(axes[0], (5.5, y), 3.5, 0.9, f'{label}\n{n:,}')
            _arrow(axes[0], (4.5, 8.25), (5.5, y + 0.45))
        axes[0].text(5.0, 1.1, 'Calibration pool reused for both temperature fitting\nand RAPS threshold estimation: empirical diagnostic.', ha='center', va='center', fontsize=9)

        axes[1].set_title('(b) Validity-preserving confirmatory protocol')
        _box(axes[1], (0.55, 7.7), 3.8, 1.0, f'Probability calibration D_prob\napproximately {len(X_cal)//2:,} rows')
        _box(axes[1], (5.65, 7.7), 3.8, 1.0, f'Conformal thresholds D_conf\napproximately {len(X_cal)-len(X_cal)//2:,} rows')
        _box(axes[1], (1.8, 5.45), 6.4, 1.0, 'Fit temperature maps on D_prob; freeze transformation')
        _box(axes[1], (1.8, 3.3), 6.4, 1.0, 'Estimate Mondrian RAPS thresholds on D_conf only')
        _box(axes[1], (2.5, 1.15), 5.0, 1.0, f'Apply frozen maps and thresholds to unchanged test set\n{len(X_test):,} rows')
        _arrow(axes[1], (2.45, 7.7), (3.5, 6.45))
        _arrow(axes[1], (7.55, 7.7), (6.5, 4.3))
        _arrow(axes[1], (5.0, 5.45), (5.0, 4.3))
        _arrow(axes[1], (5.0, 3.3), (5.0, 2.15))
        fig.suptitle('Reported evaluation and confirmatory calibration protocols', fontsize=15)
        fig.tight_layout()
        outputs['figure_03'] = save_figure(fig, PAPER_FIG_DIR / 'figure_03_calibration_protocols')
        update_coverage_status('Figure 3', 'DONE', outputs['figure_03']['png'])

        summary = {'status': 'DONE', 'outputs': outputs, 'signature': signature_hash(signature)}
        atomic_json(summary_path, summary)
        return summary


REFERENCE_SCHEMATIC_SUMMARY = run_reference_and_schematics() if PAPER.AUTO_RUN_LIGHT else {'status': 'DEFINED_NOT_RUN'}
print('Reference/schematic status:', REFERENCE_SCHEMATIC_SUMMARY.get('status'))
paper_cell_done('P05', 'generate code-aligned schematics, algorithm table, and formula map', _started)


[P05] START: generate code-aligned schematics, algorithm table, and formula map
[2026-08-28 09:54:26] [INFO] [reference_and_schematics] stage started | signature=b2782e962c11
[2026-08-28 09:54:29] [INFO] [reference_and_schematics] stage completed | elapsed_seconds=3.536
Reference/schematic status: DONE
[P05] DONE: generate code-aligned schematics, algorithm table, and formula map | elapsed=3.61s


In [ ]:
_started = paper_cell_start('P06', 'confirmatory disjoint D_prob/D_conf split-conformal analysis')


def _fit_temperature_on_indices(logits: np.ndarray, labels: np.ndarray, indices: np.ndarray) -> float:
    idx = np.asarray(indices, dtype=np.int64)
    # Smoke mode is explicitly non-paper and may cap the fitting subset.
    if PAPER.PROFILE == 'smoke' and len(idx) > PAPER.SMOKE_CAL_CAP:
        idx = idx[:PAPER.SMOKE_CAL_CAP]
    return float(fit_temperature_np(
        np.ascontiguousarray(np.asarray(logits)[idx], dtype=np.float32),
        np.ascontiguousarray(np.asarray(labels)[idx], dtype=np.int64),
    ))


def run_one_confirmatory_seed(seed: int, alphas: Sequence[float]) -> pd.DataFrame:
    seed_dir = stage_path('confirmatory_split_conformal') / f'seed_{int(seed)}'
    seed_dir.mkdir(parents=True, exist_ok=True)
    result_path = seed_dir / 'results.csv'
    meta_path = seed_dir / 'metadata.json'
    signature = {
        'base': PAPER_BASE_SIGNATURE_HASH,
        'seed': int(seed), 'alphas': list(map(float, alphas)), 'n_cal': int(len(PAPER_Y_CAL_FINE)),
        'k_reg': PAPER.RAPS_KREG, 'lambda': PAPER.RAPS_LAMBDA, 'min_group': PAPER.RAPS_MIN_GROUP,
        'profile': PAPER.PROFILE,
    }
    prior = read_json(meta_path, {})
    if (not PAPER.FORCE_RERUN and result_path.exists() and prior.get('status') == 'DONE' and
            prior.get('signature') == signature_hash(signature)):
        paper_log('confirmatory seed reused', stage='confirmatory_split_conformal', seed=seed)
        return pd.read_csv(result_path)

    rng = np.random.default_rng(int(seed))
    perm = rng.permutation(len(PAPER_Y_CAL_FINE)).astype(np.int64)
    n_prob = len(perm) // 2
    idx_prob = perm[:n_prob]
    idx_conf = perm[n_prob:]
    if np.intersect1d(idx_prob, idx_conf, assume_unique=True).size != 0:
        raise RuntimeError('D_prob and D_conf are not disjoint; refusing confirmatory analysis.')

    atomic_json(meta_path, {
        'status': 'RUNNING', 'signature': signature_hash(signature), 'signature_payload': signature,
        'seed': int(seed), 'D_prob_size': int(len(idx_prob)), 'D_conf_size': int(len(idx_conf)),
        'started_at': time.strftime('%Y-%m-%d %H:%M:%S'),
    })
    paper_log('fitting disjoint probability calibration', stage='confirmatory_split_conformal',
              seed=seed, D_prob=len(idx_prob), D_conf=len(idx_conf))

    temp_f = _fit_temperature_on_indices(PAPER_CAL_OUT['fine'], PAPER_Y_CAL_FINE, idx_prob)
    temp_c = _fit_temperature_on_indices(PAPER_CAL_OUT['coarse'], PAPER_Y_CAL_COARSE, idx_prob)
    temp_b = _fit_temperature_on_indices(PAPER_CAL_OUT['bin'], PAPER_Y_CAL_BINARY, idx_prob)
    paper_log('confirmatory temperatures fitted', stage='confirmatory_split_conformal', seed=seed,
              fine=round(temp_f, 6), coarse=round(temp_c, 6), binary=round(temp_b, 6))

    conf_probs_f = softmax_np(np.asarray(PAPER_CAL_OUT['fine'])[idx_conf] / max(temp_f, 1e-6))
    conf_probs_c = softmax_np(np.asarray(PAPER_CAL_OUT['coarse'])[idx_conf] / max(temp_c, 1e-6))
    test_probs_f = softmax_np(np.asarray(PAPER_TEST_OUT['fine']) / max(temp_f, 1e-6))
    test_probs_c = softmax_np(np.asarray(PAPER_TEST_OUT['coarse']) / max(temp_c, 1e-6))
    test_probs_b = softmax_np(np.asarray(PAPER_TEST_OUT['bin']) / max(temp_b, 1e-6))

    thresholds = build_mondrian_raps_thresholds(
        conf_probs_f, conf_probs_c, PAPER_Y_CAL_FINE[idx_conf], alphas,
        PAPER.RAPS_KREG, PAPER.RAPS_LAMBDA, PAPER.RAPS_MIN_GROUP,
    )
    atomic_json(seed_dir / 'thresholds.json', thresholds)

    rows: List[Dict[str, Any]] = []
    for alpha in alphas:
        detail = raps_evaluate_detailed_chunked(
            test_probs_f, test_probs_c, PAPER_Y_TEST_FINE, thresholds,
            float(alpha), int(benign_fine_id),
        )
        row = {
            'split': f'Seed {int(seed)}', 'seed': int(seed),
            'D_prob_size': int(len(idx_prob)), 'D_conf_size': int(len(idx_conf)),
            'alpha': float(alpha), 'coverage_percent': 100 * detail['coverage'],
            'average_set_size': detail['avg_set_size'],
            'singleton_fraction_percent': 100 * detail['singleton_fraction'],
            'singleton_risk_percent': 100 * detail['singleton_risk'],
            'benign_fpr_singletons_percent': 100 * detail['benign_fpr_singletons'],
            'committed_decisions': detail['committed_decisions'],
            'temperature_fine': temp_f, 'temperature_coarse': temp_c, 'temperature_binary': temp_b,
            'test_fine_ece': expected_calibration_error_local(PAPER_Y_TEST_FINE, test_probs_f, PAPER.ECE_BINS),
            'test_binary_malicious_f1_percent': 100 * binary_metrics(PAPER_Y_TEST_BINARY, test_probs_b)['malicious_f1'],
            'source_kind': 'measured_disjoint_split_conformal',
            'profile': PAPER.PROFILE,
        }
        rows.append(row)
        paper_log('confirmatory alpha completed', stage='confirmatory_split_conformal', seed=seed,
                  alpha=alpha, coverage_percent=round(row['coverage_percent'], 4),
                  avg_set_size=round(row['average_set_size'], 4))
        # Safe per-alpha progress file.
        atomic_dataframe(pd.DataFrame(rows), seed_dir / 'results_partial')
        check_deadline(f'confirmatory seed={seed} alpha={alpha}')

    result = pd.DataFrame(rows)
    atomic_dataframe(result, seed_dir / 'results')
    atomic_json(meta_path, {
        'status': 'DONE', 'signature': signature_hash(signature), 'signature_payload': signature,
        'seed': int(seed), 'D_prob_size': int(len(idx_prob)), 'D_conf_size': int(len(idx_conf)),
        'temperatures': {'fine': temp_f, 'coarse': temp_c, 'binary': temp_b},
        'completed_at': time.strftime('%Y-%m-%d %H:%M:%S'),
    })
    del conf_probs_f, conf_probs_c, test_probs_f, test_probs_c, test_probs_b, perm, idx_prob, idx_conf
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return result


def run_confirmatory_split_conformal() -> pd.DataFrame:
    signature = {
        'base': PAPER_BASE_SIGNATURE_HASH, 'stage_version': 4,
        'seeds': list(PAPER.CONFIRMATORY_SEEDS), 'profile': PAPER.PROFILE,
        'split_policy': 'flow-level random equal partition',
    }
    combined_path = PAPER_TABLE_DIR / 'table_s16_confirmatory_split_conformal.csv'
    with paper_stage('confirmatory_split_conformal', signature) as stage:
        if stage.skip and combined_path.exists():
            return pd.read_csv(combined_path)
        frames = []
        for seed in PAPER.CONFIRMATORY_SEEDS:
            alphas = PAPER.ALPHAS if int(seed) == 42 else (0.10,)
            frames.append(run_one_confirmatory_seed(int(seed), alphas))
            check_deadline(f'after confirmatory seed {seed}')
        detail = pd.concat(frames, ignore_index=True)
        alpha10 = detail[np.isclose(detail['alpha'].astype(float), 0.10)].copy()
        mean_row = {
            'split': 'Mean +/- SD (alpha=0.10)', 'seed': np.nan,
            'D_prob_size': np.nan, 'D_conf_size': np.nan, 'alpha': 0.10,
            'coverage_percent': float(alpha10['coverage_percent'].mean()),
            'coverage_sd_percent': float(alpha10['coverage_percent'].std(ddof=1)),
            'average_set_size': float(alpha10['average_set_size'].mean()),
            'average_set_size_sd': float(alpha10['average_set_size'].std(ddof=1)),
            'singleton_fraction_percent': float(alpha10['singleton_fraction_percent'].mean()),
            'singleton_fraction_sd_percent': float(alpha10['singleton_fraction_percent'].std(ddof=1)),
            'singleton_risk_percent': float(alpha10['singleton_risk_percent'].mean()),
            'singleton_risk_sd_percent': float(alpha10['singleton_risk_percent'].std(ddof=1)),
            'source_kind': 'aggregate_of_measured_disjoint_runs', 'profile': PAPER.PROFILE,
        }
        final = pd.concat([detail, pd.DataFrame([mean_row])], ignore_index=True, sort=False)
        atomic_dataframe(final, PAPER_TABLE_DIR / 'table_s16_confirmatory_split_conformal')
        atomic_dataframe(detail, PAPER_TABLE_DIR / 'table_s16_confirmatory_split_conformal_seed_rows')
        update_coverage_status('Table S16', 'DONE', str(combined_path),
                               'D_prob and D_conf are disjoint; validity still depends on the stated exchangeability assumptions.')
        return final


CONFIRMATORY_S16 = run_confirmatory_split_conformal() if PAPER.AUTO_RUN_LIGHT else pd.DataFrame()
if len(CONFIRMATORY_S16):
    display(CONFIRMATORY_S16)
paper_cell_done('P06', 'confirmatory disjoint D_prob/D_conf split-conformal analysis', _started)


In [ ]:
_started = paper_cell_start('P07', 'provenance-level leakage audit and capture-cluster bootstrap')


def _read_id_artifact(path_text: str) -> Optional[pd.DataFrame]:
    text = str(path_text or '').strip()
    if not text:
        return None
    path = Path(text).expanduser()
    if not path.exists():
        raise FileNotFoundError(path)
    suffix = path.suffix.lower()
    if suffix == '.csv':
        return pd.read_csv(path, low_memory=False)
    if suffix in {'.parquet', '.pq'}:
        return pd.read_parquet(path)
    if suffix == '.json':
        payload = json.loads(path.read_text(encoding='utf-8'))
        return pd.DataFrame(payload if isinstance(payload, list) else payload.get('rows', payload))
    if suffix == '.npy':
        values = np.load(path, allow_pickle=False)
        return pd.DataFrame({'root_capture_id': values.astype(str)})
    raise ValueError(f'Unsupported manifest/ID file: {path}')


def _detect_partition_column(df: pd.DataFrame) -> Optional[str]:
    normal = {_norm_label_key(c): c for c in df.columns}
    for candidate in ('partition', 'split', 'subset', 'fold'):
        if _norm_label_key(candidate) in normal:
            return normal[_norm_label_key(candidate)]
    return None


def _detect_manifest_column(df: pd.DataFrame, candidates: Sequence[str]) -> Optional[str]:
    normal = {_norm_label_key(c): c for c in df.columns}
    for candidate in candidates:
        key = _norm_label_key(candidate)
        if key in normal:
            return normal[key]
    return None


def run_provenance_audit() -> pd.DataFrame:
    manifest_text = str(PAPER.PROVENANCE_MANIFEST or '').strip()
    if not manifest_text:
        update_coverage_status('Table 3', 'BLOCKED', note='Set CAMELOT_PROVENANCE_MANIFEST to the completed split manifest.')
        return write_blocked_stage('provenance_audit', 'CAMELOT_PROVENANCE_MANIFEST is not set.')
    manifest_path = Path(manifest_text).expanduser()
    signature = {'stage_version': 3, 'manifest': str(manifest_path), 'mtime': manifest_path.stat().st_mtime if manifest_path.exists() else None}
    out_path = PAPER_TABLE_DIR / 'table_03_provenance_leakage_audit.csv'
    with paper_stage('provenance_audit', signature) as stage:
        if stage.skip and out_path.exists():
            return pd.read_csv(out_path)
        manifest = _read_id_artifact(manifest_text)
        if manifest is None or manifest.empty:
            raise RuntimeError('The provenance manifest is empty.')
        part_col = _detect_partition_column(manifest)
        if part_col is None:
            mark_stage('provenance_audit', 'BLOCKED', reason='Manifest has no partition/split column.', columns=list(manifest.columns))
            update_coverage_status('Table 3', 'BLOCKED', note='Manifest must identify train/validation/calibration/test partitions.')
            return pd.DataFrame([{'status': 'BLOCKED', 'reason': 'No partition/split column', 'columns': ', '.join(map(str, manifest.columns))}])
        manifest = manifest.copy()
        manifest['_partition'] = manifest[part_col].astype(str).str.strip().str.lower().replace({'valid': 'validation', 'val': 'validation', 'cal': 'calibration'})
        partitions = sorted(manifest['_partition'].dropna().unique().tolist())
        expected = {'train', 'validation', 'calibration', 'test'}
        missing_parts = sorted(expected.difference(partitions))
        if missing_parts:
            paper_log('manifest does not contain every expected partition label', level='WARNING', stage='provenance_audit', missing=missing_parts, observed=partitions)

        levels = [
            ('Processed feature file', ['__src_file', 'src_file', 'source_file', 'csv_file'], 'CSV/source-file identifier'),
            ('Original capture', ['root_capture_id', 'capture_id', 'pcap_id'], 'Root PCAP/capture identifier'),
            ('Attack run', ['attack_run_id', 'run_id', 'experiment_id'], 'Original experiment/attack-run identifier'),
            ('Temporal session', ['session_id', 'temporal_session_id', 'time_interval_id'], 'Original capture/run time interval'),
            ('Physical device', ['device_id', 'device_identifier', 'victim_device_id'], 'Available attacker/victim/device identifier'),
        ]
        rows: List[Dict[str, Any]] = []
        for audit_level, candidates, unit in levels:
            col = _detect_manifest_column(manifest, candidates)
            if col is None:
                rows.append({'audit_level': audit_level, 'independence_audit_unit': unit, 'column': '',
                             'cross_partition_overlap_count': np.nan, 'status': 'BLOCKED_COLUMN_MISSING',
                             'cross_partition_result': 'Required column absent from supplied manifest.'})
                continue
            sets = {
                part: set(manifest.loc[manifest['_partition'] == part, col].dropna().astype(str).tolist())
                for part in partitions
            }
            overlaps: set = set()
            pair_rows = []
            for i, left in enumerate(partitions):
                for right in partitions[i + 1:]:
                    common = sets[left].intersection(sets[right])
                    overlaps.update(common)
                    pair_rows.append({'left': left, 'right': right, 'overlap_count': len(common)})
            result_text = f'{len(overlaps)} unique values overlap across partitions; {manifest[col].nunique(dropna=True)} unique values mapped.'
            rows.append({'audit_level': audit_level, 'independence_audit_unit': unit, 'column': col,
                         'cross_partition_overlap_count': int(len(overlaps)),
                         'status': 'PASS' if len(overlaps) == 0 else 'FAIL',
                         'cross_partition_result': result_text})
            atomic_dataframe(pd.DataFrame(pair_rows), PAPER_REPORT_DIR / f'provenance_pairs_{_norm_label_key(audit_level)}')

        rows.extend([
            {'audit_level': 'Preprocessing', 'independence_audit_unit': 'Imputation/scaling/clipping parameters',
             'column': '', 'cross_partition_overlap_count': np.nan, 'status': 'CODE_AUDIT',
             'cross_partition_result': 'v4 fits preprocessing parameters on the training partition only.'},
            {'audit_level': 'Model identifiers', 'independence_audit_unit': 'IP/MAC/ports/device name/time',
             'column': '', 'cross_partition_overlap_count': np.nan, 'status': 'CODE_AUDIT',
             'cross_partition_result': 'Direct identifiers are excluded by the v4 feature-selection rules.'},
            {'audit_level': 'Released artifact', 'independence_audit_unit': 'Manifest + audit outputs',
             'column': '', 'cross_partition_overlap_count': np.nan, 'status': 'GENERATED',
             'cross_partition_result': str(PAPER_REPORT_DIR / 'provenance_manifest_normalized.csv')},
        ])
        table3 = pd.DataFrame(rows)
        atomic_dataframe(table3, PAPER_TABLE_DIR / 'table_03_provenance_leakage_audit')
        atomic_dataframe(manifest, PAPER_REPORT_DIR / 'provenance_manifest_normalized')
        audit_pass = bool((table3.loc[table3['status'].isin(['PASS', 'FAIL']), 'status'] == 'PASS').all())
        update_coverage_status('Table 3', 'DONE' if audit_pass else 'CHECK', str(out_path),
                               'Computed from supplied manifest; independently review metadata provenance before deployment.')
        return table3


def _load_row_aligned_test_cluster_ids() -> Optional[np.ndarray]:
    text = str(PAPER.TEST_CLUSTER_IDS or '').strip()
    if text:
        frame = _read_id_artifact(text)
        if frame is None or frame.empty:
            return None
        col = _detect_manifest_column(frame, ['root_capture_id', 'capture_id', 'cluster_id']) or frame.columns[0]
        values = frame[col].astype(str).values
        if len(values) != len(PAPER_Y_TEST_FINE):
            raise ValueError(f'Test cluster-ID rows {len(values)} != test prediction rows {len(PAPER_Y_TEST_FINE)}.')
        return values
    # A row-level manifest can also supply IDs directly.
    manifest_text = str(PAPER.PROVENANCE_MANIFEST or '').strip()
    if manifest_text:
        frame = _read_id_artifact(manifest_text)
        if frame is not None and not frame.empty:
            part_col = _detect_partition_column(frame)
            capture_col = _detect_manifest_column(frame, ['root_capture_id', 'capture_id', 'cluster_id'])
            if part_col and capture_col:
                test_rows = frame[frame[part_col].astype(str).str.strip().str.lower().eq('test')]
                if len(test_rows) == len(PAPER_Y_TEST_FINE):
                    return test_rows[capture_col].astype(str).values
    return None


def _metrics_from_confusion(cm: np.ndarray) -> Tuple[float, float]:
    matrix = np.asarray(cm, dtype=np.float64)
    total = matrix.sum()
    accuracy = float(np.trace(matrix) / total) if total else float('nan')
    tp = np.diag(matrix)
    fp = matrix.sum(axis=0) - tp
    fn = matrix.sum(axis=1) - tp
    denom = 2 * tp + fp + fn
    f1 = np.divide(2 * tp, denom, out=np.zeros_like(tp), where=denom > 0)
    return accuracy, float(np.mean(f1))


def _cluster_confusions(y_true: np.ndarray, y_pred: np.ndarray, clusters: np.ndarray, n_classes: int) -> Tuple[List[str], np.ndarray]:
    cluster_names = sorted(pd.unique(np.asarray(clusters).astype(str)).tolist())
    mats = np.zeros((len(cluster_names), int(n_classes), int(n_classes)), dtype=np.int64)
    cluster_index = {name: i for i, name in enumerate(cluster_names)}
    encoded = np.asarray([cluster_index[str(x)] for x in clusters], dtype=np.int64)
    # One bincount per cluster is efficient for a small number of captures.
    for i in range(len(cluster_names)):
        mask = encoded == i
        flat = np.asarray(y_true[mask], dtype=np.int64) * int(n_classes) + np.asarray(y_pred[mask], dtype=np.int64)
        mats[i] = np.bincount(flat, minlength=int(n_classes) ** 2).reshape(int(n_classes), int(n_classes))
    return cluster_names, mats


def _bootstrap_one_model(
    model_name: str,
    y_fine: np.ndarray,
    pred_fine: np.ndarray,
    y_coarse: np.ndarray,
    pred_coarse: np.ndarray,
    clusters: np.ndarray,
    reps: int,
    seed: int,
) -> Tuple[Dict[str, Any], pd.DataFrame]:
    cluster_names, fine_mats = _cluster_confusions(y_fine, pred_fine, clusters, NUM_FINE_PAPER)
    cluster_names_c, coarse_mats = _cluster_confusions(y_coarse, pred_coarse, clusters, NUM_COARSE_PAPER)
    if cluster_names != cluster_names_c:
        raise RuntimeError('Fine and coarse cluster ordering mismatch.')
    rng = np.random.default_rng(int(seed))
    k = len(cluster_names)
    draws = np.empty((int(reps), 3), dtype=np.float64)
    progress_path = stage_path('cluster_bootstrap') / f'{_norm_label_key(model_name)}_draws.npy'
    meta_path = progress_path.with_suffix('.meta.json')
    start_rep = 0
    if progress_path.exists() and meta_path.exists() and not PAPER.FORCE_RERUN:
        existing = np.load(progress_path)
        if existing.ndim == 2 and existing.shape[1] == 3 and len(existing) <= reps:
            draws[:len(existing)] = existing
            start_rep = len(existing)
            rng = np.random.default_rng(int(seed))
            # Advance exactly as if the completed draws had been generated.
            _ = rng.integers(0, k, size=(start_rep, k))
            paper_log('cluster-bootstrap partial draws resumed', stage='cluster_bootstrap', model=model_name, completed=start_rep)
    for rep in range(start_rep, int(reps)):
        sampled = rng.integers(0, k, size=k)
        cm_f = fine_mats[sampled].sum(axis=0)
        cm_c = coarse_mats[sampled].sum(axis=0)
        fine_acc, fine_mf1 = _metrics_from_confusion(cm_f)
        _, coarse_mf1 = _metrics_from_confusion(cm_c)
        draws[rep] = [fine_mf1, fine_acc, coarse_mf1]
        if (rep + 1) % 250 == 0 or rep + 1 == int(reps):
            atomic_npy(progress_path, draws[:rep + 1])
            atomic_json(meta_path, {'model': model_name, 'completed_reps': rep + 1, 'total_reps': int(reps), 'seed': int(seed)})
            paper_log('cluster-bootstrap progress', stage='cluster_bootstrap', model=model_name, reps=f'{rep+1}/{reps}')
            check_deadline(f'cluster bootstrap {model_name} rep {rep+1}')
    ci = np.quantile(draws, [0.025, 0.975], axis=0)
    point_f_acc, point_f_mf1 = _metrics_from_confusion(fine_mats.sum(axis=0))
    _, point_c_mf1 = _metrics_from_confusion(coarse_mats.sum(axis=0))
    row = {
        'model': model_name, 'clusters': int(k), 'bootstrap_reps': int(reps),
        'fine_macro_f1_percent': 100 * point_f_mf1,
        'fine_macro_f1_ci_low_percent': 100 * ci[0, 0], 'fine_macro_f1_ci_high_percent': 100 * ci[1, 0],
        'fine_accuracy_percent': 100 * point_f_acc,
        'fine_accuracy_ci_low_percent': 100 * ci[0, 1], 'fine_accuracy_ci_high_percent': 100 * ci[1, 1],
        'coarse_macro_f1_percent': 100 * point_c_mf1,
        'coarse_macro_f1_ci_low_percent': 100 * ci[0, 2], 'coarse_macro_f1_ci_high_percent': 100 * ci[1, 2],
        'source_kind': 'capture_cluster_bootstrap',
    }
    draw_df = pd.DataFrame(draws, columns=['fine_macro_f1', 'fine_accuracy', 'coarse_macro_f1'])
    draw_df.insert(0, 'replicate', np.arange(1, len(draw_df) + 1))
    draw_df.insert(0, 'model', model_name)
    return row, draw_df


def _load_baseline_prediction_files() -> List[Tuple[str, Dict[str, np.ndarray]]]:
    root = PAPER_CACHE_DIR / 'baseline_predictions'
    results: List[Tuple[str, Dict[str, np.ndarray]]] = []
    if not root.exists():
        return results
    for path in sorted(root.glob('*.npz')):
        with np.load(path, allow_pickle=False) as blob:
            keys = set(blob.files)
            if not {'pred_fine', 'pred_coarse'}.issubset(keys):
                continue
            payload = {key: np.asarray(blob[key]) for key in blob.files}
        if len(payload['pred_fine']) != len(PAPER_Y_TEST_FINE):
            paper_log('baseline predictions ignored because row count differs from test outputs', level='WARNING', stage='cluster_bootstrap', file=str(path))
            continue
        results.append((path.stem, payload))
    return results


def run_cluster_bootstrap() -> pd.DataFrame:
    clusters = _load_row_aligned_test_cluster_ids()
    if clusters is None:
        update_coverage_status('Table S17', 'BLOCKED', note='Set CAMELOT_TEST_CLUSTER_IDS to a row-aligned root_capture_id file.')
        return write_blocked_stage('cluster_bootstrap', 'No row-aligned test root-capture IDs are available.')
    unique_clusters = pd.unique(clusters.astype(str))
    baseline_root = PAPER_CACHE_DIR / 'baseline_predictions'
    baseline_files = []
    if baseline_root.exists():
        baseline_files = [(p.name, p.stat().st_size, p.stat().st_mtime_ns) for p in sorted(baseline_root.glob('*.npz'))]
    signature = {
        'stage_version': 5, 'base': PAPER_BASE_SIGNATURE_HASH, 'reps': PAPER.CLUSTER_BOOTSTRAP_REPS,
        'clusters': sorted(map(str, unique_clusters.tolist())), 'baseline_prediction_files': baseline_files,
    }
    out_path = PAPER_TABLE_DIR / 'table_s17_capture_cluster_bootstrap.csv'
    with paper_stage('cluster_bootstrap', signature) as stage:
        if stage.skip and out_path.exists():
            return pd.read_csv(out_path)
        models: List[Tuple[str, Dict[str, np.ndarray]]] = [
            ('CAMELOT-IDS', {
                'pred_fine': PAPER_TEST_PROBS_FINE.argmax(axis=1).astype(np.int64),
                'pred_coarse': PAPER_TEST_PROBS_COARSE.argmax(axis=1).astype(np.int64),
            })
        ]
        models.extend(_load_baseline_prediction_files())
        rows = []
        draws = []
        for model_name, payload in models:
            row, draw_df = _bootstrap_one_model(
                model_name, PAPER_Y_TEST_FINE, payload['pred_fine'],
                PAPER_Y_TEST_COARSE, payload['pred_coarse'], clusters,
                PAPER.CLUSTER_BOOTSTRAP_REPS, seed=42,
            )
            rows.append(row)
            draws.append(draw_df)
        table = pd.DataFrame(rows)
        atomic_dataframe(table, PAPER_TABLE_DIR / 'table_s17_capture_cluster_bootstrap')
        atomic_dataframe(pd.concat(draws, ignore_index=True), PAPER_CACHE_DIR / 'table_s17_bootstrap_draws')
        status_note = f'{len(unique_clusters)} test capture clusters. Baselines appear only after their row-aligned predictions are generated.'
        update_coverage_status('Table S17', 'DONE', str(out_path), status_note)
        return table


PROVENANCE_TABLE3 = run_provenance_audit()
CLUSTER_BOOTSTRAP_S17 = run_cluster_bootstrap()
if len(PROVENANCE_TABLE3):
    display(PROVENANCE_TABLE3)
if len(CLUSTER_BOOTSTRAP_S17):
    display(CLUSTER_BOOTSTRAP_S17)
paper_cell_done('P07', 'provenance-level leakage audit and capture-cluster bootstrap', _started)


In [10]:
_started = paper_cell_start('P08', 'optimized intra-epoch-resumable training and evaluation engine')

from dataclasses import dataclass, replace

PAPER.INTRA_EPOCH_CHECKPOINT_STEPS = _safe_int(os.environ.get('CAMELOT_CHECKPOINT_EVERY_STEPS', PAPER.INTRA_EPOCH_CHECKPOINT_STEPS), PAPER.INTRA_EPOCH_CHECKPOINT_STEPS)
PAPER.EMA_DECAY = _safe_float(os.environ.get('CAMELOT_EMA_DECAY', 0.999), 0.999)


class PaperEMA:
    def __init__(self, model: nn.Module, decay: float = 0.999):
        self.decay = float(decay)
        self.shadow = {k: v.detach().clone() for k, v in unwrap_model(model).state_dict().items()}

    @torch.no_grad()
    def update(self, model: nn.Module) -> None:
        state = unwrap_model(model).state_dict()
        for key, value in state.items():
            if torch.is_floating_point(self.shadow[key]) or torch.is_complex(self.shadow[key]):
                self.shadow[key].mul_(self.decay).add_(value.detach(), alpha=1.0 - self.decay)
            else:
                self.shadow[key].copy_(value.detach())

    def state_dict(self) -> Dict[str, Any]:
        return {'decay': self.decay, 'shadow': {k: v.detach().cpu().clone() for k, v in self.shadow.items()}}

    def load_state_dict(self, state: Mapping[str, Any], device: torch.device) -> None:
        self.decay = float(state.get('decay', self.decay))
        self.shadow = {k: v.detach().to(device).clone() for k, v in state['shadow'].items()}

    @torch.no_grad()
    def copy_to(self, model: nn.Module) -> None:
        unwrap_model(model).load_state_dict(self.shadow, strict=True)


def unwrap_model(model: nn.Module) -> nn.Module:
    return getattr(model, '_orig_mod', model)


def _class_balanced_weights_local(y: np.ndarray, n_classes: int, beta: float) -> Tuple[np.ndarray, torch.Tensor]:
    counts = np.bincount(np.asarray(y, dtype=np.int64), minlength=int(n_classes)).astype(np.float64)
    effective = 1.0 - np.power(float(beta), counts)
    weights = (1.0 - float(beta)) / np.clip(effective, 1e-12, None)
    weights /= max(float(weights.mean()), 1e-12)
    return counts, torch.tensor(weights, dtype=torch.float32)


def hierarchy_consistency_local(logits_f: torch.Tensor, logits_c: torch.Tensor, logits_b: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
    p_f = torch.softmax(logits_f, dim=1)
    p_c = torch.softmax(logits_c, dim=1)
    p_b = torch.softmax(logits_b, dim=1)
    mapping = torch.as_tensor(np.asarray(fine_to_coarse, dtype=np.int64), device=p_f.device)
    indices = mapping.unsqueeze(0).expand(p_f.size(0), -1)
    p_from_fine_c = torch.zeros(p_f.size(0), NUM_COARSE_PAPER, device=p_f.device, dtype=p_f.dtype)
    p_from_fine_c.scatter_add_(1, indices, p_f)
    p_from_fine_b = torch.stack([p_f[:, int(benign_fine_id)], 1.0 - p_f[:, int(benign_fine_id)]], dim=1)
    kl_c = F.kl_div(torch.log(p_c + 1e-8), p_from_fine_c, reduction='batchmean')
    kl_b = F.kl_div(torch.log(p_b + 1e-8), p_from_fine_b, reduction='batchmean')
    return kl_c, kl_b


def _make_adamw(parameters: Iterable[nn.Parameter], lr: float, weight_decay: float, device: torch.device) -> torch.optim.Optimizer:
    kwargs = {'lr': float(lr), 'weight_decay': float(weight_decay)}
    if device.type == 'cuda' and PAPER.USE_FUSED_ADAMW:
        try:
            return torch.optim.AdamW(parameters, fused=True, **kwargs)
        except Exception as exc:
            paper_log('fused AdamW unavailable; standard AdamW used', level='WARNING', stage='training_engine', error=str(exc))
    return torch.optim.AdamW(parameters, **kwargs)


def _gpu_resident_dataset(
    X: np.ndarray, yf: np.ndarray, yc: np.ndarray, yb: np.ndarray, device: torch.device
) -> Optional[Tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor]]:
    if device.type != 'cuda' or str(PAPER.GPU_RESIDENT_TRAIN).lower() == 'no':
        return None
    bytes_needed = int(np.asarray(X).nbytes + np.asarray(yf).nbytes + np.asarray(yc).nbytes + np.asarray(yb).nbytes)
    try:
        free_bytes, total_bytes = torch.cuda.mem_get_info(device)
    except Exception:
        free_bytes = torch.cuda.get_device_properties(device).total_memory - torch.cuda.memory_allocated(device)
        total_bytes = torch.cuda.get_device_properties(device).total_memory
    allowed = int(float(PAPER.GPU_RESIDENT_FRACTION) * free_bytes)
    force_yes = str(PAPER.GPU_RESIDENT_TRAIN).lower() == 'yes'
    if bytes_needed > allowed and not force_yes:
        paper_log('GPU-resident training skipped by memory guard', stage='training_engine',
                  dataset_gb=round(bytes_needed / 1024**3, 3), free_gb=round(free_bytes / 1024**3, 3),
                  allowed_gb=round(allowed / 1024**3, 3))
        return None
    try:
        paper_log('copying training arrays to GPU once', stage='training_engine', dataset_gb=round(bytes_needed / 1024**3, 3))
        tensors = (
            torch.from_numpy(np.asarray(X, dtype=np.float32)).to(device),
            torch.from_numpy(np.asarray(yf, dtype=np.int64)).to(device),
            torch.from_numpy(np.asarray(yc, dtype=np.int64)).to(device),
            torch.from_numpy(np.asarray(yb, dtype=np.int64)).to(device),
        )
        paper_log('GPU-resident training enabled', stage='training_engine', allocated_gb=round(torch.cuda.memory_allocated(device) / 1024**3, 3))
        return tensors
    except torch.cuda.OutOfMemoryError:
        paper_log('GPU-resident copy OOM; falling back to host arrays', level='WARNING', stage='training_engine')
        torch.cuda.empty_cache()
        return None


def _batch_from_arrays(
    indices_cpu: torch.Tensor,
    X: np.ndarray,
    yf: np.ndarray,
    yc: np.ndarray,
    yb: np.ndarray,
    device: torch.device,
    resident: Optional[Tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor]],
) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor]:
    if resident is not None:
        idx = indices_cpu.to(device, non_blocking=True)
        return resident[0].index_select(0, idx), resident[1].index_select(0, idx), resident[2].index_select(0, idx), resident[3].index_select(0, idx)
    idx_np = indices_cpu.numpy()
    xb = torch.from_numpy(np.ascontiguousarray(np.asarray(X)[idx_np], dtype=np.float32))
    yfb = torch.from_numpy(np.ascontiguousarray(np.asarray(yf)[idx_np], dtype=np.int64))
    ycb = torch.from_numpy(np.ascontiguousarray(np.asarray(yc)[idx_np], dtype=np.int64))
    ybb = torch.from_numpy(np.ascontiguousarray(np.asarray(yb)[idx_np], dtype=np.int64))
    return (
        xb.to(device, non_blocking=True), yfb.to(device, non_blocking=True),
        ycb.to(device, non_blocking=True), ybb.to(device, non_blocking=True),
    )


@torch.inference_mode()
def validate_hierarchical_model(
    model: nn.Module,
    X: np.ndarray,
    yf: np.ndarray,
    yc: np.ndarray,
    yb: np.ndarray,
    device: torch.device,
    batch_size: int,
) -> Dict[str, float]:
    model.eval()
    pred_f: List[np.ndarray] = []
    pred_c: List[np.ndarray] = []
    pred_b: List[np.ndarray] = []
    for start in range(0, len(X), int(batch_size)):
        stop = min(len(X), start + int(batch_size))
        xb = torch.from_numpy(np.array(X[start:stop], dtype=np.float32, copy=True, order='C')).to(device, non_blocking=True)
        with amp_autocast(PAPER.USE_AMP and device.type == 'cuda'):
            out = model(xb)
        pred_f.append(out['logits_fine'].argmax(dim=1).cpu().numpy())
        pred_c.append(out['logits_coarse'].argmax(dim=1).cpu().numpy())
        pred_b.append(out['logits_bin'].argmax(dim=1).cpu().numpy())
    pf = np.concatenate(pred_f) if pred_f else np.empty(0, dtype=np.int64)
    pc = np.concatenate(pred_c) if pred_c else np.empty(0, dtype=np.int64)
    pb = np.concatenate(pred_b) if pred_b else np.empty(0, dtype=np.int64)
    return {
        'fine_accuracy': float(accuracy_score(yf, pf)),
        'fine_balanced_accuracy': float(balanced_accuracy_score(yf, pf)),
        'fine_macro_f1': float(f1_score(yf, pf, average='macro', zero_division=0)),
        'coarse_accuracy': float(accuracy_score(yc, pc)),
        'coarse_balanced_accuracy': float(balanced_accuracy_score(yc, pc)),
        'coarse_macro_f1': float(f1_score(yc, pc, average='macro', zero_division=0)),
        'binary_accuracy': float(accuracy_score(yb, pb)),
        'binary_malicious_f1': float(f1_score(yb, pb, pos_label=1, zero_division=0)),
    }


def _training_data_for_profile(seed: int) -> Dict[str, np.ndarray]:
    if PAPER.PROFILE == 'strict':
        return {
            'Xtr': X_train, 'yftr': y_train_f, 'yctr': y_train_c, 'ybtr': y_train_b,
            'Xv': X_val, 'yfv': y_val_f, 'ycv': y_val_c, 'ybv': y_val_b,
        }
    # Accelerated and smoke profiles use a deterministic class-stratified
    # subset. The selected row indices are persisted so every restart and model
    # sees exactly the same profile-specific data.
    index_dir = PAPER_CACHE_DIR / 'profile_indices' / PAPER.PROFILE / f'seed_{int(seed)}'
    index_dir.mkdir(parents=True, exist_ok=True)
    train_path, val_path = index_dir / 'train.npy', index_dir / 'val.npy'
    if train_path.exists() and val_path.exists() and not PAPER.FORCE_RERUN:
        train_idx, val_idx = np.load(train_path), np.load(val_path)
    else:
        train_idx = deterministic_subset_indices(y_train_f, profile_cap('train', len(y_train_f)), seed)
        val_idx = deterministic_subset_indices(y_val_f, profile_cap('val', len(y_val_f)), seed + 1)
        atomic_npy(train_path, train_idx.astype(np.int64))
        atomic_npy(val_path, val_idx.astype(np.int64))
    return {
        'Xtr': X_train[train_idx], 'yftr': y_train_f[train_idx], 'yctr': y_train_c[train_idx], 'ybtr': y_train_b[train_idx],
        'Xv': X_val[val_idx], 'yfv': y_val_f[val_idx], 'ycv': y_val_c[val_idx], 'ybv': y_val_b[val_idx],
    }


def _make_training_losses(
    epoch: int,
    fine_loss_mode: str,
    counts_f: np.ndarray,
    weights_f: torch.Tensor,
    weights_c: torch.Tensor,
    weights_b: torch.Tensor,
    device: torch.device,
) -> Tuple[nn.Module, nn.Module, nn.Module]:
    drw_warm = int(epoch) <= int(cfg.DRW_EPOCHS)
    mode = str(fine_loss_mode).lower()
    if mode == 'ldam_drw':
        fine_loss = LDAMLoss(
            counts_f, max_m=float(cfg.LDAM_MAX_M), s=float(cfg.LDAM_S),
            weight=None if drw_warm else weights_f.to(device),
        ).to(device)
    elif mode in {'weighted_ce', 'ce_weighted', 'standard_weighted_ce'}:
        fine_loss = nn.CrossEntropyLoss(weight=weights_f.to(device))
    elif mode in {'plain_ce', 'ce'}:
        fine_loss = nn.CrossEntropyLoss()
    elif mode == 'balanced_softmax':
        fine_loss = BalancedSoftmaxLoss(counts_f).to(device)
    elif mode == 'logit_adj':
        fine_loss = LogitAdjustedCELoss(counts_f, tau=float(cfg.LOGIT_ADJ_TAU)).to(device)
    else:
        raise ValueError(f'Unknown fine_loss_mode={fine_loss_mode!r}')
    aux_c = nn.CrossEntropyLoss(weight=None if drw_warm else weights_c.to(device))
    aux_b = nn.CrossEntropyLoss(weight=None if drw_warm else weights_b.to(device))
    return fine_loss, aux_c, aux_b


def _save_training_checkpoint(
    path: Path,
    model: nn.Module,
    optimizer: torch.optim.Optimizer,
    scheduler: torch.optim.lr_scheduler.LRScheduler,
    scaler: Any,
    ema: PaperEMA,
    epoch: int,
    step_in_epoch: int,
    global_step: int,
    best_score: float,
    best_epoch: int,
    bad_epochs: int,
    history: List[Dict[str, Any]],
    signature: str,
) -> None:
    payload = {
        'signature': signature,
        'model_state': model_state_cpu(unwrap_model(model)),
        'optimizer_state': optimizer.state_dict(),
        'scheduler_state': scheduler.state_dict(),
        'scaler_state': scaler.state_dict() if scaler is not None else None,
        'ema_state': ema.state_dict(),
        'epoch': int(epoch),
        'step_in_epoch': int(step_in_epoch),
        'global_step': int(global_step),
        'best_score': float(best_score),
        'best_epoch': int(best_epoch),
        'bad_epochs': int(bad_epochs),
        'history': history,
        'rng_state': get_rng_bundle(),
        'saved_at': time.strftime('%Y-%m-%d %H:%M:%S'),
    }
    atomic_torch(path, payload)


@dataclass
class HierTrainResult:
    run_name: str
    run_dir: Path
    best_checkpoint: Path
    last_checkpoint: Path
    history: pd.DataFrame
    status: str
    best_epoch: int
    best_val_fine_macro_f1: float
    profile: str


def train_resumable_hierarchical(
    run_name: str,
    model_factory: Callable[[], nn.Module],
    *,
    seed: int = 42,
    epochs: Optional[int] = None,
    fine_loss_mode: str = 'ldam_drw',
    w_coarse: float = 0.30,
    w_binary: float = 0.20,
    w_kl_coarse: float = 0.10,
    w_kl_binary: float = 0.05,
    select_metric: str = 'fine_macro_f1',
    model_spec: Optional[Mapping[str, Any]] = None,
    data_override: Optional[Mapping[str, np.ndarray]] = None,
) -> HierTrainResult:
    data = dict(data_override) if data_override is not None else _training_data_for_profile(int(seed))
    required_data = {'Xtr', 'yftr', 'yctr', 'ybtr', 'Xv', 'yfv', 'ycv', 'ybv'}
    missing_data = sorted(required_data.difference(data))
    if missing_data:
        raise KeyError('data_override missing keys: ' + ', '.join(missing_data))
    requested_epochs = int(PAPER.EPOCHS if epochs is None else epochs)
    run_dir = PAPER_MODEL_DIR / re.sub(r'[^A-Za-z0-9_.-]+', '_', str(run_name))
    run_dir.mkdir(parents=True, exist_ok=True)
    last_path = run_dir / 'last_checkpoint.pt'
    best_path = run_dir / 'best_raw_checkpoint.pt'
    last_ema_path = run_dir / 'last_ema_checkpoint.pt'
    history_path = run_dir / 'history.csv'
    done_path = run_dir / 'done.json'
    signature_payload = {
        'engine_version': 6, 'base': PAPER_BASE_SIGNATURE_HASH, 'run_name': run_name,
        'seed': int(seed), 'epochs': requested_epochs, 'batch_size': int(PAPER.BATCH_SIZE),
        'profile': PAPER.PROFILE, 'fine_loss_mode': fine_loss_mode,
        'weights': [w_coarse, w_binary, w_kl_coarse, w_kl_binary],
        'model_spec': dict(model_spec or {}),
        'train_rows': int(len(data['Xtr'])), 'val_rows': int(len(data['Xv'])),
    }
    signature = signature_hash(signature_payload)
    done = read_json(done_path, {})
    if (not PAPER.FORCE_RERUN and done.get('status') == 'DONE' and done.get('signature') == signature and
            best_path.exists() and history_path.exists()):
        history = pd.read_csv(history_path)
        paper_log('training run reused', stage=run_name, best_epoch=done.get('best_epoch'),
                  best_val_fine_macro_f1=done.get('best_val_fine_macro_f1'))
        return HierTrainResult(run_name, run_dir, best_path, last_path, history, 'DONE',
                               int(done.get('best_epoch', 0)), float(done.get('best_val_fine_macro_f1', np.nan)), PAPER.PROFILE)

    seed_everything(int(seed), deterministic=(PAPER.PROFILE == 'strict'))
    device = DEVICE_T
    model = model_factory().to(device)
    if PAPER.USE_TORCH_COMPILE and hasattr(torch, 'compile') and PAPER.PROFILE != 'strict':
        try:
            model = torch.compile(model, mode=PAPER.TORCH_COMPILE_MODE)
            paper_log('torch.compile enabled for run', stage=run_name, mode=PAPER.TORCH_COMPILE_MODE)
        except Exception as exc:
            paper_log('torch.compile failed; eager execution retained', level='WARNING', stage=run_name, error=str(exc))

    counts_f, weights_f = _class_balanced_weights_local(data['yftr'], NUM_FINE_PAPER, float(cfg.CB_BETA))
    _, weights_c = _class_balanced_weights_local(data['yctr'], NUM_COARSE_PAPER, float(cfg.CB_BETA))
    _, weights_b = _class_balanced_weights_local(data['ybtr'], 2, float(cfg.CB_BETA))
    optimizer = _make_adamw(model.parameters(), float(cfg.LR), float(cfg.WEIGHT_DECAY), device)
    steps_per_epoch = int(math.ceil(len(data['Xtr']) / int(PAPER.BATCH_SIZE)))
    total_steps = max(1, requested_epochs * steps_per_epoch)
    warmup_steps = max(1, int(float(PAPER.WARMUP_EPOCHS) * steps_per_epoch))

    def lr_lambda(step: int) -> float:
        if step < warmup_steps:
            return max(1e-6, float(step + 1) / warmup_steps)
        progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
        return 0.5 * (1.0 + math.cos(math.pi * min(max(progress, 0.0), 1.0)))

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lr_lambda)
    use_amp = bool(PAPER.USE_AMP and device.type == 'cuda')
    scaler = make_grad_scaler(use_amp)
    ema = PaperEMA(model, PAPER.EMA_DECAY)
    resident = _gpu_resident_dataset(data['Xtr'], data['yftr'], data['yctr'], data['ybtr'], device)

    start_epoch = 1
    start_step = 0
    global_step = 0
    best_score = -float('inf')
    best_epoch = 0
    bad_epochs = 0
    history: List[Dict[str, Any]] = []
    if last_path.exists() and not PAPER.FORCE_RERUN:
        checkpoint = safe_torch_load(last_path, map_location='cpu')
        if checkpoint.get('signature') == signature:
            unwrap_model(model).load_state_dict(checkpoint['model_state'], strict=True)
            optimizer.load_state_dict(checkpoint['optimizer_state'])
            optimizer_to_device(optimizer, device)
            scheduler.load_state_dict(checkpoint['scheduler_state'])
            if scaler is not None and checkpoint.get('scaler_state') is not None:
                scaler.load_state_dict(checkpoint['scaler_state'])
            ema.load_state_dict(checkpoint['ema_state'], device)
            start_epoch = int(checkpoint['epoch'])
            start_step = int(checkpoint.get('step_in_epoch', 0))
            global_step = int(checkpoint.get('global_step', 0))
            best_score = float(checkpoint.get('best_score', -float('inf')))
            best_epoch = int(checkpoint.get('best_epoch', 0))
            bad_epochs = int(checkpoint.get('bad_epochs', 0))
            history = list(checkpoint.get('history', []))
            set_rng_bundle(checkpoint.get('rng_state'))
            if start_step >= steps_per_epoch:
                start_epoch += 1
                start_step = 0
            paper_log('intra-epoch checkpoint resumed', stage=run_name,
                      epoch=start_epoch, step=f'{start_step}/{steps_per_epoch}', global_step=global_step)
        else:
            paper_log('existing checkpoint signature differs; clean run started', level='WARNING', stage=run_name)

    atomic_json(run_dir / 'run_spec.json', {'signature': signature, 'signature_payload': signature_payload})
    patience = int(PAPER.EARLY_STOPPING_PATIENCE)
    status = 'RUNNING'
    try:
        for epoch in range(start_epoch, requested_epochs + 1):
            model.train()
            fine_loss_fn, coarse_loss_fn, binary_loss_fn = _make_training_losses(
                epoch, fine_loss_mode, counts_f, weights_f, weights_c, weights_b, device
            )
            generator = torch.Generator(device='cpu')
            generator.manual_seed(int(seed) * 1_000_003 + int(epoch))
            permutation = torch.randperm(len(data['Xtr']), generator=generator)
            epoch_started = time.perf_counter()
            epoch_loss_sum = 0.0
            epoch_seen = 0
            first_step = start_step if epoch == start_epoch else 0
            for step_idx in range(first_step, steps_per_epoch):
                lo = step_idx * int(PAPER.BATCH_SIZE)
                hi = min(len(permutation), lo + int(PAPER.BATCH_SIZE))
                indices = permutation[lo:hi]
                xb, yfb, ycb, ybb = _batch_from_arrays(
                    indices, data['Xtr'], data['yftr'], data['yctr'], data['ybtr'], device, resident
                )
                optimizer.zero_grad(set_to_none=True)
                with amp_autocast(use_amp):
                    out = model(xb)
                    kl_c, kl_b = hierarchy_consistency_local(out['logits_fine'], out['logits_coarse'], out['logits_bin'])
                    loss_f = fine_loss_fn(out['logits_fine'], yfb)
                    loss_c = coarse_loss_fn(out['logits_coarse'], ycb)
                    loss_b = binary_loss_fn(out['logits_bin'], ybb)
                    loss = loss_f + float(w_coarse) * loss_c + float(w_binary) * loss_b + float(w_kl_coarse) * kl_c + float(w_kl_binary) * kl_b
                if scaler is not None:
                    scaler.scale(loss).backward()
                    scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(model.parameters(), float(cfg.GRAD_CLIP))
                    scaler.step(optimizer)
                    scaler.update()
                else:
                    loss.backward()
                    torch.nn.utils.clip_grad_norm_(model.parameters(), float(cfg.GRAD_CLIP))
                    optimizer.step()
                scheduler.step()
                ema.update(model)
                global_step += 1
                batch_n = int(len(yfb))
                epoch_seen += batch_n
                epoch_loss_sum += float(loss.detach().cpu()) * batch_n

                completed_step = step_idx + 1
                if completed_step == 1 or completed_step % int(PAPER.LOG_EVERY_STEPS) == 0 or completed_step == steps_per_epoch:
                    elapsed = time.perf_counter() - epoch_started
                    effective_seen = max(1, epoch_seen)
                    rate = effective_seen / max(elapsed, 1e-9)
                    eta = (len(data['Xtr']) - (hi if first_step == 0 else lo + batch_n)) / max(rate, 1e-9)
                    gpu_mem = torch.cuda.memory_allocated(device) / 1024**3 if device.type == 'cuda' else 0.0
                    paper_log('training progress', stage=run_name, epoch=f'{epoch}/{requested_epochs}',
                              step=f'{completed_step}/{steps_per_epoch}', loss=round(epoch_loss_sum / effective_seen, 6),
                              lr=optimizer.param_groups[0]['lr'], samples_s=round(rate, 1), eta_minutes=round(max(0.0, eta) / 60, 1),
                              gpu_allocated_gb=round(gpu_mem, 3))

                must_checkpoint = (
                    completed_step % max(1, int(PAPER.INTRA_EPOCH_CHECKPOINT_STEPS)) == 0 or
                    completed_step == steps_per_epoch or
                    remaining_budget_seconds() <= 120.0
                )
                if must_checkpoint:
                    _save_training_checkpoint(
                        last_path, model, optimizer, scheduler, scaler, ema, epoch, completed_step,
                        global_step, best_score, best_epoch, bad_epochs, history, signature,
                    )
                    paper_log('atomic training checkpoint saved', stage=run_name, epoch=epoch,
                              step=f'{completed_step}/{steps_per_epoch}', path=str(last_path))
                    check_deadline(f'{run_name} epoch={epoch} step={completed_step}')

            start_step = 0
            validation = validate_hierarchical_model(
                model, data['Xv'], data['yfv'], data['ycv'], data['ybv'], device,
                int(PAPER.VALIDATION_BATCH_SIZE),
            )
            score = float(validation[select_metric])
            epoch_row = {
                'epoch': int(epoch), 'train_loss': epoch_loss_sum / max(epoch_seen, 1),
                **validation, 'selection_metric': select_metric, 'selection_score': score,
                'epoch_seconds': time.perf_counter() - epoch_started,
                'global_step': int(global_step), 'learning_rate_end': float(optimizer.param_groups[0]['lr']),
                'profile': PAPER.PROFILE,
            }
            history.append(epoch_row)
            atomic_dataframe(pd.DataFrame(history), run_dir / 'history')
            if score > best_score:
                best_score = score
                best_epoch = int(epoch)
                bad_epochs = 0
                atomic_torch(best_path, {
                    'signature': signature, 'model_state': model_state_cpu(unwrap_model(model)),
                    'epoch': int(epoch), 'val_metrics': validation, 'profile': PAPER.PROFILE,
                })
                paper_log('new best raw checkpoint', stage=run_name, epoch=epoch, score=round(score, 6))
            else:
                bad_epochs += 1
            _save_training_checkpoint(
                last_path, model, optimizer, scheduler, scaler, ema, epoch, steps_per_epoch,
                global_step, best_score, best_epoch, bad_epochs, history, signature,
            )
            atomic_torch(last_ema_path, {'signature': signature, 'model_state': ema.state_dict()['shadow'], 'epoch': int(epoch), 'profile': PAPER.PROFILE})
            paper_log('epoch completed', stage=run_name, epoch=f'{epoch}/{requested_epochs}',
                      train_loss=round(epoch_row['train_loss'], 6), val_fine_macro_f1=round(validation['fine_macro_f1'], 6),
                      val_coarse_macro_f1=round(validation['coarse_macro_f1'], 6),
                      val_binary_f1=round(validation['binary_malicious_f1'], 6), checkpoint='OK')
            if patience > 0 and bad_epochs >= patience:
                paper_log('early stopping reached', stage=run_name, epoch=epoch, patience=patience, best_epoch=best_epoch)
                break
            check_deadline(f'after {run_name} epoch {epoch}')
        status = 'DONE'
    except TimeBudgetReached:
        status = 'PAUSED'
        paper_log('training paused by time budget; rerun the orchestrator to resume', level='WARNING', stage=run_name)
    finally:
        if resident is not None:
            del resident
        gc.collect()
        if device.type == 'cuda':
            torch.cuda.empty_cache()

    history_df = pd.DataFrame(history)
    completed_epochs = int(history_df['epoch'].max()) if len(history_df) else 0
    if status == 'DONE':
        atomic_json(done_path, {
            'status': 'DONE', 'signature': signature, 'signature_payload': signature_payload,
            'best_epoch': int(best_epoch), 'best_val_fine_macro_f1': float(best_score),
            'completed_epochs': completed_epochs, 'requested_epochs': requested_epochs,
            'profile': PAPER.PROFILE, 'completed_at': time.strftime('%Y-%m-%d %H:%M:%S'),
        })
    else:
        atomic_json(run_dir / 'paused.json', {
            'status': 'PAUSED', 'signature': signature, 'completed_epochs': completed_epochs,
            'best_epoch': int(best_epoch), 'best_val_fine_macro_f1': float(best_score),
        })
    return HierTrainResult(run_name, run_dir, best_path, last_path, history_df, status,
                           int(best_epoch), float(best_score), PAPER.PROFILE)


def load_hierarchical_checkpoint(model_factory: Callable[[], nn.Module], checkpoint_path: Path, device: torch.device = DEVICE_T) -> nn.Module:
    blob = safe_torch_load(checkpoint_path, map_location='cpu')
    model = model_factory()
    state = blob.get('model_state') or blob.get('state_dict')
    if state is None:
        raise KeyError(f'No model_state/state_dict in {checkpoint_path}')
    model.load_state_dict(state, strict=True)
    return model.to(device).eval()


def evaluate_hierarchical_run(
    run_name: str,
    model_factory: Callable[[], nn.Module],
    checkpoint_path: Path,
    *,
    include_logits: bool = False,
    X_eval: Optional[np.ndarray] = None,
    y_fine_eval: Optional[np.ndarray] = None,
    y_coarse_eval: Optional[np.ndarray] = None,
    y_binary_eval: Optional[np.ndarray] = None,
) -> Dict[str, Any]:
    slug = re.sub(r'[^A-Za-z0-9_.-]+', '_', run_name)
    cache_dir = PAPER_CACHE_DIR / 'variant_evaluations' / slug
    cache_dir.mkdir(parents=True, exist_ok=True)
    metrics_path = cache_dir / 'metrics.json'
    pred_path = cache_dir / 'predictions.npz'
    X_use = X_test if X_eval is None else X_eval
    yf_use = y_test_f if y_fine_eval is None else y_fine_eval
    yc_use = y_test_c if y_coarse_eval is None else y_coarse_eval
    yb_use = y_test_b if y_binary_eval is None else y_binary_eval
    signature = signature_hash({'run_name': run_name, 'checkpoint': str(checkpoint_path), 'mtime': Path(checkpoint_path).stat().st_mtime,
                                'eval_rows': int(len(X_use)), 'eval_features': int(X_use.shape[1])})
    prior = read_json(metrics_path, {})
    if not PAPER.FORCE_RERUN and prior.get('signature') == signature and pred_path.exists():
        return prior
    model = load_hierarchical_checkpoint(model_factory, checkpoint_path)
    outputs = predict_hier_logits_array(model, X_use, batch_size=PAPER.INFERENCE_BATCH_SIZE)
    probs_f = softmax_np(outputs['fine'])
    probs_c = softmax_np(outputs['coarse'])
    probs_b = softmax_np(outputs['bin'])
    fine = multiclass_metrics(yf_use, probs_f)
    coarse = multiclass_metrics(yc_use, probs_c)
    binary = binary_metrics(yb_use, probs_b)
    payload = {
        'signature': signature, 'run_name': run_name, 'checkpoint': str(checkpoint_path),
        'fine': fine, 'coarse': coarse, 'binary': binary, 'profile': PAPER.PROFILE,
    }
    atomic_npz(
        pred_path,
        pred_fine=probs_f.argmax(axis=1).astype(np.int16),
        pred_coarse=probs_c.argmax(axis=1).astype(np.int8),
        pred_binary=probs_b.argmax(axis=1).astype(np.int8),
    )
    if include_logits:
        atomic_npz(cache_dir / 'logits.npz', fine=outputs['fine'], coarse=outputs['coarse'], binary=outputs['bin'])
    atomic_json(metrics_path, payload)
    del model, outputs, probs_f, probs_c, probs_b
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return payload


paper_log('training engine ready', stage='training_engine', profile=PAPER.PROFILE,
          checkpoint_every_steps=PAPER.INTRA_EPOCH_CHECKPOINT_STEPS,
          epochs=PAPER.EPOCHS, batch_size=PAPER.BATCH_SIZE)
paper_cell_done('P08', 'optimized intra-epoch-resumable training and evaluation engine', _started)


[P08] START: optimized intra-epoch-resumable training and evaluation engine
[2026-08-28 09:55:48] [INFO] [training_engine] training engine ready | profile=accelerated checkpoint_every_steps=500 epochs=12 batch_size=1024
[P08] DONE: optimized intra-epoch-resumable training and evaluation engine | elapsed=0.02s


In [11]:
_started = paper_cell_start('P09', 'paper-aligned ablation and baseline model definitions')


class FlatHierTransformer(nn.Module):
    """Flat 46-token hierarchical Transformer used by Table 10 and S10/S11."""
    def __init__(self, n_features: int = 46, n_fine: int = NUM_FINE_PAPER, n_coarse: int = NUM_COARSE_PAPER, cfg_local: Any = cfg):
        super().__init__()
        self.tokenizer = FeatureTokenizer(int(n_features), int(cfg_local.D_MODEL))
        self.cls = nn.Parameter(torch.zeros(1, 1, int(cfg_local.D_MODEL)))
        self.pos = nn.Parameter(torch.zeros(1, 1 + int(n_features), int(cfg_local.D_MODEL)))
        nn.init.trunc_normal_(self.cls, std=0.02)
        nn.init.trunc_normal_(self.pos, std=0.02)
        layer = make_encoder_layer(cfg_local.D_MODEL, cfg_local.NHEAD, cfg_local.FF_DIM, cfg_local.DROPOUT, cfg_local.ATTN_DROPOUT)
        self.encoder = nn.TransformerEncoder(layer, num_layers=int(cfg_local.GLOBAL_LAYERS))
        self.head_fine = nn.Sequential(nn.LayerNorm(cfg_local.D_MODEL), nn.Dropout(cfg_local.DROPOUT), nn.Linear(cfg_local.D_MODEL, int(n_fine)))
        self.head_coarse = nn.Sequential(nn.LayerNorm(cfg_local.D_MODEL), nn.Dropout(cfg_local.DROPOUT), nn.Linear(cfg_local.D_MODEL, int(n_coarse)))
        self.head_bin = nn.Sequential(nn.LayerNorm(cfg_local.D_MODEL), nn.Dropout(cfg_local.DROPOUT), nn.Linear(cfg_local.D_MODEL, 2))

    def forward(self, x: torch.Tensor) -> Dict[str, torch.Tensor]:
        tokens = self.tokenizer(x)
        cls = self.cls.expand(x.size(0), -1, -1)
        encoded = self.encoder(torch.cat([cls, tokens], dim=1) + self.pos[:, :1 + x.size(1)])
        h = encoded[:, 0]
        return {'logits_fine': self.head_fine(h), 'logits_coarse': self.head_coarse(h), 'logits_bin': self.head_bin(h)}


class FineFromHier(nn.Module):
    def __init__(self, base: nn.Module):
        super().__init__()
        self.base = base

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.base(x)['logits_fine']


class DNNBaseline(nn.Module):
    def __init__(self, n_features: int, n_classes: int, dropout: float = 0.10):
        super().__init__()
        dims = [int(n_features), 512, 512, 256, 128, 64]
        blocks: List[nn.Module] = []
        for inp, out in zip(dims[:-1], dims[1:]):
            blocks.extend([nn.Linear(inp, out), nn.LayerNorm(out), nn.GELU(), nn.Dropout(float(dropout))])
        blocks.append(nn.Linear(dims[-1], int(n_classes)))
        self.net = nn.Sequential(*blocks)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


class CNNGRUBaseline(nn.Module):
    def __init__(self, n_features: int, n_classes: int, dropout: float = 0.10):
        super().__init__()
        self.conv1 = nn.Conv1d(1, 64, kernel_size=3, padding=1)
        self.conv2 = nn.Conv1d(64, 128, kernel_size=3, padding=1)
        self.pool = nn.MaxPool1d(2)
        self.gru = nn.GRU(input_size=128, hidden_size=256, num_layers=1, batch_first=True)
        self.norm = nn.LayerNorm(256)
        self.dropout = nn.Dropout(float(dropout))
        self.head = nn.Linear(256, int(n_classes))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        z = F.gelu(self.conv1(x.unsqueeze(1)))
        z = F.gelu(self.conv2(z))
        z = self.pool(z).transpose(1, 2)
        _, hidden = self.gru(z)
        h = self.dropout(self.norm(hidden[-1]))
        return self.head(h)


class FTTransformerBaseline(nn.Module):
    def __init__(self, n_features: int, n_classes: int, d_model: int = 192, nhead: int = 8, layers: int = 4, ff_dim: int = 768, dropout: float = 0.10):
        super().__init__()
        self.tokenizer = FeatureTokenizer(int(n_features), int(d_model))
        self.cls = nn.Parameter(torch.zeros(1, 1, int(d_model)))
        self.pos = nn.Parameter(torch.zeros(1, 1 + int(n_features), int(d_model)))
        nn.init.trunc_normal_(self.cls, std=0.02)
        nn.init.trunc_normal_(self.pos, std=0.02)
        layer = nn.TransformerEncoderLayer(d_model=int(d_model), nhead=int(nhead), dim_feedforward=int(ff_dim), dropout=float(dropout), activation='gelu', batch_first=True, norm_first=True)
        self.encoder = nn.TransformerEncoder(layer, num_layers=int(layers))
        self.head = nn.Sequential(nn.LayerNorm(int(d_model)), nn.Dropout(float(dropout)), nn.Linear(int(d_model), int(n_classes)))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        tokens = self.tokenizer(x)
        cls = self.cls.expand(x.size(0), -1, -1)
        h = self.encoder(torch.cat([cls, tokens], dim=1) + self.pos[:, :1 + x.size(1)])[:, 0]
        return self.head(h)


class TabTransformerNumericBaseline(nn.Module):
    def __init__(self, n_features: int, n_classes: int, d_model: int = 128, nhead: int = 8, layers: int = 4, ff_dim: int = 512, dropout: float = 0.10):
        super().__init__()
        self.scalar_weight = nn.Parameter(torch.empty(int(n_features), int(d_model)))
        self.scalar_bias = nn.Parameter(torch.empty(int(n_features), int(d_model)))
        self.feature_id = nn.Embedding(int(n_features), int(d_model))
        nn.init.trunc_normal_(self.scalar_weight, std=0.02)
        nn.init.trunc_normal_(self.scalar_bias, std=0.02)
        layer = nn.TransformerEncoderLayer(d_model=int(d_model), nhead=int(nhead), dim_feedforward=int(ff_dim), dropout=float(dropout), activation='gelu', batch_first=True, norm_first=True)
        self.encoder = nn.TransformerEncoder(layer, num_layers=int(layers))
        self.head = nn.Sequential(nn.LayerNorm(int(d_model)), nn.Dropout(float(dropout)), nn.Linear(int(d_model), int(n_classes)))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        ids = torch.arange(x.size(1), device=x.device)
        tokens = x.unsqueeze(-1) * self.scalar_weight.unsqueeze(0) + self.scalar_bias.unsqueeze(0) + self.feature_id(ids).unsqueeze(0)
        h = self.encoder(tokens).mean(dim=1)
        return self.head(h)


class SAINTBaseline(nn.Module):
    """Paper-specified SAINT-style feature attention plus one minibatch row-attention layer."""
    def __init__(self, n_features: int, n_classes: int, d_model: int = 128, nhead: int = 8, ff_dim: int = 512, dropout: float = 0.10):
        super().__init__()
        self.tokenizer = FeatureTokenizer(int(n_features), int(d_model))
        self.feature_pos = nn.Parameter(torch.zeros(1, int(n_features), int(d_model)))
        nn.init.trunc_normal_(self.feature_pos, std=0.02)
        feature_layer = nn.TransformerEncoderLayer(d_model=int(d_model), nhead=int(nhead), dim_feedforward=int(ff_dim), dropout=float(dropout), activation='gelu', batch_first=True, norm_first=True)
        self.feature_encoder = nn.TransformerEncoder(feature_layer, num_layers=2)
        self.row_attention = nn.MultiheadAttention(int(d_model), int(nhead), dropout=float(dropout), batch_first=True)
        self.row_norm = nn.LayerNorm(int(d_model))
        self.row_ffn = nn.Sequential(nn.Linear(int(d_model), int(ff_dim)), nn.GELU(), nn.Dropout(float(dropout)), nn.Linear(int(ff_dim), int(d_model)))
        self.head = nn.Sequential(nn.LayerNorm(int(d_model)), nn.Dropout(float(dropout)), nn.Linear(int(d_model), int(n_classes)))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        tokens = self.feature_encoder(self.tokenizer(x) + self.feature_pos[:, :x.size(1)])
        rows = tokens.mean(dim=1)
        # Treat the current minibatch as one row-attention sequence exactly as documented.
        seq = rows.unsqueeze(0)
        attended, _ = self.row_attention(seq, seq, seq, need_weights=False)
        rows = self.row_norm(rows + attended.squeeze(0))
        rows = self.row_norm(rows + self.row_ffn(rows))
        return self.head(rows)


class GraphSAGEBatchKNNBaseline(nn.Module):
    def __init__(self, n_features: int, n_classes: int, hidden: int = 256, k: int = 10, dropout: float = 0.10):
        super().__init__()
        self.k = int(k)
        self.input_mlp = nn.Sequential(nn.Linear(int(n_features), int(hidden)), nn.GELU(), nn.LayerNorm(int(hidden)))
        self.layer1 = nn.Linear(2 * int(hidden), int(hidden))
        self.layer2 = nn.Linear(2 * int(hidden), int(hidden))
        self.dropout = nn.Dropout(float(dropout))
        self.head = nn.Linear(int(hidden), int(n_classes))

    def _aggregate(self, h: torch.Tensor) -> torch.Tensor:
        if h.size(0) <= 1:
            return h
        # Batch-local graph: no edge can cross minibatches or data partitions.
        distances = torch.cdist(h.float(), h.float(), p=2)
        distances.fill_diagonal_(float('inf'))
        k_use = min(self.k, h.size(0) - 1)
        neighbors = distances.topk(k=k_use, largest=False, dim=1).indices
        return h[neighbors].mean(dim=1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        h = self.input_mlp(x)
        n1 = self._aggregate(h)
        h = self.dropout(F.gelu(self.layer1(torch.cat([h, n1], dim=1))))
        n2 = self._aggregate(h)
        h = self.dropout(F.gelu(self.layer2(torch.cat([h, n2], dim=1))))
        return self.head(h)


def random_grouping(n_features: int, n_groups: int, seed: int) -> List[List[int]]:
    rng = np.random.default_rng(int(seed))
    values = rng.permutation(int(n_features))
    return [sorted(part.astype(int).tolist()) for part in np.array_split(values, int(n_groups)) if len(part)]


def derive_auxiliary_predictions_from_fine(pred_fine: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    pred = np.asarray(pred_fine, dtype=np.int64)
    coarse = np.asarray(fine_to_coarse, dtype=np.int64)[pred]
    binary = (pred != int(benign_fine_id)).astype(np.int64)
    return coarse, binary


def model_parameter_count(model: nn.Module) -> int:
    return int(sum(p.numel() for p in model.parameters() if p.requires_grad))


MODEL_CONFIG_TABLE = pd.DataFrame([
    ['CAMELOT-IDS', 'Grouped local-global Transformer', model_parameter_count(build_fresh_camelot_model()), '34/8/2 heads'],
    ['Flat Transformer', '46 feature tokens + CLS, d=256, 8 heads, 6 layers', model_parameter_count(FlatHierTransformer()), '34/8/2 heads'],
    ['DNN+SMOTE', '46-512-512-256-128-64-34 MLP', model_parameter_count(DNNBaseline(len(FEATURE_COLS), NUM_FINE_PAPER)), 'fine head; auxiliary labels derived'],
    ['CNN-GRU', 'Conv1d 1->64->128, max-pool, GRU hidden 256', model_parameter_count(CNNGRUBaseline(len(FEATURE_COLS), NUM_FINE_PAPER)), 'fine head; auxiliary labels derived'],
    ['FT-Transformer', 'd=192, 8 heads, 4 layers, FFN=768', model_parameter_count(FTTransformerBaseline(len(FEATURE_COLS), NUM_FINE_PAPER)), 'fine head; auxiliary labels derived'],
    ['TabTransformer adaptation', 'numeric tokens + feature IDs, d=128, 4 layers', model_parameter_count(TabTransformerNumericBaseline(len(FEATURE_COLS), NUM_FINE_PAPER)), 'fine head; auxiliary labels derived'],
    ['SAINT', '2 feature-attention + 1 minibatch row-attention layer', model_parameter_count(SAINTBaseline(len(FEATURE_COLS), NUM_FINE_PAPER)), 'fine head; auxiliary labels derived'],
    ['GraphSAGE-GNN', 'batch-local kNN k=10, 2 mean-aggregation layers', model_parameter_count(GraphSAGEBatchKNNBaseline(len(FEATURE_COLS), NUM_FINE_PAPER)), 'fine head; auxiliary labels derived'],
], columns=['model', 'architecture', 'trainable_parameters', 'output_policy'])
MODEL_CONFIG_TABLE['source_kind'] = 'paper_configuration_implemented_in_code'
atomic_dataframe(MODEL_CONFIG_TABLE, PAPER_REPORT_DIR / 'implemented_model_registry')
display(MODEL_CONFIG_TABLE)
paper_cell_done('P09', 'paper-aligned ablation and baseline model definitions', _started)


[P09] START: paper-aligned ablation and baseline model definitions


,model,architecture,trainable_parameters,output_policy,source_kind
0,CAMELOT-IDS,Grouped local-global Transformer,6358828,34/8/2 heads,paper_configuration_implemented_in_code
1,Flat Transformer,"46 feature tokens + CLS, d=256, 8 heads, 6 layers",4787244,34/8/2 heads,paper_configuration_implemented_in_code
2,DNN+SMOTE,46-512-512-256-128-64-34 MLP,464354,fine head; auxiliary labels derived,paper_configuration_implemented_in_code
3,CNN-GRU,"Conv1d 1->64->128, max-pool, GRU hidden 256",330658,fine head; auxiliary labels derived,paper_configuration_implemented_in_code
4,FT-Transformer,"d=192, 8 heads, 4 layers, FFN=768",1813282,fine head; auxiliary labels derived,paper_configuration_implemented_in_code
5,TabTransformer adaptation,"numeric tokens + feature IDs, d=128, 4 layers",815394,fine head; auxiliary labels derived,paper_configuration_implemented_in_code
6,SAINT,2 feature-attention + 1 minibatch row-attentio...,616866,fine head; auxiliary labels derived,paper_configuration_implemented_in_code
7,GraphSAGE-GNN,"batch-local kNN k=10, 2 mean-aggregation layers",283938,fine head; auxiliary labels derived,paper_configuration_implemented_in_code


[P09] DONE: paper-aligned ablation and baseline model definitions | elapsed=0.21s


In [12]:
_started = paper_cell_start('P10', 'resumable fine-class baseline trainer, calibration, and evaluation')


def _gpu_resident_fine(X: np.ndarray, y: np.ndarray, device: torch.device) -> Optional[Tuple[torch.Tensor, torch.Tensor]]:
    if device.type != 'cuda' or str(PAPER.GPU_RESIDENT_TRAIN).lower() == 'no':
        return None
    bytes_needed = int(np.asarray(X).nbytes + np.asarray(y).nbytes)
    free_bytes, _ = torch.cuda.mem_get_info(device)
    if bytes_needed > float(PAPER.GPU_RESIDENT_FRACTION) * free_bytes and str(PAPER.GPU_RESIDENT_TRAIN).lower() != 'yes':
        return None
    try:
        return (
            torch.from_numpy(np.asarray(X, dtype=np.float32)).to(device),
            torch.from_numpy(np.asarray(y, dtype=np.int64)).to(device),
        )
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache()
        return None


def _fine_batch(indices_cpu: torch.Tensor, X: np.ndarray, y: np.ndarray, device: torch.device,
                resident: Optional[Tuple[torch.Tensor, torch.Tensor]]) -> Tuple[torch.Tensor, torch.Tensor]:
    if resident is not None:
        idx = indices_cpu.to(device, non_blocking=True)
        return resident[0].index_select(0, idx), resident[1].index_select(0, idx)
    idx_np = indices_cpu.numpy()
    xb = torch.from_numpy(np.ascontiguousarray(np.asarray(X)[idx_np], dtype=np.float32)).to(device, non_blocking=True)
    yb = torch.from_numpy(np.ascontiguousarray(np.asarray(y)[idx_np], dtype=np.int64)).to(device, non_blocking=True)
    return xb, yb


@torch.inference_mode()
def predict_fine_logits_array(model: nn.Module, X: np.ndarray, batch_size: Optional[int] = None,
                              device: torch.device = DEVICE_T, log_stage: str = 'fine_predict') -> np.ndarray:
    bs = int(PAPER.INFERENCE_BATCH_SIZE if batch_size is None else batch_size)
    model = model.to(device).eval()
    parts: List[np.ndarray] = []
    total_batches = int(math.ceil(len(X) / bs)) if len(X) else 0
    started = time.perf_counter()
    for batch_idx, start in enumerate(range(0, len(X), bs), start=1):
        stop = min(len(X), start + bs)
        xb = torch.from_numpy(np.array(X[start:stop], dtype=np.float32, copy=True, order='C')).to(device, non_blocking=True)
        with amp_autocast(PAPER.USE_AMP and device.type == 'cuda'):
            logits = model(xb)
        parts.append(logits.float().cpu().numpy())
        if batch_idx == 1 or batch_idx % 100 == 0 or batch_idx == total_batches:
            elapsed = time.perf_counter() - started
            paper_log('fine-model inference progress', stage=log_stage, batch=f'{batch_idx}/{total_batches}',
                      rows=stop, samples_s=round(stop / max(elapsed, 1e-9), 1))
        check_deadline(f'{log_stage} batch {batch_idx}')
    return np.concatenate(parts, axis=0) if parts else np.empty((0, NUM_FINE_PAPER), dtype=np.float32)


@torch.inference_mode()
def validate_fine_model(model: nn.Module, X: np.ndarray, y: np.ndarray, device: torch.device,
                        batch_size: int) -> Dict[str, float]:
    logits = predict_fine_logits_array(model, X, batch_size=batch_size, device=device, log_stage='fine_validation')
    pred = logits.argmax(axis=1)
    return {
        'fine_accuracy': float(accuracy_score(y, pred)),
        'fine_balanced_accuracy': float(balanced_accuracy_score(y, pred)),
        'fine_macro_f1': float(f1_score(y, pred, average='macro', zero_division=0)),
        'fine_weighted_f1': float(f1_score(y, pred, average='weighted', zero_division=0)),
        'fine_mcc': float(matthews_corrcoef(y, pred)),
    }


def _fine_loss_for_epoch(epoch: int, loss_mode: str, counts: np.ndarray, weights: torch.Tensor, device: torch.device) -> nn.Module:
    mode = str(loss_mode).lower()
    if mode in {'class_balanced_ce', 'weighted_ce', 'ce_weighted'}:
        return nn.CrossEntropyLoss(weight=weights.to(device))
    if mode in {'plain_ce', 'ce'}:
        return nn.CrossEntropyLoss()
    if mode == 'ldam_drw':
        return LDAMLoss(counts, max_m=float(cfg.LDAM_MAX_M), s=float(cfg.LDAM_S),
                        weight=None if int(epoch) <= int(cfg.DRW_EPOCHS) else weights.to(device)).to(device)
    if mode == 'balanced_softmax':
        return BalancedSoftmaxLoss(counts).to(device)
    raise ValueError(f'Unknown fine loss_mode={loss_mode!r}')


@dataclass
class FineTrainResult:
    run_name: str
    run_dir: Path
    best_checkpoint: Path
    last_checkpoint: Path
    history: pd.DataFrame
    status: str
    best_epoch: int
    best_val_fine_macro_f1: float


def train_resumable_fine(
    run_name: str,
    model_factory: Callable[[], nn.Module],
    *,
    seed: int = 42,
    epochs: Optional[int] = None,
    loss_mode: str = 'class_balanced_ce',
    Xtr: Optional[np.ndarray] = None,
    ytr: Optional[np.ndarray] = None,
    Xv: Optional[np.ndarray] = None,
    yv: Optional[np.ndarray] = None,
    model_spec: Optional[Mapping[str, Any]] = None,
) -> FineTrainResult:
    if Xtr is None or ytr is None or Xv is None or yv is None:
        data = _training_data_for_profile(int(seed))
        Xtr, ytr, Xv, yv = data['Xtr'], data['yftr'], data['Xv'], data['yfv']
    requested_epochs = int(PAPER.EPOCHS if epochs is None else epochs)
    run_dir = PAPER_MODEL_DIR / re.sub(r'[^A-Za-z0-9_.-]+', '_', str(run_name))
    run_dir.mkdir(parents=True, exist_ok=True)
    last_path = run_dir / 'last_checkpoint.pt'
    best_path = run_dir / 'best_raw_checkpoint.pt'
    ema_path = run_dir / 'last_ema_checkpoint.pt'
    history_path = run_dir / 'history.csv'
    done_path = run_dir / 'done.json'
    signature_payload = {
        'engine_version': 5, 'base': PAPER_BASE_SIGNATURE_HASH, 'run_name': run_name,
        'seed': int(seed), 'epochs': requested_epochs, 'batch_size': int(PAPER.BATCH_SIZE),
        'loss_mode': loss_mode, 'profile': PAPER.PROFILE,
        'train_rows': int(len(Xtr)), 'val_rows': int(len(Xv)), 'model_spec': dict(model_spec or {}),
    }
    signature = signature_hash(signature_payload)
    done = read_json(done_path, {})
    if (not PAPER.FORCE_RERUN and done.get('status') == 'DONE' and done.get('signature') == signature and
            best_path.exists() and history_path.exists()):
        return FineTrainResult(run_name, run_dir, best_path, last_path, pd.read_csv(history_path), 'DONE',
                               int(done.get('best_epoch', 0)), float(done.get('best_val_fine_macro_f1', np.nan)))

    seed_everything(int(seed), deterministic=(PAPER.PROFILE == 'strict'))
    device = DEVICE_T
    model = model_factory().to(device)
    if PAPER.USE_TORCH_COMPILE and hasattr(torch, 'compile') and PAPER.PROFILE != 'strict' and not isinstance(model, (SAINTBaseline, GraphSAGEBatchKNNBaseline)):
        try:
            model = torch.compile(model, mode=PAPER.TORCH_COMPILE_MODE)
        except Exception as exc:
            paper_log('torch.compile skipped for fine model', level='WARNING', stage=run_name, error=str(exc))
    counts, weights = _class_balanced_weights_local(ytr, NUM_FINE_PAPER, float(cfg.CB_BETA))
    optimizer = _make_adamw(model.parameters(), float(cfg.LR), float(cfg.WEIGHT_DECAY), device)
    steps_per_epoch = int(math.ceil(len(Xtr) / int(PAPER.BATCH_SIZE)))
    total_steps = max(1, steps_per_epoch * requested_epochs)
    warmup_steps = max(1, int(float(PAPER.WARMUP_EPOCHS) * steps_per_epoch))

    def lr_lambda(step: int) -> float:
        if step < warmup_steps:
            return max(1e-6, float(step + 1) / warmup_steps)
        progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
        return 0.5 * (1.0 + math.cos(math.pi * min(max(progress, 0.0), 1.0)))

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
    scaler = make_grad_scaler(PAPER.USE_AMP and device.type == 'cuda')
    ema = PaperEMA(model, PAPER.EMA_DECAY)
    resident = _gpu_resident_fine(Xtr, ytr, device)
    start_epoch, start_step, global_step = 1, 0, 0
    best_score, best_epoch, bad_epochs = -float('inf'), 0, 0
    history: List[Dict[str, Any]] = []
    if last_path.exists() and not PAPER.FORCE_RERUN:
        checkpoint = safe_torch_load(last_path, map_location='cpu')
        if checkpoint.get('signature') == signature:
            unwrap_model(model).load_state_dict(checkpoint['model_state'], strict=True)
            optimizer.load_state_dict(checkpoint['optimizer_state'])
            optimizer_to_device(optimizer, device)
            scheduler.load_state_dict(checkpoint['scheduler_state'])
            if scaler is not None and checkpoint.get('scaler_state') is not None:
                scaler.load_state_dict(checkpoint['scaler_state'])
            ema.load_state_dict(checkpoint['ema_state'], device)
            start_epoch = int(checkpoint['epoch'])
            start_step = int(checkpoint.get('step_in_epoch', 0))
            global_step = int(checkpoint.get('global_step', 0))
            best_score = float(checkpoint.get('best_score', -float('inf')))
            best_epoch = int(checkpoint.get('best_epoch', 0))
            bad_epochs = int(checkpoint.get('bad_epochs', 0))
            history = list(checkpoint.get('history', []))
            set_rng_bundle(checkpoint.get('rng_state'))
            if start_step >= steps_per_epoch:
                start_epoch += 1
                start_step = 0
            paper_log('fine-model checkpoint resumed', stage=run_name, epoch=start_epoch, step=f'{start_step}/{steps_per_epoch}')

    atomic_json(run_dir / 'run_spec.json', {'signature': signature, 'signature_payload': signature_payload})
    status = 'RUNNING'
    try:
        for epoch in range(start_epoch, requested_epochs + 1):
            model.train()
            loss_fn = _fine_loss_for_epoch(epoch, loss_mode, counts, weights, device)
            generator = torch.Generator(device='cpu')
            generator.manual_seed(int(seed) * 1_000_003 + int(epoch))
            permutation = torch.randperm(len(Xtr), generator=generator)
            epoch_started = time.perf_counter()
            loss_sum, seen = 0.0, 0
            first_step = start_step if epoch == start_epoch else 0
            for step_idx in range(first_step, steps_per_epoch):
                lo = step_idx * int(PAPER.BATCH_SIZE)
                hi = min(len(permutation), lo + int(PAPER.BATCH_SIZE))
                xb, yb = _fine_batch(permutation[lo:hi], Xtr, ytr, device, resident)
                optimizer.zero_grad(set_to_none=True)
                with amp_autocast(PAPER.USE_AMP and device.type == 'cuda'):
                    logits = model(xb)
                    loss = loss_fn(logits, yb)
                if scaler is not None:
                    scaler.scale(loss).backward()
                    scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(model.parameters(), float(cfg.GRAD_CLIP))
                    scaler.step(optimizer)
                    scaler.update()
                else:
                    loss.backward()
                    torch.nn.utils.clip_grad_norm_(model.parameters(), float(cfg.GRAD_CLIP))
                    optimizer.step()
                scheduler.step()
                ema.update(model)
                global_step += 1
                n_batch = int(len(yb))
                seen += n_batch
                loss_sum += float(loss.detach().cpu()) * n_batch
                completed_step = step_idx + 1
                if completed_step == 1 or completed_step % int(PAPER.LOG_EVERY_STEPS) == 0 or completed_step == steps_per_epoch:
                    elapsed = time.perf_counter() - epoch_started
                    paper_log('fine-model training progress', stage=run_name, epoch=f'{epoch}/{requested_epochs}',
                              step=f'{completed_step}/{steps_per_epoch}', loss=round(loss_sum / max(seen, 1), 6),
                              samples_s=round(seen / max(elapsed, 1e-9), 1), lr=optimizer.param_groups[0]['lr'])
                if (completed_step % max(1, int(PAPER.INTRA_EPOCH_CHECKPOINT_STEPS)) == 0 or
                        completed_step == steps_per_epoch or remaining_budget_seconds() <= 120):
                    _save_training_checkpoint(last_path, model, optimizer, scheduler, scaler, ema,
                                              epoch, completed_step, global_step, best_score, best_epoch,
                                              bad_epochs, history, signature)
                    check_deadline(f'{run_name} epoch={epoch} step={completed_step}')
            start_step = 0
            val = validate_fine_model(model, Xv, yv, device, int(PAPER.VALIDATION_BATCH_SIZE))
            score = val['fine_macro_f1']
            row = {'epoch': epoch, 'train_loss': loss_sum / max(seen, 1), **val,
                   'epoch_seconds': time.perf_counter() - epoch_started, 'global_step': global_step,
                   'profile': PAPER.PROFILE}
            history.append(row)
            atomic_dataframe(pd.DataFrame(history), run_dir / 'history')
            if score > best_score:
                best_score, best_epoch, bad_epochs = float(score), int(epoch), 0
                atomic_torch(best_path, {'signature': signature, 'model_state': model_state_cpu(unwrap_model(model)),
                                         'epoch': epoch, 'val_metrics': val, 'profile': PAPER.PROFILE})
            else:
                bad_epochs += 1
            _save_training_checkpoint(last_path, model, optimizer, scheduler, scaler, ema,
                                      epoch, steps_per_epoch, global_step, best_score, best_epoch,
                                      bad_epochs, history, signature)
            atomic_torch(ema_path, {'signature': signature, 'model_state': ema.state_dict()['shadow'], 'epoch': epoch})
            paper_log('fine-model epoch completed', stage=run_name, epoch=f'{epoch}/{requested_epochs}',
                      val_fine_macro_f1=round(score, 6), best_epoch=best_epoch, checkpoint='OK')
            if PAPER.EARLY_STOPPING_PATIENCE > 0 and bad_epochs >= PAPER.EARLY_STOPPING_PATIENCE:
                paper_log('fine-model early stopping reached', stage=run_name, epoch=epoch, best_epoch=best_epoch)
                break
            check_deadline(f'after {run_name} epoch {epoch}')
        status = 'DONE'
    except TimeBudgetReached:
        status = 'PAUSED'
        paper_log('fine-model training paused by time budget', level='WARNING', stage=run_name)
    finally:
        if resident is not None:
            del resident
        gc.collect()
        if device.type == 'cuda':
            torch.cuda.empty_cache()

    history_df = pd.DataFrame(history)
    if status == 'DONE':
        atomic_json(done_path, {'status': 'DONE', 'signature': signature, 'signature_payload': signature_payload,
                                'best_epoch': best_epoch, 'best_val_fine_macro_f1': best_score,
                                'completed_epochs': int(history_df['epoch'].max()) if len(history_df) else 0,
                                'profile': PAPER.PROFILE})
    return FineTrainResult(run_name, run_dir, best_path, last_path, history_df, status, best_epoch, best_score)


def load_fine_checkpoint(model_factory: Callable[[], nn.Module], checkpoint_path: Path,
                         device: torch.device = DEVICE_T) -> nn.Module:
    blob = safe_torch_load(checkpoint_path, map_location='cpu')
    model = model_factory()
    state = blob.get('model_state') or blob.get('state_dict')
    if state is None:
        raise KeyError(f'No model state in {checkpoint_path}')
    model.load_state_dict(state, strict=True)
    return model.to(device).eval()


def evaluate_fine_baseline(
    run_name: str,
    model_factory: Callable[[], nn.Module],
    checkpoint_path: Path,
    *,
    fit_calibration: bool = True,
    benchmark: bool = True,
) -> Dict[str, Any]:
    slug = re.sub(r'[^A-Za-z0-9_.-]+', '_', run_name)
    cache_dir = PAPER_CACHE_DIR / 'baseline_evaluations' / slug
    cache_dir.mkdir(parents=True, exist_ok=True)
    result_path = cache_dir / 'metrics.json'
    pred_path = PAPER_CACHE_DIR / 'baseline_predictions' / f'{slug}.npz'
    pred_path.parent.mkdir(parents=True, exist_ok=True)
    signature = signature_hash({'run_name': run_name, 'checkpoint': str(checkpoint_path),
                                'mtime': Path(checkpoint_path).stat().st_mtime,
                                'fit_calibration': fit_calibration, 'benchmark': benchmark})
    prior = read_json(result_path, {})
    if not PAPER.FORCE_RERUN and prior.get('signature') == signature and pred_path.exists():
        return prior
    model = load_fine_checkpoint(model_factory, checkpoint_path)
    test_logits = predict_fine_logits_array(model, X_test, log_stage=f'{slug}_test')
    cal_logits = predict_fine_logits_array(model, X_cal, log_stage=f'{slug}_cal') if fit_calibration else None
    temperature = fit_temperature_np(cal_logits, y_cal_f) if fit_calibration else 1.0
    test_probs = softmax_np(test_logits / max(float(temperature), 1e-6))
    pred_fine = test_probs.argmax(axis=1).astype(np.int64)
    pred_coarse, pred_binary = derive_auxiliary_predictions_from_fine(pred_fine)
    fine = {
        'accuracy': float(accuracy_score(y_test_f, pred_fine)),
        'balanced_accuracy': float(balanced_accuracy_score(y_test_f, pred_fine)),
        'macro_f1': float(f1_score(y_test_f, pred_fine, average='macro', zero_division=0)),
        'weighted_f1': float(f1_score(y_test_f, pred_fine, average='weighted', zero_division=0)),
        'mcc': float(matthews_corrcoef(y_test_f, pred_fine)),
    }
    coarse = {
        'accuracy': float(accuracy_score(y_test_c, pred_coarse)),
        'balanced_accuracy': float(balanced_accuracy_score(y_test_c, pred_coarse)),
        'macro_f1': float(f1_score(y_test_c, pred_coarse, average='macro', zero_division=0)),
    }
    binary = {
        'accuracy': float(accuracy_score(y_test_b, pred_binary)),
        'balanced_accuracy': float(balanced_accuracy_score(y_test_b, pred_binary)),
        'malicious_f1': float(f1_score(y_test_b, pred_binary, pos_label=1, zero_division=0)),
    }
    cal_metrics = calibration_metrics(y_test_f, test_probs, PAPER.ECE_BINS)
    throughput = benchmark_native_pytorch(model, X_test, DEVICE_T, batch_size=4096, max_rows=200_000) if benchmark else {}
    payload = {'signature': signature, 'run_name': run_name, 'temperature': float(temperature),
               'fine': fine, 'coarse': coarse, 'binary': binary, 'calibration': cal_metrics,
               'throughput': throughput, 'profile': PAPER.PROFILE}
    atomic_npz(pred_path, pred_fine=pred_fine.astype(np.int16), pred_coarse=pred_coarse.astype(np.int8),
               pred_binary=pred_binary.astype(np.int8))
    atomic_json(result_path, payload)
    del model, test_logits, test_probs
    if cal_logits is not None:
        del cal_logits
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return payload


paper_log('fine-model training/evaluation engine ready', stage='fine_training_engine')
paper_cell_done('P10', 'resumable fine-class baseline trainer, calibration, and evaluation', _started)


[P10] START: resumable fine-class baseline trainer, calibration, and evaluation
[2026-08-28 09:55:49] [INFO] [fine_training_engine] fine-model training/evaluation engine ready
[P10] DONE: resumable fine-class baseline trainer, calibration, and evaluation | elapsed=0.02s


In [13]:
_started = paper_cell_start('P11', 'architectural/training ablation suite for Table 10')


def _camelot_factory_with_groups(groups: Sequence[Sequence[int]]) -> Callable[[], nn.Module]:
    groups_clean = [list(map(int, g)) for g in groups]
    return lambda: build_fresh_camelot_model(group_idxs=groups_clean)


def _flat_factory() -> nn.Module:
    return FlatHierTransformer(len(FEATURE_COLS), NUM_FINE_PAPER, NUM_COARSE_PAPER, copy.deepcopy(cfg))


def _ablation_metrics_from_eval(name: str, evaluation: Mapping[str, Any], derive_aux_from_fine: bool = False) -> Dict[str, Any]:
    if not derive_aux_from_fine:
        return {
            'variant': name,
            'fine_macro_f1_percent': 100 * float(evaluation['fine']['macro_f1']),
            'coarse_macro_f1_percent': 100 * float(evaluation['coarse']['macro_f1']),
            'binary_f1_percent': 100 * float(evaluation['binary']['malicious_f1']),
        }
    slug = re.sub(r'[^A-Za-z0-9_.-]+', '_', name)
    pred_file = PAPER_CACHE_DIR / 'variant_evaluations' / slug / 'predictions.npz'
    with np.load(pred_file, allow_pickle=False) as blob:
        pred_fine = np.asarray(blob['pred_fine'], dtype=np.int64)
    _, pred_binary = derive_auxiliary_predictions_from_fine(pred_fine)
    return {
        'variant': name,
        'fine_macro_f1_percent': 100 * float(evaluation['fine']['macro_f1']),
        'coarse_macro_f1_percent': np.nan,
        'binary_f1_percent': 100 * float(f1_score(y_test_b, pred_binary, pos_label=1, zero_division=0)),
    }


def run_ablation_study() -> pd.DataFrame:
    signature = {'stage_version': 5, 'base': PAPER_BASE_SIGNATURE_HASH, 'profile': PAPER.PROFILE,
                 'epochs': PAPER.EPOCHS, 'seed': 42}
    out_path = PAPER_TABLE_DIR / 'table_10_architectural_training_ablation.csv'
    with paper_stage('ablations', signature) as stage:
        if stage.skip and out_path.exists():
            return pd.read_csv(out_path)
        rows: List[Dict[str, Any]] = []
        core = ARCHIVED_CORE_SUMMARY if isinstance(ARCHIVED_CORE_SUMMARY, Mapping) else read_json(PAPER_REPORT_DIR / 'archived_core_summary.json', {})
        rows.append({
            'variant': 'CAMELOT-IDS (full)',
            'fine_macro_f1_percent': 100 * float(core['fine_metrics']['macro_f1']),
            'coarse_macro_f1_percent': 100 * float(core['coarse_metrics']['macro_f1']),
            'binary_f1_percent': 100 * float(core['binary_metrics']['malicious_f1']),
            'training_status': 'REUSED_ARCHIVED_V4', 'checkpoint': 'v4 publication selected model',
            'profile': 'archived_v4', 'source_kind': 'archived_executed_result',
        })
        atomic_dataframe(pd.DataFrame(rows), PAPER_TABLE_DIR / 'table_10_architectural_training_ablation_partial')

        variant_specs = [
            {
                'display': 'Flat-46 (grouping removed)', 'run': 'ablation_flat_46_seed_42',
                'factory': _flat_factory, 'fine_loss_mode': 'ldam_drw',
                'weights': (0.30, 0.20, 0.10, 0.05), 'derive_aux': False,
                'model_spec': {'architecture': 'flat_46_tokens', 'layers': cfg.GLOBAL_LAYERS},
            },
            {
                'display': 'Random grouping', 'run': 'ablation_random_grouping_seed_42',
                'factory': _camelot_factory_with_groups(random_grouping(len(FEATURE_COLS), len(GROUP_IDXS), 42)),
                'fine_loss_mode': 'ldam_drw', 'weights': (0.30, 0.20, 0.10, 0.05), 'derive_aux': False,
                'model_spec': {'architecture': 'camelot_random_groups', 'groups': random_grouping(len(FEATURE_COLS), len(GROUP_IDXS), 42)},
            },
            {
                'display': 'Fine-head only', 'run': 'ablation_fine_head_only_seed_42',
                'factory': _camelot_factory_with_groups(GROUP_IDXS), 'fine_loss_mode': 'ldam_drw',
                'weights': (0.0, 0.0, 0.0, 0.0), 'derive_aux': True,
                'model_spec': {'architecture': 'camelot_backbone_fine_supervision_only'},
            },
            {
                'display': 'Hierarchy without KL consistency', 'run': 'ablation_hierarchy_no_kl_seed_42',
                'factory': _camelot_factory_with_groups(GROUP_IDXS), 'fine_loss_mode': 'ldam_drw',
                'weights': (0.30, 0.20, 0.0, 0.0), 'derive_aux': False,
                'model_spec': {'architecture': 'camelot_no_kl'},
            },
            {
                'display': 'Standard weighted CE', 'run': 'ablation_weighted_ce_seed_42',
                'factory': _camelot_factory_with_groups(GROUP_IDXS), 'fine_loss_mode': 'weighted_ce',
                'weights': (0.30, 0.20, 0.10, 0.05), 'derive_aux': False,
                'model_spec': {'architecture': 'camelot_weighted_ce'},
            },
        ]

        for spec in variant_specs:
            check_deadline(f'before {spec["display"]}')
            wc, wb, wkc, wkb = spec['weights']
            result = train_resumable_hierarchical(
                spec['run'], spec['factory'], seed=42, fine_loss_mode=spec['fine_loss_mode'],
                w_coarse=wc, w_binary=wb, w_kl_coarse=wkc, w_kl_binary=wkb,
                model_spec=spec['model_spec'],
            )
            if result.status != 'DONE' or not result.best_checkpoint.exists():
                rows.append({
                    'variant': spec['display'], 'fine_macro_f1_percent': np.nan,
                    'coarse_macro_f1_percent': np.nan, 'binary_f1_percent': np.nan,
                    'training_status': result.status, 'checkpoint': str(result.best_checkpoint),
                    'profile': PAPER.PROFILE, 'source_kind': 'incomplete_resumable_run',
                })
                atomic_dataframe(pd.DataFrame(rows), PAPER_TABLE_DIR / 'table_10_architectural_training_ablation_partial')
                if result.status == 'PAUSED':
                    raise TimeBudgetReached(f'Ablation {spec["display"]} paused; rerun to resume.')
                continue
            evaluation = evaluate_hierarchical_run(spec['run'], spec['factory'], result.best_checkpoint)
            row = _ablation_metrics_from_eval(spec['run'], evaluation, derive_aux_from_fine=spec['derive_aux'])
            row['variant'] = spec['display']
            row.update({
                'training_status': result.status, 'checkpoint': str(result.best_checkpoint),
                'best_epoch': result.best_epoch, 'best_val_fine_macro_f1_percent': 100 * result.best_val_fine_macro_f1,
                'profile': PAPER.PROFILE,
                'source_kind': 'strict_measured' if PAPER.PROFILE == 'strict' else f'{PAPER.PROFILE}_profile_measured',
            })
            rows.append(row)
            atomic_dataframe(pd.DataFrame(rows), PAPER_TABLE_DIR / 'table_10_architectural_training_ablation_partial')

        table = pd.DataFrame(rows)
        atomic_dataframe(table, PAPER_TABLE_DIR / 'table_10_architectural_training_ablation')
        status = 'DONE_STRICT' if PAPER.PROFILE == 'strict' else 'DONE_NONSTRICT_PROFILE'
        update_coverage_status('Table 10', status, str(out_path),
                               'Each trainable variant is epoch- and intra-epoch-resumable; only strict profile is manuscript-protocol equivalent.')
        return table


print('Ablation suite defined. It runs from the final orchestrator when CAMELOT_RUN_ALL=1.')
paper_cell_done('P11', 'architectural/training ablation suite for Table 10', _started)


[P11] START: architectural/training ablation suite for Table 10
Ablation suite defined. It runs from the final orchestrator when CAMELOT_RUN_ALL=1.
[P11] DONE: architectural/training ablation suite for Table 10 | elapsed=0.01s


In [14]:
_started = paper_cell_start('P12', 'feature-count scalability and hyperparameter/group sensitivity suites')


def profile_hierarchical_model(model: nn.Module, X_reference: np.ndarray, batch_size: int = 4096) -> Dict[str, Any]:
    model = model.to(DEVICE_T).eval()
    result: Dict[str, Any] = {'parameters': model_parameter_count(model), 'flops_per_sample': np.nan,
                              'gpu_peak_memory_gb': np.nan, 'throughput_samples_s': np.nan}
    if importlib.util.find_spec('fvcore') is not None:
        try:
            from fvcore.nn import FlopCountAnalysis
            class FineWrapper(nn.Module):
                def __init__(self, base: nn.Module):
                    super().__init__(); self.base = base
                def forward(self, x: torch.Tensor) -> torch.Tensor:
                    return self.base(x)['logits_fine']
            wrapper = FineWrapper(copy.deepcopy(model).cpu().eval())
            result['flops_per_sample'] = float(FlopCountAnalysis(wrapper, torch.zeros(1, X_reference.shape[1])).total())
            del wrapper
        except Exception as exc:
            result['flops_error'] = str(exc)
    if DEVICE_T.type == 'cuda':
        torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats(DEVICE_T)
    bench = benchmark_native_pytorch(model, X_reference, DEVICE_T, batch_size=batch_size, max_rows=200_000)
    result['throughput_samples_s'] = bench['throughput_samples_s']
    if DEVICE_T.type == 'cuda':
        result['gpu_peak_memory_gb'] = torch.cuda.max_memory_allocated(DEVICE_T) / 1024**3
    return result


def _load_feature_scaling_artifact(feature_count: int) -> Tuple[Optional[Dict[str, np.ndarray]], Optional[List[List[int]]], str]:
    root_text = str(PAPER.FEATURE_MAP_DIR or '').strip()
    if not root_text:
        return None, None, 'CAMELOT_FEATURE_MAP_DIR is not set.'
    root = Path(root_text).expanduser() / str(int(feature_count))
    required = ['X_train.npy', 'X_val.npy', 'X_test.npy', 'groups.json']
    missing = [name for name in required if not (root / name).exists()]
    if missing:
        return None, None, f'Missing {missing} under {root}. Exact feature-scaling arrays/mappings are required.'
    Xtr = np.load(root / 'X_train.npy', mmap_mode='r')
    Xv = np.load(root / 'X_val.npy', mmap_mode='r')
    Xte = np.load(root / 'X_test.npy', mmap_mode='r')
    if Xtr.shape[1] != int(feature_count) or Xv.shape[1] != int(feature_count) or Xte.shape[1] != int(feature_count):
        return None, None, f'Feature dimension mismatch in {root}.'
    if len(Xtr) != len(y_train_f) or len(Xv) != len(y_val_f) or len(Xte) != len(y_test_f):
        return None, None, 'Feature-scaling arrays must be row-aligned with the v4 split labels.'
    groups_raw = json.loads((root / 'groups.json').read_text(encoding='utf-8'))
    groups = groups_raw.get('groups', groups_raw) if isinstance(groups_raw, dict) else groups_raw
    groups = [list(map(int, g)) for g in groups]
    flat = sorted(i for g in groups for i in g)
    if flat != list(range(int(feature_count))):
        return None, None, 'groups.json must partition every transformed feature index exactly once.'
    data = {'Xtr': Xtr, 'yftr': y_train_f, 'yctr': y_train_c, 'ybtr': y_train_b,
            'Xv': Xv, 'yfv': y_val_f, 'ycv': y_val_c, 'ybv': y_val_b, 'Xte': Xte}
    if PAPER.PROFILE != 'strict':
        tr_idx = deterministic_subset_indices(y_train_f, profile_cap('train', len(y_train_f)), 42)
        va_idx = deterministic_subset_indices(y_val_f, profile_cap('val', len(y_val_f)), 43)
        te_idx = deterministic_subset_indices(y_test_f, profile_cap('test', len(y_test_f)), 44)
        data = {'Xtr': np.asarray(Xtr[tr_idx]), 'yftr': y_train_f[tr_idx], 'yctr': y_train_c[tr_idx], 'ybtr': y_train_b[tr_idx],
                'Xv': np.asarray(Xv[va_idx]), 'yfv': y_val_f[va_idx], 'ycv': y_val_c[va_idx], 'ybv': y_val_b[va_idx],
                'Xte': np.asarray(Xte[te_idx]), 'yfte': y_test_f[te_idx], 'ycte': y_test_c[te_idx], 'ybte': y_test_b[te_idx]}
    return data, groups, ''


def run_feature_scalability() -> pd.DataFrame:
    signature = {'stage_version': 4, 'base': PAPER_BASE_SIGNATURE_HASH, 'feature_counts': [23, 46, 92, 138],
                 'feature_map_dir': PAPER.FEATURE_MAP_DIR, 'profile': PAPER.PROFILE}
    out_path = PAPER_TABLE_DIR / 'table_s05_feature_count_scalability.csv'
    with paper_stage('scalability', signature) as stage:
        if stage.skip and out_path.exists():
            return pd.read_csv(out_path)
        rows: List[Dict[str, Any]] = []
        core = ARCHIVED_CORE_SUMMARY
        archived_profile = profile_hierarchical_model(PAPER_MODEL, X_test)
        rows.append({
            'feature_count': 46, 'groups': len(GROUP_IDXS), 'parameters_m': archived_profile['parameters'] / 1e6,
            'flops_g': archived_profile['flops_per_sample'] / 1e9 if np.isfinite(archived_profile['flops_per_sample']) else np.nan,
            'gpu_memory_gb': archived_profile['gpu_peak_memory_gb'],
            'throughput_samples_s': archived_profile['throughput_samples_s'],
            'fine_macro_f1_percent': 100 * float(core['fine_metrics']['macro_f1']),
            'status': 'REUSED_ARCHIVED_V4', 'profile': 'archived_v4',
            'source_kind': 'executed_reference_46_feature_model',
        })
        atomic_dataframe(pd.DataFrame(rows), PAPER_TABLE_DIR / 'table_s05_feature_count_scalability_partial')
        for n_features in [23, 92, 138]:
            data, groups, reason = _load_feature_scaling_artifact(n_features)
            if data is None or groups is None:
                rows.append({'feature_count': n_features, 'groups': np.nan, 'parameters_m': np.nan,
                             'flops_g': np.nan, 'gpu_memory_gb': np.nan, 'throughput_samples_s': np.nan,
                             'fine_macro_f1_percent': np.nan, 'status': 'BLOCKED_ARTIFACT_MISSING',
                             'profile': PAPER.PROFILE, 'source_kind': 'not_run', 'note': reason})
                paper_log('feature-scaling run blocked', level='WARNING', stage='scalability', features=n_features, reason=reason)
                continue
            factory = lambda nf=n_features, gs=groups: build_fresh_camelot_model(group_idxs=gs, n_features=nf)
            train_data = {k: data[k] for k in ['Xtr', 'yftr', 'yctr', 'ybtr', 'Xv', 'yfv', 'ycv', 'ybv']}
            result = train_resumable_hierarchical(
                f'scalability_features_{n_features}_seed_42', factory, seed=42,
                model_spec={'feature_count': n_features, 'groups': groups}, data_override=train_data,
            )
            if result.status != 'DONE':
                rows.append({'feature_count': n_features, 'groups': len(groups), 'status': result.status,
                             'profile': PAPER.PROFILE, 'source_kind': 'incomplete_resumable_run'})
                atomic_dataframe(pd.DataFrame(rows), PAPER_TABLE_DIR / 'table_s05_feature_count_scalability_partial')
                if result.status == 'PAUSED':
                    raise TimeBudgetReached(f'Scalability {n_features} paused.')
                continue
            Xte = data['Xte']
            yfte = data.get('yfte', y_test_f); ycte = data.get('ycte', y_test_c); ybte = data.get('ybte', y_test_b)
            evaluation = evaluate_hierarchical_run(
                f'scalability_features_{n_features}_seed_42', factory, result.best_checkpoint,
                X_eval=Xte, y_fine_eval=yfte, y_coarse_eval=ycte, y_binary_eval=ybte,
            )
            model = load_hierarchical_checkpoint(factory, result.best_checkpoint)
            profile = profile_hierarchical_model(model, Xte)
            rows.append({
                'feature_count': n_features, 'groups': len(groups), 'parameters_m': profile['parameters'] / 1e6,
                'flops_g': profile['flops_per_sample'] / 1e9 if np.isfinite(profile['flops_per_sample']) else np.nan,
                'gpu_memory_gb': profile['gpu_peak_memory_gb'], 'throughput_samples_s': profile['throughput_samples_s'],
                'fine_macro_f1_percent': 100 * float(evaluation['fine']['macro_f1']),
                'status': 'DONE', 'profile': PAPER.PROFILE,
                'source_kind': 'strict_measured_exact_mapping' if PAPER.PROFILE == 'strict' else f'{PAPER.PROFILE}_profile_measured_exact_mapping',
            })
            del model
            atomic_dataframe(pd.DataFrame(rows), PAPER_TABLE_DIR / 'table_s05_feature_count_scalability_partial')
        table = pd.DataFrame(rows).sort_values('feature_count').reset_index(drop=True)
        atomic_dataframe(table, PAPER_TABLE_DIR / 'table_s05_feature_count_scalability')
        complete = table['status'].isin(['REUSED_ARCHIVED_V4', 'DONE']).all()
        update_coverage_status('Table S5', 'DONE' if complete else 'PARTIAL_BLOCKED', str(out_path),
                               '23/92/138 require exact transformed arrays and group mappings; no feature duplication is invented.')
        return table


def _load_grouping_plan(k: int) -> Tuple[Optional[List[List[int]]], str]:
    root_text = str(PAPER.FEATURE_MAP_DIR or '').strip()
    if not root_text:
        return None, 'CAMELOT_FEATURE_MAP_DIR is not set.'
    path = Path(root_text).expanduser() / f'grouping_k{k}.json'
    if not path.exists():
        return None, f'Missing {path}.'
    payload = json.loads(path.read_text(encoding='utf-8'))
    groups = payload.get('groups', payload) if isinstance(payload, dict) else payload
    groups = [list(map(int, g)) for g in groups]
    if len(groups) != int(k) or sorted(i for g in groups for i in g) != list(range(len(FEATURE_COLS))):
        return None, f'{path} must define exactly {k} groups partitioning all {len(FEATURE_COLS)} features.'
    return groups, ''


def run_hyperparameter_sensitivity() -> pd.DataFrame:
    signature = {'stage_version': 4, 'base': PAPER_BASE_SIGNATURE_HASH, 'profile': PAPER.PROFILE,
                 'coarse_weights': [0.2, 0.3, 0.4], 'group_counts': [3, 5, 7, 10],
                 'feature_map_dir': PAPER.FEATURE_MAP_DIR}
    out_path = PAPER_TABLE_DIR / 'table_s06_hyperparameter_sensitivity.csv'
    with paper_stage('sensitivity', signature) as stage:
        if stage.skip and out_path.exists():
            return pd.read_csv(out_path)
        rows: List[Dict[str, Any]] = []
        # Coarse-loss sensitivity. Lambda_c=0.30 reuses the archived full model.
        rows.append({'factor': 'Coarse loss weight lambda_c', 'setting': 0.30,
                     'fine_macro_f1_percent': 100 * ARCHIVED_CORE_SUMMARY['fine_metrics']['macro_f1'],
                     'coarse_macro_f1_percent': 100 * ARCHIVED_CORE_SUMMARY['coarse_metrics']['macro_f1'],
                     'status': 'REUSED_ARCHIVED_V4', 'source_kind': 'archived_executed_result'})
        for weight in [0.20, 0.40]:
            factory = _camelot_factory_with_groups(GROUP_IDXS)
            result = train_resumable_hierarchical(
                f'sensitivity_lambda_c_{weight:.2f}_seed_42', factory, seed=42,
                w_coarse=weight, w_binary=0.20, w_kl_coarse=0.10, w_kl_binary=0.05,
                model_spec={'factor': 'lambda_c', 'value': weight},
            )
            if result.status != 'DONE':
                rows.append({'factor': 'Coarse loss weight lambda_c', 'setting': weight,
                             'fine_macro_f1_percent': np.nan, 'coarse_macro_f1_percent': np.nan,
                             'status': result.status, 'source_kind': 'incomplete_resumable_run'})
                if result.status == 'PAUSED':
                    atomic_dataframe(pd.DataFrame(rows), PAPER_TABLE_DIR / 'table_s06_hyperparameter_sensitivity_partial')
                    raise TimeBudgetReached(f'Sensitivity lambda_c={weight} paused.')
                continue
            evaluation = evaluate_hierarchical_run(f'sensitivity_lambda_c_{weight:.2f}_seed_42', factory, result.best_checkpoint)
            rows.append({'factor': 'Coarse loss weight lambda_c', 'setting': weight,
                         'fine_macro_f1_percent': 100 * evaluation['fine']['macro_f1'],
                         'coarse_macro_f1_percent': 100 * evaluation['coarse']['macro_f1'],
                         'status': 'DONE', 'source_kind': 'measured_run', 'profile': PAPER.PROFILE})
            atomic_dataframe(pd.DataFrame(rows), PAPER_TABLE_DIR / 'table_s06_hyperparameter_sensitivity_partial')

        # K=5 semantic and random are already represented by archived/full and Table 10.
        rows.append({'factor': 'Number/grouping of semantic groups K', 'setting': 'K=5 semantic',
                     'fine_macro_f1_percent': 100 * ARCHIVED_CORE_SUMMARY['fine_metrics']['macro_f1'],
                     'status': 'REUSED_ARCHIVED_V4', 'source_kind': 'archived_executed_result'})
        table10_path = PAPER_TABLE_DIR / 'table_10_architectural_training_ablation.csv'
        if table10_path.exists():
            ab = pd.read_csv(table10_path)
            random_row = ab[ab['variant'] == 'Random grouping']
            if len(random_row):
                rows.append({'factor': 'Grouping at K=5', 'setting': 'K=5 random',
                             'fine_macro_f1_percent': float(random_row.iloc[0]['fine_macro_f1_percent']),
                             'status': str(random_row.iloc[0].get('training_status', 'DONE')),
                             'source_kind': 'measured_ablation_run'})
        for k in [3, 7, 10]:
            groups, reason = _load_grouping_plan(k)
            if groups is None:
                rows.append({'factor': 'Number/grouping of semantic groups K', 'setting': f'K={k}',
                             'fine_macro_f1_percent': np.nan, 'status': 'BLOCKED_MAPPING_MISSING',
                             'source_kind': 'not_run', 'note': reason})
                continue
            factory = _camelot_factory_with_groups(groups)
            result = train_resumable_hierarchical(
                f'sensitivity_groups_k{k}_seed_42', factory, seed=42,
                model_spec={'factor': 'semantic_group_count', 'K': k, 'groups': groups},
            )
            if result.status != 'DONE':
                rows.append({'factor': 'Number/grouping of semantic groups K', 'setting': f'K={k}',
                             'fine_macro_f1_percent': np.nan, 'status': result.status,
                             'source_kind': 'incomplete_resumable_run'})
                if result.status == 'PAUSED':
                    atomic_dataframe(pd.DataFrame(rows), PAPER_TABLE_DIR / 'table_s06_hyperparameter_sensitivity_partial')
                    raise TimeBudgetReached(f'Sensitivity K={k} paused.')
                continue
            evaluation = evaluate_hierarchical_run(f'sensitivity_groups_k{k}_seed_42', factory, result.best_checkpoint)
            rows.append({'factor': 'Number/grouping of semantic groups K', 'setting': f'K={k}',
                         'fine_macro_f1_percent': 100 * evaluation['fine']['macro_f1'],
                         'coarse_macro_f1_percent': 100 * evaluation['coarse']['macro_f1'],
                         'status': 'DONE', 'source_kind': ('strict_measured_exact_mapping' if PAPER.PROFILE == 'strict' else f'{PAPER.PROFILE}_profile_measured_exact_mapping'), 'profile': PAPER.PROFILE})
            atomic_dataframe(pd.DataFrame(rows), PAPER_TABLE_DIR / 'table_s06_hyperparameter_sensitivity_partial')

        # The manuscript does not provide an exact grid for the qualitative KL statement.
        rows.append({'factor': 'KL-consistency weights', 'setting': 'exact grid not specified in manuscript',
                     'fine_macro_f1_percent': np.nan, 'status': 'BLOCKED_PROTOCOL_UNDERSPECIFIED',
                     'source_kind': 'not_run', 'note': 'Provide a sensitivity_plan.json to add exact configurations; no grid is invented.'})
        table = pd.DataFrame(rows)
        atomic_dataframe(table, PAPER_TABLE_DIR / 'table_s06_hyperparameter_sensitivity')
        complete = not table['status'].astype(str).str.startswith('BLOCKED').any()
        update_coverage_status('Table S6', 'DONE' if complete else 'PARTIAL_BLOCKED', str(out_path),
                               'Fully specified lambda_c settings run; unspecified K mappings/KL grid are not invented.')
        return table


print('Scalability and sensitivity suites defined; exact missing mappings are reported as BLOCKED, not synthesized.')
paper_cell_done('P12', 'feature-count scalability and hyperparameter/group sensitivity suites', _started)


[P12] START: feature-count scalability and hyperparameter/group sensitivity suites
Scalability and sensitivity suites defined; exact missing mappings are reported as BLOCKED, not synthesized.
[P12] DONE: feature-count scalability and hyperparameter/group sensitivity suites | elapsed=0.01s


In [15]:
_started = paper_cell_start('P13', 'robustness suite: missingness, noise, FGSM, and holdout protocols')


def _balanced_accuracy_from_cm(cm: np.ndarray) -> float:
    matrix = np.asarray(cm, dtype=np.float64)
    support = matrix.sum(axis=1)
    recall = np.divide(np.diag(matrix), support, out=np.zeros_like(support), where=support > 0)
    return float(recall[support > 0].mean()) if np.any(support > 0) else float('nan')


def _macro_f1_from_cm(cm: np.ndarray) -> float:
    _, mf1 = _metrics_from_confusion(cm)
    return mf1


def _raps_chunk_counts(probs_f: np.ndarray, probs_c: np.ndarray, y: np.ndarray,
                       thresholds: Mapping[str, Any], alpha: float = 0.10) -> Dict[str, int]:
    akey = f'{float(alpha):.12g}'
    q = np.asarray([row['threshold'] for row in thresholds['groups'][akey]], dtype=np.float32)
    group = probs_c.argmax(axis=1)
    q_row = q[group]
    order = np.argsort(-probs_f, axis=1)
    sorted_p = np.take_along_axis(probs_f, order, axis=1)
    cumulative = np.cumsum(sorted_p, axis=1, dtype=np.float32)
    ranks = np.arange(1, probs_f.shape[1] + 1, dtype=np.float32)[None, :]
    score = cumulative + float(thresholds['lambda']) * np.maximum(ranks - float(thresholds['k_reg']), 0.0)
    k_star = np.maximum((score <= q_row[:, None]).sum(axis=1), 1)
    inverse = np.empty_like(order)
    rows = np.arange(len(y))[:, None]
    inverse[rows, order] = np.arange(probs_f.shape[1], dtype=order.dtype)[None, :]
    rank0 = inverse[np.arange(len(y)), y]
    covered = rank0 < k_star
    singleton = k_star == 1
    top1 = order[:, 0]
    return {
        'n': int(len(y)), 'covered': int(covered.sum()), 'set_size_sum': int(k_star.sum()),
        'singleton': int(singleton.sum()), 'singleton_correct': int((singleton & (top1 == y)).sum()),
    }


def _perturbation_chunk(X_chunk: np.ndarray, kind: str, severity: float, seed: int, start: int) -> np.ndarray:
    x = np.asarray(X_chunk, dtype=np.float32).copy()
    rng = np.random.default_rng(int(seed) + int(start) * 1009)
    if kind == 'missing':
        mask = rng.random(x.shape) < float(severity)
        x[mask] = 0.0  # standardized train-median value
    elif kind == 'gaussian':
        x += rng.normal(0.0, float(severity), size=x.shape).astype(np.float32)
        x = np.clip(x, -float(cfg.POST_SCALE_CLIP), float(cfg.POST_SCALE_CLIP))
    elif kind == 'clean':
        pass
    else:
        raise ValueError(kind)
    return x


def _evaluate_robustness_condition(name: str, kind: str, severity: float, seed: int = 42,
                                   chunk_size: int = 16_384) -> Dict[str, Any]:
    condition_dir = stage_path('robustness') / re.sub(r'[^A-Za-z0-9_.-]+', '_', name)
    condition_dir.mkdir(parents=True, exist_ok=True)
    result_path = condition_dir / 'result.json'
    progress_path = condition_dir / 'progress.json'
    signature = signature_hash({'base': PAPER_BASE_SIGNATURE_HASH, 'name': name, 'kind': kind,
                                'severity': severity, 'seed': seed, 'chunk_size': chunk_size})
    prior = read_json(result_path, {})
    if not PAPER.FORCE_RERUN and prior.get('signature') == signature:
        return prior
    thresholds = globals().get('ARCHIVED_RAPS_THRESHOLDS') or read_json(PAPER_CACHE_DIR / 'archived_empirical_raps_thresholds.json', {})
    if not thresholds:
        raise RuntimeError('Run P04 archived core first to create empirical RAPS thresholds.')
    progress = read_json(progress_path, {})
    if progress.get('signature') == signature and not PAPER.FORCE_RERUN:
        start_row = int(progress.get('next_row', 0))
        cm = np.asarray(progress.get('fine_confusion', np.zeros((NUM_FINE_PAPER, NUM_FINE_PAPER), dtype=np.int64)), dtype=np.int64)
        raps = {k: int(progress.get('raps', {}).get(k, 0)) for k in ['n', 'covered', 'set_size_sum', 'singleton', 'singleton_correct']}
        paper_log('robustness condition resumed', stage='robustness', condition=name, next_row=start_row)
    else:
        start_row = 0
        cm = np.zeros((NUM_FINE_PAPER, NUM_FINE_PAPER), dtype=np.int64)
        raps = {k: 0 for k in ['n', 'covered', 'set_size_sum', 'singleton', 'singleton_correct']}
    model = PAPER_MODEL.to(DEVICE_T).eval()
    total = len(X_test)
    for start in range(start_row, total, int(chunk_size)):
        stop = min(total, start + int(chunk_size))
        yy = np.asarray(y_test_f[start:stop], dtype=np.int64)
        if kind == 'fgsm':
            xb = torch.from_numpy(np.array(X_test[start:stop], dtype=np.float32, copy=True, order='C')).to(DEVICE_T)
            xb.requires_grad_(True)
            model.zero_grad(set_to_none=True)
            # Full precision for a reproducible gradient-sign perturbation.
            out_clean = model(xb)
            loss = F.cross_entropy(out_clean['logits_fine'], torch.from_numpy(yy).to(DEVICE_T))
            gradient = torch.autograd.grad(loss, xb, only_inputs=True)[0]
            adv = torch.clamp(xb + float(severity) * gradient.sign(), -float(cfg.POST_SCALE_CLIP), float(cfg.POST_SCALE_CLIP)).detach()
            with torch.inference_mode():
                out = model(adv)
            logits_f = out['logits_fine'].float().cpu().numpy()
            logits_c = out['logits_coarse'].float().cpu().numpy()
            del xb, out_clean, gradient, adv, out
        else:
            perturbed = _perturbation_chunk(X_test[start:stop], kind, severity, seed, start)
            xb = torch.from_numpy(perturbed).to(DEVICE_T)
            with torch.inference_mode(), amp_autocast(PAPER.USE_AMP and DEVICE_T.type == 'cuda'):
                out = model(xb)
            logits_f = out['logits_fine'].float().cpu().numpy()
            logits_c = out['logits_coarse'].float().cpu().numpy()
            del xb, out, perturbed
        probs_f = softmax_np(logits_f / max(PAPER_TEMP_FINE, 1e-6))
        probs_c = softmax_np(logits_c / max(PAPER_TEMP_COARSE, 1e-6))
        pred = probs_f.argmax(axis=1)
        flat = yy * NUM_FINE_PAPER + pred
        cm += np.bincount(flat, minlength=NUM_FINE_PAPER ** 2).reshape(NUM_FINE_PAPER, NUM_FINE_PAPER)
        counts = _raps_chunk_counts(probs_f, probs_c, yy, thresholds, alpha=0.10)
        for key in raps:
            raps[key] += counts[key]
        atomic_json(progress_path, {'signature': signature, 'next_row': stop, 'fine_confusion': cm,
                                    'raps': raps, 'condition': name, 'updated_at': time.strftime('%Y-%m-%d %H:%M:%S')})
        if stop == total or (start // int(chunk_size)) % 10 == 0:
            paper_log('robustness progress', stage='robustness', condition=name, rows=f'{stop}/{total}')
        check_deadline(f'robustness {name} row {stop}')
    fine_accuracy, fine_mf1 = _metrics_from_confusion(cm)
    singleton_acc = raps['singleton_correct'] / raps['singleton'] if raps['singleton'] else float('nan')
    result = {
        'signature': signature, 'condition': name, 'kind': kind, 'severity_numeric': severity,
        'fine_accuracy': fine_accuracy, 'fine_macro_f1': fine_mf1,
        'balanced_accuracy': _balanced_accuracy_from_cm(cm),
        'coverage': raps['covered'] / raps['n'], 'avg_set_size': raps['set_size_sum'] / raps['n'],
        'singleton_fraction': raps['singleton'] / raps['n'],
        'singleton_risk': 1.0 - singleton_acc,
        'source_kind': 'measured_perturbation_on_archived_checkpoint', 'profile': PAPER.PROFILE,
    }
    atomic_json(result_path, result)
    return result


def _mirai_coarse_id() -> Optional[int]:
    for i, name in enumerate(CANONICAL_COARSE_NAMES):
        if 'mirai' in str(name).lower():
            return int(i)
    return None


def run_mirai_family_holdout() -> Dict[str, Any]:
    mirai_id = _mirai_coarse_id()
    if mirai_id is None:
        return {'status': 'BLOCKED', 'reason': 'No Mirai coarse-family label was found.'}
    keep_train = np.asarray(y_train_c) != int(mirai_id)
    keep_val = np.asarray(y_val_c) != int(mirai_id)
    if PAPER.PROFILE != 'strict':
        tr_pool = np.flatnonzero(keep_train)
        va_pool = np.flatnonzero(keep_val)
        tr_sel = tr_pool[deterministic_subset_indices(y_train_f[tr_pool], min(profile_cap('train', len(tr_pool)), len(tr_pool)), 442)]
        va_sel = va_pool[deterministic_subset_indices(y_val_f[va_pool], min(profile_cap('val', len(va_pool)), len(va_pool)), 443)]
    else:
        tr_sel = np.flatnonzero(keep_train)
        va_sel = np.flatnonzero(keep_val)
    data = {
        'Xtr': X_train[tr_sel], 'yftr': y_train_f[tr_sel], 'yctr': y_train_c[tr_sel], 'ybtr': y_train_b[tr_sel],
        'Xv': X_val[va_sel], 'yfv': y_val_f[va_sel], 'ycv': y_val_c[va_sel], 'ybv': y_val_b[va_sel],
    }
    factory = _camelot_factory_with_groups(GROUP_IDXS)
    result = train_resumable_hierarchical(
        'robustness_mirai_family_holdout_seed_42', factory, seed=42,
        model_spec={'holdout': 'Mirai coarse family', 'mirai_coarse_id': mirai_id}, data_override=data,
    )
    if result.status != 'DONE':
        return {'status': result.status, 'reason': 'Resumable training incomplete.'}
    evaluation = evaluate_hierarchical_run('robustness_mirai_family_holdout_seed_42', factory, result.best_checkpoint)
    return {'status': 'DONE', 'fine_macro_f1': evaluation['fine']['macro_f1'],
            'balanced_accuracy': evaluation['fine']['balanced_accuracy'],
            'source_kind': 'measured_label-defined_family_holdout', 'profile': PAPER.PROFILE}


def run_robustness_suite() -> pd.DataFrame:
    signature = {'stage_version': 5, 'base': PAPER_BASE_SIGNATURE_HASH, 'profile': PAPER.PROFILE,
                 'conditions': ['clean', 'missing .1/.3/.5', 'gaussian .05/.10', 'fgsm .01', 'mqtt', 'mirai', '200shot']}
    out_path = PAPER_TABLE_DIR / 'table_s07_robustness.csv'
    with paper_stage('robustness', signature) as stage:
        if stage.skip and out_path.exists():
            return pd.read_csv(out_path)
        rows: List[Dict[str, Any]] = []
        # Clean row reuses P04 exactly.
        a10 = pd.read_csv(PAPER_TABLE_DIR / 'table_07_empirical_mondrian_raps.csv')
        a10 = a10[np.isclose(a10['alpha'].astype(float), 0.10)].iloc[0]
        rows.append({'perturbation': 'None', 'setting': 'Clean',
                     'fine_macro_f1_percent': 100 * ARCHIVED_CORE_SUMMARY['fine_metrics']['macro_f1'],
                     'balanced_accuracy_percent': 100 * ARCHIVED_CORE_SUMMARY['fine_metrics']['balanced_accuracy'],
                     'coverage_percent': float(a10['coverage_percent']), 'average_set_size': float(a10['average_set_size']),
                     'singleton_risk_percent': float(a10['singleton_risk_percent']),
                     'status': 'REUSED_ARCHIVED_V4', 'source_kind': 'archived_executed_result'})
        conditions = [
            ('Missing features', '10%', 'missing', 0.10),
            ('Missing features', '30%', 'missing', 0.30),
            ('Missing features', '50%', 'missing', 0.50),
            ('Gaussian noise', 'sigma=0.05', 'gaussian', 0.05),
            ('Gaussian noise', 'sigma=0.10', 'gaussian', 0.10),
            ('FGSM', 'epsilon=0.01', 'fgsm', 0.01),
        ]
        for perturbation, setting, kind, severity in conditions:
            result = _evaluate_robustness_condition(f'{perturbation}_{setting}', kind, severity)
            rows.append({'perturbation': perturbation, 'setting': setting,
                         'fine_macro_f1_percent': 100 * result['fine_macro_f1'],
                         'balanced_accuracy_percent': 100 * result['balanced_accuracy'],
                         'coverage_percent': 100 * result['coverage'], 'average_set_size': result['avg_set_size'],
                         'singleton_risk_percent': 100 * result['singleton_risk'],
                         'status': 'DONE', 'source_kind': result['source_kind'], 'profile': PAPER.PROFILE})
            atomic_dataframe(pd.DataFrame(rows), PAPER_TABLE_DIR / 'table_s07_robustness_partial')

        # MQTT requires a protocol-holdout split/manifest; it is not inferred from one-hot feature names.
        mqtt_root = Path(PAPER.ROBUSTNESS_ARTIFACT_DIR).expanduser() / 'mqtt_holdout' if str(PAPER.ROBUSTNESS_ARTIFACT_DIR).strip() else None
        if mqtt_root is None or not (mqtt_root / 'protocol.json').exists():
            rows.append({'perturbation': 'Protocol holdout', 'setting': 'MQTT', 'fine_macro_f1_percent': np.nan,
                         'balanced_accuracy_percent': np.nan, 'coverage_percent': np.nan, 'average_set_size': np.nan,
                         'singleton_risk_percent': np.nan, 'status': 'BLOCKED_PROTOCOL_ARTIFACT_MISSING',
                         'source_kind': 'not_run', 'note': 'Provide CAMELOT_ROBUSTNESS_ARTIFACT_DIR/mqtt_holdout protocol files.'})
        else:
            rows.append({'perturbation': 'Protocol holdout', 'setting': 'MQTT', 'status': 'READY_PROTOCOL_ARTIFACT',
                         'source_kind': 'run delegated to supplied protocol artifact',
                         'note': 'Use the protocol artifact runner described in protocol.json; no split is inferred.'})

        mirai = run_mirai_family_holdout()
        rows.append({'perturbation': 'Mirai family held out', 'setting': 'No support',
                     'fine_macro_f1_percent': 100 * mirai.get('fine_macro_f1', np.nan),
                     'balanced_accuracy_percent': 100 * mirai.get('balanced_accuracy', np.nan),
                     'coverage_percent': np.nan, 'average_set_size': np.nan, 'singleton_risk_percent': np.nan,
                     'status': mirai.get('status', 'UNKNOWN'), 'source_kind': mirai.get('source_kind', 'not_run'),
                     'profile': mirai.get('profile', PAPER.PROFILE), 'note': mirai.get('reason', '')})
        if mirai.get('status') == 'PAUSED':
            atomic_dataframe(pd.DataFrame(rows), PAPER_TABLE_DIR / 'table_s07_robustness_partial')
            raise TimeBudgetReached('Mirai family-holdout training paused.')

        shot_file = (Path(PAPER.ROBUSTNESS_ARTIFACT_DIR).expanduser() / 'mirai_200shot_indices.npy') if str(PAPER.ROBUSTNESS_ARTIFACT_DIR).strip() else None
        if shot_file is None or not shot_file.exists():
            rows.append({'perturbation': 'Mirai few-shot support', 'setting': '200 labeled flows',
                         'fine_macro_f1_percent': np.nan, 'balanced_accuracy_percent': np.nan,
                         'coverage_percent': np.nan, 'average_set_size': np.nan, 'singleton_risk_percent': np.nan,
                         'status': 'BLOCKED_LABEL_ACCESS_PROTOCOL_MISSING', 'source_kind': 'not_run',
                         'note': 'Provide exact 200-shot indices and label-access protocol; no sample is invented.'})
        else:
            rows.append({'perturbation': 'Mirai few-shot support', 'setting': '200 labeled flows',
                         'status': 'READY_EXACT_INDICES', 'source_kind': 'pending_head_tuning_runner',
                         'note': str(shot_file)})

        table = pd.DataFrame(rows)
        atomic_dataframe(table, PAPER_TABLE_DIR / 'table_s07_robustness')
        blocked = table['status'].astype(str).str.startswith('BLOCKED').any()
        update_coverage_status('Table S7', 'PARTIAL_BLOCKED' if blocked else 'DONE', str(out_path),
                               'Inference perturbations are measured; holdout/few-shot rows require exact split/label-access artifacts.')
        return table


print('Robustness suite defined. Long FGSM and Mirai-holdout stages resume from chunk/epoch checkpoints.')
paper_cell_done('P13', 'robustness suite: missingness, noise, FGSM, and holdout protocols', _started)


[P13] START: robustness suite: missingness, noise, FGSM, and holdout protocols
Robustness suite defined. Long FGSM and Mirai-holdout stages resume from chunk/epoch checkpoints.
[P13] DONE: robustness suite: missingness, noise, FGSM, and holdout protocols | elapsed=0.02s


In [16]:
_started = paper_cell_start('P14', 'archived stream stationarity replay and trigger-gated adaptation pathway')


class PaperConformalMartingale:
    def __init__(self, reference: np.ndarray, epsilon: float = 0.5, threshold: float = 25.0):
        self.reference = np.asarray(reference, dtype=np.float32)
        self.epsilon = float(epsilon)
        self.threshold = float(threshold)
        self.value = 1.0

    def p_value(self, statistic: float) -> float:
        return float((1.0 + np.sum(self.reference >= float(statistic))) / (len(self.reference) + 1.0))

    def update(self, statistic: float) -> Dict[str, Any]:
        p = self.p_value(statistic)
        self.value *= self.epsilon * (p ** (self.epsilon - 1.0))
        triggered = bool(self.value > self.threshold)
        observed = float(self.value)
        if triggered:
            self.value = 1.0
        return {'p_value': p, 'martingale': observed, 'triggered': triggered}


class TentLayerNormAdapter:
    def __init__(self, model: nn.Module, lr: float = 1e-4, passes: int = 5):
        self.model = model
        self.passes = int(passes)
        for parameter in self.model.parameters():
            parameter.requires_grad_(False)
        parameters: List[nn.Parameter] = []
        for module in self.model.modules():
            if isinstance(module, nn.LayerNorm):
                for parameter in module.parameters(recurse=False):
                    parameter.requires_grad_(True)
                    parameters.append(parameter)
        self.parameters = parameters
        self.optimizer = torch.optim.Adam(parameters, lr=float(lr), weight_decay=0.0) if parameters else None
        for module in self.model.modules():
            if isinstance(module, nn.Dropout):
                module.eval()

    def adapt(self, batches: Sequence[np.ndarray], micro_batch: int) -> None:
        if self.optimizer is None:
            return
        self.model.train()
        for pass_idx in range(self.passes):
            for array in batches:
                for start in range(0, len(array), int(micro_batch)):
                    xb = torch.from_numpy(np.array(array[start:start + int(micro_batch)], dtype=np.float32, copy=True, order='C')).to(DEVICE_T)
                    output = self.model(xb)
                    probs = torch.softmax(output['logits_fine'], dim=1)
                    entropy = -(probs * torch.log(probs + 1e-8)).sum(dim=1).mean()
                    self.optimizer.zero_grad(set_to_none=True)
                    entropy.backward()
                    self.optimizer.step()
        self.model.eval()


class PseudoLabelHeadTuner:
    def __init__(self, model: nn.Module, lr: float = 5e-5, passes: int = 10):
        self.model = model
        self.passes = int(passes)
        for parameter in self.model.parameters():
            parameter.requires_grad_(False)
        parameters: List[nn.Parameter] = []
        for head_name in ('head_fine', 'head_coarse', 'head_bin'):
            head = getattr(self.model, head_name)
            for parameter in head.parameters():
                parameter.requires_grad_(True)
                parameters.append(parameter)
        self.optimizer = torch.optim.Adam(parameters, lr=float(lr), weight_decay=0.0)

    def tune(self, X_selected: np.ndarray, y_fine: np.ndarray, micro_batch: int) -> None:
        if len(X_selected) == 0:
            return
        y_coarse = np.asarray(fine_to_coarse, dtype=np.int64)[np.asarray(y_fine, dtype=np.int64)]
        y_binary = (np.asarray(y_fine, dtype=np.int64) != int(benign_fine_id)).astype(np.int64)
        self.model.train()
        for pass_idx in range(self.passes):
            for start in range(0, len(X_selected), int(micro_batch)):
                stop = min(len(X_selected), start + int(micro_batch))
                xb = torch.from_numpy(np.array(X_selected[start:stop], dtype=np.float32, copy=True, order='C')).to(DEVICE_T)
                yf = torch.from_numpy(y_fine[start:stop].astype(np.int64)).to(DEVICE_T)
                yc = torch.from_numpy(y_coarse[start:stop].astype(np.int64)).to(DEVICE_T)
                yb = torch.from_numpy(y_binary[start:stop].astype(np.int64)).to(DEVICE_T)
                output = self.model(xb)
                loss = (F.cross_entropy(output['logits_fine'], yf) + 0.30 * F.cross_entropy(output['logits_coarse'], yc)
                        + 0.20 * F.cross_entropy(output['logits_bin'], yb))
                self.optimizer.zero_grad(set_to_none=True)
                loss.backward()
                self.optimizer.step()
        self.model.eval()


def _stream_arrays() -> Tuple[Optional[np.ndarray], Optional[np.ndarray]]:
    if 'X_stream_test' in globals() and globals()['X_stream_test'] is not None:
        return np.asarray(globals()['X_stream_test']), np.asarray(globals()['y_stream_f'])
    root = Path(cfg.OUT_DIR) / 'cache' / 'stream'
    xp, yp = root / 'X_stream_test.npy', root / 'y_stream_f.npy'
    if xp.exists() and yp.exists():
        return np.load(xp), np.load(yp)
    return None, None


@torch.inference_mode()
def _stream_predict(model: nn.Module, X_batch: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    fine_parts, coarse_parts = [], []
    for start in range(0, len(X_batch), int(cfg.STREAM_MICRO_BATCH)):
        xb = torch.from_numpy(np.array(X_batch[start:start + int(cfg.STREAM_MICRO_BATCH)], dtype=np.float32, copy=True, order='C')).to(DEVICE_T)
        with amp_autocast(PAPER.USE_AMP and DEVICE_T.type == 'cuda'):
            output = model(xb)
        fine_parts.append(torch.softmax(output['logits_fine'] / max(PAPER_TEMP_FINE, 1e-6), dim=1).float().cpu().numpy())
        coarse_parts.append(torch.softmax(output['logits_coarse'] / max(PAPER_TEMP_COARSE, 1e-6), dim=1).float().cpu().numpy())
    return np.concatenate(fine_parts), np.concatenate(coarse_parts)


def _make_adwin() -> Any:
    if importlib.util.find_spec('river') is None:
        return None
    from river import drift
    return drift.ADWIN(delta=float(cfg.DRIFT_DELTA))


def _replay_detector_history(adwin: Any, martingale: PaperConformalMartingale, history: Sequence[Mapping[str, Any]]) -> None:
    for row in history:
        statistic = float(row['mean_entropy'])
        if adwin is not None:
            adwin.update(statistic)
        martingale.update(statistic)


def run_stream_stationarity() -> pd.DataFrame:
    Xs, ys = _stream_arrays()
    if Xs is None or ys is None:
        update_coverage_status('Table 9', 'BLOCKED', note='Run original v4 preprocessing with CACHE_STREAM_INPUTS=True.')
        return write_blocked_stage('stream_stationarity', 'The row-ordered stream arrays are unavailable.')
    existing_path = Path(cfg.OUT_DIR) / 'stream_results.json'
    signature = {'stage_version': 4, 'base': PAPER_BASE_SIGNATURE_HASH, 'rows': len(Xs),
                 'logical_batch': cfg.STREAM_BATCH, 'micro_batch': cfg.STREAM_MICRO_BATCH,
                 'delta': cfg.DRIFT_DELTA, 'mart_eps': cfg.MART_EPS, 'mart_threshold': cfg.MART_THRESHOLD}
    out_path = PAPER_TABLE_DIR / 'table_09_stream_stationarity.csv'
    with paper_stage('stream_stationarity', signature) as stage:
        if stage.skip and out_path.exists():
            return pd.read_csv(out_path)
        # Reuse the exact archived result when it is present; still export a code-aligned table.
        if existing_path.exists() and not PAPER.FORCE_RERUN:
            archived = read_json(existing_path, {})
            drift_points = list(archived.get('drift_points', []))
            logical_batches = int(math.ceil(len(Xs) / int(cfg.STREAM_BATCH)))
            table = pd.DataFrame([
                ['Test-stream flows', int(len(Xs))], ['Logical batches', logical_batches],
                ['Logical/micro-batch size', f'{int(cfg.STREAM_BATCH):,}/{int(cfg.STREAM_MICRO_BATCH):,}'],
                ['Monitored statistic', 'Mean 34-class fine-head entropy'],
                ['Detector', f'ADWIN (delta={cfg.DRIFT_DELTA}) + conformal martingale (epsilon={cfg.MART_EPS}; threshold {cfg.MART_THRESHOLD:g})'],
                ['Combined drift points', len(drift_points)], ['Adaptation events', len(drift_points)],
            ], columns=['item', 'archived_value'])
            table['source_kind'] = 'reused_exact_v4_stream_result'
            atomic_dataframe(table, PAPER_TABLE_DIR / 'table_09_stream_stationarity')
            update_coverage_status('Table 9', 'DONE', str(out_path), 'Existing v4 stream result reused; no new adaptation claim.')
            return table

        adwin = _make_adwin()
        if adwin is None:
            update_coverage_status('Table 9', 'BLOCKED', note='Install river or retain the original v4 stream_results.json cache.')
            return write_blocked_stage('stream_stationarity', 'river is unavailable and no cached v4 stream result exists.')
        reference_entropy = -np.sum(PAPER_CAL_PROBS_FINE * np.log(PAPER_CAL_PROBS_FINE + 1e-12), axis=1)
        reference_entropy = reference_entropy[:min(len(reference_entropy), 200_000)]
        martingale = PaperConformalMartingale(reference_entropy, cfg.MART_EPS, cfg.MART_THRESHOLD)
        progress_path = stage_path('stream_stationarity') / 'progress.json'
        model_path = stage_path('stream_stationarity') / 'adapted_model_checkpoint.pt'
        progress = read_json(progress_path, {})
        history: List[Dict[str, Any]] = list(progress.get('history', [])) if progress.get('signature') == signature_hash(signature) else []
        start_batch = len(history)
        model = copy.deepcopy(PAPER_MODEL).to(DEVICE_T).eval()
        if start_batch and model_path.exists():
            blob = safe_torch_load(model_path, map_location='cpu')
            model.load_state_dict(blob['model_state'], strict=True)
            _replay_detector_history(adwin, martingale, history)
            paper_log('stream replay resumed', stage='stream_stationarity', next_batch=start_batch)
        thresholds = globals().get('ARCHIVED_RAPS_THRESHOLDS') or read_json(PAPER_CACHE_DIR / 'archived_empirical_raps_thresholds.json', {})
        drift_points: List[int] = [int(r['batch_index']) for r in history if bool(r.get('combined_trigger'))]
        adaptation_events = int(sum(bool(r.get('adaptation_invoked')) for r in history))
        total_batches = int(math.ceil(len(Xs) / int(cfg.STREAM_BATCH)))
        for batch_index in range(start_batch, total_batches):
            lo = batch_index * int(cfg.STREAM_BATCH)
            hi = min(len(Xs), lo + int(cfg.STREAM_BATCH))
            X_batch = np.asarray(Xs[lo:hi], dtype=np.float32)
            y_batch = np.asarray(ys[lo:hi], dtype=np.int64)
            probs_f, probs_c = _stream_predict(model, X_batch)
            mean_entropy = float(np.mean(-np.sum(probs_f * np.log(probs_f + 1e-12), axis=1)))
            adwin.update(mean_entropy)
            adwin_trigger = bool(getattr(adwin, 'drift_detected', False))
            mart = martingale.update(mean_entropy)
            combined = bool(adwin_trigger or mart['triggered'])
            adaptation_invoked = False
            singleton_count = 0
            if combined:
                drift_points.append(batch_index)
                # Rebuild the three-batch buffer from immutable stream arrays; this is resume-safe.
                first_buffer_batch = max(0, batch_index - int(cfg.ADAPT_BUFFER_BATCHES) + 1)
                buffer_arrays = [np.asarray(Xs[b * int(cfg.STREAM_BATCH):min(len(Xs), (b + 1) * int(cfg.STREAM_BATCH))], dtype=np.float32)
                                 for b in range(first_buffer_batch, batch_index + 1)]
                tent = TentLayerNormAdapter(model, lr=cfg.ADAPT_LR, passes=cfg.ADAPT_STEPS)
                tent.adapt(buffer_arrays, int(cfg.STREAM_MICRO_BATCH))
                if thresholds:
                    akey = f'{float(cfg.ADAPT_ALPHA):.12g}'
                    q = np.asarray([row['threshold'] for row in thresholds['groups'][akey]], dtype=np.float32)
                    group = probs_c.argmax(axis=1)
                    order = np.argsort(-probs_f, axis=1)
                    sorted_p = np.take_along_axis(probs_f, order, axis=1)
                    cumulative = np.cumsum(sorted_p, axis=1)
                    ranks = np.arange(1, probs_f.shape[1] + 1, dtype=np.float32)[None, :]
                    score = cumulative + float(PAPER.RAPS_LAMBDA) * np.maximum(ranks - float(PAPER.RAPS_KREG), 0.0)
                    k_star = np.maximum((score <= q[group, None]).sum(axis=1), 1)
                    singleton = k_star == 1
                    singleton_count = int(singleton.sum())
                    if singleton_count >= int(cfg.MIN_PSEUDO):
                        tuner = PseudoLabelHeadTuner(model, lr=cfg.HEAD_TUNE_LR, passes=cfg.HEAD_TUNE_STEPS)
                        tuner.tune(X_batch[singleton], order[singleton, 0].astype(np.int64), int(cfg.STREAM_MICRO_BATCH))
                adaptation_invoked = True
                adaptation_events += 1
            row = {
                'batch_index': batch_index, 'row_start': lo, 'row_stop': hi,
                'mean_entropy': mean_entropy, 'display_accuracy_using_labels_only': float((probs_f.argmax(axis=1) == y_batch).mean()),
                'adwin_trigger': adwin_trigger, 'martingale_p': mart['p_value'],
                'martingale_value': mart['martingale'], 'martingale_trigger': mart['triggered'],
                'combined_trigger': combined, 'adaptation_invoked': adaptation_invoked,
                'singleton_pseudo_labels': singleton_count,
            }
            history.append(row)
            atomic_dataframe(pd.DataFrame(history), PAPER_TABLE_DIR / 'stream_batch_log')
            atomic_torch(model_path, {'model_state': model_state_cpu(model), 'batch_index': batch_index})
            atomic_json(progress_path, {'signature': signature_hash(signature), 'history': history,
                                        'updated_at': time.strftime('%Y-%m-%d %H:%M:%S')})
            paper_log('stream logical batch completed', stage='stream_stationarity', batch=f'{batch_index+1}/{total_batches}',
                      mean_entropy=round(mean_entropy, 6), trigger=combined, adaptation=adaptation_invoked)
            check_deadline(f'stream batch {batch_index+1}')
        table = pd.DataFrame([
            ['Test-stream flows', int(len(Xs))], ['Logical batches', total_batches],
            ['Logical/micro-batch size', f'{int(cfg.STREAM_BATCH):,}/{int(cfg.STREAM_MICRO_BATCH):,}'],
            ['Monitored statistic', 'Mean 34-class fine-head entropy'],
            ['Detector', f'ADWIN (delta={cfg.DRIFT_DELTA}) + conformal martingale (epsilon={cfg.MART_EPS}; threshold {cfg.MART_THRESHOLD:g})'],
            ['Combined drift points', len(drift_points)], ['Adaptation events', adaptation_events],
        ], columns=['item', 'archived_value'])
        table['source_kind'] = 'measured_stream_replay'
        atomic_dataframe(table, PAPER_TABLE_DIR / 'table_09_stream_stationarity')
        atomic_json(PAPER_REPORT_DIR / 'stream_stationarity_result.json', {'history': history, 'drift_points': drift_points,
                                                                          'adaptation_events': adaptation_events})
        update_coverage_status('Table 9', 'DONE', str(out_path),
                               'Ground-truth labels are used only for displayed accuracy, never by the detector.')
        return table


def run_exploratory_shift_unit_tests() -> pd.DataFrame:
    if not PAPER.RUN_EXPLORATORY_SYNTHETIC_SHIFT_TESTS:
        return pd.DataFrame([{'status': 'DISABLED', 'note': 'Exploratory shift tests do not populate manuscript evidence tables.'}])
    # Deliberately small unit tests verify trigger/adaptation plumbing only.
    rng = np.random.default_rng(42)
    base = np.asarray(X_test[:50_000], dtype=np.float32)
    shifted = np.clip(base + rng.normal(0, 0.25, size=base.shape).astype(np.float32), -cfg.POST_SCALE_CLIP, cfg.POST_SCALE_CLIP)
    return pd.DataFrame([{'scenario': 'Gaussian feature-space unit test', 'rows': len(shifted),
                          'status': 'GENERATED_EXPLORATORY_ONLY', 'manuscript_claim': False}])


print('Stream replay defined with threshold=25, immediate OR trigger, and no five-batch persistence rule.')
paper_cell_done('P14', 'archived stream stationarity replay and trigger-gated adaptation pathway', _started)


[P14] START: archived stream stationarity replay and trigger-gated adaptation pathway
Stream replay defined with threshold=25, immediate OR trigger, and no five-batch persistence rule.
[P14] DONE: archived stream stationarity replay and trigger-gated adaptation pathway | elapsed=0.02s


In [ ]:
_started = paper_cell_start('P15', 'native PyTorch GPU/CPU throughput benchmark for Table 8')


def _normalize_archived_throughput(payload: Any) -> Optional[Dict[str, float]]:
    if not isinstance(payload, Mapping):
        return None
    throughput = payload.get('samples_per_sec', payload.get('throughput_samples_s'))
    latency = payload.get('ms_per_sample')
    if throughput is None:
        return None
    if latency is None:
        latency = 1000.0 / float(throughput)
    return {'throughput_samples_s': float(throughput), 'ms_per_sample': float(latency)}


def run_throughput_table() -> pd.DataFrame:
    signature = {'stage_version': 3, 'base': PAPER_BASE_SIGNATURE_HASH, 'batch_size': 4096,
                 'max_rows': min(200_000, len(X_test)), 'warmup_passes': 5,
                 'runtime': runtime_info.get('torch'), 'gpu': runtime_info.get('gpu_name')}
    out_path = PAPER_TABLE_DIR / 'table_08_native_pytorch_throughput.csv'
    with paper_stage('throughput_profile', signature) as stage:
        if stage.skip and out_path.exists():
            return pd.read_csv(out_path)
        gpu = _normalize_archived_throughput(globals().get('throughput_gpu'))
        cpu = _normalize_archived_throughput(globals().get('throughput_cpu'))
        source = 'reused_exact_v4_benchmark'
        if gpu is None:
            if DEVICE_T.type == 'cuda':
                gpu = benchmark_native_pytorch(PAPER_MODEL, X_test, DEVICE_T, batch_size=4096, max_rows=200_000, warmup_passes=5)
            else:
                gpu = {'throughput_samples_s': np.nan, 'ms_per_sample': np.nan}
            source = 'measured_by_appended_benchmark'
        if cpu is None:
            cpu_model = copy.deepcopy(PAPER_MODEL).cpu().eval()
            cpu = benchmark_native_pytorch(cpu_model, X_test, torch.device('cpu'), batch_size=4096, max_rows=200_000, warmup_passes=5)
            del cpu_model
            source = 'measured_by_appended_benchmark'
        rows = [
            {'platform': f'GPU ({runtime_info.get("gpu_name", "CUDA device")})', 'model': 'CAMELOT-IDS',
             'throughput_samples_s': gpu['throughput_samples_s'], 'ms_per_sample': gpu['ms_per_sample'],
             'batch_size': 4096, 'max_rows': min(200_000, len(X_test)), 'source_kind': source},
            {'platform': 'CPU (host platform)', 'model': 'CAMELOT-IDS',
             'throughput_samples_s': cpu['throughput_samples_s'], 'ms_per_sample': cpu['ms_per_sample'],
             'batch_size': 4096, 'max_rows': min(200_000, len(X_test)), 'source_kind': source},
        ]
        table = pd.DataFrame(rows)
        atomic_dataframe(table, PAPER_TABLE_DIR / 'table_08_native_pytorch_throughput')
        update_coverage_status('Table 8', 'DONE', str(out_path),
                               'Batch throughput on the tested desktop-class platforms; not single-flow network latency.')
        return table


THROUGHPUT_TABLE8 = run_throughput_table() if PAPER.AUTO_RUN_LIGHT else pd.DataFrame()
if len(THROUGHPUT_TABLE8):
    display(THROUGHPUT_TABLE8)
paper_cell_done('P15', 'native PyTorch GPU/CPU throughput benchmark for Table 8', _started)


In [18]:
_started = paper_cell_start('P16', 'paper baseline suite: resumable trees, neural models, Table S10/S11/S13')

import joblib
from sklearn.model_selection import GroupKFold


# -----------------------------------------------------------------------------
# Supplementary Table S11: code-defined configurations. This table is exported
# immediately; result rows are populated only from executed checkpoints.
# -----------------------------------------------------------------------------
TABLE_S11_CONFIG = pd.DataFrame([
    {
        'model': 'XGBoost',
        'configuration': 'multi:softprob; 34 classes; hist; sample weight N/(34*N_c); depth=8; trees=500; LR=.08; subsample=.90; colsample=.90; seed=42',
        'selection_protocol': '5-fold group-aware CV by root_capture_id; exact search grid/log required to reproduce selection',
        'implementation': 'native xgboost.train in resumable boosting-round chunks',
    },
    {
        'model': 'LightGBM',
        'configuration': 'multiclass; 34 classes; sample weight N/(34*N_c); leaves=127; trees=700; LR=.05; max_depth=-1; seed=42',
        'selection_protocol': '5-fold group-aware CV by root_capture_id; exact search grid/log required to reproduce selection',
        'implementation': 'native lightgbm.train in resumable boosting-round chunks',
    },
    {
        'model': 'DNN+SMOTE',
        'configuration': '46-512-512-256-128-64-34; Linear-LayerNorm-GELU-Dropout(.10); SMOTE k=5 to 10% of largest class; no post-SMOTE class weight; AdamW',
        'selection_protocol': 'fixed capture-disjoint validation; best validation fine macro-F1',
        'implementation': 'exact SMOTE cache builder is opt-in because full-data materialization is memory/disk intensive',
    },
    {
        'model': 'CNN-GRU',
        'configuration': 'Conv1d 1->64 (k=3), 64->128 (k=3), GELU, max-pool 2; one-layer GRU hidden=256; LayerNorm; Dropout(.10); class-balanced CE',
        'selection_protocol': 'best validation fine macro-F1',
        'implementation': 'intra-epoch model/optimizer/scheduler/scaler/EMA/RNG checkpoints',
    },
    {
        'model': 'Flat Transformer',
        'configuration': '46 feature tokens + CLS; d=256; 8 heads; 6 layers; FFN=1024; Dropout(.10); LDAM-DRW; hierarchical heads',
        'selection_protocol': 'best validation fine macro-F1',
        'implementation': 'shared with Table 10 flat-46 ablation; intra-epoch resume',
    },
    {
        'model': 'FT-Transformer',
        'configuration': 'feature-specific scalar tokenizer; CLS; d=192; 8 heads; 4 layers; FFN=768; Dropout(.10); class-balanced CE',
        'selection_protocol': 'best validation fine macro-F1',
        'implementation': 'intra-epoch resume',
    },
    {
        'model': 'TabTransformer adaptation',
        'configuration': 'numeric tokens x_i*w_i+b_i plus feature-ID embedding; d=128; 8 heads; 4 layers; FFN=512; mean pooling; class-balanced CE',
        'selection_protocol': 'best validation fine macro-F1',
        'implementation': 'intra-epoch resume',
    },
    {
        'model': 'SAINT',
        'configuration': 'two feature-attention layers + one row-attention layer over current minibatch; d=128; 8 heads; FFN=512; no contrastive pretraining; class-balanced CE',
        'selection_protocol': 'best validation fine macro-F1',
        'implementation': 'intra-epoch resume; batch-order-sensitive by design',
    },
    {
        'model': 'TabNet',
        'configuration': 'n_d=n_a=64; 5 steps; gamma=1.5; batch=1024; virtual batch=256; AdamW; class-weighted samples; validation balanced accuracy',
        'selection_protocol': 'fixed capture-disjoint validation because group-aware CV is incompatible with documented virtual-batch path',
        'implementation': 'optional pytorch-tabnet; epoch-boundary weight resume (optimizer state is not exposed by its public save/load API)',
    },
    {
        'model': 'GraphSAGE-GNN',
        'configuration': 'batch-local standardized-feature kNN graph; Euclidean k=10; hidden=256; two mean-aggregation layers; class-balanced CE',
        'selection_protocol': 'best validation fine macro-F1',
        'implementation': 'intra-epoch resume; no edges across minibatches or partitions',
    },
], dtype=object)
TABLE_S11_CONFIG['profile'] = PAPER.PROFILE
TABLE_S11_CONFIG['source_kind'] = 'implemented_configuration_not_result'
atomic_dataframe(TABLE_S11_CONFIG, PAPER_TABLE_DIR / 'table_s11_baseline_configurations')


def _class_sample_weights(y: np.ndarray, n_classes: int) -> np.ndarray:
    y = np.asarray(y, dtype=np.int64)
    counts = np.bincount(y, minlength=int(n_classes)).astype(np.float64)
    weights = np.zeros_like(counts)
    nonzero = counts > 0
    weights[nonzero] = len(y) / (float(n_classes) * counts[nonzero])
    return weights[y].astype(np.float32)


def _baseline_eval_indices(seed: int = 42) -> Tuple[np.ndarray, np.ndarray]:
    if PAPER.PROFILE == 'strict':
        return np.arange(len(y_cal_f), dtype=np.int64), np.arange(len(y_test_f), dtype=np.int64)
    cal_path = PAPER_CACHE_DIR / 'profile_indices' / PAPER.PROFILE / f'baseline_cal_seed_{seed}.npy'
    test_path = PAPER_CACHE_DIR / 'profile_indices' / PAPER.PROFILE / f'baseline_test_seed_{seed}.npy'
    cal_path.parent.mkdir(parents=True, exist_ok=True)
    if cal_path.exists() and test_path.exists() and not PAPER.FORCE_RERUN:
        return np.load(cal_path), np.load(test_path)
    ci = deterministic_subset_indices(y_cal_f, profile_cap('cal', len(y_cal_f)), seed + 200)
    # Always evaluate every baseline on the complete archived test order. This is
    # inexpensive relative to training and preserves row alignment for S17.
    ti = np.arange(len(y_test_f), dtype=np.int64)
    atomic_npy(cal_path, ci)
    atomic_npy(test_path, ti)
    return ci, ti


def _tree_predict_proba_chunked(model: Any, X: np.ndarray, kind: str, chunk_size: int = 100_000) -> np.ndarray:
    parts: List[np.ndarray] = []
    started = time.perf_counter()
    for start in range(0, len(X), int(chunk_size)):
        stop = min(len(X), start + int(chunk_size))
        block = np.ascontiguousarray(np.asarray(X[start:stop]), dtype=np.float32)
        if kind == 'xgboost':
            import xgboost as xgb
            pred = model.predict(xgb.DMatrix(block))
        elif kind == 'lightgbm':
            pred = model.predict(block, num_iteration=model.current_iteration())
        else:
            pred = model.predict_proba(block)
        parts.append(np.asarray(pred, dtype=np.float32))
        paper_log('tree prediction progress', stage=f'{kind}_predict', rows=stop,
                  samples_s=round(stop / max(time.perf_counter() - started, 1e-9), 1))
        check_deadline(f'{kind} prediction row {stop}')
    return np.concatenate(parts, axis=0) if parts else np.empty((0, NUM_FINE_PAPER), dtype=np.float32)


def _evaluate_probability_baseline(
    model_name: str,
    cal_probs_raw: np.ndarray,
    test_probs_raw: np.ndarray,
    y_cal_local: np.ndarray,
    y_test_f_local: np.ndarray,
    y_test_c_local: np.ndarray,
    y_test_b_local: np.ndarray,
    throughput: Mapping[str, Any],
    *,
    source_kind: str,
    status: str = 'MEASURED',
    note: str = '',
    cache_slug: Optional[str] = None,
) -> Dict[str, Any]:
    cal_probs_raw = np.clip(np.asarray(cal_probs_raw, dtype=np.float64), 1e-12, 1.0)
    test_probs_raw = np.clip(np.asarray(test_probs_raw, dtype=np.float64), 1e-12, 1.0)
    # Tree/package probabilities are converted to log-probability logits before
    # scalar temperature fitting, preserving the probability simplex.
    temperature = float(fit_temperature_np(np.log(cal_probs_raw), np.asarray(y_cal_local, dtype=np.int64)))
    test_probs = softmax_np(np.log(test_probs_raw) / max(temperature, 1e-6))
    pred_f = test_probs.argmax(axis=1).astype(np.int64)
    pred_c, pred_b = derive_auxiliary_predictions_from_fine(pred_f)
    fine = {
        'accuracy': float(accuracy_score(y_test_f_local, pred_f)),
        'balanced_accuracy': float(balanced_accuracy_score(y_test_f_local, pred_f)),
        'macro_f1': float(f1_score(y_test_f_local, pred_f, average='macro', zero_division=0)),
        'weighted_f1': float(f1_score(y_test_f_local, pred_f, average='weighted', zero_division=0)),
        'mcc': float(matthews_corrcoef(y_test_f_local, pred_f)),
    }
    coarse = {
        'accuracy': float(accuracy_score(y_test_c_local, pred_c)),
        'balanced_accuracy': float(balanced_accuracy_score(y_test_c_local, pred_c)),
        'macro_f1': float(f1_score(y_test_c_local, pred_c, average='macro', zero_division=0)),
    }
    binary = {
        'accuracy': float(accuracy_score(y_test_b_local, pred_b)),
        'balanced_accuracy': float(balanced_accuracy_score(y_test_b_local, pred_b)),
        'malicious_f1': float(f1_score(y_test_b_local, pred_b, pos_label=1, zero_division=0)),
    }
    slug = cache_slug or re.sub(r'[^A-Za-z0-9_.-]+', '_', model_name.lower())
    pred_path = PAPER_CACHE_DIR / 'baseline_predictions' / f'{slug}.npz'
    atomic_npz(pred_path, pred_fine=pred_f.astype(np.int16), pred_coarse=pred_c.astype(np.int8),
               pred_binary=pred_b.astype(np.int8))
    payload = {
        'run_name': model_name,
        'temperature': temperature,
        'fine': fine,
        'coarse': coarse,
        'binary': binary,
        'calibration': calibration_metrics(y_test_f_local, test_probs, PAPER.ECE_BINS),
        'throughput': dict(throughput),
        'profile': PAPER.PROFILE,
        'source_kind': source_kind,
        'status': status,
        'note': note,
        'evaluation_rows': int(len(y_test_f_local)),
        'calibration_rows': int(len(y_cal_local)),
        'prediction_cache': str(pred_path),
    }
    atomic_json(PAPER_CACHE_DIR / 'baseline_evaluations' / f'{slug}.json', payload)
    return payload


def _train_xgboost_resumable(seed: int = 42) -> Tuple[Any, Dict[str, Any]]:
    if importlib.util.find_spec('xgboost') is None:
        raise RuntimeError('xgboost is not installed')
    import xgboost as xgb
    data = _training_data_for_profile(seed)
    run_dir = PAPER_MODEL_DIR / 'baseline_xgboost'
    run_dir.mkdir(parents=True, exist_ok=True)
    model_path = run_dir / 'booster.ubj'
    progress_path = run_dir / 'progress.json'
    total_rounds, chunk = 500, 25
    params = {
        'objective': 'multi:softprob', 'num_class': NUM_FINE_PAPER, 'tree_method': 'hist',
        'max_depth': 8, 'eta': 0.08, 'subsample': 0.90, 'colsample_bytree': 0.90,
        'eval_metric': 'mlogloss', 'seed': int(seed), 'nthread': int(PAPER.TREE_THREADS),
    }
    sig_payload = {'engine': 2, 'model': 'xgboost', 'profile': PAPER.PROFILE, 'seed': seed,
                   'params': params, 'rows': len(data['Xtr']), 'base': PAPER_BASE_SIGNATURE_HASH}
    sig = signature_hash(sig_payload)
    progress = read_json(progress_path, {})
    completed = int(progress.get('completed_rounds', 0)) if progress.get('signature') == sig else 0
    booster = None
    if completed > 0 and model_path.exists() and not PAPER.FORCE_RERUN:
        booster = xgb.Booster()
        booster.load_model(model_path)
        paper_log('XGBoost boosting-round checkpoint resumed', stage='baseline_xgboost', completed_rounds=completed)
    else:
        completed = 0
    dtrain = xgb.DMatrix(np.ascontiguousarray(data['Xtr'], dtype=np.float32), label=data['yftr'],
                         weight=_class_sample_weights(data['yftr'], NUM_FINE_PAPER), nthread=int(PAPER.TREE_THREADS))
    while completed < total_rounds:
        add = min(chunk, total_rounds - completed)
        booster = xgb.train(params, dtrain, num_boost_round=add, xgb_model=booster, verbose_eval=False)
        completed += add
        tmp = model_path.with_name(model_path.name + '.tmp')
        booster.save_model(tmp)
        os.replace(tmp, model_path)
        atomic_json(progress_path, {'signature': sig, 'signature_payload': sig_payload,
                                    'completed_rounds': completed, 'status': 'DONE' if completed >= total_rounds else 'RUNNING'})
        paper_log('XGBoost boosting chunk checkpointed', stage='baseline_xgboost', rounds=f'{completed}/{total_rounds}')
        check_deadline(f'XGBoost round {completed}')
    return booster, {'checkpoint': str(model_path), 'rounds': completed, 'signature': sig}


def _train_lightgbm_resumable(seed: int = 42) -> Tuple[Any, Dict[str, Any]]:
    if importlib.util.find_spec('lightgbm') is None:
        raise RuntimeError('lightgbm is not installed')
    import lightgbm as lgb
    data = _training_data_for_profile(seed)
    run_dir = PAPER_MODEL_DIR / 'baseline_lightgbm'
    run_dir.mkdir(parents=True, exist_ok=True)
    model_path = run_dir / 'booster.txt'
    progress_path = run_dir / 'progress.json'
    total_rounds, chunk = 700, 25
    params = {
        'objective': 'multiclass', 'num_class': NUM_FINE_PAPER, 'num_leaves': 127,
        'learning_rate': 0.05, 'max_depth': -1, 'seed': int(seed),
        'num_threads': int(PAPER.TREE_THREADS), 'verbosity': -1,
    }
    sig_payload = {'engine': 2, 'model': 'lightgbm', 'profile': PAPER.PROFILE, 'seed': seed,
                   'params': params, 'rows': len(data['Xtr']), 'base': PAPER_BASE_SIGNATURE_HASH}
    sig = signature_hash(sig_payload)
    progress = read_json(progress_path, {})
    completed = int(progress.get('completed_rounds', 0)) if progress.get('signature') == sig else 0
    booster = lgb.Booster(model_file=str(model_path)) if completed > 0 and model_path.exists() and not PAPER.FORCE_RERUN else None
    if booster is None:
        completed = 0
    else:
        paper_log('LightGBM boosting-round checkpoint resumed', stage='baseline_lightgbm', completed_rounds=completed)
    train_set = lgb.Dataset(np.ascontiguousarray(data['Xtr'], dtype=np.float32), label=data['yftr'],
                            weight=_class_sample_weights(data['yftr'], NUM_FINE_PAPER), free_raw_data=False)
    while completed < total_rounds:
        add = min(chunk, total_rounds - completed)
        booster = lgb.train(params, train_set, num_boost_round=add, init_model=booster,
                            keep_training_booster=True, callbacks=[lgb.log_evaluation(period=0)])
        completed += add
        tmp = model_path.with_name(model_path.name + '.tmp')
        booster.save_model(str(tmp))
        os.replace(tmp, model_path)
        atomic_json(progress_path, {'signature': sig, 'signature_payload': sig_payload,
                                    'completed_rounds': completed, 'status': 'DONE' if completed >= total_rounds else 'RUNNING'})
        paper_log('LightGBM boosting chunk checkpointed', stage='baseline_lightgbm', rounds=f'{completed}/{total_rounds}')
        check_deadline(f'LightGBM round {completed}')
    return booster, {'checkpoint': str(model_path), 'rounds': completed, 'signature': sig}


def _tree_throughput(model: Any, kind: str, X: np.ndarray, max_rows: int = 200_000) -> Dict[str, float]:
    n = min(int(max_rows), len(X))
    sample = np.ascontiguousarray(X[:n], dtype=np.float32)
    for _ in range(2):
        _ = _tree_predict_proba_chunked(model, sample[:min(n, 20_000)], kind, chunk_size=20_000)
    t0 = time.perf_counter()
    _ = _tree_predict_proba_chunked(model, sample, kind, chunk_size=100_000)
    elapsed = time.perf_counter() - t0
    return {'samples_per_second': n / max(elapsed, 1e-9), 'ms_per_sample': 1000.0 * elapsed / max(n, 1),
            'rows': n, 'kind': kind}


def run_tree_baseline(kind: str, seed: int = 42) -> Dict[str, Any]:
    kind = str(kind).lower()
    stage_name = f'baseline_{kind}'
    with paper_stage(stage_name, {'base': PAPER_BASE_SIGNATURE_HASH, 'profile': PAPER.PROFILE, 'seed': seed, 'version': 3}) as stage:
        if stage.skip:
            cached = read_json(PAPER_CACHE_DIR / 'baseline_evaluations' / f'{kind}.json', {})
            if cached:
                return cached
        if kind == 'xgboost':
            model, train_meta = _train_xgboost_resumable(seed)
        elif kind == 'lightgbm':
            model, train_meta = _train_lightgbm_resumable(seed)
        else:
            raise ValueError(kind)
        cal_idx, test_idx = _baseline_eval_indices(seed)
        cal_raw = _tree_predict_proba_chunked(model, X_cal[cal_idx], kind)
        test_raw = _tree_predict_proba_chunked(model, X_test[test_idx], kind)
        throughput = _tree_throughput(model, kind, X_test[test_idx])
        result = _evaluate_probability_baseline(
            'XGBoost' if kind == 'xgboost' else 'LightGBM', cal_raw, test_raw,
            y_cal_f[cal_idx], y_test_f[test_idx], y_test_c[test_idx], y_test_b[test_idx], throughput,
            source_kind='strict_measured' if PAPER.PROFILE == 'strict' else f'{PAPER.PROFILE}_profile_measured',
            note='Selected paper configuration executed. Exact historical hyperparameter-search grid is not supplied; selection search is not claimed.',
            cache_slug=kind,
        )
        result['training'] = train_meta
        atomic_json(PAPER_CACHE_DIR / 'baseline_evaluations' / f'{kind}.json', result)
        return result


def prepare_reference_smote_cache(seed: int = 42) -> Path:
    """Materialize the exact documented SMOTE training set with explicit opt-in.

    Full strict data can require many gigabytes of transient memory and disk. To
    prevent an accidental workstation crash, CAMELOT_ALLOW_FULL_SMOTE=1 is
    required. Accelerated/smoke profiles operate on their deterministic subset.
    """
    if importlib.util.find_spec('imblearn') is None:
        raise RuntimeError('imbalanced-learn is not installed')
    if PAPER.PROFILE == 'strict' and not _env_flag('CAMELOT_ALLOW_FULL_SMOTE', False):
        raise RuntimeError('Strict full-data SMOTE requires CAMELOT_ALLOW_FULL_SMOTE=1 after confirming available RAM/disk.')
    from imblearn.over_sampling import SMOTE
    data = _training_data_for_profile(seed)
    run_dir = PAPER_CACHE_DIR / 'smote' / PAPER.PROFILE
    run_dir.mkdir(parents=True, exist_ok=True)
    x_path, y_path, meta_path = run_dir / 'X_train_smote.npy', run_dir / 'y_train_smote.npy', run_dir / 'metadata.json'
    counts = np.bincount(data['yftr'], minlength=NUM_FINE_PAPER)
    target = max(1, int(math.ceil(0.10 * counts.max())))
    strategy = {int(cls): int(target) for cls, n in enumerate(counts) if 0 < int(n) < target}
    sig = signature_hash({'base': PAPER_BASE_SIGNATURE_HASH, 'profile': PAPER.PROFILE, 'seed': seed,
                          'k': 5, 'target': target, 'rows': len(data['Xtr']), 'strategy': strategy})
    prior = read_json(meta_path, {})
    if prior.get('signature') == sig and x_path.exists() and y_path.exists() and not PAPER.FORCE_RERUN:
        return run_dir
    paper_log('SMOTE materialization started; this operation is not interrupt-resumable inside nearest-neighbor fitting',
              level='WARNING', stage='baseline_dnn_smote', rows=len(data['Xtr']), target=target)
    sampler = SMOTE(sampling_strategy=strategy, k_neighbors=5, random_state=int(seed))
    Xr, yr = sampler.fit_resample(np.asarray(data['Xtr'], dtype=np.float32), np.asarray(data['yftr'], dtype=np.int64))
    atomic_npy(x_path, np.asarray(Xr, dtype=np.float32))
    atomic_npy(y_path, np.asarray(yr, dtype=np.int64))
    atomic_json(meta_path, {'signature': sig, 'rows_before': len(data['Xtr']), 'rows_after': len(yr),
                            'sampling_strategy': strategy, 'profile': PAPER.PROFILE})
    return run_dir


def _neural_baseline_spec(name: str) -> Tuple[Callable[[], nn.Module], str, Dict[str, Any]]:
    key = name.strip().lower()
    if key == 'cnn-gru':
        return (lambda: CNNGRUBaseline(len(FEATURE_COLS), NUM_FINE_PAPER)), 'class_balanced_ce', {'architecture': 'cnn_gru'}
    if key == 'ft-transformer':
        return (lambda: FTTransformerBaseline(len(FEATURE_COLS), NUM_FINE_PAPER)), 'class_balanced_ce', {'architecture': 'ft_transformer'}
    if key in {'tabtransformer', 'tabtransformer adaptation'}:
        return (lambda: TabTransformerNumericBaseline(len(FEATURE_COLS), NUM_FINE_PAPER)), 'class_balanced_ce', {'architecture': 'tabtransformer_numeric'}
    if key == 'saint':
        return (lambda: SAINTBaseline(len(FEATURE_COLS), NUM_FINE_PAPER)), 'class_balanced_ce', {'architecture': 'saint'}
    if key in {'graphsage', 'graphsage-gnn'}:
        return (lambda: GraphSAGEBatchKNNBaseline(len(FEATURE_COLS), NUM_FINE_PAPER)), 'class_balanced_ce', {'architecture': 'graphsage_batch_knn'}
    if key == 'dnn+smote':
        return (lambda: DNNBaseline(len(FEATURE_COLS), NUM_FINE_PAPER)), 'plain_ce', {'architecture': 'dnn_smote'}
    raise KeyError(name)


def run_neural_fine_baseline(name: str, seed: int = 42) -> Dict[str, Any]:
    factory, loss_mode, spec = _neural_baseline_spec(name)
    slug = re.sub(r'[^A-Za-z0-9_.-]+', '_', name.lower())
    kwargs: Dict[str, Any] = {}
    if name.strip().lower() == 'dnn+smote':
        cache_dir = Path(os.environ.get('CAMELOT_SMOTE_CACHE_DIR', '')) if os.environ.get('CAMELOT_SMOTE_CACHE_DIR') else None
        if cache_dir is None or not (cache_dir / 'X_train_smote.npy').exists() or not (cache_dir / 'y_train_smote.npy').exists():
            cache_dir = prepare_reference_smote_cache(seed)
        profile_data = _training_data_for_profile(seed)
        kwargs = {
            'Xtr': np.load(cache_dir / 'X_train_smote.npy', mmap_mode='r'),
            'ytr': np.load(cache_dir / 'y_train_smote.npy', mmap_mode='r'),
            'Xv': profile_data['Xv'], 'yv': profile_data['yfv'],
        }
    result = train_resumable_fine(f'baseline_{slug}_seed_{seed}', factory, seed=seed,
                                  loss_mode=loss_mode, model_spec=spec, **kwargs)
    if result.status != 'DONE' or not result.best_checkpoint.exists():
        raise TimeBudgetReached(f'{name} training is {result.status}; restart to continue from {result.last_checkpoint}')
    evaluated = evaluate_fine_baseline(name, factory, result.best_checkpoint, fit_calibration=True, benchmark=True)
    evaluated['source_kind'] = 'strict_measured' if PAPER.PROFILE == 'strict' else f'{PAPER.PROFILE}_profile_measured'
    evaluated['status'] = 'MEASURED'
    evaluated['training_profile_equivalent_to_manuscript'] = PAPER.PROFILE == 'strict'
    evaluated['best_epoch'] = result.best_epoch
    atomic_json(PAPER_CACHE_DIR / 'baseline_evaluations' / f'{slug}.json', evaluated)
    return evaluated


def run_flat_transformer_baseline(seed: int = 42) -> Dict[str, Any]:
    # Reuse the exact flat-46 ablation checkpoint when available; otherwise train
    # it here. This avoids duplicating the most expensive baseline.
    result = train_resumable_hierarchical(
        f'ablation_flat_46_seed_{seed}', lambda: FlatHierTransformer(), seed=seed,
        fine_loss_mode='ldam_drw', model_spec={'variant': 'flat_46', 'shared_with': 'Table 10'},
    )
    if result.status != 'DONE' or not result.best_checkpoint.exists():
        raise TimeBudgetReached(f'Flat Transformer training is {result.status}; restart to continue.')
    payload = evaluate_hierarchical_run('Flat Transformer', lambda: FlatHierTransformer(), result.best_checkpoint)
    flat_model_for_bench = load_hierarchical_checkpoint(lambda: FlatHierTransformer(), result.best_checkpoint)
    payload['throughput'] = benchmark_native_pytorch(flat_model_for_bench, X_test, DEVICE_T, batch_size=4096, max_rows=200_000)
    del flat_model_for_bench
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    payload['source_kind'] = 'strict_measured' if PAPER.PROFILE == 'strict' else f'{PAPER.PROFILE}_profile_measured'
    payload['status'] = 'MEASURED'
    payload['training_profile_equivalent_to_manuscript'] = PAPER.PROFILE == 'strict'
    atomic_json(PAPER_CACHE_DIR / 'baseline_evaluations' / 'flat_transformer.json', payload)
    return payload


def _archived_camelot_baseline() -> Dict[str, Any]:
    fine = multiclass_metrics(PAPER_Y_TEST_FINE, PAPER_TEST_PROBS_FINE)
    coarse = multiclass_metrics(PAPER_Y_TEST_COARSE, PAPER_TEST_PROBS_COARSE)
    binary = binary_metrics(PAPER_Y_TEST_BINARY, PAPER_TEST_PROBS_BINARY)
    throughput = {}
    table8_path = PAPER_TABLE_DIR / 'table_08_measured_throughput.csv'
    if table8_path.exists():
        try:
            t8 = pd.read_csv(table8_path)
            gpu = t8[t8['platform'].astype(str).str.contains('GPU', case=False, na=False)]
            if len(gpu):
                throughput = {'samples_per_second': float(gpu.iloc[0]['throughput_samples_s']),
                              'ms_per_sample': float(gpu.iloc[0]['ms_per_sample'])}
        except Exception:
            pass
    return {
        'run_name': 'CAMELOT-IDS', 'fine': fine, 'coarse': coarse, 'binary': binary,
        'calibration': calibration_metrics(PAPER_Y_TEST_FINE, PAPER_TEST_PROBS_FINE, PAPER.ECE_BINS),
        'throughput': throughput, 'profile': 'archived_v4_full_run', 'source_kind': 'archived_v4_measured',
        'status': 'MEASURED', 'evaluation_rows': len(PAPER_Y_TEST_FINE),
    }


def _baseline_table_row(name: str, payload: Mapping[str, Any]) -> Dict[str, Any]:
    fine, coarse, binary = payload.get('fine', {}), payload.get('coarse', {}), payload.get('binary', {})
    throughput = payload.get('throughput', {}) or {}
    return {
        'model': name,
        'fine_accuracy_pct': 100.0 * float(fine.get('accuracy', np.nan)),
        'fine_macro_f1_pct': 100.0 * float(fine.get('macro_f1', np.nan)),
        'coarse_macro_f1_pct': 100.0 * float(coarse.get('macro_f1', np.nan)),
        'binary_malicious_f1_pct': 100.0 * float(binary.get('malicious_f1', np.nan)),
        'throughput_samples_s': float(throughput.get('samples_per_second', np.nan)),
        'status': payload.get('status', 'UNKNOWN'),
        'source_kind': payload.get('source_kind', 'unknown'),
        'profile': payload.get('profile', PAPER.PROFILE),
        'evaluation_rows': payload.get('evaluation_rows', np.nan),
        'note': payload.get('note', ''),
    }


def run_baseline_suite(models: Optional[Sequence[str]] = None) -> pd.DataFrame:
    requested = list(models or [
        'XGBoost', 'LightGBM', 'DNN+SMOTE', 'CNN-GRU', 'Flat Transformer',
        'FT-Transformer', 'TabTransformer adaptation', 'SAINT', 'CAMELOT-IDS',
        'TabNet', 'GraphSAGE-GNN',
    ])
    rows: List[Dict[str, Any]] = []
    for name in requested:
        check_deadline(f'before baseline {name}')
        paper_log('baseline dispatch', stage='baselines', model=name)
        try:
            if name == 'CAMELOT-IDS':
                payload = _archived_camelot_baseline()
            elif name == 'XGBoost':
                payload = run_tree_baseline('xgboost')
            elif name == 'LightGBM':
                payload = run_tree_baseline('lightgbm')
            elif name == 'Flat Transformer':
                payload = run_flat_transformer_baseline()
            elif name == 'TabNet':
                payload = run_tabnet_baseline()
            else:
                payload = run_neural_fine_baseline(name)
            rows.append(_baseline_table_row(name, payload))
        except TimeBudgetReached:
            raise
        except Exception as exc:
            paper_log('baseline blocked/failed without substituting a reference value', level='WARNING',
                      stage='baselines', model=name, error=repr(exc))
            rows.append({
                'model': name, 'fine_accuracy_pct': np.nan, 'fine_macro_f1_pct': np.nan,
                'coarse_macro_f1_pct': np.nan, 'binary_malicious_f1_pct': np.nan,
                'throughput_samples_s': np.nan, 'status': 'BLOCKED', 'source_kind': 'not_executed',
                'profile': PAPER.PROFILE, 'evaluation_rows': np.nan, 'note': repr(exc),
            })
        partial = pd.DataFrame(rows)
        if len(partial):
            camelot_value = partial.loc[partial['model'] == 'CAMELOT-IDS', 'fine_macro_f1_pct']
            if len(camelot_value):
                partial['delta_fine_mf1_vs_camelot_pp'] = partial['fine_macro_f1_pct'] - float(camelot_value.iloc[0])
            atomic_dataframe(partial, PAPER_TABLE_DIR / 'table_s10_baseline_summary_partial')
    table = pd.DataFrame(rows)
    if len(table):
        camelot_value = table.loc[table['model'] == 'CAMELOT-IDS', 'fine_macro_f1_pct']
        table['delta_fine_mf1_vs_camelot_pp'] = table['fine_macro_f1_pct'] - (float(camelot_value.iloc[0]) if len(camelot_value) else np.nan)
    atomic_dataframe(table, PAPER_TABLE_DIR / 'table_s10_baseline_summary')
    s13 = table.loc[table['model'].isin(['XGBoost', 'LightGBM', 'Flat Transformer', 'FT-Transformer', 'SAINT', 'CAMELOT-IDS']),
                    ['model', 'fine_accuracy_pct', 'throughput_samples_s', 'status', 'source_kind', 'profile']].copy()
    atomic_dataframe(s13, PAPER_TABLE_DIR / 'table_s13_seed42_accuracy_throughput')
    update_coverage_status('Tables S10-S11', 'DONE' if (table['status'] == 'MEASURED').all() else 'PARTIAL',
                           str(PAPER_TABLE_DIR / 'table_s10_baseline_summary.csv'),
                           'No manuscript reference number is substituted for a blocked model.')
    update_coverage_status('Table S13', 'DONE' if len(s13) and (s13['status'] == 'MEASURED').all() else 'PARTIAL',
                           str(PAPER_TABLE_DIR / 'table_s13_seed42_accuracy_throughput.csv'))
    display(table)
    return table


# Optional TabNet path kept separate because its public API does not expose a
# serializable optimizer state. It is still restartable at completed epochs.
def run_tabnet_baseline(seed: int = 42) -> Dict[str, Any]:
    if importlib.util.find_spec('pytorch_tabnet') is None:
        raise RuntimeError('pytorch-tabnet is not installed')
    from pytorch_tabnet.tab_model import TabNetClassifier
    data = _training_data_for_profile(seed)
    run_dir = PAPER_MODEL_DIR / 'baseline_tabnet'
    run_dir.mkdir(parents=True, exist_ok=True)
    progress_path = run_dir / 'progress.json'
    model_prefix = run_dir / 'tabnet_epoch'
    progress = read_json(progress_path, {})
    completed = int(progress.get('completed_epochs', 0)) if progress.get('profile') == PAPER.PROFILE else 0
    clf = TabNetClassifier(n_d=64, n_a=64, n_steps=5, gamma=1.5,
                           optimizer_fn=torch.optim.AdamW,
                           optimizer_params={'lr': float(cfg.LR), 'weight_decay': float(cfg.WEIGHT_DECAY)},
                           seed=int(seed), verbose=1, device_name='cuda' if torch.cuda.is_available() else 'cpu')
    if completed > 0:
        zip_path = Path(str(model_prefix) + '.zip')
        if zip_path.exists():
            clf.load_model(str(zip_path))
            paper_log('TabNet epoch-boundary weights resumed', stage='baseline_tabnet', completed_epochs=completed)
        else:
            completed = 0
    counts = np.bincount(np.asarray(data['yftr'], dtype=np.int64), minlength=NUM_FINE_PAPER).astype(np.float64)
    class_weight_map = {int(i): float(len(data['yftr']) / (NUM_FINE_PAPER * n)) for i, n in enumerate(counts) if n > 0}
    while completed < int(PAPER.EPOCHS):
        clf.fit(
            X_train=np.asarray(data['Xtr'], dtype=np.float32), y_train=np.asarray(data['yftr'], dtype=np.int64),
            eval_set=[(np.asarray(data['Xv'], dtype=np.float32), np.asarray(data['yfv'], dtype=np.int64))],
            eval_name=['val'], eval_metric=['balanced_accuracy'], weights=class_weight_map,
            max_epochs=1, patience=0, batch_size=int(PAPER.BATCH_SIZE), virtual_batch_size=256,
            num_workers=0, drop_last=False, warm_start=(completed > 0),
        )
        completed += 1
        # save_model appends .zip; remove old file first to avoid package-specific suffix duplication.
        zip_path = Path(str(model_prefix) + '.zip')
        if zip_path.exists():
            zip_path.unlink()
        clf.save_model(str(model_prefix))
        atomic_json(progress_path, {'completed_epochs': completed, 'profile': PAPER.PROFILE,
                                    'status': 'DONE' if completed >= int(PAPER.EPOCHS) else 'RUNNING',
                                    'resume_boundary': 'epoch_weights_only_optimizer_reinitialized'})
        paper_log('TabNet epoch checkpointed', stage='baseline_tabnet', epoch=f'{completed}/{PAPER.EPOCHS}')
        check_deadline(f'TabNet epoch {completed}')
    cal_idx, test_idx = _baseline_eval_indices(seed)
    cal_raw = clf.predict_proba(np.asarray(X_cal[cal_idx], dtype=np.float32))
    test_raw = clf.predict_proba(np.asarray(X_test[test_idx], dtype=np.float32))
    n = min(200_000, len(test_idx))
    for _ in range(2):
        _ = clf.predict_proba(np.asarray(X_test[test_idx[:min(n, 20_000)]], dtype=np.float32))
    t0 = time.perf_counter()
    _ = clf.predict_proba(np.asarray(X_test[test_idx[:n]], dtype=np.float32))
    elapsed = time.perf_counter() - t0
    throughput = {'samples_per_second': n / max(elapsed, 1e-9), 'ms_per_sample': 1000 * elapsed / max(n, 1), 'rows': n}
    return _evaluate_probability_baseline(
        'TabNet', cal_raw, test_raw, y_cal_f[cal_idx], y_test_f[test_idx], y_test_c[test_idx], y_test_b[test_idx],
        throughput, source_kind='strict_measured' if PAPER.PROFILE == 'strict' else f'{PAPER.PROFILE}_profile_measured',
        note='Epoch-boundary weights are resumable; pytorch-tabnet public API does not expose optimizer-state restoration.',
        cache_slug='tabnet',
    )


paper_log('baseline suite definitions ready; long execution is delegated to the orchestrator', stage='baselines')
paper_cell_done('P16', 'paper baseline suite: resumable trees, neural models, Table S10/S11/S13', _started)


[P16] START: paper baseline suite: resumable trees, neural models, Table S10/S11/S13
[2026-08-28 09:55:49] [INFO] [baselines] baseline suite definitions ready; long execution is delegated to the orchestrator
[P16] DONE: paper baseline suite: resumable trees, neural models, Table S10/S11/S13 | elapsed=0.05s


In [19]:
_started = paper_cell_start('P17', 'five-seed matched evaluation and paired confidence intervals (Tables S12/S12b)')


def _aggregate_fine_probabilities(probs_fine: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    pf = np.asarray(probs_fine, dtype=np.float32)
    mapping = np.asarray(fine_to_coarse, dtype=np.int64)
    coarse = np.zeros((len(pf), NUM_COARSE_PAPER), dtype=np.float32)
    for fine_id in range(NUM_FINE_PAPER):
        coarse[:, mapping[fine_id]] += pf[:, fine_id]
    binary = np.empty((len(pf), 2), dtype=np.float32)
    binary[:, 0] = pf[:, int(benign_fine_id)]
    binary[:, 1] = 1.0 - binary[:, 0]
    return coarse, binary


def _resumable_predict_fine_memmap(
    model: nn.Module,
    X: np.ndarray,
    output_path: Path,
    signature_payload: Mapping[str, Any],
    *,
    batch_size: Optional[int] = None,
) -> np.ndarray:
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    progress_path = output_path.with_suffix(output_path.suffix + '.progress.json')
    sig = signature_hash(dict(signature_payload))
    prior = read_json(progress_path, {})
    if output_path.exists() and prior.get('signature') == sig and prior.get('status') == 'DONE':
        return np.load(output_path, mmap_mode='r')
    start_row = int(prior.get('completed_rows', 0)) if prior.get('signature') == sig and output_path.exists() else 0
    mode = 'r+' if start_row > 0 else 'w+'
    mmap = np.lib.format.open_memmap(output_path, mode=mode, dtype=np.float32,
                                     shape=(len(X), NUM_FINE_PAPER))
    model = model.to(DEVICE_T).eval()
    bs = int(PAPER.INFERENCE_BATCH_SIZE if batch_size is None else batch_size)
    started = time.perf_counter()
    for start in range(start_row, len(X), bs):
        stop = min(len(X), start + bs)
        xb = torch.from_numpy(np.array(X[start:stop], dtype=np.float32, copy=True, order='C')).to(DEVICE_T, non_blocking=True)
        with torch.inference_mode(), amp_autocast(PAPER.USE_AMP and DEVICE_T.type == 'cuda'):
            logits = model(xb)
        mmap[start:stop] = logits.float().cpu().numpy()
        mmap.flush()
        atomic_json(progress_path, {'signature': sig, 'completed_rows': stop,
                                    'status': 'DONE' if stop >= len(X) else 'RUNNING',
                                    'shape': [len(X), NUM_FINE_PAPER]})
        if start == start_row or stop == len(X) or (stop // bs) % 100 == 0:
            paper_log('resumable fine inference', stage='multi_seed_inference', rows=f'{stop}/{len(X)}',
                      samples_s=round((stop - start_row) / max(time.perf_counter() - started, 1e-9), 1))
        check_deadline(f'fine inference row {stop}')
    del mmap
    return np.load(output_path, mmap_mode='r')


def _resumable_predict_hier_memmap(
    model: nn.Module,
    X: np.ndarray,
    output_dir: Path,
    signature_payload: Mapping[str, Any],
    *,
    batch_size: Optional[int] = None,
) -> Dict[str, np.ndarray]:
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    paths = {k: output_dir / f'{k}.npy' for k in ('fine', 'coarse', 'binary')}
    progress_path = output_dir / 'progress.json'
    sig = signature_hash(dict(signature_payload))
    prior = read_json(progress_path, {})
    if prior.get('signature') == sig and prior.get('status') == 'DONE' and all(p.exists() for p in paths.values()):
        return {k: np.load(p, mmap_mode='r') for k, p in paths.items()}
    start_row = int(prior.get('completed_rows', 0)) if prior.get('signature') == sig and all(p.exists() for p in paths.values()) else 0
    mode = 'r+' if start_row > 0 else 'w+'
    maps = {
        'fine': np.lib.format.open_memmap(paths['fine'], mode=mode, dtype=np.float32, shape=(len(X), NUM_FINE_PAPER)),
        'coarse': np.lib.format.open_memmap(paths['coarse'], mode=mode, dtype=np.float32, shape=(len(X), NUM_COARSE_PAPER)),
        'binary': np.lib.format.open_memmap(paths['binary'], mode=mode, dtype=np.float32, shape=(len(X), 2)),
    }
    model = model.to(DEVICE_T).eval()
    bs = int(PAPER.INFERENCE_BATCH_SIZE if batch_size is None else batch_size)
    started = time.perf_counter()
    for start in range(start_row, len(X), bs):
        stop = min(len(X), start + bs)
        xb = torch.from_numpy(np.array(X[start:stop], dtype=np.float32, copy=True, order='C')).to(DEVICE_T, non_blocking=True)
        with torch.inference_mode(), amp_autocast(PAPER.USE_AMP and DEVICE_T.type == 'cuda'):
            out = model(xb)
        maps['fine'][start:stop] = out['logits_fine'].float().cpu().numpy()
        maps['coarse'][start:stop] = out['logits_coarse'].float().cpu().numpy()
        maps['binary'][start:stop] = out['logits_bin'].float().cpu().numpy()
        for mmap in maps.values():
            mmap.flush()
        atomic_json(progress_path, {'signature': sig, 'completed_rows': stop,
                                    'status': 'DONE' if stop >= len(X) else 'RUNNING'})
        if start == start_row or stop == len(X) or (stop // bs) % 100 == 0:
            paper_log('resumable hierarchical inference', stage='multi_seed_inference', rows=f'{stop}/{len(X)}',
                      samples_s=round((stop - start_row) / max(time.perf_counter() - started, 1e-9), 1))
        check_deadline(f'hierarchical inference row {stop}')
    del maps
    return {k: np.load(p, mmap_mode='r') for k, p in paths.items()}


def _evaluate_multiseed_probabilities(
    model_name: str,
    seed: int,
    cal_logits_f: np.ndarray,
    test_logits_f: np.ndarray,
    *,
    cal_logits_c: Optional[np.ndarray] = None,
    test_logits_c: Optional[np.ndarray] = None,
) -> Dict[str, Any]:
    temp_f = float(fit_temperature_np(np.asarray(cal_logits_f), y_cal_f))
    cal_pf = softmax_np(np.asarray(cal_logits_f) / max(temp_f, 1e-6))
    test_pf = softmax_np(np.asarray(test_logits_f) / max(temp_f, 1e-6))
    if cal_logits_c is not None and test_logits_c is not None:
        temp_c = float(fit_temperature_np(np.asarray(cal_logits_c), y_cal_c))
        cal_pc = softmax_np(np.asarray(cal_logits_c) / max(temp_c, 1e-6))
        test_pc = softmax_np(np.asarray(test_logits_c) / max(temp_c, 1e-6))
    else:
        temp_c = 1.0
        cal_pc, _ = _aggregate_fine_probabilities(cal_pf)
        test_pc, _ = _aggregate_fine_probabilities(test_pf)
    fine = multiclass_metrics(y_test_f, test_pf)
    coarse = multiclass_metrics(y_test_c, test_pc)
    thresholds = build_mondrian_raps_thresholds(
        cal_pf, cal_pc, y_cal_f, [0.10], PAPER.RAPS_KREG, PAPER.RAPS_LAMBDA,
        min_group=PAPER.RAPS_MIN_GROUP,
    )
    raps = raps_evaluate_detailed_chunked(
        test_pf, test_pc, y_test_f, thresholds, 0.10, int(benign_fine_id)
    )
    return {
        'model': model_name, 'seed': int(seed),
        'fine_macro_f1': float(fine['macro_f1']),
        'fine_accuracy': float(fine['accuracy']),
        'coarse_macro_f1': float(coarse['macro_f1']),
        'ece': expected_calibration_error_local(y_test_f, test_pf, PAPER.ECE_BINS),
        'singleton_risk': float(raps['singleton_risk']),
        'singleton_fraction': float(raps['singleton_fraction']),
        'raps_coverage': float(raps['coverage']),
        'raps_mean_set_size': float(raps['avg_set_size']),
        'temperature_fine': temp_f, 'temperature_coarse': temp_c,
        'profile': 'archived_v4_full_run' if model_name == 'CAMELOT-IDS' and int(seed) == 42 else PAPER.PROFILE,
        'strict_manuscript_equivalent_training': bool(PAPER.PROFILE == 'strict' or (model_name == 'CAMELOT-IDS' and int(seed) == 42)),
        'source_kind': 'archived_v4_measured' if model_name == 'CAMELOT-IDS' and int(seed) == 42 else ('strict_measured' if PAPER.PROFILE == 'strict' else f'{PAPER.PROFILE}_profile_measured'),
    }


def run_multiseed_model_seed(model_name: str, seed: int) -> Dict[str, Any]:
    slug = re.sub(r'[^A-Za-z0-9_.-]+', '_', model_name.lower())
    result_path = PAPER_CACHE_DIR / 'multi_seed' / slug / f'seed_{int(seed)}.json'
    result_path.parent.mkdir(parents=True, exist_ok=True)
    cached = read_json(result_path, {})
    if cached.get('status') == 'DONE' and not PAPER.FORCE_RERUN:
        return cached['result']

    if model_name == 'CAMELOT-IDS' and int(seed) == 42:
        result = _evaluate_multiseed_probabilities(
            model_name, seed, PAPER_CAL_OUT['fine'], PAPER_TEST_OUT['fine'],
            cal_logits_c=PAPER_CAL_OUT['coarse'], test_logits_c=PAPER_TEST_OUT['coarse'],
        )
        atomic_json(result_path, {'status': 'DONE', 'result': result})
        return result

    if model_name == 'CAMELOT-IDS':
        factory = lambda: build_fresh_camelot_model()
        trained = train_resumable_hierarchical(
            f'multiseed_camelot_seed_{seed}', factory, seed=seed, fine_loss_mode='ldam_drw',
            model_spec={'model': 'CAMELOT-IDS', 'multi_seed': True},
        )
        if trained.status != 'DONE':
            raise TimeBudgetReached(f'CAMELOT seed {seed} paused; restart to continue.')
        model = load_hierarchical_checkpoint(factory, trained.best_checkpoint)
        base = PAPER_CACHE_DIR / 'multi_seed' / slug / f'seed_{seed}'
        cal = _resumable_predict_hier_memmap(model, X_cal, base / 'cal',
                                             {'checkpoint': str(trained.best_checkpoint), 'mtime': trained.best_checkpoint.stat().st_mtime, 'split': 'cal'})
        test = _resumable_predict_hier_memmap(model, X_test, base / 'test',
                                              {'checkpoint': str(trained.best_checkpoint), 'mtime': trained.best_checkpoint.stat().st_mtime, 'split': 'test'})
        result = _evaluate_multiseed_probabilities(model_name, seed, cal['fine'], test['fine'],
                                                   cal_logits_c=cal['coarse'], test_logits_c=test['coarse'])
    elif model_name == 'Flat Transformer':
        factory = lambda: FlatHierTransformer()
        trained = train_resumable_hierarchical(
            f'ablation_flat_46_seed_{seed}', factory, seed=seed, fine_loss_mode='ldam_drw',
            model_spec={'variant': 'flat_46', 'multi_seed': True},
        )
        if trained.status != 'DONE':
            raise TimeBudgetReached(f'Flat Transformer seed {seed} paused; restart to continue.')
        model = load_hierarchical_checkpoint(factory, trained.best_checkpoint)
        base = PAPER_CACHE_DIR / 'multi_seed' / slug / f'seed_{seed}'
        cal = _resumable_predict_hier_memmap(model, X_cal, base / 'cal',
                                             {'checkpoint': str(trained.best_checkpoint), 'mtime': trained.best_checkpoint.stat().st_mtime, 'split': 'cal'})
        test = _resumable_predict_hier_memmap(model, X_test, base / 'test',
                                              {'checkpoint': str(trained.best_checkpoint), 'mtime': trained.best_checkpoint.stat().st_mtime, 'split': 'test'})
        result = _evaluate_multiseed_probabilities(model_name, seed, cal['fine'], test['fine'],
                                                   cal_logits_c=cal['coarse'], test_logits_c=test['coarse'])
    elif model_name in {'FT-Transformer', 'SAINT'}:
        if model_name == 'FT-Transformer':
            factory = lambda: FTTransformerBaseline(len(FEATURE_COLS), NUM_FINE_PAPER)
            slug_train = 'ft-transformer'
            spec = {'architecture': 'ft_transformer', 'multi_seed': True}
        else:
            factory = lambda: SAINTBaseline(len(FEATURE_COLS), NUM_FINE_PAPER)
            slug_train = 'saint'
            spec = {'architecture': 'saint', 'multi_seed': True}
        trained = train_resumable_fine(
            f'baseline_{slug_train}_seed_{seed}', factory, seed=seed,
            loss_mode='class_balanced_ce', model_spec=spec,
        )
        if trained.status != 'DONE':
            raise TimeBudgetReached(f'{model_name} seed {seed} paused; restart to continue.')
        model = load_fine_checkpoint(factory, trained.best_checkpoint)
        base = PAPER_CACHE_DIR / 'multi_seed' / slug / f'seed_{seed}'
        cal_logits = _resumable_predict_fine_memmap(
            model, X_cal, base / 'cal_fine_logits.npy',
            {'checkpoint': str(trained.best_checkpoint), 'mtime': trained.best_checkpoint.stat().st_mtime, 'split': 'cal'},
        )
        test_logits = _resumable_predict_fine_memmap(
            model, X_test, base / 'test_fine_logits.npy',
            {'checkpoint': str(trained.best_checkpoint), 'mtime': trained.best_checkpoint.stat().st_mtime, 'split': 'test'},
        )
        result = _evaluate_multiseed_probabilities(model_name, seed, cal_logits, test_logits)
    else:
        raise KeyError(model_name)

    atomic_json(result_path, {'status': 'DONE', 'result': result})
    try:
        del model
    except Exception:
        pass
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return result


def _mean_sd_text(values: Sequence[float], scale: float = 100.0, decimals: int = 2) -> str:
    arr = np.asarray(values, dtype=np.float64) * float(scale)
    return f'{arr.mean():.{decimals}f} ± {arr.std(ddof=1):.{decimals}f}'


def _paired_ci_direct(a: np.ndarray, b: np.ndarray, scale: float = 100.0) -> Tuple[float, float, float]:
    diff = (np.asarray(a, dtype=np.float64) - np.asarray(b, dtype=np.float64)) * float(scale)
    n = len(diff)
    mean = float(diff.mean())
    if n < 2:
        return mean, float('nan'), float('nan')
    half = float(student_t.ppf(0.975, df=n - 1) * diff.std(ddof=1) / math.sqrt(n))
    return mean, mean - half, mean + half


def _paired_ci_assumed_rho(a: np.ndarray, b: np.ndarray, rho: float, scale: float = 100.0) -> Tuple[float, float, float]:
    av = np.asarray(a, dtype=np.float64) * float(scale)
    bv = np.asarray(b, dtype=np.float64) * float(scale)
    n = min(len(av), len(bv))
    mean = float(av.mean() - bv.mean())
    if n < 2:
        return mean, float('nan'), float('nan')
    sd_diff = math.sqrt(max(0.0, av.std(ddof=1) ** 2 + bv.std(ddof=1) ** 2 - 2.0 * float(rho) * av.std(ddof=1) * bv.std(ddof=1)))
    half = float(student_t.ppf(0.975, df=n - 1) * sd_diff / math.sqrt(n))
    return mean, mean - half, mean + half


def run_multiseed_suite() -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    models = ['CAMELOT-IDS', 'Flat Transformer', 'FT-Transformer', 'SAINT']
    records: List[Dict[str, Any]] = []
    for model_name in models:
        for seed in PAPER.SEEDS:
            check_deadline(f'before {model_name} seed {seed}')
            paper_log('multi-seed dispatch', stage='multi_seed', model=model_name, seed=seed)
            try:
                row = run_multiseed_model_seed(model_name, int(seed))
                records.append(row)
            except TimeBudgetReached:
                atomic_dataframe(pd.DataFrame(records), PAPER_TABLE_DIR / 'table_s12_per_seed_partial')
                raise
            except Exception as exc:
                paper_log('multi-seed run failed/blocked; no reference value substituted', level='ERROR',
                          stage='multi_seed', model=model_name, seed=seed, error=repr(exc))
                records.append({'model': model_name, 'seed': int(seed), 'status': 'FAILED', 'error': repr(exc),
                                'profile': PAPER.PROFILE, 'source_kind': 'not_executed'})
            atomic_dataframe(pd.DataFrame(records), PAPER_TABLE_DIR / 'table_s12_per_seed_partial')

    per_seed = pd.DataFrame(records)
    atomic_dataframe(per_seed, PAPER_TABLE_DIR / 'table_s12_per_seed')
    complete = per_seed.dropna(subset=['fine_macro_f1', 'coarse_macro_f1', 'ece', 'singleton_risk']) if len(per_seed) else per_seed
    summary_rows = []
    for model_name in models:
        sub = complete[complete['model'] == model_name].sort_values('seed')
        if len(sub) == len(PAPER.SEEDS):
            summary_rows.append({
                'model': model_name,
                'fine_macro_f1_pct_mean_sd': _mean_sd_text(sub['fine_macro_f1']),
                'coarse_macro_f1_pct_mean_sd': _mean_sd_text(sub['coarse_macro_f1']),
                'ece_mean_sd': _mean_sd_text(sub['ece'], scale=1.0, decimals=3),
                'singleton_risk_pct_mean_sd': _mean_sd_text(sub['singleton_risk'], decimals=3),
                'n_seeds': len(sub),
                'profile': PAPER.PROFILE,
                'strict_all_seeds': bool(sub['strict_manuscript_equivalent_training'].all()),
                'status': 'DONE',
            })
        else:
            summary_rows.append({'model': model_name, 'n_seeds': len(sub), 'profile': PAPER.PROFILE,
                                 'strict_all_seeds': False, 'status': 'PARTIAL'})
    summary = pd.DataFrame(summary_rows)
    atomic_dataframe(summary, PAPER_TABLE_DIR / 'table_s12_five_seed_summary')

    ci_rows = []
    camelot = complete[complete['model'] == 'CAMELOT-IDS'].set_index('seed')
    for baseline in ['Flat Transformer', 'FT-Transformer', 'SAINT']:
        base = complete[complete['model'] == baseline].set_index('seed')
        common = sorted(set(camelot.index).intersection(base.index))
        if len(common) < 2:
            ci_rows.append({'comparison': f'CAMELOT-IDS vs. {baseline}', 'n_pairs': len(common), 'status': 'PARTIAL'})
            continue
        cf = camelot.loc[common, 'fine_macro_f1'].to_numpy()
        bf = base.loc[common, 'fine_macro_f1'].to_numpy()
        cc = camelot.loc[common, 'coarse_macro_f1'].to_numpy()
        bc = base.loc[common, 'coarse_macro_f1'].to_numpy()
        fm, fl, fh = _paired_ci_direct(cf, bf)
        cm, cl, ch = _paired_ci_direct(cc, bc)
        fm_r, fl_r, fh_r = _paired_ci_assumed_rho(cf, bf, PAPER.PAIRED_SEED_CORRELATION)
        cm_r, cl_r, ch_r = _paired_ci_assumed_rho(cc, bc, PAPER.PAIRED_SEED_CORRELATION)
        ci_rows.append({
            'comparison': f'CAMELOT-IDS vs. {baseline}', 'n_pairs': len(common),
            'mean_delta_fine_mf1_pp': fm, 'direct_paired_95ci_fine_low': fl, 'direct_paired_95ci_fine_high': fh,
            'mean_delta_coarse_mf1_pp': cm, 'direct_paired_95ci_coarse_low': cl, 'direct_paired_95ci_coarse_high': ch,
            'paper_rho_0.6_mean_delta_fine_pp': fm_r, 'paper_rho_0.6_95ci_fine_low': fl_r, 'paper_rho_0.6_95ci_fine_high': fh_r,
            'paper_rho_0.6_mean_delta_coarse_pp': cm_r, 'paper_rho_0.6_95ci_coarse_low': cl_r, 'paper_rho_0.6_95ci_coarse_high': ch_r,
            'status': 'DONE', 'profile': PAPER.PROFILE,
            'note': 'Direct paired CI uses the observed matched seed differences. rho=0.6 columns reproduce the manuscript-specified summary formula separately.',
        })
    ci_table = pd.DataFrame(ci_rows)
    atomic_dataframe(ci_table, PAPER_TABLE_DIR / 'table_s12b_paired_confidence_intervals')
    status = 'DONE' if len(summary) and (summary['status'] == 'DONE').all() else 'PARTIAL'
    update_coverage_status('Tables S12-S12b', status,
                           str(PAPER_TABLE_DIR / 'table_s12_five_seed_summary.csv'),
                           'Accelerated/smoke runs are not substituted for strict manuscript values.')
    display(summary)
    display(ci_table)
    return per_seed, summary, ci_table


paper_log('multi-seed suite definitions ready; each model/seed has independent checkpoints and logits progress', stage='multi_seed')
paper_cell_done('P17', 'five-seed matched evaluation and paired confidence intervals (Tables S12/S12b)', _started)


[P17] START: five-seed matched evaluation and paired confidence intervals (Tables S12/S12b)
[2026-08-28 09:55:49] [INFO] [multi_seed] multi-seed suite definitions ready; each model/seed has independent checkpoints and logits progress
[P17] DONE: five-seed matched evaluation and paired confidence intervals (Tables S12/S12b) | elapsed=0.01s


In [20]:
_started = paper_cell_start('P18', 'external-dataset preparation, mappings, retraining, and Tables S8-S9')

from contextlib import contextmanager as _contextmanager


EXTERNAL_SPECS: Dict[str, Dict[str, Any]] = {
    'cicids2017': {
        'display_name': 'CIC-IDS2017', 'root': PAPER.CIC_IDS2017_ROOT,
        'label_candidates': ['Label', 'label', 'attack_type', 'category'],
        'coarse_protocol': ['Benign', 'DDoS', 'DoS/Heartbleed', 'BruteForce', 'Web', 'Recon', 'Botnet', 'Infiltration', 'Other'],
    },
    'cicids2018': {
        'display_name': 'CSE-CIC-IDS2018', 'root': PAPER.CSE_CIC_IDS2018_ROOT,
        'label_candidates': ['Label', 'label', 'attack_type', 'category'],
        'coarse_protocol': ['Benign', 'DDoS', 'DoS/Heartbleed', 'BruteForce', 'Web', 'Recon', 'Botnet', 'Infiltration', 'Other'],
    },
    'toniot': {
        'display_name': 'TON-IoT', 'root': PAPER.TON_IOT_ROOT,
        'label_candidates': ['type', 'attack_type', 'category', 'label'],
        'coarse_protocol': ['Benign', 'DDoS', 'DoS', 'BruteForce', 'Web', 'Recon', 'Botnet', 'Spoofing', 'Malware', 'Other'],
    },
    'unsw_nb15': {
        'display_name': 'UNSW-NB15', 'root': PAPER.UNSW_NB15_ROOT,
        'label_candidates': ['attack_cat', 'attack_category', 'category', 'label'],
        'coarse_protocol': ['Benign', 'DDoS', 'DoS', 'BruteForce', 'Web', 'Recon', 'Botnet', 'Exploit', 'Generic', 'Fuzzers', 'Analysis', 'Spoofing', 'Other'],
    },
}

_IDENTIFIER_TERMS = {
    'timestamp', 'time_stamp', 'flow_id', 'flowid', 'src_ip', 'dst_ip', 'source_ip', 'destination_ip',
    'srcip', 'dstip', 'ip_src', 'ip_dst', 'src_mac', 'dst_mac', 'mac', 'device', 'device_id',
    'src_port', 'dst_port', 'source_port', 'destination_port', 'sport', 'dport', 'port',
    'id', 'record_id', 'index', 'unnamed',
}


def _norm_col(name: Any) -> str:
    return re.sub(r'[^a-z0-9]+', '_', str(name).strip().lower()).strip('_')


def _is_identifier_column(name: Any) -> bool:
    norm = _norm_col(name)
    if norm in _IDENTIFIER_TERMS:
        return True
    terms = set(norm.split('_'))
    if {'ip', 'mac', 'timestamp', 'address', 'addr', 'port', 'device'}.intersection(terms):
        return True
    return norm.startswith('unnamed') or norm.endswith('_id') or norm == 'id'


def _read_header(path: Path) -> List[str]:
    if path.suffix.lower() == '.csv':
        return list(pd.read_csv(path, nrows=0).columns)
    if path.suffix.lower() in {'.parquet', '.pq'}:
        try:
            import pyarrow.parquet as pq
            return list(pq.ParquetFile(path).schema.names)
        except Exception:
            return list(pd.read_parquet(path).columns)
    raise ValueError(f'Unsupported external file type: {path}')


def _read_external_chunks(path: Path, usecols: Optional[Sequence[str]] = None,
                          chunksize: int = 250_000) -> Iterator[pd.DataFrame]:
    if path.suffix.lower() == '.csv':
        yield from pd.read_csv(path, usecols=list(usecols) if usecols is not None else None,
                               chunksize=int(chunksize), low_memory=False)
    elif path.suffix.lower() in {'.parquet', '.pq'}:
        # Parquet files are normally columnar and already partitioned. pyarrow
        # batches avoid loading a very large file in one DataFrame.
        try:
            import pyarrow.parquet as pq
            pf = pq.ParquetFile(path)
            for batch in pf.iter_batches(batch_size=int(chunksize), columns=list(usecols) if usecols is not None else None):
                yield batch.to_pandas()
        except Exception:
            yield pd.read_parquet(path, columns=list(usecols) if usecols is not None else None)
    else:
        raise ValueError(f'Unsupported external file type: {path}')


def _discover_external_files(root: str) -> List[Path]:
    if not root:
        return []
    p = Path(root).expanduser().resolve()
    if not p.exists():
        return []
    if p.is_file():
        return [p] if p.suffix.lower() in {'.csv', '.parquet', '.pq'} else []
    files = sorted([x for x in p.rglob('*') if x.is_file() and x.suffix.lower() in {'.csv', '.parquet', '.pq'}])
    return files


def _choose_label_column(files: Sequence[Path], candidates: Sequence[str]) -> str:
    header_maps = [{_norm_col(c): c for c in _read_header(path)} for path in files]
    for candidate in candidates:
        norm = _norm_col(candidate)
        if all(norm in mapping for mapping in header_maps):
            # Return spelling from first file; later files are resolved by normalized name.
            return header_maps[0][norm]
    common_norm = set(header_maps[0])
    for mapping in header_maps[1:]:
        common_norm.intersection_update(mapping)
    likely = [x for x in common_norm if any(token in x for token in ('label', 'attack', 'category', 'type'))]
    if len(likely) == 1:
        return header_maps[0][likely[0]]
    raise RuntimeError(f'Could not identify one common label column; candidates={candidates}, likely={sorted(likely)}')


def _resolve_actual_column(path: Path, canonical: str) -> str:
    mapping = {_norm_col(c): c for c in _read_header(path)}
    norm = _norm_col(canonical)
    if norm not in mapping:
        raise KeyError(f'{canonical!r} not present in {path}')
    return mapping[norm]


def _select_external_numeric_features(files: Sequence[Path], label_column: str) -> List[str]:
    header_maps = [{_norm_col(c): c for c in _read_header(path)} for path in files]
    common = set(header_maps[0])
    for mapping in header_maps[1:]:
        common.intersection_update(mapping)
    label_norm = _norm_col(label_column)
    candidates = [name for name in sorted(common) if name != label_norm and not _is_identifier_column(name)]
    if not candidates:
        raise RuntimeError('No common non-identifier feature columns remain.')
    numeric_votes = {name: [] for name in candidates}
    for path in files[:min(8, len(files))]:
        actual = {_norm_col(c): c for c in _read_header(path)}
        use = [actual[name] for name in candidates if name in actual]
        sample = next(_read_external_chunks(path, usecols=use, chunksize=5_000), pd.DataFrame())
        for name in candidates:
            if name not in actual or actual[name] not in sample:
                continue
            series = sample[actual[name]]
            if pd.api.types.is_numeric_dtype(series):
                numeric_votes[name].append(1.0)
            else:
                converted = pd.to_numeric(series, errors='coerce')
                numeric_votes[name].append(float(converted.notna().mean()))
    selected_norm = [name for name in candidates if numeric_votes[name] and min(numeric_votes[name]) >= 0.90]
    selected = [header_maps[0][name] for name in selected_norm]
    if len(selected) < 5:
        raise RuntimeError(f'Only {len(selected)} robust numeric features identified; inspect raw columns and mapping.')
    return selected


def _semantic_groups_for_columns(columns: Sequence[str]) -> Tuple[List[str], List[List[int]]]:
    groups: Dict[str, List[int]] = {'rates': [], 'flags_counts': [], 'protocols': [], 'statistics': [], 'other': []}
    for idx, col in enumerate(columns):
        norm = _norm_col(col)
        if any(key in norm for key in ('rate', 'bytes_s', 'byts_s', 'pkts_s', 'packets_s', 'load')):
            groups['rates'].append(idx)
        elif any(key in norm for key in ('flag', 'count', 'cnt', 'syn', 'ack', 'fin', 'rst', 'psh', 'urg', 'packet', 'pkt', 'tot_')):
            groups['flags_counts'].append(idx)
        elif any(key in norm for key in ('protocol', 'proto', 'service', 'state', 'ttl', 'tcp', 'udp', 'icmp')):
            groups['protocols'].append(idx)
        elif any(key in norm for key in ('mean', 'std', 'min', 'max', 'var', 'median', 'size', 'len', 'duration', 'iat', 'jitter')):
            groups['statistics'].append(idx)
        else:
            groups['other'].append(idx)
    # The architecture requires five local sequences. Move one feature from the
    # largest group into any empty group; the deterministic repair is recorded.
    for target in list(groups):
        if groups[target]:
            continue
        donor = max(groups, key=lambda key: len(groups[key]))
        if not groups[donor]:
            raise RuntimeError('Cannot construct non-empty semantic groups.')
        groups[target].append(groups[donor].pop())
    names = ['rates', 'flags_counts', 'protocols', 'statistics', 'other']
    return names, [sorted(groups[name]) for name in names]


def _canonical_external_label(value: Any) -> str:
    raw = str(value).strip()
    if raw.lower() in {'nan', 'none', ''}:
        return 'Unknown'
    return re.sub(r'\s+', ' ', raw)


def _map_external_coarse(label: str, dataset_key: str) -> str:
    s = _norm_col(label)
    if any(k in s for k in ('benign', 'normal')) or s in {'0', 'false'}:
        return 'Benign'
    if 'ddos' in s or 'distributed_denial' in s:
        return 'DDoS'
    if any(k in s for k in ('brute', 'patator', 'password', 'credential')):
        return 'BruteForce'
    if any(k in s for k in ('heartbleed',)):
        return 'DoS/Heartbleed' if dataset_key.startswith('cicids') else 'DoS'
    if re.search(r'(^|_)dos($|_)', s) or 'denial_of_service' in s:
        return 'DoS/Heartbleed' if dataset_key.startswith('cicids') else 'DoS'
    if any(k in s for k in ('xss', 'sql', 'injection', 'web_attack', 'http')):
        return 'Web'
    if any(k in s for k in ('scan', 'recon', 'portscan', 'enumeration')):
        return 'Recon'
    if any(k in s for k in ('botnet', 'bot')):
        return 'Botnet'
    if 'infiltration' in s:
        return 'Infiltration'
    if any(k in s for k in ('spoof', 'mitm')):
        return 'Spoofing'
    if any(k in s for k in ('malware', 'ransom', 'backdoor', 'trojan', 'worm')):
        return 'Malware'
    if 'exploit' in s:
        return 'Exploit'
    if 'generic' in s:
        return 'Generic'
    if 'fuzz' in s:
        return 'Fuzzers'
    if 'analysis' in s:
        return 'Analysis'
    return 'Other'


def _file_level_partition(files: Sequence[Path], seed: int = 42) -> Dict[str, List[Path]]:
    if len(files) < 4:
        raise RuntimeError('At least four physical files are required for non-empty train/validation/calibration/test file holdouts.')
    rng = np.random.default_rng(int(seed))
    shuffled = [files[i] for i in rng.permutation(len(files))]
    n = len(shuffled)
    raw = np.asarray([0.80, 0.10, 0.05, 0.05]) * n
    counts = np.floor(raw).astype(int)
    counts = np.maximum(counts, 1)
    while counts.sum() > n:
        choices = np.flatnonzero(counts > 1)
        counts[choices[np.argmax(counts[choices] - raw[choices])]] -= 1
    while counts.sum() < n:
        counts[int(np.argmax(raw - counts))] += 1
    boundaries = np.cumsum(counts)
    return {
        'train': shuffled[:boundaries[0]],
        'val': shuffled[boundaries[0]:boundaries[1]],
        'cal': shuffled[boundaries[1]:boundaries[2]],
        'test': shuffled[boundaries[2]:boundaries[3]],
    }


def _scan_labels_and_rows(files: Sequence[Path], label_column: str) -> Tuple[Dict[str, int], List[str]]:
    row_counts: Dict[str, int] = {}
    labels: set[str] = set()
    for path in files:
        actual_label = _resolve_actual_column(path, label_column)
        count = 0
        for chunk in _read_external_chunks(path, usecols=[actual_label]):
            count += len(chunk)
            labels.update(_canonical_external_label(x) for x in chunk[actual_label].tolist())
        row_counts[str(path)] = int(count)
        paper_log('external file scanned', stage='external_prepare', file=path.name, rows=count, labels=len(labels))
    return row_counts, sorted(labels)


def _prepare_raw_external_file(
    dataset_key: str, path: Path, label_column: str, feature_columns: Sequence[str],
    fine_to_id: Mapping[str, int], output_dir: Path, expected_rows: int,
) -> Tuple[Path, Path]:
    token = hashlib.sha256(str(path.resolve()).encode()).hexdigest()[:16]
    x_path, y_path = output_dir / f'{token}_X.npy', output_dir / f'{token}_yf.npy'
    progress_path = output_dir / f'{token}_progress.json'
    sig = signature_hash({'path': str(path), 'mtime': path.stat().st_mtime, 'rows': expected_rows,
                          'features': list(feature_columns), 'label': label_column, 'labels': dict(fine_to_id)})
    prior = read_json(progress_path, {})
    if prior.get('signature') == sig and prior.get('status') == 'DONE' and x_path.exists() and y_path.exists():
        return x_path, y_path
    actual_map = {_norm_col(c): c for c in _read_header(path)}
    actual_features = [actual_map[_norm_col(c)] for c in feature_columns]
    actual_label = actual_map[_norm_col(label_column)]
    start_row = int(prior.get('completed_rows', 0)) if prior.get('signature') == sig and x_path.exists() and y_path.exists() else 0
    mode = 'r+' if start_row > 0 else 'w+'
    xm = np.lib.format.open_memmap(x_path, mode=mode, dtype=np.float32, shape=(int(expected_rows), len(feature_columns)))
    ym = np.lib.format.open_memmap(y_path, mode=mode, dtype=np.int32, shape=(int(expected_rows),))
    cursor = 0
    for chunk in _read_external_chunks(path, usecols=actual_features + [actual_label]):
        next_cursor = cursor + len(chunk)
        if next_cursor <= start_row:
            cursor = next_cursor
            continue
        if cursor < start_row:
            chunk = chunk.iloc[start_row - cursor:].copy()
            cursor = start_row
            next_cursor = cursor + len(chunk)
        numeric = chunk[actual_features].apply(pd.to_numeric, errors='coerce').to_numpy(dtype=np.float32, copy=False)
        numeric = np.clip(numeric, -float(cfg.CLIP_INF), float(cfg.CLIP_INF)).astype(np.float32, copy=False)
        labels = np.asarray([fine_to_id[_canonical_external_label(x)] for x in chunk[actual_label].tolist()], dtype=np.int32)
        xm[cursor:next_cursor] = numeric
        ym[cursor:next_cursor] = labels
        xm.flush(); ym.flush()
        cursor = next_cursor
        atomic_json(progress_path, {'signature': sig, 'completed_rows': cursor,
                                    'status': 'DONE' if cursor >= expected_rows else 'RUNNING'})
        paper_log('external file cache progress', stage=f'external_{dataset_key}', file=path.name,
                  rows=f'{cursor}/{expected_rows}')
        check_deadline(f'external {dataset_key} file {path.name} row {cursor}')
    if cursor != expected_rows:
        raise RuntimeError(f'Row count changed for {path}: expected {expected_rows}, wrote {cursor}')
    del xm, ym
    return x_path, y_path


def _concatenate_external_split(split: str, file_pairs: Sequence[Tuple[Path, Path]], out_dir: Path) -> Tuple[Path, Path]:
    x_path, y_path = out_dir / f'X_{split}_raw.npy', out_dir / f'y_{split}_fine.npy'
    progress_path = out_dir / f'{split}_concat_progress.json'
    rows = sum(int(np.load(xp, mmap_mode='r').shape[0]) for xp, _ in file_pairs)
    n_features = int(np.load(file_pairs[0][0], mmap_mode='r').shape[1])
    sig = signature_hash({'split': split, 'files': [(str(x), x.stat().st_mtime) for x, _ in file_pairs], 'rows': rows})
    prior = read_json(progress_path, {})
    if prior.get('signature') == sig and prior.get('status') == 'DONE' and x_path.exists() and y_path.exists():
        return x_path, y_path
    completed_files = int(prior.get('completed_files', 0)) if prior.get('signature') == sig and x_path.exists() and y_path.exists() else 0
    completed_rows = int(prior.get('completed_rows', 0)) if completed_files else 0
    mode = 'r+' if completed_files else 'w+'
    Xout = np.lib.format.open_memmap(x_path, mode=mode, dtype=np.float32, shape=(rows, n_features))
    yout = np.lib.format.open_memmap(y_path, mode=mode, dtype=np.int32, shape=(rows,))
    cursor = completed_rows
    for file_index, (xp, yp) in enumerate(file_pairs[completed_files:], start=completed_files):
        xin, yin = np.load(xp, mmap_mode='r'), np.load(yp, mmap_mode='r')
        stop = cursor + len(yin)
        Xout[cursor:stop] = xin
        yout[cursor:stop] = yin
        Xout.flush(); yout.flush()
        cursor = stop
        atomic_json(progress_path, {'signature': sig, 'completed_files': file_index + 1,
                                    'completed_rows': cursor,
                                    'status': 'DONE' if file_index + 1 >= len(file_pairs) else 'RUNNING'})
        paper_log('external split concatenation', stage='external_prepare', split=split,
                  files=f'{file_index + 1}/{len(file_pairs)}', rows=cursor)
        check_deadline(f'external concatenate {split} file {file_index + 1}')
    del Xout, yout
    return x_path, y_path


def _fit_external_preprocessor(X_train_raw: np.ndarray, out_path: Path) -> Dict[str, Any]:
    if out_path.exists() and not PAPER.FORCE_RERUN:
        return read_json(out_path, {})
    n_features = X_train_raw.shape[1]
    median = np.empty(n_features, dtype=np.float64)
    for j in range(n_features):
        column = np.asarray(X_train_raw[:, j], dtype=np.float64)
        column[~np.isfinite(column)] = np.nan
        med = np.nanmedian(column)
        median[j] = 0.0 if not np.isfinite(med) else med
        paper_log('external median progress', stage='external_preprocess', feature=f'{j + 1}/{n_features}')
        check_deadline(f'external median feature {j}')
    total = np.zeros(n_features, dtype=np.float64)
    total_sq = np.zeros(n_features, dtype=np.float64)
    n_rows = 0
    for start in range(0, len(X_train_raw), 250_000):
        stop = min(len(X_train_raw), start + 250_000)
        block = np.asarray(X_train_raw[start:stop], dtype=np.float64)
        block = np.where(np.isfinite(block), block, median[None, :])
        total += block.sum(axis=0)
        total_sq += np.square(block).sum(axis=0)
        n_rows += len(block)
        check_deadline(f'external mean/std row {stop}')
    mean = total / max(n_rows, 1)
    variance = np.maximum(total_sq / max(n_rows, 1) - np.square(mean), 1e-12)
    std = np.sqrt(variance)
    payload = {'median': median.tolist(), 'mean': mean.tolist(), 'std': std.tolist(),
               'post_scale_clip': float(cfg.POST_SCALE_CLIP), 'rows_fit': int(n_rows),
               'fit_partition': 'train_only'}
    atomic_json(out_path, payload)
    return payload


def _transform_external_split(raw_path: Path, out_path: Path, preprocessor: Mapping[str, Any]) -> Path:
    raw = np.load(raw_path, mmap_mode='r')
    progress_path = out_path.with_suffix(out_path.suffix + '.progress.json')
    sig = signature_hash({'raw': str(raw_path), 'mtime': raw_path.stat().st_mtime, 'preprocessor': dict(preprocessor)})
    prior = read_json(progress_path, {})
    if prior.get('signature') == sig and prior.get('status') == 'DONE' and out_path.exists():
        return out_path
    start_row = int(prior.get('completed_rows', 0)) if prior.get('signature') == sig and out_path.exists() else 0
    mode = 'r+' if start_row > 0 else 'w+'
    out = np.lib.format.open_memmap(out_path, mode=mode, dtype=np.float32, shape=raw.shape)
    median = np.asarray(preprocessor['median'], dtype=np.float64)
    mean = np.asarray(preprocessor['mean'], dtype=np.float64)
    std = np.asarray(preprocessor['std'], dtype=np.float64)
    clip = float(preprocessor['post_scale_clip'])
    for start in range(start_row, len(raw), 250_000):
        stop = min(len(raw), start + 250_000)
        block = np.asarray(raw[start:stop], dtype=np.float64)
        block = np.where(np.isfinite(block), block, median[None, :])
        block = np.clip((block - mean[None, :]) / std[None, :], -clip, clip)
        out[start:stop] = np.nan_to_num(block, nan=0.0, posinf=clip, neginf=-clip).astype(np.float32)
        out.flush()
        atomic_json(progress_path, {'signature': sig, 'completed_rows': stop,
                                    'status': 'DONE' if stop >= len(raw) else 'RUNNING'})
        check_deadline(f'external transform {out_path.name} row {stop}')
    del out
    return out_path


def prepare_external_dataset(dataset_key: str) -> Dict[str, Any]:
    spec = dict(EXTERNAL_SPECS[dataset_key])
    files = _discover_external_files(spec['root'])
    if not files:
        raise RuntimeError(f"Dataset root is missing or contains no CSV/Parquet files: {spec['root']!r}")
    out_dir = PAPER_CACHE_DIR / 'external' / dataset_key
    raw_file_dir = out_dir / 'raw_files'
    raw_file_dir.mkdir(parents=True, exist_ok=True)
    prepared_meta = out_dir / 'prepared_metadata.json'
    split = _file_level_partition(files, 42)
    label_column = _choose_label_column(files, spec['label_candidates'])
    feature_columns = _select_external_numeric_features(files, label_column)
    group_names, group_indices = _semantic_groups_for_columns(feature_columns)
    row_counts, labels = _scan_labels_and_rows(files, label_column)
    fine_to_id = {label: i for i, label in enumerate(labels)}
    coarse_by_fine_name = {label: _map_external_coarse(label, dataset_key) for label in labels}
    observed_coarse = [name for name in spec['coarse_protocol'] if name in set(coarse_by_fine_name.values())]
    for name in sorted(set(coarse_by_fine_name.values())):
        if name not in observed_coarse:
            observed_coarse.append(name)
    coarse_to_id = {name: i for i, name in enumerate(observed_coarse)}
    fine_to_coarse_ext = np.asarray([coarse_to_id[coarse_by_fine_name[label]] for label in labels], dtype=np.int64)
    benign_candidates = [i for i, label in enumerate(labels) if coarse_by_fine_name[label] == 'Benign']
    if len(benign_candidates) != 1:
        raise RuntimeError(f'Expected exactly one benign/normal fine label; found {[(labels[i], i) for i in benign_candidates]}')
    benign_ext = int(benign_candidates[0])

    file_pairs: Dict[str, List[Tuple[Path, Path]]] = {part: [] for part in split}
    manifest_rows = []
    for part, part_files in split.items():
        for path in part_files:
            xp, yp = _prepare_raw_external_file(dataset_key, path, label_column, feature_columns,
                                                fine_to_id, raw_file_dir, row_counts[str(path)])
            file_pairs[part].append((xp, yp))
            manifest_rows.append({'dataset': spec['display_name'], 'source_file': str(path), 'split': part,
                                  'rows': row_counts[str(path)], 'source_file_disjoint': True})
    manifest = pd.DataFrame(manifest_rows)
    atomic_dataframe(manifest, out_dir / 'split_manifest')

    raw_paths: Dict[str, Tuple[Path, Path]] = {}
    for part in ('train', 'val', 'cal', 'test'):
        raw_paths[part] = _concatenate_external_split(part, file_pairs[part], out_dir)
    preprocessor = _fit_external_preprocessor(np.load(raw_paths['train'][0], mmap_mode='r'), out_dir / 'preprocessor.json')
    normalized_paths: Dict[str, Path] = {}
    for part in ('train', 'val', 'cal', 'test'):
        normalized_paths[part] = _transform_external_split(raw_paths[part][0], out_dir / f'X_{part}.npy', preprocessor)

    metadata = {
        'dataset_key': dataset_key, 'display_name': spec['display_name'], 'root': str(Path(spec['root']).expanduser()),
        'files': [str(x) for x in files], 'label_column': label_column,
        'feature_columns': feature_columns, 'group_names': group_names, 'group_indices': group_indices,
        'fine_labels': labels, 'coarse_labels': observed_coarse,
        'fine_to_coarse': fine_to_coarse_ext.tolist(), 'benign_fine_id': benign_ext,
        'coarse_protocol': spec['coarse_protocol'], 'split_paths': {k: str(v) for k, v in normalized_paths.items()},
        'label_paths': {k: str(raw_paths[k][1]) for k in raw_paths},
        'manifest_path': str(out_dir / 'split_manifest.csv'), 'preprocessor_path': str(out_dir / 'preprocessor.json'),
        'profile': PAPER.PROFILE, 'source_kind': 'generated_from_supplied_external_dataset_root',
    }
    metadata['signature'] = signature_hash(metadata)
    atomic_json(prepared_meta, metadata)
    atomic_json(PAPER_REPORT_DIR / f'{dataset_key}_feature_groups.json',
                {'selected_columns': feature_columns, 'group_names': group_names, 'group_indices': group_indices})
    atomic_json(PAPER_REPORT_DIR / f'{dataset_key}_hierarchy.json',
                {'fine_labels': labels, 'coarse_labels': observed_coarse, 'fine_to_coarse': fine_to_coarse_ext.tolist(),
                 'benign_fine_id': benign_ext, 'coarse_protocol': spec['coarse_protocol']})
    return metadata


@_contextmanager
def _temporary_external_hierarchy(metadata: Mapping[str, Any]):
    keys = ['NUM_FINE_PAPER', 'NUM_COARSE_PAPER', 'fine_to_coarse', 'benign_fine_id',
            'CANONICAL_FINE_NAMES', 'CANONICAL_COARSE_NAMES']
    old = {key: globals()[key] for key in keys}
    try:
        globals()['NUM_FINE_PAPER'] = len(metadata['fine_labels'])
        globals()['NUM_COARSE_PAPER'] = len(metadata['coarse_labels'])
        globals()['fine_to_coarse'] = np.asarray(metadata['fine_to_coarse'], dtype=np.int64)
        globals()['benign_fine_id'] = int(metadata['benign_fine_id'])
        globals()['CANONICAL_FINE_NAMES'] = list(metadata['fine_labels'])
        globals()['CANONICAL_COARSE_NAMES'] = list(metadata['coarse_labels'])
        yield
    finally:
        globals().update(old)


def _external_profile_data(metadata: Mapping[str, Any], seed: int = 42) -> Dict[str, np.ndarray]:
    arrays = {part: np.load(metadata['split_paths'][part], mmap_mode='r') for part in ('train', 'val', 'cal', 'test')}
    labels = {part: np.load(metadata['label_paths'][part], mmap_mode='r').astype(np.int64, copy=False) for part in ('train', 'val', 'cal', 'test')}
    mapping = np.asarray(metadata['fine_to_coarse'], dtype=np.int64)
    benign = int(metadata['benign_fine_id'])
    if PAPER.PROFILE == 'strict':
        Xtr, yftr = arrays['train'], labels['train']
        Xv, yfv = arrays['val'], labels['val']
    else:
        ti = deterministic_subset_indices(labels['train'], profile_cap('train', len(labels['train'])), seed)
        vi = deterministic_subset_indices(labels['val'], profile_cap('val', len(labels['val'])), seed + 1)
        Xtr, yftr = np.asarray(arrays['train'][ti], dtype=np.float32), np.asarray(labels['train'][ti], dtype=np.int64)
        Xv, yfv = np.asarray(arrays['val'][vi], dtype=np.float32), np.asarray(labels['val'][vi], dtype=np.int64)
    # Exact hierarchy requires every declared class to be represented in train.
    missing_fine = sorted(set(range(len(metadata['fine_labels']))) - set(np.unique(yftr).tolist()))
    if missing_fine:
        raise RuntimeError('External training split lacks declared fine classes: ' + ', '.join(metadata['fine_labels'][i] for i in missing_fine))
    yftr_i = np.asarray(yftr, dtype=np.int64)
    yfv_i = np.asarray(yfv, dtype=np.int64)
    return {
        'Xtr': Xtr, 'yftr': yftr_i, 'yctr': mapping[yftr_i], 'ybtr': (yftr_i != benign).astype(np.int64),
        'Xv': Xv, 'yfv': yfv_i, 'ycv': mapping[yfv_i], 'ybv': (yfv_i != benign).astype(np.int64),
        'Xcal': arrays['cal'], 'yfcal': labels['cal'], 'Xtest': arrays['test'], 'yftest': labels['test'],
    }


def run_external_dataset(dataset_key: str) -> Dict[str, Any]:
    metadata = prepare_external_dataset(dataset_key)
    with _temporary_external_hierarchy(metadata):
        data = _external_profile_data(metadata, 42)
        mapping = np.asarray(metadata['fine_to_coarse'], dtype=np.int64)
        benign = int(metadata['benign_fine_id'])
        factory = lambda: build_fresh_camelot_model(
            group_idxs=metadata['group_indices'], n_features=len(metadata['feature_columns']),
            n_fine=len(metadata['fine_labels']), n_coarse=len(metadata['coarse_labels']),
        )
        trained = train_resumable_hierarchical(
            f'external_{dataset_key}_seed_42', factory, seed=42, fine_loss_mode='ldam_drw',
            data_override={k: data[k] for k in ('Xtr', 'yftr', 'yctr', 'ybtr', 'Xv', 'yfv', 'ycv', 'ybv')},
            model_spec={'dataset': dataset_key, 'metadata_signature': metadata['signature'],
                        'n_features': len(metadata['feature_columns']), 'n_fine': len(metadata['fine_labels']),
                        'n_coarse': len(metadata['coarse_labels'])},
        )
        if trained.status != 'DONE':
            raise TimeBudgetReached(f'External dataset {dataset_key} training paused; restart to continue.')
        model = load_hierarchical_checkpoint(factory, trained.best_checkpoint)
        cal_out = predict_hier_logits_array(model, data['Xcal'], log_every_batches=50)
        test_out = predict_hier_logits_array(model, data['Xtest'], log_every_batches=50)
        ycal = np.asarray(data['yfcal'], dtype=np.int64)
        ytest = np.asarray(data['yftest'], dtype=np.int64)
        yctest, ybtest = mapping[ytest], (ytest != benign).astype(np.int64)
        tf = float(fit_temperature_np(cal_out['fine'], ycal))
        tc = float(fit_temperature_np(cal_out['coarse'], mapping[ycal]))
        tb = float(fit_temperature_np(cal_out['bin'], (ycal != benign).astype(np.int64)))
        pf = softmax_np(test_out['fine'] / max(tf, 1e-6))
        pc = softmax_np(test_out['coarse'] / max(tc, 1e-6))
        pb = softmax_np(test_out['bin'] / max(tb, 1e-6))
        fine = multiclass_metrics(ytest, pf)
        coarse = multiclass_metrics(yctest, pc)
        binary = binary_metrics(ybtest, pb)
        payload = {
            'dataset': metadata['display_name'], 'dataset_key': dataset_key,
            'fine_accuracy': fine['accuracy'], 'fine_macro_f1': fine['macro_f1'],
            'coarse_macro_f1': coarse['macro_f1'], 'binary_malicious_f1': binary['malicious_f1'],
            'temperatures': {'fine': tf, 'coarse': tc, 'binary': tb},
            'test_rows': len(ytest), 'fine_classes': len(metadata['fine_labels']),
            'coarse_classes': len(metadata['coarse_labels']), 'features': len(metadata['feature_columns']),
            'profile': PAPER.PROFILE, 'strict_manuscript_equivalent_training': PAPER.PROFILE == 'strict',
            'source_kind': 'strict_measured' if PAPER.PROFILE == 'strict' else f'{PAPER.PROFILE}_profile_measured',
            'status': 'MEASURED', 'metadata_path': str(PAPER_CACHE_DIR / 'external' / dataset_key / 'prepared_metadata.json'),
        }
        atomic_json(PAPER_CACHE_DIR / 'external' / dataset_key / 'evaluation.json', payload)
        del model, cal_out, test_out, pf, pc, pb
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        return payload


def run_external_dataset_suite() -> Tuple[pd.DataFrame, pd.DataFrame]:
    result_rows: List[Dict[str, Any]] = []
    mapping_rows: List[Dict[str, Any]] = []
    for key, spec in EXTERNAL_SPECS.items():
        check_deadline(f'before external dataset {key}')
        try:
            payload = run_external_dataset(key)
            result_rows.append({
                'dataset': payload['dataset'], 'fine_accuracy_pct': 100 * payload['fine_accuracy'],
                'fine_macro_f1_pct': 100 * payload['fine_macro_f1'],
                'coarse_macro_f1_pct': 100 * payload['coarse_macro_f1'],
                'binary_malicious_f1_pct': 100 * payload['binary_malicious_f1'],
                'features': payload['features'], 'fine_classes': payload['fine_classes'],
                'coarse_classes': payload['coarse_classes'], 'test_rows': payload['test_rows'],
                'status': payload['status'], 'source_kind': payload['source_kind'], 'profile': payload['profile'],
            })
            meta = read_json(Path(payload['metadata_path']), {})
            mapping_rows.append({
                'dataset': payload['dataset'], 'label_column': meta.get('label_column'),
                'selected_features': len(meta.get('feature_columns', [])),
                'semantic_group_sizes': '/'.join(str(len(x)) for x in meta.get('group_indices', [])),
                'fine_to_coarse_hierarchy': '; '.join(meta.get('coarse_labels', [])),
                'split_protocol': 'seed 42 shuffled physical-file holdout ~80/10/5/5; train-only median/standardization/clipping',
                'manifest': meta.get('manifest_path'), 'feature_mapping': str(PAPER_REPORT_DIR / f'{key}_feature_groups.json'),
                'hierarchy_mapping': str(PAPER_REPORT_DIR / f'{key}_hierarchy.json'), 'status': 'GENERATED',
            })
        except TimeBudgetReached:
            atomic_dataframe(pd.DataFrame(result_rows), PAPER_TABLE_DIR / 'table_s08_external_results_partial')
            raise
        except Exception as exc:
            paper_log('external dataset blocked; no paper reference value substituted', level='WARNING',
                      stage='external_datasets', dataset=key, error=repr(exc))
            result_rows.append({'dataset': spec['display_name'], 'status': 'BLOCKED', 'source_kind': 'not_executed',
                                'profile': PAPER.PROFILE, 'note': repr(exc)})
            mapping_rows.append({'dataset': spec['display_name'], 'status': 'BLOCKED', 'root': spec['root'],
                                 'coarse_protocol': '; '.join(spec['coarse_protocol']), 'note': repr(exc)})
        atomic_dataframe(pd.DataFrame(result_rows), PAPER_TABLE_DIR / 'table_s08_external_results_partial')
    results = pd.DataFrame(result_rows)
    mappings = pd.DataFrame(mapping_rows)
    atomic_dataframe(results, PAPER_TABLE_DIR / 'table_s08_external_dataset_results')
    atomic_dataframe(mappings, PAPER_TABLE_DIR / 'table_s09_external_mapping_protocol')
    status = 'DONE' if len(results) and (results['status'] == 'MEASURED').all() else 'PARTIAL_OR_BLOCKED'
    update_coverage_status('Tables S8-S9', status,
                           str(PAPER_TABLE_DIR / 'table_s08_external_dataset_results.csv'),
                           'Raw datasets are never auto-downloaded; missing roots remain BLOCKED.')
    display(results)
    display(mappings)
    return results, mappings


paper_log('external-dataset pipeline ready; no dataset is downloaded automatically', stage='external_datasets',
          configured_roots={key: bool(spec['root']) for key, spec in EXTERNAL_SPECS.items()})
paper_cell_done('P18', 'external-dataset preparation, mappings, retraining, and Tables S8-S9', _started)


[P18] START: external-dataset preparation, mappings, retraining, and Tables S8-S9
[2026-08-28 09:55:50] [INFO] [external_datasets] external-dataset pipeline ready; no dataset is downloaded automatically | configured_roots={'cicids2017': False, 'cicids2018': False, 'toniot': False, 'unsw_nb15': False}
[P18] DONE: external-dataset preparation, mappings, retraining, and Tables S8-S9 | elapsed=0.03s


In [21]:
_started = paper_cell_start('P19', 'execution plan, evidence audit, artifact manifest, and restart guide')


PIPELINE_STEPS: List[Dict[str, Any]] = [
    {'name': 'archived_core', 'group': 'light', 'function': 'run_archived_core_tables', 'artifacts': 'Tables 2,4-7,S1-S4,S14-S15; Figures 1,4'},
    {'name': 'schematics', 'group': 'light', 'function': 'run_reference_and_schematics', 'artifacts': 'Table 1; Equations 1-11; Algorithm 1; Figures 2-3'},
    {'name': 'confirmatory_split_conformal', 'group': 'light', 'function': 'run_confirmatory_split_conformal', 'artifacts': 'Table S16'},
    {'name': 'provenance_audit', 'group': 'light', 'function': 'run_provenance_audit', 'artifacts': 'Table 3'},
    {'name': 'throughput', 'group': 'light', 'function': 'run_throughput_table', 'artifacts': 'Table 8'},
    {'name': 'stream_stationarity', 'group': 'medium', 'function': 'run_stream_stationarity', 'artifacts': 'Table 9'},
    {'name': 'ablations', 'group': 'long', 'function': 'run_ablation_study', 'artifacts': 'Table 10'},
    {'name': 'robustness', 'group': 'long', 'function': 'run_robustness_suite', 'artifacts': 'Table S7'},
    {'name': 'baselines', 'group': 'long', 'function': 'run_baseline_suite', 'artifacts': 'Tables S10-S11,S13'},
    {'name': 'cluster_bootstrap_post_baselines', 'group': 'medium', 'function': 'run_cluster_bootstrap', 'artifacts': 'Table S17'},
    {'name': 'sensitivity', 'group': 'long', 'function': 'run_hyperparameter_sensitivity', 'artifacts': 'Table S6'},
    {'name': 'scalability', 'group': 'long', 'function': 'run_feature_scalability', 'artifacts': 'Table S5'},
    {'name': 'multi_seed', 'group': 'long', 'function': 'run_multiseed_suite', 'artifacts': 'Tables S12-S12b'},
    {'name': 'external_datasets', 'group': 'long', 'function': 'run_external_dataset_suite', 'artifacts': 'Tables S8-S9'},
]
EXECUTION_PLAN = pd.DataFrame(PIPELINE_STEPS)
EXECUTION_PLAN['enabled'] = EXECUTION_PLAN['group'].isin(PAPER.RUN_GROUPS)
EXECUTION_PLAN['profile'] = PAPER.PROFILE
EXECUTION_PLAN['strict_protocol_equivalent'] = (PAPER.PROFILE == 'strict')
atomic_dataframe(EXECUTION_PLAN, PAPER_REPORT_DIR / 'execution_plan')


def _summarize_result_object(value: Any) -> Dict[str, Any]:
    if isinstance(value, pd.DataFrame):
        return {'type': 'DataFrame', 'rows': int(len(value)), 'columns': list(map(str, value.columns))}
    if isinstance(value, tuple):
        return {'type': 'tuple', 'items': [_summarize_result_object(x) for x in value]}
    if isinstance(value, Mapping):
        return {'type': 'mapping', 'keys': sorted(map(str, value.keys()))[:100],
                'status': value.get('status') if 'status' in value else None}
    return {'type': type(value).__name__, 'repr': repr(value)[:500]}


def _artifact_sha256(path: Path, block_size: int = 8 * 1024 * 1024) -> str:
    h = hashlib.sha256()
    with open(path, 'rb') as handle:
        while True:
            block = handle.read(block_size)
            if not block:
                break
            h.update(block)
    return h.hexdigest()


def build_output_manifest() -> pd.DataFrame:
    excluded = {'artifact_manifest.csv', 'artifact_manifest.xlsx'}
    rows: List[Dict[str, Any]] = []
    for path in sorted(PAPER_DIR.rglob('*')):
        if not path.is_file() or path.name in excluded or path.name.endswith('.tmp'):
            continue
        try:
            rows.append({
                'relative_path': str(path.relative_to(PAPER_DIR)),
                'bytes': int(path.stat().st_size),
                'modified_at': time.strftime('%Y-%m-%d %H:%M:%S', time.localtime(path.stat().st_mtime)),
                'sha256': _artifact_sha256(path),
            })
        except FileNotFoundError:
            continue
    manifest = pd.DataFrame(rows)
    atomic_dataframe(manifest, PAPER_REPORT_DIR / 'artifact_manifest')
    return manifest


def _collect_stage_statuses() -> pd.DataFrame:
    rows = []
    for status_file in sorted(PAPER_STAGE_DIR.glob('*/status.json')):
        payload = read_json(status_file, {})
        rows.append({
            'stage': payload.get('stage', status_file.parent.name),
            'status': payload.get('status', 'UNKNOWN'),
            'attempts': payload.get('attempts', np.nan),
            'updated_at': payload.get('updated_at', ''),
            'elapsed_seconds': payload.get('elapsed_seconds', np.nan),
            'reason': payload.get('reason', ''),
            'error': payload.get('error', ''),
        })
    return pd.DataFrame(rows)


def _collect_table_provenance() -> pd.DataFrame:
    rows: List[Dict[str, Any]] = []
    for path in sorted(PAPER_TABLE_DIR.glob('*.csv')):
        try:
            table = pd.read_csv(path)
        except Exception as exc:
            rows.append({'table_file': path.name, 'status': 'READ_ERROR', 'note': repr(exc)})
            continue
        source_counts = {}
        status_counts = {}
        if 'source_kind' in table.columns:
            source_counts = table['source_kind'].fillna('NA').astype(str).value_counts().to_dict()
        if 'status' in table.columns:
            status_counts = table['status'].fillna('NA').astype(str).value_counts().to_dict()
        elif 'training_status' in table.columns:
            status_counts = table['training_status'].fillna('NA').astype(str).value_counts().to_dict()
        rows.append({
            'table_file': path.name, 'rows': len(table), 'columns': len(table.columns),
            'source_kind_counts': stable_json(source_counts), 'status_counts': stable_json(status_counts),
            'contains_manuscript_reference_only': bool(
                'source_kind' in table.columns and table['source_kind'].astype(str).str.contains('reference', case=False, na=False).any()
            ),
        })
    return pd.DataFrame(rows)


def _write_runtime_readme(coverage: pd.DataFrame, stages: pd.DataFrame, queue: Optional[pd.DataFrame] = None) -> Path:
    coverage_md = coverage[['artifact', 'status', 'output', 'note']].fillna('').to_markdown(index=False) if len(coverage) else 'No coverage rows yet.'
    stage_md = stages.fillna('').to_markdown(index=False) if len(stages) else 'No stage status files yet.'
    queue_md = queue.fillna('').to_markdown(index=False) if queue is not None and len(queue) else 'No orchestrator execution rows yet.'
    text = f"""# CAMELOT-IDS v4 merged paper-completion run

Generated by the append-only section of the merged notebook.

## Active execution profile

- Profile: **{PAPER.PROFILE}**
- Run-all: **{PAPER.RUN_ALL}**
- Per-invocation time budget: **{PAPER.TIME_BUDGET_HOURS} hours**
- Requested training epochs for this profile: **{PAPER.EPOCHS}**
- Training batch size: **{PAPER.BATCH_SIZE}**
- Output root: `{PAPER_DIR}`

Only `strict` uses the complete 21,705,973-row training partition and the paper's 70-epoch budget. `accelerated` and `smoke` outputs are tagged in every result table and are never substituted for manuscript reference numbers.

## Restart after shutdown or power failure

1. Start Jupyter in the same working directory with the same dataset paths.
2. Open the merged notebook and run from the original v4 definition/main cell through the appended cells.
3. Keep the same `CAMELOT_PROFILE` and artifact environment variables. Signatures prevent incompatible checkpoints from being mixed.
4. The final orchestrator reloads completed stages and resumes the first unfinished epoch, optimizer step block, inference chunk, boosting-round chunk, stream batch, or bootstrap block.
5. Repeat until the queue reports `DONE`, `BLOCKED`, or a documented protocol limitation for every artifact.

The checkpoints include model, optimizer, scheduler, AMP scaler, EMA, and Python/NumPy/PyTorch/CUDA RNG states for the custom PyTorch trainers. The optional TabNet package exposes weight-level epoch restart but not optimizer-state restoration; that limitation is logged in Table S11.

## Required external artifacts for currently non-self-contained experiments

- `CAMELOT_PROVENANCE_MANIFEST`: split manifest with `__src_file`, `root_capture_id`, `attack_run_id`, `session_id`, and `device_id` or equivalent columns.
- `CAMELOT_TEST_CLUSTER_IDS`: row-aligned test `root_capture_id` values for Table S17.
- `CAMELOT_TRAIN_CLUSTER_IDS`: training capture IDs for any requested group-aware baseline search rerun.
- `CAMELOT_FEATURE_MAP_DIR`: exact 23/92/138 feature arrays and K=3/7/10 grouping JSON files for Tables S5-S6.
- `CAMELOT_ROBUSTNESS_ARTIFACT_DIR`: exact MQTT holdout and 200-shot label-access artifacts for Table S7.
- `CIC_IDS2017_ROOT`, `CSE_CIC_IDS2018_ROOT`, `TON_IOT_ROOT`, `UNSW_NB15_ROOT`: local external dataset roots for Tables S8-S9. Data are never downloaded automatically.

A `BLOCKED` row means the current files or manuscript do not specify enough information to execute the exact experiment. It is not a fabricated zero, a substituted paper value, or a silent approximation.

## Artifact coverage

{coverage_md}

## Stage status

{stage_md}

## Orchestrator queue

{queue_md}
"""
    path = PAPER_DIR / 'README.md'
    atomic_write_text(path, text)
    return path


def build_final_audit(queue: Optional[pd.DataFrame] = None) -> Dict[str, Any]:
    coverage_path = PAPER_REPORT_DIR / 'paper_artifact_coverage_registry.csv'
    coverage = pd.read_csv(coverage_path) if coverage_path.exists() else PAPER_COVERAGE.copy()
    for column in ('output', 'note'):
        if column not in coverage.columns:
            coverage[column] = ''
    stages = _collect_stage_statuses()
    provenance = _collect_table_provenance()
    atomic_dataframe(stages, PAPER_REPORT_DIR / 'stage_status_summary')
    atomic_dataframe(provenance, PAPER_REPORT_DIR / 'table_result_provenance_audit')
    if queue is not None:
        atomic_dataframe(queue, PAPER_REPORT_DIR / 'orchestrator_queue')
    readme_path = _write_runtime_readme(coverage, stages, queue)
    manifest = build_output_manifest()
    status_counts = coverage['status'].fillna('UNKNOWN').astype(str).value_counts().to_dict()
    blocked = coverage[coverage['status'].astype(str).str.contains('BLOCK', case=False, na=False)].copy()
    pending = coverage[coverage['status'].astype(str).isin(['REGISTERED', 'PENDING', 'PARTIAL', 'PARTIAL_OR_BLOCKED'])].copy()
    atomic_dataframe(blocked, PAPER_REPORT_DIR / 'blocked_artifacts')
    atomic_dataframe(pending, PAPER_REPORT_DIR / 'pending_artifacts')
    summary = {
        'profile': PAPER.PROFILE,
        'strict_protocol_equivalent_for_new_training': PAPER.PROFILE == 'strict',
        'coverage_status_counts': status_counts,
        'blocked_artifacts': blocked['artifact'].astype(str).tolist(),
        'pending_artifacts': pending['artifact'].astype(str).tolist(),
        'generated_files': int(len(manifest)),
        'readme': str(readme_path),
        'coverage_registry': str(coverage_path),
        'artifact_manifest': str(PAPER_REPORT_DIR / 'artifact_manifest.csv'),
        'no_reference_value_substitution_policy': True,
        'built_at': time.strftime('%Y-%m-%d %H:%M:%S'),
    }
    atomic_json(PAPER_REPORT_DIR / 'final_audit_summary.json', summary)
    return summary


print('Execution plan:')
display(EXECUTION_PLAN)
paper_log('finalization/audit helpers ready', stage='finalization', steps=len(EXECUTION_PLAN))
paper_cell_done('P19', 'execution plan, evidence audit, artifact manifest, and restart guide', _started)


[P19] START: execution plan, evidence audit, artifact manifest, and restart guide
Execution plan:


,name,group,function,artifacts,enabled,profile,strict_protocol_equivalent
0,archived_core,light,run_archived_core_tables,"Tables 2,4-7,S1-S4,S14-S15; Figures 1,4",True,accelerated,False
1,schematics,light,run_reference_and_schematics,Table 1; Equations 1-11; Algorithm 1; Figures 2-3,True,accelerated,False
2,confirmatory_split_conformal,light,run_confirmatory_split_conformal,Table S16,True,accelerated,False
3,provenance_audit,light,run_provenance_audit,Table 3,True,accelerated,False
4,throughput,light,run_throughput_table,Table 8,True,accelerated,False
5,stream_stationarity,medium,run_stream_stationarity,Table 9,True,accelerated,False
6,ablations,long,run_ablation_study,Table 10,True,accelerated,False
7,robustness,long,run_robustness_suite,Table S7,True,accelerated,False
8,baselines,long,run_baseline_suite,"Tables S10-S11,S13",True,accelerated,False
9,cluster_bootstrap_post_baselines,medium,run_cluster_bootstrap,Table S17,True,accelerated,False


[2026-08-28 09:55:50] [INFO] [finalization] finalization/audit helpers ready | steps=14
[P19] DONE: execution plan, evidence audit, artifact manifest, and restart guide | elapsed=0.08s


In [ ]:
_started = paper_cell_start('P20', 'time-bounded, restartable execution of the complete paper experiment graph')


def run_complete_paper_pipeline() -> Tuple[pd.DataFrame, Dict[str, Any]]:
    set_pipeline_deadline(PAPER.TIME_BUDGET_HOURS)
    queue_rows: List[Dict[str, Any]] = []
    pipeline_state_path = PAPER_REPORT_DIR / 'orchestrator_state.json'
    overall_status = 'DONE'
    for order, step in enumerate(PIPELINE_STEPS, start=1):
        name = str(step['name'])
        group = str(step['group'])
        enabled = bool(group in PAPER.RUN_GROUPS)
        if not enabled:
            row = {'order': order, 'stage': name, 'group': group, 'status': 'SKIPPED_GROUP_FILTER',
                   'function': step['function'], 'artifacts': step['artifacts'], 'elapsed_seconds': 0.0}
            queue_rows.append(row)
            atomic_dataframe(pd.DataFrame(queue_rows), PAPER_REPORT_DIR / 'orchestrator_queue_partial')
            continue
        if not PAPER.RUN_ALL and group in {'medium', 'long'}:
            row = {'order': order, 'stage': name, 'group': group, 'status': 'SKIPPED_RUN_ALL_FALSE',
                   'function': step['function'], 'artifacts': step['artifacts'], 'elapsed_seconds': 0.0,
                   'note': 'Set CAMELOT_RUN_ALL=1 or PAPER.RUN_ALL=True.'}
            queue_rows.append(row)
            atomic_dataframe(pd.DataFrame(queue_rows), PAPER_REPORT_DIR / 'orchestrator_queue_partial')
            continue
        fn = globals().get(str(step['function']))
        if not callable(fn):
            row = {'order': order, 'stage': name, 'group': group, 'status': 'FAILED_MISSING_FUNCTION',
                   'function': step['function'], 'artifacts': step['artifacts'], 'elapsed_seconds': 0.0}
            queue_rows.append(row)
            overall_status = 'PARTIAL'
            continue
        stage_started = time.perf_counter()
        paper_log('orchestrator stage dispatch', stage='orchestrator', queue_stage=name,
                  order=f'{order}/{len(PIPELINE_STEPS)}', group=group,
                  remaining_hours=round(remaining_budget_seconds() / 3600, 3))
        try:
            check_deadline(f'before orchestrator stage {name}')
            value = fn()
            row = {
                'order': order, 'stage': name, 'group': group, 'status': 'COMPLETED_OR_REUSED',
                'function': step['function'], 'artifacts': step['artifacts'],
                'elapsed_seconds': time.perf_counter() - stage_started,
                'result_summary': stable_json(_summarize_result_object(value)),
                'remaining_hours': remaining_budget_seconds() / 3600,
            }
        except TimeBudgetReached as exc:
            row = {
                'order': order, 'stage': name, 'group': group, 'status': 'PAUSED_TIME_BUDGET',
                'function': step['function'], 'artifacts': step['artifacts'],
                'elapsed_seconds': time.perf_counter() - stage_started,
                'note': str(exc), 'remaining_hours': 0.0,
            }
            queue_rows.append(row)
            overall_status = 'PAUSED'
            atomic_dataframe(pd.DataFrame(queue_rows), PAPER_REPORT_DIR / 'orchestrator_queue_partial')
            atomic_json(pipeline_state_path, {'status': overall_status, 'next_stage': name,
                                              'queue': queue_rows, 'updated_at': time.strftime('%Y-%m-%d %H:%M:%S')})
            paper_log('pipeline paused; restart notebook to continue from checkpoints', level='WARNING',
                      stage='orchestrator', paused_stage=name, reason=str(exc))
            break
        except KeyboardInterrupt:
            row = {
                'order': order, 'stage': name, 'group': group, 'status': 'PAUSED_KEYBOARD_INTERRUPT',
                'function': step['function'], 'artifacts': step['artifacts'],
                'elapsed_seconds': time.perf_counter() - stage_started,
                'note': 'User interrupted execution; latest safe checkpoint is retained.',
            }
            queue_rows.append(row)
            overall_status = 'PAUSED'
            atomic_dataframe(pd.DataFrame(queue_rows), PAPER_REPORT_DIR / 'orchestrator_queue_partial')
            atomic_json(pipeline_state_path, {'status': overall_status, 'next_stage': name,
                                              'queue': queue_rows, 'updated_at': time.strftime('%Y-%m-%d %H:%M:%S')})
            paper_log('pipeline interrupted; restart to resume', level='WARNING', stage='orchestrator', paused_stage=name)
            break
        except Exception as exc:
            row = {
                'order': order, 'stage': name, 'group': group, 'status': 'FAILED_CONTINUING',
                'function': step['function'], 'artifacts': step['artifacts'],
                'elapsed_seconds': time.perf_counter() - stage_started,
                'note': repr(exc), 'traceback': traceback.format_exc(),
                'remaining_hours': remaining_budget_seconds() / 3600,
            }
            overall_status = 'PARTIAL'
            paper_log('orchestrator stage failed; independent stages continue', level='ERROR',
                      stage='orchestrator', failed_stage=name, error=repr(exc))
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
        queue_rows.append(row)
        atomic_dataframe(pd.DataFrame(queue_rows), PAPER_REPORT_DIR / 'orchestrator_queue_partial')
        atomic_json(pipeline_state_path, {'status': overall_status, 'last_completed_or_attempted_stage': name,
                                          'queue': queue_rows, 'updated_at': time.strftime('%Y-%m-%d %H:%M:%S')})

    queue = pd.DataFrame(queue_rows)
    # A queue can finish while individual artifacts remain BLOCKED because exact
    # manifests, mappings, raw datasets, packages, or label-access protocols were
    # not supplied. The audit keeps that distinction explicit.
    audit = build_final_audit(queue)
    atomic_json(pipeline_state_path, {'status': overall_status, 'queue': queue_rows, 'audit': audit,
                                      'updated_at': time.strftime('%Y-%m-%d %H:%M:%S')})
    display(queue)
    print('Final audit summary:', json.dumps(audit, indent=2, default=_jsonable))
    return queue, audit


ORCHESTRATOR_QUEUE, FINAL_AUDIT = run_complete_paper_pipeline()
if any(ORCHESTRATOR_QUEUE.get('status', pd.Series(dtype=str)).astype(str).str.startswith('PAUSED')):
    print('\nPAUSED SAFELY: restart the kernel/notebook with the same profile and paths; the first unfinished stage resumes automatically.')
elif any(ORCHESTRATOR_QUEUE.get('status', pd.Series(dtype=str)).astype(str).str.startswith('FAILED')):
    print('\nPIPELINE COMPLETED PARTIALLY: inspect reports/orchestrator_queue.csv and logs/events.jsonl; independent completed stages are reusable.')
else:
    print('\nPIPELINE PASS COMPLETE: inspect the coverage registry for DONE, PARTIAL, and BLOCKED evidence boundaries.')
paper_cell_done('P20', 'time-bounded, restartable execution of the complete paper experiment graph', _started)


[P20] START: time-bounded, restartable execution of the complete paper experiment graph
[2026-08-28 09:55:50] [INFO] [orchestrator] orchestrator stage dispatch | queue_stage=archived_core order=1/14 group=light remaining_hours=96.0
[2026-08-28 09:55:50] [INFO] [archived_core] completed stage reused | signature=94d322c5346c
[2026-08-28 09:55:50] [INFO] [orchestrator] orchestrator stage dispatch | queue_stage=schematics order=2/14 group=light remaining_hours=96.0
[2026-08-28 09:55:50] [INFO] [reference_and_schematics] completed stage reused | signature=b2782e962c11
[2026-08-28 09:55:50] [INFO] [orchestrator] orchestrator stage dispatch | queue_stage=confirmatory_split_conformal order=3/14 group=light remaining_hours=96.0
[2026-08-28 09:55:50] [INFO] [confirmatory_split_conformal] completed stage reused | signature=258f9bc4bb38
[2026-08-28 09:55:50] [INFO] [orchestrator] orchestrator stage dispatch | queue_stage=provenance_audit order=4/14 group=light remaining_hours=96.0
[2026-08-28 09:5